In [ ]:
def main(datasources, start_date, end_date):
    """【1616：非线性堆叠深度五四种子平均多期目标零点四平台口径】= 1600（公榜 0.87851）的升级版：四个深度 5 的树（各 100 棵，四个种子）取平均（学习率 ÷4），训练目标加多期目标 0.40，H1363 按平台口径；四个种子单独也都五项全好；浅层 GBDT（400 棵树、深度 5、学习率 0.015，只在 2020-01-01～2022-12-31 拟合）作用在 27 列特征上。
      H1363（第 12 列）：A = 1363（九根合成，精简按月、头部按季；与已交付的 1363 逐值相同；平台上 Y_sd5、X_n5d 为零、N_dir 活着，与树的训练口径一致）的原始值
      L_lab10（第 0 列）先做 EWMA α=0.75 再排名：同时段季节性（= 1232 的根）
      M_q_lin（第 1 列）排名：聪明钱（= 1236）
      W_vdir（第 2 列）排名：量价同向（= 1260）
      W_zero（第 3 列）排名：零收益分钟占比（= 1263）
      Z_mfr60（第 4 列）排名：近 60 分钟上涨分钟量占比（= 1334）
      Z_mfr90（第 5 列）排名：近 90 分钟上涨分钟量占比（= 1342）
      s155（第 6 列）排名：队友 sushirui 的 155
      s139（第 7 列）排名：队友 sushirui 的 139
      s161（第 8 列）排名：队友 sushirui 的 161
      k1381（第 9 列）排名：日内已实现峰度反向（= 1381）
      k1385（第 10 列）排名：近 5 日已实现偏度反向（= 1385）
      k1406（第 11 列）排名：前日收盘竞价价格冲击（= 1406）
      H152（第 13 列）排名：队友 sushirui 的 152（排名）
      CLS20（第 25 列）排名：收盘段季节性（排名）
      CLOSEPOS20（第 26 列）排名：收盘位置锚（排名）
      slot1000…slot1430：截面时刻 one-hot
      L_5／VSH20／PACE20／DOW10：同时段季节性族（下一段 lab 过去 5 日均值、下一段成交额占比过去 20 日均值、今日累计成交额 ÷ 过去 20 日同一时刻均值、过去 10 个同星期几的下一段 lab 均值），再排名
    排名轴 = 对齐到 A 的键之后逐截面百分位排名 − 0.5（只在有值的行上排），缺失记 0；09:30 截面全部特征与输出置 0。
    分箱边界与树都写死在 _GBDT 里（拟合窗上算好，不重算）。每个分量按自然季度循环计算，控制内存。
    """
    import gc
    import numpy as np
    import pandas as pd

    a = _main_a(datasources, start_date, end_date)[["date", "instrument", "factor"]].copy()
    a["date"] = pd.to_datetime(a["date"])
    a["instrument"] = a["instrument"].astype(str)
    key = a[["date", "instrument"]]
    n = len(a)
    hm = a["date"].dt.strftime("%H%M").to_numpy()
    z930 = hm == "0930"
    feat = _GBDT["feat_order"]
    X = np.zeros((n, len(feat)), np.float32)
    X[:, 12] = np.nan_to_num(pd.to_numeric(a["factor"], errors="coerce").to_numpy(float), nan=0.0)
    for _s in ("1000", "1030", "1100", "1130", "1330", "1400", "1430"):
        X[:, feat.index("slot" + _s)] = (hm == _s)
    del a
    gc.collect()

    def _quarters(fn):
        parts = []
        for _q in pd.period_range(pd.Timestamp(start_date).to_period("Q"), pd.Timestamp(end_date).to_period("Q"), freq="Q"):
            _st = max(pd.Timestamp(start_date), _q.start_time)
            _en = min(pd.Timestamp(end_date), _q.end_time)
            if _st > _en:
                continue
            parts.append(fn(datasources, _st.strftime("%Y-%m-%d %H:%M:%S"), _en.strftime("%Y-%m-%d %H:%M:%S")))
            gc.collect()
        b = pd.concat(parts, ignore_index=True)
        b["date"] = pd.to_datetime(b["date"])
        b["instrument"] = b["instrument"].astype(str)
        return b.drop_duplicates(["date", "instrument"])

    def _rank_on_key(b, col):
        """对齐到 A 的键，再逐截面百分位排名 − 0.5（只在有值的行上排），缺失记 0。"""
        m = key.merge(b[["date", "instrument", col]], on=["date", "instrument"], how="left")
        x = pd.to_numeric(m[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        r = x.groupby(m["date"]).rank(pct=True) - 0.5
        return r.fillna(0.0).to_numpy(np.float32)

    for _j, _fn, _ew in [
        (0, main_b1, 0.75),
        (1, main_b2, 0.0),
        (2, main_b3, 0.0),
        (3, main_b4, 0.0),
        (4, main_b5, 0.0),
        (5, main_b6, 0.0),
        (6, main_b7, 0.0),
        (7, main_b8, 0.0),
        (8, main_b9, 0.0),
        (9, main_b10, 0.0),
        (10, main_b11, 0.0),
        (11, main_b12, 0.0),
        (13, main_b13, 0.0),
        (25, main_b14, 0.0),
        (26, main_b15, 0.0),
    ]:
        b = _quarters(_fn)[["date", "instrument", "factor"]]
        b["factor"] = pd.to_numeric(b["factor"], errors="coerce")
        if _ew > 0:                                                       # 同一股票按时间连续的 EWMA（跨日），只用过去
            b = b.sort_values(["instrument", "date"]).reset_index(drop=True)
            b["factor"] = b.groupby("instrument", sort=False)["factor"].transform(lambda x: x.ewm(alpha=1 - _ew, adjust=False, ignore_na=True).mean())
        X[:, _j] = _rank_on_key(b, "factor")
        del b
        gc.collect()

    s = _quarters(_seas_impl)
    for _c in ("L_5", "VSH20", "PACE20", "DOW10"):
        X[:, feat.index(_c)] = _rank_on_key(s, _c)
    del s
    gc.collect()

    X[z930] = 0.0
    v = _gbdt_score(X)
    v[z930] = 0.0
    out = pd.DataFrame({"date": key["date"].to_numpy(), "instrument": key["instrument"].to_numpy(), "factor": np.nan_to_num(v, nan=0.0)})
    return out.sort_values(["date", "instrument"]).reset_index(drop=True)


def _gbdt_score(X):
    """纯 numpy 推理：逐列按拟合窗分箱（searchsorted, side=left），每棵树从根往下走（bin <= 切分点走左 2n+1，否则右 2n+2）。"""
    import numpy as np
    E = np.asarray(_GBDT["edges"], np.float32)
    B = np.empty(X.shape, np.uint8)
    for j in range(X.shape[1]):
        B[:, j] = np.searchsorted(E[j], X[:, j], side="left").astype(np.uint8)
    score = np.full(len(X), float(_GBDT["base"]))
    ar = np.arange(len(X))
    for tr in _GBDT["trees"]:
        m = max(int(it[1]) for it in tr) + 1
        LV, SF, SK, HS = np.zeros(m), np.zeros(m, np.int64), np.zeros(m, np.int64), np.zeros(m, bool)
        for it in tr:
            if it[0] == "split":
                SF[int(it[1])], SK[int(it[1])], HS[int(it[1])] = int(it[2]), int(it[3]), True
            else:
                LV[int(it[1])] = float(it[2])
        node = np.zeros(len(X), np.int64)
        for _ in range(int(_GBDT["depth"])):
            hs = HS[node]
            if not hs.any():
                break
            left = B[ar, SF[node]] <= SK[node]
            node = np.where(hs, np.where(left, 2 * node + 1, 2 * node + 2), node)
        score += float(_GBDT["lr"]) * LV[node]
    return score


def _seas_impl(datasources, start_date, end_date):
    """同时段季节性族的原始值（截面 10:00…14:30），口径同 因子实验/h1_season.py：
    30 分钟段 (09:30,10:00] … (14:30,15:00] 由 1 分钟 bar 聚合（与 1232 同一段 SQL）；多日统计按同一股票自己的交易日滚动再错开 1 日。
      L_5     下一段 lab（段收盘 ÷ 段 VWAP − 1）过去 5 日均值（至少 3 日）
      VSH20   下一段成交额占全天比重的过去 20 日均值（至少 10 日）
      PACE20  今日至本段累计成交额 ÷ 过去 20 日同一时刻累计成交额均值（至少 10 日）
      DOW10   过去 10 个同星期几的下一段 lab 均值（至少 5 个）
    """
    import numpy as np
    import pandas as pd
    import dai

    SEGS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
    CS = SEGS[:-1]
    NX = SEGS[1:]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
    sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
    sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
    cols = ["date", "instrument"] + ["L_5", "VSH20", "PACE20", "DOW10"]
    if not len(sg):
        return pd.DataFrame(columns=cols)
    sg["instrument"] = sg["instrument"].astype(str)
    sg["d"] = pd.to_datetime(sg["d"])
    amt = sg.pivot_table(index=["instrument", "d"], columns="seg", values="amt", aggfunc="last").reindex(columns=SEGS)
    vol = sg.pivot_table(index=["instrument", "d"], columns="seg", values="vol", aggfunc="last").reindex(columns=SEGS)
    cl = sg.pivot_table(index=["instrument", "d"], columns="seg", values="c_last", aggfunc="last").reindex(columns=SEGS)
    del sg
    amt, vol, cl = amt.sort_index(), vol.sort_index(), cl.sort_index()
    with np.errstate(all="ignore"):
        lab = cl / (amt / vol.where(vol > 0)) - 1.0
        daytot = amt.sum(axis=1)
        vsh = amt.div(daytot.where(daytot > 0), axis=0)
        cum = amt.cumsum(axis=1)

    def roll_prev(fr, N, minp):
        g = fr.groupby(level=0, sort=False)
        m = g.rolling(N, min_periods=minp).mean()
        m.index = m.index.droplevel(0)
        m = m.reindex(fr.index)
        return m.groupby(level=0, sort=False).shift(1)

    def nxt(fr):
        x = fr[NX].copy()
        x.columns = CS
        return x

    feats = {}
    feats["L_5"] = roll_prev(nxt(lab), 5, 3)
    feats["VSH20"] = roll_prev(nxt(vsh), 20, 10)
    pc = roll_prev(cum[CS], 20, 10)
    with np.errstate(all="ignore"):
        feats["PACE20"] = cum[CS] / pc.where(pc > 0)
    dw = nxt(lab)                                                          # 同一股票、同一星期几的序列上：先错开 1 个，再取 10 个的均值
    wd = pd.DatetimeIndex(dw.index.get_level_values(1)).weekday
    dw.index = pd.MultiIndex.from_arrays([dw.index.get_level_values(0), wd, dw.index.get_level_values(1)], names=["instrument", "wd", "d"])
    dw = dw.sort_index()
    sh = dw.groupby(level=[0, 1], sort=False).shift(1)
    rr = sh.groupby(level=[0, 1], sort=False).rolling(10, min_periods=5).mean()
    rr.index = rr.index.droplevel([0, 1])
    feats["DOW10"] = rr.reindex(dw.index).droplevel(1).sort_index()
    d0 = pd.Timestamp(pd.to_datetime(start_date).normalize())
    out = None
    for k, fr in feats.items():
        fr = fr[fr.index.get_level_values(1) >= d0]
        st = fr.stack().rename(k).reset_index()
        st.columns = ["instrument", "d", "slot", k]
        st["date"] = st["d"] + pd.to_timedelta(st["slot"].str[:2].astype(int), "h") + pd.to_timedelta(st["slot"].str[2:].astype(int), "m")
        st = st[["date", "instrument", k]]
        out = st if out is None else out.merge(st, on=["date", "instrument"], how="outer")
    out = out[(out["date"] >= pd.to_datetime(start_date)) & (out["date"] <= pd.to_datetime(end_date))]
    return out[cols].reset_index(drop=True)


_GBDT = __import__("json").loads(r"""{"feat_order":["L_lab10","M_q_lin","W_vdir","W_zero","Z_mfr60","Z_mfr90","s155","s139","s161","k1381","k1385","k1406","H1363","H152","slot1000","slot1030","slot1100","slot1130","slot1330","slot1400","slot1430","L_5","VSH20","PACE20","DOW10","CLS20","CLOSEPOS20"],"edges":[[-0.4679679572582245,-0.4369369447231293,-0.40490490198135376,-0.37387385964393616,-0.34184184670448303,-0.31081080436706543,-0.2787787914276123,-0.2477477490901947,-0.2167167216539383,-0.18468467891216278,-0.15365365147590637,-0.12162162363529205,-0.09059058874845505,-0.05855855718255043,-0.026526525616645813,0.0005005004932172596,0.027527527883648872,0.06056056171655655,0.09159158915281296,0.12262262403964996,0.15465465188026428,0.1856856793165207,0.2177177220582962,0.2487487494945526,0.2807807922363281,0.31181180477142334,0.34284284710884094,0.37487486004829407,0.40590590238571167,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1866866797208786,-0.1556556522846222,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0,0.03153153136372566,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.1556556522846222,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0,0.03153153136372566,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18803222477436066,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.46496495604515076,-0.4389389455318451,-0.4054054021835327,-0.3733733594417572,-0.3433433473110199,-0.3133133053779602,-0.28178179264068604,-0.24924924969673157,-0.2187187224626541,-0.1876876950263977,-0.1566566526889801,-0.12512512505054474,-0.09359358996152878,-0.06256256252527237,-0.03153153136372566,0.0005005004932172596,0.03253253176808357,0.06406406313180923,0.09509509801864624,0.12612612545490265,0.15715715289115906,0.1876876950263977,0.21921922266483307,0.2502502501010895,0.2812812924385071,0.31281280517578125,0.3443443477153778,0.3753753900527954,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34284284710884094,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1866866797208786,-0.1556556522846222,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.03153153136372566,0.0,0.03153153136372566,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34284284710884094,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1866866797208786,-0.1556556522846222,-0.12462462484836578,-0.09459459781646729,-0.06256256252527237,-0.03153153136372566,0.0,0.03153153136372566,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.15615615248680115,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0015015015378594398,0.03253253176808357,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.15715715289115906,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.15615615248680115,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.03153153136372566,0.0005005004932172596,0.03253253176808357,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.15715715289115906,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.15615615248680115,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0015015015378594398,0.03253253176808357,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.15715715289115906,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.1556556522846222,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0010010009864345193,0.03153153136372566,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.1566566526889801,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.03153153136372566,0.0005005004932172596,0.03253253176808357,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.1566566526889801,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37387385964393616,-0.34284284710884094,-0.31081080436706543,-0.2787787914276123,-0.2432432472705841,-0.19969969987869263,-0.1301301270723343,-0.08408408612012863,-0.06156156212091446,-0.046046044677495956,-0.027027027681469917,-0.01601601578295231,-0.0015015015378594398,0.011011010967195034,0.024024024605751038,0.041041042655706406,0.060060061514377594,0.08908908814191818,0.13963964581489563,0.20270270109176636,0.24474474787712097,0.2797797918319702,0.3123123049736023,0.34384384751319885,0.37487486004829407,0.4069069027900696,0.4379379451274872,0.4689689576625824],[-0.7377176284790039,-0.6132471561431885,-0.5277278423309326,-0.45955225825309753,-0.4011535048484802,-0.34951719641685486,-0.3028116524219513,-0.25979745388031006,-0.2196551114320755,-0.18162286281585693,-0.14548207819461823,-0.11100007593631744,-0.07755781710147858,-0.04516949504613876,-0.013478175736963749,0.01763116382062435,0.04839535057544708,0.07901275157928467,0.10969006270170212,0.1406002789735794,0.17197896540164948,0.2041686475276947,0.23722714185714722,0.2717551589012146,0.308149516582489,0.3471025228500366,0.38955414295196533,0.43762126564979553,0.49394744634628296,0.5654506683349609,0.6721454858779907],[-0.4679679572582245,-0.4369369447231293,-0.40590590238571167,-0.37487486004829407,-0.34384384751319885,-0.31181180477142334,-0.2807807922363281,-0.24974974989891052,-0.2187187224626541,-0.1876876950263977,-0.15615615248680115,-0.12462462484836578,-0.09359358996152878,-0.06256256252527237,-0.030530530959367752,0.0015015015378594398,0.03253253176808357,0.06356356292963028,0.09459459781646729,0.1256256252527237,0.15715715289115906,0.18868869543075562,0.21971972286701202,0.25075075030326843,0.28178179264068604,0.31281280517578125,0.34484484791755676,0.37587589025497437,0.4069069027900696,0.4379379451274872,0.4689689576625824],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0],[-0.4679679572582245,-0.4369369447231293,-0.40490490198135376,-0.37387385964393616,-0.34284284710884094,-0.31081080436706543,-0.2797797918319702,-0.2487487494945526,-0.2167167216539383,-0.1856856793165207,-0.15465465188026428,-0.12262262403964996,-0.09159158915281296,-0.06056056171655655,-0.028528528288006783,0.0005005004932172596,0.029529528692364693,0.06156156212091446,0.09259258955717087,0.12362362444400787,0.1556556522846222,0.1866866797208786,0.2187187224626541,0.24974974989891052,0.2807807922363281,0.31181180477142334,0.34384384751319885,0.37487486004829407,0.40590590238571167,0.4379379451274872,0.4689689576625824],[-0.46496495604515076,-0.4389389455318451,-0.40490490198135376,-0.37287285923957825,-0.3408408463001251,-0.3088088035583496,-0.2777777910232544,-0.24574574828147888,-0.21471472084522247,-0.18268267810344696,-0.15065065026283264,-0.11961962282657623,-0.08758758753538132,-0.0555555559694767,-0.024524524807929993,0.0005005004932172596,0.025525525212287903,0.05655655637383461,0.08858858793973923,0.12062062323093414,0.15165165066719055,0.18368367850780487,0.21564450860023499,0.2467467486858368,0.2787787914276123,0.3098098039627075,0.34184184670448303,0.37387385964393616,0.40490490198135376,0.4369369447231293,0.4689689576625824],[-0.46396395564079285,-0.4369369447231293,-0.40290290117263794,-0.37187185883522034,-0.3398398458957672,-0.3078078031539917,-0.2767767906188965,-0.24474474787712097,-0.21371372044086456,-0.18168167769908905,-0.14964964985847473,-0.11861861497163773,-0.08658658713102341,-0.0555555559694767,-0.023523524403572083,0.0005005004932172596,0.025525525212287903,0.05755755677819252,0.08958958834409714,0.12062062323093414,0.15265265107154846,0.18368367850780487,0.21571572124958038,0.2477477490901947,0.2787787914276123,0.31081080436706543,0.34184184670448303,0.37387385964393616,0.40590590238571167,0.4369369447231293,0.4689689576625824],[-0.4669669568538666,-0.43493494391441345,-0.40190190076828003,-0.3698698580265045,-0.3378378450870514,-0.30480480194091797,-0.27277275919914246,-0.24074074625968933,-0.2077077031135559,-0.1756756752729416,-0.14364364743232727,-0.11061061173677444,-0.07857857644557953,-0.04654654487967491,-0.013513513840734959,0.0005005004932172596,0.015515515580773354,0.04754754900932312,0.07957957684993744,0.11261261254549026,0.14464464783668518,0.1766766756772995,0.20870870351791382,0.24174174666404724,0.27377375960350037,0.3068068027496338,0.3388388454914093,0.3708708584308624,0.40290290117263794,0.43593594431877136,0.4679679572582245],[-0.4679679572582245,-0.43593594431877136,-0.40390390157699585,-0.37287285923957825,-0.3408408463001251,-0.3088088035583496,-0.2767767906188965,-0.24574574828147888,-0.21371372044086456,-0.18268267810344696,-0.15065065026283264,-0.11961962282657623,-0.08758758753538132,-0.0555555559694767,-0.023523524403572083,0.0005005004932172596,0.025525525212287903,0.05655655637383461,0.08858858793973923,0.12062062323093414,0.15165165066719055,0.18368367850780487,0.21571572124958038,0.2467467486858368,0.2787787914276123,0.31081080436706543,0.34184184670448303,0.37387385964393616,0.40590590238571167,0.4369369447231293,0.4689689576625824],[-0.4679679572582245,-0.43593594431877136,-0.40490490198135376,-0.37287285923957825,-0.3408408463001251,-0.3098098039627075,-0.2777777910232544,-0.24574574828147888,-0.21471472084522247,-0.18268267810344696,-0.15065065026283264,-0.11961962282657623,-0.08758758753538132,-0.05655655637383461,-0.024524524807929993,0.0005005004932172596,0.024524524807929993,0.05655655637383461,0.08858858793973923,0.11961962282657623,0.15165165066719055,0.18368367850780487,0.21471472084522247,0.2467467486858368,0.2787787914276123,0.3098098039627075,0.34184184670448303,0.37387385964393616,0.40490490198135376,0.4389389455318451,0.46596595644950867]],"base":0.0006448723388783821,"lr":0.015,"depth":5,"trees":[[["split",0,8.0,11,0],["split",1,6.0,6,1],["split",2,6.0,24,1],["split",3,4.0,23,2],["split",4,7.0,8,2],["split",5,4.0,14,2],["split",6,5.0,8,2],["split",7,7.0,7,3],["split",8,23.0,25,3],["split",9,4.0,20,3],["split",10,21.0,11,3],["split",11,8.0,18,3],["split",12,13.0,16,3],["split",13,20.0,0,3],["split",14,4.0,19,3],["split",15,20.0,0,4],["split",16,25.0,12,4],["split",17,3.0,5,4],["split",18,7.0,3,4],["split",19,0.0,7,4],["split",20,7.0,4,4],["split",21,2.0,24,4],["split",22,5.0,13,4],["split",23,7.0,8,4],["split",24,20.0,0,4],["split",25,5.0,19,4],["split",26,10.0,7,4],["split",27,6.0,29,4],["split",28,25.0,14,4],["split",29,7.0,28,4],["split",30,22.0,21,4],["leaf",31,-0.0501897327457191],["leaf",32,-0.013840845753956207],["leaf",33,-0.032448211689385095],["leaf",34,-0.01535135820432258],["leaf",35,-0.04715575106532133],["leaf",36,-0.06438302592008313],["leaf",37,-0.08870234595219824],["leaf",38,-0.047048817183122156],["leaf",39,-0.03636284377702655],["leaf",40,-0.016204067495743126],["leaf",41,-0.06451626933079614],["leaf",42,-0.03751495481680537],["leaf",43,-0.01692459891377642],["leaf",44,-0.03312759024099008],["leaf",45,0.012768579512435027],["leaf",46,-0.01075993391048643],["leaf",47,-0.004928425290902435],["leaf",48,0.015329601331887039],["leaf",49,0.032283619369262107],["leaf",50,0.012635483124658385],["leaf",51,-0.0021026268768576358],["leaf",52,-0.013757986766991517],["leaf",53,-0.006179892572290701],["leaf",54,0.01124087855824775],["leaf",55,0.04998391661617032],["leaf",56,0.06778145415562288],["leaf",57,0.0071804568855679575],["leaf",58,0.03478964995008728],["leaf",59,0.032422498777247066],["leaf",60,0.04677253475474191],["leaf",61,0.029305252418288005],["leaf",62,0.005410294876928448]],[["split",0,8.0,12,0],["split",1,7.0,3,1],["split",2,13.0,22,1],["split",3,7.0,0,2],["split",4,7.0,8,2],["split",5,5.0,11,2],["split",6,4.0,8,2],["split",7,22.0,11,3],["split",8,20.0,0,3],["split",9,4.0,16,3],["split",10,4.0,13,3],["split",11,13.0,17,3],["split",12,12.0,9,3],["split",13,6.0,25,3],["split",14,8.0,29,3],["leaf",15,-0.098289570017009],["split",16,5.0,29,4],["split",17,19.0,0,4],["leaf",18,-0.02472307886427735],["split",19,6.0,8,4],["split",20,10.0,6,4],["split",21,11.0,24,4],["split",22,24.0,4,4],["split",23,12.0,19,4],["split",24,10.0,7,4],["split",25,0.0,9,4],["split",26,26.0,6,4],["split",27,2.0,2,4],["split",28,20.0,0,4],["split",29,7.0,28,4],["split",30,13.0,30,4],["leaf",33,-0.07036385386522712],["leaf",34,-0.08933156413486387],["leaf",35,-0.06635294594599711],["leaf",36,-0.04041103648044617],["leaf",39,-0.02630079502920543],["leaf",40,-0.006954449182980568],["leaf",41,-0.05165672143951886],["leaf",42,-0.03360746251174961],["leaf",43,0.0016913744145963573],["leaf",44,-0.013168065895835926],["leaf",45,-0.031387504382600254],["leaf",46,-0.016476758882721865],["leaf",47,0.004975589477432557],["leaf",48,0.020350836351994077],["leaf",49,0.012613651621560041],["leaf",50,0.030488480622047207],["leaf",51,-0.023846624058436308],["leaf",52,-0.0016471146934993392],["leaf",53,-0.004958485242627856],["leaf",54,0.008711908873626674],["leaf",55,0.04721689389671254],["leaf",56,0.03167231745606154],["leaf",57,0.05892216876262522],["leaf",58,0.0263902215874463],["leaf",59,0.020909188043885026],["leaf",60,0.03614498020756603],["leaf",61,0.03915380521972221],["leaf",62,0.05363457620437008]],[["split",0,6.0,11,0],["split",1,7.0,4,1],["split",2,8.0,24,1],["split",3,4.0,16,2],["split",4,7.0,8,2],["split",5,6.0,17,2],["split",6,6.0,27,2],["split",7,3.0,25,3],["split",8,20.0,0,3],["split",9,6.0,8,3],["split",10,10.0,10,3],["split",11,4.0,11,3],["split",12,4.0,18,3],["split",13,20.0,0,3],["split",14,12.0,22,3],["leaf",15,-0.0366121829650491],["leaf",16,-0.015164999859418306],["split",17,2.0,27,4],["split",18,21.0,17,4],["split",19,21.0,2,4],["split",20,10.0,11,4],["split",21,24.0,21,4],["split",22,4.0,11,4],["split",23,21.0,5,4],["split",24,26.0,20,4],["split",25,12.0,21,4],["split",26,21.0,11,4],["split",27,5.0,7,4],["leaf",28,0.007786139044245288],["split",29,4.0,8,4],["split",30,20.0,0,4],["leaf",35,-0.0592286234141512],["leaf",36,-0.08250662540607091],["leaf",37,-0.05368151633347735],["leaf",38,-0.014311028112828106],["leaf",39,-0.05457114230627395],["leaf",40,-0.03206585742496466],["leaf",41,-0.031169155989674794],["leaf",42,-0.008146531737377411],["leaf",43,-0.02490961340532496],["leaf",44,-0.009108565099594951],["leaf",45,0.014055003083783239],["leaf",46,-0.011072882435004854],["leaf",47,-0.0046102886140338235],["leaf",48,0.01277844797268943],["leaf",49,-0.011550694455325439],["leaf",50,0.0031037626411867143],["leaf",51,0.01894758096978716],["leaf",52,0.030434554061615177],["leaf",53,-0.005829987085904336],["leaf",54,0.008328685577286617],["leaf",55,0.04517848847520976],["leaf",56,0.030441936441348745],["leaf",59,0.0430816145591197],["leaf",60,0.026072586693100704],["leaf",61,0.05629381337884439],["leaf",62,0.03519338985342325]],[["split",0,8.0,10,0],["split",1,4.0,25,1],["split",2,6.0,22,1],["split",3,7.0,2,2],["split",4,13.0,4,2],["split",5,4.0,13,2],["split",6,5.0,10,2],["split",7,3.0,22,3],["split",8,7.0,13,3],["split",9,7.0,1,3],["split",10,7.0,5,3],["split",11,8.0,16,3],["split",12,6.0,15,3],["split",13,20.0,0,3],["split",14,5.0,20,3],["split",15,10.0,7,4],["leaf",16,-0.028860522209394106],["split",17,20.0,0,4],["split",18,25.0,16,4],["split",19,10.0,12,4],["split",20,24.0,18,4],["split",21,2.0,26,4],["split",22,7.0,11,4],["split",23,12.0,17,4],["split",24,21.0,9,4],["split",25,4.0,19,4],["split",26,2.0,10,4],["split",27,2.0,6,4],["split",28,25.0,18,4],["split",29,7.0,29,4],["split",30,3.0,9,4],["leaf",31,-0.07627428895289605],["leaf",32,-0.048958733477698066],["leaf",35,-0.025512665322206786],["leaf",36,-0.003062187665781878],["leaf",37,-0.013235281835119888],["leaf",38,0.005079425885178182],["leaf",39,-0.08249412769419655],["leaf",40,-0.06625159161636983],["leaf",41,-0.051358444729771066],["leaf",42,-0.03046351155044985],["leaf",43,-0.036906806543092904],["leaf",44,-0.0679335055225851],["leaf",45,-0.03090337716356143],["leaf",46,-0.009395000596557493],["leaf",47,0.002231798539651679],["leaf",48,0.012335214752711664],["leaf",49,0.00712679207635129],["leaf",50,0.025559199282655386],["leaf",51,-0.0007915309745178797],["leaf",52,-0.014030751447930713],["leaf",53,0.013368672428788102],["leaf",54,-0.0004511106299322448],["leaf",55,0.054507725594189926],["leaf",56,0.03797007849762529],["leaf",57,0.007918384683930342],["leaf",58,0.03618637210131687],["leaf",59,0.02274468350417589],["leaf",60,0.03963964508552299],["leaf",61,-0.005925356359900804],["leaf",62,0.01727470794688357]],[["split",0,6.0,15,0],["split",1,7.0,6,1],["split",2,8.0,23,1],["split",3,7.0,1,2],["split",4,4.0,20,2],["split",5,12.0,15,2],["split",6,21.0,4,2],["split",7,3.0,21,3],["split",8,10.0,6,3],["split",9,12.0,18,3],["split",10,10.0,9,3],["split",11,4.0,5,3],["split",12,4.0,16,3],["split",13,2.0,13,3],["split",14,13.0,29,3],["split",15,10.0,5,4],["leaf",16,-0.030230234624188026],["split",17,4.0,17,4],["split",18,20.0,0,4],["split",19,21.0,5,4],["split",20,10.0,4,4],["split",21,22.0,7,4],["split",22,7.0,12,4],["leaf",23,0.017106582814747792],["split",24,21.0,2,4],["split",25,21.0,4,4],["split",26,25.0,4,4],["split",27,1.0,17,4],["leaf",28,-0.0025421451959819316],["split",29,2.0,8,4],["split",30,25.0,18,4],["leaf",31,-0.0847070518850553],["leaf",32,-0.059302538872414975],["leaf",35,-0.03376032185137019],["leaf",36,-0.05230769590060136],["leaf",37,-0.03176191784573108],["leaf",38,-0.011137773949712933],["leaf",39,-0.015517598407113547],["leaf",40,-0.0029260239076028807],["leaf",41,-0.004095932669547321],["leaf",42,0.011704761829368297],["leaf",43,-0.015302235690171834],["leaf",44,-0.030436799556930256],["leaf",45,-0.01764897898879297],["leaf",46,-0.006923975821980485],["leaf",49,-0.01640639388264939],["leaf",50,0.0020576062094569466],["leaf",51,0.008649076252257087],["leaf",52,0.024474531185597052],["leaf",53,-0.013146860353084782],["leaf",54,0.010122040503391288],["leaf",55,0.01019870237613807],["leaf",56,0.036292261881216495],["leaf",59,0.04085688047211193],["leaf",60,0.029484353453985042],["leaf",61,0.04221034741528743],["leaf",62,0.058382066463002785]],[["split",0,7.0,8,0],["split",1,7.0,1,1],["split",2,8.0,22,1],["split",3,3.0,3,2],["split",4,4.0,15,2],["split",5,8.0,12,2],["split",6,6.0,30,2],["split",7,10.0,12,3],["split",8,4.0,20,3],["split",9,8.0,9,3],["split",10,20.0,0,3],["split",11,21.0,2,3],["split",12,4.0,18,3],["split",13,12.0,23,3],["split",14,21.0,9,3],["leaf",15,-0.06255169011329889],["leaf",16,-0.032758274107853376],["leaf",17,-0.04079234924081928],["split",18,21.0,11,4],["split",19,25.0,23,4],["split",20,24.0,21,4],["split",21,10.0,4,4],["split",22,25.0,17,4],["split",23,4.0,20,4],["split",24,5.0,6,4],["split",25,12.0,25,4],["split",26,22.0,24,4],["split",27,2.0,5,4],["split",28,5.0,6,4],["leaf",29,0.04148599091199026],["split",30,22.0,20,4],["leaf",37,-0.08748832917931748],["leaf",38,-0.06909552797846662],["leaf",39,-0.02269316073704393],["leaf",40,-0.007116490056405086],["leaf",41,-0.008414794102207107],["leaf",42,0.006533129598995932],["leaf",43,-0.05173876635127941],["leaf",44,-0.032842220975491436],["leaf",45,-0.03122432918513207],["leaf",46,0.0018844350085123727],["leaf",47,-0.008548987326515822],["leaf",48,-0.043409020821166676],["leaf",49,0.01318842364743204],["leaf",50,-0.00715933355477002],["leaf",51,0.010169968528556839],["leaf",52,0.023035160591396337],["leaf",53,0.0021645488040060532],["leaf",54,-0.01252392984576119],["leaf",55,0.03074034703189266],["leaf",56,0.0165389139371993],["leaf",57,0.04153317250747828],["leaf",58,0.025991019139431755],["leaf",61,0.06558210717668644],["leaf",62,0.04852600404123154]],[["split",0,6.0,11,0],["split",1,7.0,4,1],["split",2,13.0,21,1],["split",3,7.0,0,2],["split",4,4.0,16,2],["split",5,4.0,13,2],["split",6,13.0,27,2],["split",7,24.0,17,3],["split",8,20.0,0,3],["split",9,21.0,8,3],["split",10,21.0,15,3],["split",11,4.0,2,3],["split",12,26.0,8,3],["split",13,5.0,13,3],["split",14,2.0,3,3],["split",15,4.0,29,4],["leaf",16,-0.05156240917750474],["split",17,10.0,10,4],["split",18,25.0,13,4],["split",19,25.0,15,4],["split",20,7.0,15,4],["split",21,19.0,0,4],["split",22,20.0,0,4],["split",23,12.0,18,4],["split",24,24.0,21,4],["split",25,7.0,21,4],["split",26,7.0,9,4],["split",27,20.0,0,4],["split",28,21.0,13,4],["split",29,13.0,30,4],["split",30,3.0,19,4],["leaf",31,-0.062131158988644915],["leaf",32,-0.07494898841096886],["leaf",35,-0.05427489555648929],["leaf",36,-0.03518777936854536],["leaf",37,-0.03554509158694028],["leaf",38,0.002794458300401167],["leaf",39,-0.022499018966845124],["leaf",40,-0.008203173773006906],["leaf",41,-0.003622399946305278],["leaf",42,0.01637192839024176],["leaf",43,-0.02539031554055833],["leaf",44,-0.006488520133714464],["leaf",45,-0.015501752794871598],["leaf",46,0.013907438326711204],["leaf",47,0.014822351359614705],["leaf",48,0.029830092965391814],["leaf",49,0.005420856318731652],["leaf",50,0.017988514386470898],["leaf",51,-0.01970717735029971],["leaf",52,-0.004812181484035891],["leaf",53,-0.01542062007101757],["leaf",54,0.004044352028452301],["leaf",55,0.030615781021651717],["leaf",56,0.004861842913675887],["leaf",57,0.0007519558004262794],["leaf",58,0.019731894939534308],["leaf",59,0.04150508936991637],["leaf",60,0.05893230792175795],["leaf",61,0.0216577326921107],["leaf",62,0.03636770657857725]],[["split",0,13.0,10,0],["split",1,7.0,3,1],["split",2,6.0,21,1],["split",3,20.0,0,2],["split",4,7.0,8,2],["split",5,4.0,11,2],["split",6,6.0,28,2],["split",7,3.0,26,3],["split",8,25.0,16,3],["split",9,4.0,20,3],["split",10,23.0,29,3],["split",11,7.0,12,3],["split",12,4.0,20,3],["split",13,5.0,17,3],["split",14,20.0,0,3],["split",15,7.0,1,4],["leaf",16,-0.01204829359239799],["leaf",17,-0.03582540931211974],["leaf",18,0.006886452428643835],["split",19,24.0,10,4],["split",20,20.0,0,4],["split",21,7.0,15,4],["leaf",22,-0.023664196654787684],["split",23,12.0,17,4],["split",24,2.0,14,4],["split",25,21.0,14,4],["split",26,1.0,3,4],["split",27,7.0,27,4],["split",28,21.0,20,4],["split",29,4.0,15,4],["leaf",30,0.021355689845792565],["leaf",31,-0.06596454076001419],["leaf",32,-0.04201165546607072],["leaf",39,-0.02481902129053084],["leaf",40,-0.01031067584971652],["leaf",41,-0.03524290736144663],["leaf",42,-0.012359825302220035],["leaf",43,-0.011784108267226449],["leaf",44,0.0008200902907565289],["leaf",47,-0.003959350086434347],["leaf",48,0.013313415580871195],["leaf",49,0.0230290067913728],["leaf",50,0.008331031297147865],["leaf",51,-0.004652104130601376],["leaf",52,0.007630799475277545],["leaf",53,-0.027024842255053052],["leaf",54,-0.007844011477067899],["leaf",55,0.022389908514411822],["leaf",56,0.03361448957903469],["leaf",57,0.0035883048457730866],["leaf",58,0.020485774297176292],["leaf",59,0.044311216770250975],["leaf",60,0.020872658234617276]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,8.0,26,1],["split",3,20.0,0,2],["split",4,7.0,10,2],["split",5,12.0,14,2],["split",6,20.0,0,2],["split",7,4.0,15,3],["split",8,21.0,18,3],["split",9,5.0,14,3],["split",10,24.0,1,3],["split",11,4.0,26,3],["split",12,5.0,21,3],["split",13,4.0,5,3],["split",14,25.0,18,3],["leaf",15,-0.019974160560069728],["split",16,10.0,13,4],["leaf",17,-0.03281839242740105],["leaf",18,0.004825435148838847],["split",19,0.0,22,4],["split",20,20.0,0,4],["leaf",21,-0.02694900503413975],["split",22,21.0,5,4],["split",23,8.0,21,4],["split",24,26.0,13,4],["split",25,10.0,14,4],["split",26,24.0,7,4],["split",27,13.0,30,4],["split",28,0.0,6,4],["leaf",29,0.003767103013642248],["leaf",30,0.03177802555146533],["leaf",33,-0.05946570538724261],["leaf",34,-0.04023521585524509],["leaf",39,-0.0180506748884326],["leaf",40,0.005490386644177756],["leaf",41,-0.02733655984465097],["leaf",42,-0.007327529849578614],["leaf",45,-0.01843670334177236],["leaf",46,-0.00199451906085644],["leaf",47,-0.00017645739132451214],["leaf",48,0.016104765263532803],["leaf",49,-0.03495513585486413],["leaf",50,-0.00891320357106463],["leaf",51,0.00984281329737746],["leaf",52,0.020522226357461728],["leaf",53,-0.012019938734327868],["leaf",54,0.0037582170818372183],["leaf",55,0.03936548725243392],["leaf",56,0.05553369281237839],["leaf",57,0.014964666264094996],["leaf",58,0.03272376788802972]],[["split",0,8.0,9,0],["split",1,7.0,3,1],["split",2,6.0,20,1],["split",3,4.0,19,2],["split",4,20.0,0,2],["split",5,4.0,20,2],["split",6,4.0,5,2],["split",7,3.0,23,3],["split",8,20.0,0,3],["split",9,2.0,4,3],["split",10,25.0,8,3],["split",11,10.0,6,3],["split",12,21.0,7,3],["split",13,20.0,0,3],["split",14,1.0,8,3],["leaf",15,-0.02943256412204015],["leaf",16,-0.0058038971386909785],["split",17,2.0,26,4],["leaf",18,-0.022985671786678243],["leaf",19,0.008636421539775613],["split",20,4.0,28,4],["leaf",21,-0.023791727641175566],["split",22,2.0,18,4],["split",23,7.0,17,4],["split",24,7.0,7,4],["split",25,23.0,11,4],["split",26,26.0,6,4],["split",27,12.0,30,4],["split",28,7.0,23,4],["split",29,7.0,26,4],["split",30,10.0,7,4],["leaf",35,-0.03951463175984781],["leaf",36,-0.06094949026503958],["leaf",41,-0.0180906143855509],["leaf",42,-0.03437056007029546],["leaf",45,-0.006135225119483321],["leaf",46,0.017680403843877975],["leaf",47,-0.011469177571854663],["leaf",48,0.0034997701283562073],["leaf",49,-0.00704577630143617],["leaf",50,0.010042227420069097],["leaf",51,-0.005321755735954662],["leaf",52,-0.021992397146533164],["leaf",53,-0.01575193390453589],["leaf",54,0.00011864850737033661],["leaf",55,0.03316327577024946],["leaf",56,0.05317758797591171],["leaf",57,-0.0012715282473840612],["leaf",58,0.021017700042444788],["leaf",59,-0.0026640824692521535],["leaf",60,0.01419291315400419],["leaf",61,0.009067044415710719],["leaf",62,0.021509419169720177]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,8.0,23,1],["split",3,7.0,0,2],["split",4,4.0,20,2],["split",5,4.0,16,2],["split",6,4.0,6,2],["split",7,2.0,19,3],["split",8,4.0,12,3],["split",9,10.0,14,3],["split",10,7.0,8,3],["split",11,26.0,20,3],["split",12,22.0,25,3],["split",13,20.0,0,3],["split",14,12.0,19,3],["leaf",15,-0.027235814870182104],["split",16,4.0,28,4],["leaf",17,-0.007286028232987436],["split",18,24.0,28,4],["split",19,6.0,6,4],["split",20,25.0,20,4],["split",21,2.0,25,4],["split",22,20.0,0,4],["split",23,12.0,17,4],["split",24,6.0,18,4],["split",25,1.0,29,4],["split",26,23.0,20,4],["split",27,19.0,0,4],["split",28,23.0,13,4],["split",29,4.0,15,4],["split",30,5.0,19,4],["leaf",33,-0.043129987501815535],["leaf",34,-0.06550074041439288],["leaf",37,-0.0372472594713229],["leaf",38,-0.01607250063971699],["leaf",39,-0.018263217986057324],["leaf",40,-0.006993542555027656],["leaf",41,-0.005185617443237333],["leaf",42,0.008121409813845596],["leaf",43,-0.019310693488552237],["leaf",44,-0.031686881111747],["leaf",45,-0.015506386304683395],["leaf",46,0.0020719595079985885],["leaf",47,-0.00015583840193711813],["leaf",48,0.011023252153992845],["leaf",49,0.01101916282361987],["leaf",50,0.021645319768155077],["leaf",51,0.0019186992651546964],["leaf",52,-0.018716262712562517],["leaf",53,-0.006961998225154057],["leaf",54,-0.025570531070431158],["leaf",55,0.03969462778722121],["leaf",56,0.012103350655529786],["leaf",57,0.002927738647846172],["leaf",58,0.019117259719541983],["leaf",59,0.016445666307546226],["leaf",60,0.001285528902453565],["leaf",61,0.024539678588500962],["leaf",62,0.011377481268308088]],[["split",0,7.0,7,0],["split",1,7.0,1,1],["split",2,6.0,18,1],["split",3,7.0,0,2],["split",4,3.0,20,2],["split",5,4.0,9,2],["split",6,8.0,27,2],["split",7,25.0,15,3],["split",8,8.0,0,3],["split",9,21.0,7,3],["split",10,1.0,7,3],["split",11,25.0,15,3],["split",12,10.0,2,3],["split",13,4.0,13,3],["split",14,8.0,30,3],["split",15,21.0,16,4],["split",16,0.0,18,4],["leaf",17,-0.02290559597803535],["split",18,3.0,15,4],["split",19,23.0,18,4],["split",20,24.0,25,4],["split",21,2.0,27,4],["split",22,3.0,27,4],["split",23,21.0,7,4],["split",24,3.0,23,4],["split",25,9.0,12,4],["split",26,13.0,14,4],["split",27,20.0,0,4],["split",28,23.0,24,4],["split",29,25.0,4,4],["split",30,2.0,3,4],["leaf",31,-0.06494212732769086],["leaf",32,-0.043700813141518795],["leaf",33,-0.03090850500876499],["leaf",34,-0.05468752507200694],["leaf",37,-0.046073426703377586],["leaf",38,-0.03054390606092678],["leaf",39,-0.027336284875568412],["leaf",40,-0.0418474431976114],["leaf",41,-0.023805763622314662],["leaf",42,-0.009945830890872344],["leaf",43,-0.014074685300480885],["leaf",44,-0.030164396207338094],["leaf",45,-0.010805419611697608],["leaf",46,0.004567652725397086],["leaf",47,-0.012881432253824286],["leaf",48,0.007421441476705338],["leaf",49,0.005482230608033302],["leaf",50,0.02065441961217644],["leaf",51,-0.02421613018937221],["leaf",52,-0.004606722616903888],["leaf",53,-0.005227392302647605],["leaf",54,0.0036929780660060136],["leaf",55,0.02030025157588228],["leaf",56,0.002389963688288644],["leaf",57,0.010000222295352664],["leaf",58,-0.006221610488325393],["leaf",59,0.011203949397772478],["leaf",60,0.02564702680532023],["leaf",61,0.047719042430411],["leaf",62,0.029713495629407997]],[["split",0,6.0,11,0],["split",1,4.0,24,1],["split",2,8.0,20,1],["split",3,7.0,5,2],["split",4,7.0,1,2],["split",5,4.0,10,2],["split",6,6.0,28,2],["split",7,20.0,0,3],["split",8,3.0,22,3],["split",9,2.0,26,3],["split",10,21.0,13,3],["split",11,10.0,7,3],["split",12,0.0,2,3],["split",13,3.0,24,3],["split",14,5.0,3,3],["split",15,3.0,27,4],["leaf",16,0.007120473424750802],["split",17,23.0,23,4],["split",18,9.0,19,4],["split",19,9.0,17,4],["split",20,0.0,20,4],["split",21,17.0,0,4],["split",22,20.0,0,4],["split",23,24.0,14,4],["split",24,7.0,13,4],["split",25,7.0,17,4],["split",26,7.0,7,4],["split",27,21.0,10,4],["split",28,3.0,30,4],["split",29,1.0,22,4],["split",30,1.0,7,4],["leaf",31,-0.032430472143730064],["leaf",32,0.0010793804825107846],["leaf",35,-0.00610120047416015],["leaf",36,-0.016450927815707592],["leaf",37,-0.005483001183915326],["leaf",38,0.01614002508616945],["leaf",39,-0.043718941066483774],["leaf",40,-0.025225525737208147],["leaf",41,-0.06557457925761526],["leaf",42,-0.038809865352347796],["leaf",43,-0.02470749642911245],["leaf",44,-0.042881918914691716],["leaf",45,-0.018715019414271936],["leaf",46,0.007336887443803371],["leaf",47,-0.010264677540043511],["leaf",48,0.009653449544965949],["leaf",49,0.004798950237530825],["leaf",50,0.016788659415984248],["leaf",51,-0.025208230051533234],["leaf",52,-0.004713901871480872],["leaf",53,-0.01669401342640837],["leaf",54,0.002601907497598596],["leaf",55,0.0063708295478729585],["leaf",56,0.014191194156337886],["leaf",57,0.022032843065416523],["leaf",58,0.03610108486116471],["leaf",59,0.02888277669629338],["leaf",60,0.04779487661037918],["leaf",61,0.0032295088297452403],["leaf",62,0.025907448592661433]],[["split",0,8.0,13,0],["split",1,7.0,3,1],["split",2,6.0,20,1],["split",3,20.0,0,2],["split",4,4.0,22,2],["split",5,5.0,15,2],["split",6,4.0,13,2],["split",7,10.0,10,3],["split",8,3.0,12,3],["split",9,7.0,14,3],["split",10,13.0,1,3],["split",11,20.0,0,3],["split",12,0.0,9,3],["split",13,20.0,0,3],["split",14,4.0,25,3],["split",15,6.0,1,4],["split",16,4.0,18,4],["leaf",17,-0.026257052498205407],["leaf",18,0.009030382523132985],["split",19,3.0,21,4],["split",20,25.0,6,4],["leaf",21,0.00697498109409979],["split",22,12.0,3,4],["split",23,8.0,16,4],["split",24,25.0,16,4],["split",25,7.0,17,4],["split",26,21.0,6,4],["split",27,19.0,0,4],["split",28,25.0,15,4],["split",29,23.0,27,4],["leaf",30,-0.00977441955120128],["leaf",31,-0.035056079675834344],["leaf",32,-0.053984297716295794],["leaf",33,-0.00382416076876748],["leaf",34,-0.029457284589147613],["leaf",39,-0.010214760192943328],["leaf",40,0.0015142466132040022],["leaf",41,-0.0088657213026472],["leaf",42,0.008721640447554687],["leaf",45,-0.028131914808058642],["leaf",46,-0.013253684797883622],["leaf",47,0.005221041020561565],["leaf",48,0.014153104630227977],["leaf",49,-0.026938500093597644],["leaf",50,0.0030479822515196396],["leaf",51,-0.018427047450721205],["leaf",52,-0.005810441702806321],["leaf",53,-0.01351631295513454],["leaf",54,0.003962090089618752],["leaf",55,0.028006123052381808],["leaf",56,0.013324020634333992],["leaf",57,-0.014746909542122997],["leaf",58,0.021313277723732666],["leaf",59,0.011906188470743373],["leaf",60,-0.006547298682492604]],[["split",0,6.0,14,0],["split",1,7.0,5,1],["split",2,4.0,12,1],["split",3,7.0,0,2],["split",4,10.0,4,2],["split",5,20.0,0,2],["split",6,8.0,19,2],["split",7,24.0,19,3],["split",8,10.0,10,3],["split",9,23.0,26,3],["split",10,12.0,7,3],["split",11,19.0,0,3],["split",12,21.0,10,3],["split",13,23.0,25,3],["split",14,21.0,8,3],["split",15,4.0,28,4],["leaf",16,-0.034915795035006886],["split",17,20.0,0,4],["split",18,2.0,8,4],["split",19,26.0,17,4],["split",20,4.0,19,4],["split",21,4.0,17,4],["split",22,3.0,20,4],["split",23,12.0,28,4],["split",24,0.0,24,4],["leaf",25,-0.017277809582061435],["split",26,25.0,20,4],["split",27,12.0,7,4],["leaf",28,-0.017334724362765754],["split",29,12.0,24,4],["split",30,26.0,6,4],["leaf",31,-0.04342662296071779],["leaf",32,-0.06285779454199776],["leaf",35,-0.0320058385407714],["leaf",36,-0.008717237679428738],["leaf",37,0.0001931841103427146],["leaf",38,-0.021233945906338612],["leaf",39,-0.0252138477186544],["leaf",40,-0.008876315999113298],["leaf",41,0.001893052005187706],["leaf",42,-0.014170958089413137],["leaf",43,-0.0030210695382042255],["leaf",44,-0.014561333077404821],["leaf",45,-0.00344731820012956],["leaf",46,0.005794012635715933],["leaf",47,0.0207793335230298],["leaf",48,0.03529810268181104],["leaf",49,0.005208261918004959],["leaf",50,0.017752014728591597],["leaf",53,0.0035991352992348845],["leaf",54,0.022264367820145604],["leaf",55,-0.01399690523059674],["leaf",56,0.005334328735066242],["leaf",59,-0.004276489346472238],["leaf",60,0.011582118930281472],["leaf",61,0.004639904439267169],["leaf",62,0.0170098115292577]],[["split",0,6.0,9,0],["split",1,4.0,15,1],["split",2,6.0,24,1],["split",3,18.0,0,2],["split",4,7.0,2,2],["split",5,5.0,17,2],["split",6,20.0,0,2],["split",7,10.0,4,3],["leaf",8,0.006954960199157711],["split",9,10.0,12,3],["split",10,20.0,0,3],["split",11,12.0,22,3],["split",12,21.0,14,3],["split",13,2.0,14,3],["split",14,21.0,9,3],["leaf",15,-0.019493038740535346],["split",16,1.0,4,4],["split",19,12.0,6,4],["split",20,21.0,16,4],["split",21,23.0,26,4],["split",22,25.0,13,4],["split",23,24.0,3,4],["split",24,5.0,7,4],["split",25,4.0,24,4],["split",26,20.0,0,4],["split",27,12.0,26,4],["split",28,3.0,15,4],["leaf",29,-0.012958426958613076],["split",30,7.0,27,4],["leaf",33,-0.01959936682451569],["leaf",34,-0.0028795749333764974],["leaf",39,-0.03581460195613563],["leaf",40,-0.058706614525954694],["leaf",41,-0.03331270266293046],["leaf",42,-0.014554203480133358],["leaf",43,-0.014962967062919968],["leaf",44,-0.027864374379927743],["leaf",45,-0.02468438852168556],["leaf",46,0.011843243843579435],["leaf",47,-0.00650967430460606],["leaf",48,0.004661283696278385],["leaf",49,0.018043923875514033],["leaf",50,0.006775441868110105],["leaf",51,-0.005689168276522025],["leaf",52,-0.01756683494648062],["leaf",53,-0.002899403170986826],["leaf",54,0.016860749568424222],["leaf",55,0.022982991752697333],["leaf",56,0.033459138916063016],["leaf",57,-0.0018480614434054179],["leaf",58,0.018435867949761632],["leaf",61,0.00116098426199517],["leaf",62,0.027638189549462226]],[["split",0,8.0,15,0],["split",1,7.0,2,1],["split",2,6.0,23,1],["split",3,17.0,0,2],["split",4,7.0,6,2],["split",5,2.0,6,2],["split",6,4.0,12,2],["split",7,7.0,0,3],["leaf",8,-0.04554838451904076],["split",9,3.0,26,3],["split",10,2.0,5,3],["split",11,20.0,0,3],["split",12,3.0,23,3],["split",13,7.0,30,3],["split",14,23.0,24,3],["split",15,9.0,20,4],["split",16,6.0,0,4],["split",19,20.0,0,4],["split",20,4.0,10,4],["split",21,24.0,18,4],["split",22,26.0,21,4],["split",23,7.0,26,4],["leaf",24,-0.006294629247410189],["split",25,21.0,2,4],["split",26,0.0,26,4],["split",27,20.0,0,4],["split",28,0.0,15,4],["split",29,21.0,7,4],["split",30,2.0,12,4],["leaf",31,-0.04313242691432954],["leaf",32,-0.024439016729619224],["leaf",33,-0.0026584825622268196],["leaf",34,-0.024748370050175588],["leaf",39,-0.02007551172611037],["leaf",40,0.0018327969586366018],["leaf",41,0.008138248538785624],["leaf",42,-0.009952493478004119],["leaf",43,0.0010634202162704894],["leaf",44,0.014439545713492284],["leaf",45,-0.008823773761445455],["leaf",46,0.0020626580559139278],["leaf",47,0.015914805326134936],["leaf",48,0.030497865060913978],["leaf",51,-0.013194976344236465],["leaf",52,0.0012484612046210057],["leaf",53,0.007694790239977735],["leaf",54,0.027476137030084963],["leaf",55,0.022326091681763076],["leaf",56,0.002589105151172809],["leaf",57,0.03607021872434528],["leaf",58,0.042970598408819255],["leaf",59,0.0030618626316592156],["leaf",60,0.017219114529323468],["leaf",61,0.006617425429356089],["leaf",62,-0.015789783620112966]],[["split",0,7.0,6,0],["split",1,4.0,22,1],["split",2,12.0,16,1],["split",3,4.0,3,2],["split",4,2.0,28,2],["split",5,26.0,10,2],["split",6,6.0,24,2],["leaf",7,0.006225623095602117],["split",8,21.0,3,3],["split",9,10.0,3,3],["split",10,10.0,4,3],["split",11,21.0,2,3],["split",12,8.0,14,3],["split",13,21.0,2,3],["split",14,21.0,4,3],["leaf",17,-0.030500998608165818],["split",18,6.0,5,4],["leaf",19,-0.038613668617019974],["split",20,25.0,22,4],["leaf",21,-0.055233320560124276],["split",22,21.0,21,4],["split",23,25.0,6,4],["split",24,20.0,0,4],["split",25,21.0,16,4],["split",26,17.0,0,4],["split",27,2.0,14,4],["split",28,22.0,26,4],["split",29,10.0,20,4],["split",30,19.0,0,4],["leaf",37,-0.019191824328913955],["leaf",38,-0.006770278997027776],["leaf",41,-0.022565318435623725],["leaf",42,-0.008479623312990733],["leaf",45,-0.04218600024510576],["leaf",46,-0.01707099501816447],["leaf",47,-0.03327723881272585],["leaf",48,-0.011521691499789526],["leaf",49,-0.007084272612744977],["leaf",50,0.00794683061319715],["leaf",51,-0.006648653727213455],["leaf",52,0.0024217980840709353],["leaf",53,0.0041109659826663435],["leaf",54,0.021923360436248263],["leaf",55,-0.0026404317557876446],["leaf",56,-0.019370289986497608],["leaf",57,0.009090530796089394],["leaf",58,-0.002799502357821423],["leaf",59,0.009020388424544392],["leaf",60,-0.0017675974388498522],["leaf",61,0.02113377870484294],["leaf",62,0.007515275863900821]],[["split",0,8.0,15,0],["split",1,8.0,7,1],["split",2,6.0,27,1],["split",3,7.0,3,2],["split",4,10.0,1,2],["split",5,0.0,5,2],["split",6,2.0,14,2],["split",7,10.0,2,3],["split",8,17.0,0,3],["leaf",9,-0.022399615161314973],["split",10,21.0,2,3],["split",11,12.0,20,3],["split",12,12.0,20,3],["split",13,20.0,0,3],["split",14,4.0,16,3],["split",15,3.0,14,4],["split",16,20.0,0,4],["split",17,19.0,0,4],["split",18,23.0,19,4],["split",21,7.0,15,4],["split",22,1.0,4,4],["split",23,9.0,5,4],["split",24,9.0,9,4],["split",25,14.0,0,4],["split",26,4.0,10,4],["split",27,19.0,0,4],["leaf",28,-0.0018626359246485572],["split",29,0.0,17,4],["leaf",30,0.0022287203438569056],["leaf",31,-0.0529797670424946],["leaf",32,-0.019181828819914454],["leaf",33,-0.026616507546801293],["leaf",34,-0.0015923162635607556],["leaf",35,-0.008672515332013485],["leaf",36,0.0040786984852231815],["leaf",37,-0.006756275976218253],["leaf",38,-0.04074174465783606],["leaf",43,-0.02585645321182966],["leaf",44,-0.0033792542558757615],["leaf",45,-0.01251814685134051],["leaf",46,-9.2934880263908e-05],["leaf",47,-0.0179466410714566],["leaf",48,-0.002521773288190444],["leaf",49,-0.00310974936644952],["leaf",50,0.013593748352156015],["leaf",51,0.003675076908314194],["leaf",52,0.016855076690136202],["leaf",53,0.01817016261877185],["leaf",54,0.009488035041505126],["leaf",55,0.03156039524459042],["leaf",56,0.013799694486444893],["leaf",59,0.010300809138594876],["leaf",60,0.022423267914760146]],[["split",0,13.0,12,0],["split",1,7.0,2,1],["split",2,7.0,16,1],["split",3,7.0,0,2],["split",4,4.0,19,2],["split",5,3.0,23,2],["split",6,2.0,8,2],["split",7,10.0,16,3],["split",8,23.0,27,3],["split",9,10.0,6,3],["split",10,10.0,16,3],["split",11,26.0,20,3],["split",12,20.0,0,3],["split",13,20.0,0,3],["split",14,24.0,1,3],["split",15,23.0,26,4],["leaf",16,-0.020597079928491562],["split",17,13.0,2,4],["split",18,10.0,4,4],["split",19,4.0,11,4],["split",20,25.0,14,4],["split",21,20.0,0,4],["split",22,2.0,19,4],["split",23,23.0,14,4],["split",24,24.0,15,4],["split",25,9.0,12,4],["leaf",26,-0.012444840319757846],["split",27,1.0,16,4],["split",28,25.0,11,4],["split",29,26.0,8,4],["split",30,3.0,25,4],["leaf",31,-0.05391759074823212],["leaf",32,-0.02792984827789881],["leaf",35,-0.007253069435764208],["leaf",36,-0.02076316787974075],["leaf",37,-0.018404892805133573],["leaf",38,-0.037390851359408624],["leaf",39,-0.003047820376225862],["leaf",40,-0.016610302932616],["leaf",41,-0.004384880880227093],["leaf",42,0.0056547019245884985],["leaf",43,-0.01790424499876321],["leaf",44,0.0016766569018478363],["leaf",45,0.002620973147109914],["leaf",46,-0.011580452158624747],["leaf",47,-0.005235467138970176],["leaf",48,-0.020665998813914403],["leaf",49,-0.006391562989659783],["leaf",50,0.005620198823111141],["leaf",51,-0.0012731155406868623],["leaf",52,0.011574146045312182],["leaf",55,0.013621113830905867],["leaf",56,0.025853488189445196],["leaf",57,-0.02356958685044129],["leaf",58,0.0073914895963384745],["leaf",59,-0.01821599180727504],["leaf",60,-0.000985041669096813],["leaf",61,0.006330750356772987],["leaf",62,0.01552460444721989]],[["split",0,8.0,14,0],["split",1,7.0,2,1],["split",2,12.0,25,1],["split",3,20.0,0,2],["split",4,10.0,4,2],["split",5,21.0,8,2],["split",6,13.0,26,2],["split",7,4.0,30,3],["leaf",8,-0.0007432506288253312],["split",9,17.0,0,3],["split",10,21.0,4,3],["split",11,20.0,0,3],["split",12,12.0,13,3],["split",13,19.0,0,3],["split",14,9.0,25,3],["split",15,10.0,11,4],["split",16,7.0,0,4],["split",19,25.0,16,4],["leaf",20,-0.04126311816498471],["split",21,5.0,27,4],["split",22,7.0,12,4],["split",23,4.0,17,4],["split",24,3.0,15,4],["split",25,4.0,20,4],["split",26,22.0,22,4],["split",27,10.0,5,4],["leaf",28,-0.0019336908028829895],["split",29,1.0,16,4],["split",30,21.0,21,4],["leaf",31,-0.02803986958207634],["leaf",32,-0.012031034985431361],["leaf",33,-0.047546323423037745],["leaf",34,-0.03320987850088565],["leaf",39,-0.018865367637396234],["leaf",40,-0.005343503765466173],["leaf",43,-0.016298818594521376],["leaf",44,-0.0017369407478663623],["leaf",45,-0.005431291483960665],["leaf",46,0.004539125364832627],["leaf",47,0.005731820069264244],["leaf",48,-0.00802791921112566],["leaf",49,-0.013433214615762093],["leaf",50,-0.026849222696648065],["leaf",51,0.005190805450044184],["leaf",52,-0.007548852837531421],["leaf",53,0.01412019044899241],["leaf",54,0.004593503018286471],["leaf",55,0.00019157443642554522],["leaf",56,0.016177474014091656],["leaf",59,0.011437522373398672],["leaf",60,0.024856336659506838],["leaf",61,0.02424180656179846],["leaf",62,0.045682545716604614]],[["split",0,7.0,9,0],["split",1,4.0,15,1],["split",2,12.0,21,1],["split",3,5.0,10,2],["split",4,7.0,0,2],["split",5,26.0,9,2],["split",6,4.0,4,2],["split",7,1.0,14,3],["split",8,26.0,16,3],["split",9,1.0,10,3],["split",10,10.0,5,3],["split",11,4.0,14,3],["split",12,7.0,14,3],["split",13,7.0,25,3],["split",14,6.0,23,3],["split",15,25.0,18,4],["split",16,6.0,9,4],["split",17,9.0,14,4],["split",18,11.0,17,4],["split",19,25.0,12,4],["leaf",20,-0.018959518785820462],["split",21,23.0,16,4],["split",22,15.0,0,4],["split",23,4.0,12,4],["split",24,0.0,14,4],["split",25,10.0,6,4],["split",26,12.0,10,4],["split",27,26.0,17,4],["split",28,9.0,16,4],["split",29,3.0,27,4],["split",30,25.0,23,4],["leaf",31,0.0038489755760188504],["leaf",32,-0.012475936707590205],["leaf",33,0.021116063068312763],["leaf",34,0.004056805430429521],["leaf",35,-0.006152886015010171],["leaf",36,-0.021152831794273835],["leaf",37,0.00664017907521188],["leaf",38,-0.009005692582373197],["leaf",39,-0.05285956772612253],["leaf",40,-0.026705235348777123],["leaf",43,-0.011621011299378075],["leaf",44,-0.027336760450397225],["leaf",45,-0.01045468605539144],["leaf",46,-0.02241863218810507],["leaf",47,5.608416798208857e-06],["leaf",48,0.014213781576415644],["leaf",49,-0.0034684303161880104],["leaf",50,-0.01475333673094856],["leaf",51,-0.009535513253248284],["leaf",52,0.0019598404892846757],["leaf",53,0.00013985323144999023],["leaf",54,0.010029016742393515],["leaf",55,0.00642199573625811],["leaf",56,0.01826492623179859],["leaf",57,0.021843447971754557],["leaf",58,0.0363777208063331],["leaf",59,0.003136314595966098],["leaf",60,0.01737286612308321],["leaf",61,0.009267528899896082],["leaf",62,0.02342110931449408]],[["split",0,7.0,9,0],["split",1,3.0,27,1],["split",2,8.0,26,1],["split",3,4.0,29,2],["split",4,21.0,22,2],["split",5,4.0,16,2],["split",6,4.0,3,2],["split",7,10.0,2,3],["split",8,0.0,25,3],["split",9,22.0,13,3],["leaf",10,0.01171172946562464],["split",11,20.0,0,3],["split",12,25.0,20,3],["split",13,9.0,12,3],["split",14,24.0,7,3],["split",15,12.0,2,4],["split",16,2.0,9,4],["split",17,7.0,0,4],["leaf",18,-0.0087991303582075],["leaf",19,0.008225233542812672],["leaf",20,-0.008667213329560114],["split",23,2.0,2,4],["split",24,25.0,21,4],["split",25,3.0,8,4],["split",26,7.0,21,4],["leaf",27,0.010536608783360575],["split",28,25.0,23,4],["split",29,5.0,11,4],["split",30,23.0,25,4],["leaf",31,-0.01517731018109026],["leaf",32,-0.031669805639585796],["leaf",33,0.00031498046577418243],["leaf",34,-0.01304982459575458],["leaf",35,-0.04334859405625276],["leaf",36,-0.024920577470010946],["leaf",47,0.023232375442255086],["leaf",48,0.00786724517447647],["leaf",49,-0.01279542560678462],["leaf",50,0.019207784928752178],["leaf",51,-0.008149098594860056],["leaf",52,0.00019993502190426086],["leaf",53,0.0008979100811671654],["leaf",54,0.01277896688298394],["leaf",57,0.0238395253731065],["leaf",58,0.03701910705880461],["leaf",59,0.00932430607361532],["leaf",60,-0.00932302252370429],["leaf",61,0.017636744826925923],["leaf",62,0.006113705660705531]],[["split",0,8.0,11,0],["split",1,7.0,5,1],["split",2,12.0,23,1],["split",3,21.0,15,2],["split",4,21.0,18,2],["split",5,21.0,6,2],["split",6,19.0,0,2],["split",7,22.0,7,3],["split",8,20.0,0,3],["split",9,10.0,5,3],["split",10,20.0,0,3],["split",11,25.0,5,3],["split",12,23.0,27,3],["split",13,5.0,10,3],["split",14,24.0,12,3],["split",15,12.0,5,4],["split",16,7.0,0,4],["split",17,10.0,16,4],["leaf",18,0.011682257268687258],["split",19,6.0,7,4],["split",20,19.0,0,4],["split",21,17.0,0,4],["leaf",22,0.027579028697719787],["split",23,2.0,16,4],["split",24,1.0,8,4],["split",25,17.0,0,4],["split",26,9.0,7,4],["split",27,7.0,25,4],["split",28,1.0,11,4],["leaf",29,-0.010333096871733843],["split",30,3.0,20,4],["leaf",31,-0.024581471744475444],["leaf",32,-0.04287208529534599],["leaf",33,-0.03840903706451906],["leaf",34,-0.01645180299619272],["leaf",35,-0.02227244244777768],["leaf",36,-0.006347266873798105],["leaf",39,-0.010417742967433615],["leaf",40,-0.026134550926115684],["leaf",41,-0.009365213875918373],["leaf",42,0.006589536103608704],["leaf",43,-0.0015648858539854674],["leaf",44,-0.01849804940113074],["leaf",47,-0.023092840538203447],["leaf",48,-0.007406797985544712],["leaf",49,-0.013451037531498686],["leaf",50,0.001507439498221863],["leaf",51,0.003730114478676721],["leaf",52,0.014521397164469612],["leaf",53,-0.026473953238177066],["leaf",54,-0.0024427271709620848],["leaf",55,0.012730739625106175],["leaf",56,0.025910721006951475],["leaf",57,-0.003448726163155296],["leaf",58,0.010271763993735085],["leaf",61,0.0005848966735594124],["leaf",62,0.010397901344744014]],[["split",0,7.0,9,0],["split",1,7.0,0,1],["split",2,6.0,22,1],["split",3,4.0,25,2],["split",4,10.0,6,2],["split",5,24.0,1,2],["split",6,7.0,29,2],["leaf",7,-0.01167836762298949],["split",8,11.0,15,3],["split",9,17.0,0,3],["split",10,21.0,6,3],["split",11,13.0,11,3],["split",12,26.0,1,3],["split",13,3.0,18,3],["split",14,2.0,11,3],["leaf",17,-0.023561761770900125],["leaf",18,-0.04816446057701279],["split",19,14.0,0,4],["leaf",20,-0.03506977672984956],["split",21,3.0,5,4],["split",22,11.0,6,4],["leaf",23,-0.0023961895862805805],["split",24,23.0,14,4],["split",25,7.0,20,4],["split",26,22.0,22,4],["split",27,4.0,14,4],["split",28,19.0,0,4],["split",29,2.0,4,4],["split",30,23.0,18,4],["leaf",39,-0.014095012260679501],["leaf",40,-0.030571604056736094],["leaf",43,-0.004897312911267892],["leaf",44,-0.019076661419272948],["leaf",45,-0.012227592072830529],["leaf",46,-0.0029529335694814416],["leaf",49,-0.0208918190379203],["leaf",50,-0.007470057978798114],["leaf",51,-0.01607794684015231],["leaf",52,-0.003216223806172627],["leaf",53,0.004988831072302317],["leaf",54,-0.0016286692680633063],["leaf",55,0.010175466580569968],["leaf",56,-0.002571258612959023],["leaf",57,0.01628303038872198],["leaf",58,-0.0009942694519969194],["leaf",59,0.037668099624840995],["leaf",60,0.02398825497464136],["leaf",61,0.018550750863382745],["leaf",62,-0.0021600500362940075]],[["split",0,12.0,13,0],["split",1,8.0,9,1],["split",2,7.0,26,1],["split",3,20.0,0,2],["split",4,5.0,17,2],["split",5,10.0,16,2],["split",6,16.0,0,2],["split",7,2.0,15,3],["split",8,2.0,12,3],["split",9,22.0,28,3],["split",10,21.0,5,3],["split",11,8.0,15,3],["split",12,3.0,25,3],["split",13,1.0,7,3],["split",14,7.0,29,3],["split",15,10.0,8,4],["split",16,14.0,0,4],["leaf",17,-0.03407635980397624],["split",18,25.0,16,4],["split",19,5.0,5,4],["leaf",20,-0.011736573976158444],["split",21,10.0,15,4],["split",22,11.0,10,4],["split",23,20.0,0,4],["split",24,25.0,28,4],["split",25,23.0,25,4],["split",26,20.0,0,4],["split",27,22.0,17,4],["split",28,5.0,1,4],["leaf",29,0.019521540995818407],["leaf",30,0.041650516803525654],["leaf",31,-0.014514314731905852],["leaf",32,-0.0012505872564542435],["leaf",33,-0.014448557566635364],["leaf",34,-0.0295273598823495],["leaf",37,-0.0005429924924199247],["leaf",38,0.02517490858925429],["leaf",39,0.011166965815414324],["leaf",40,0.0006733557511277344],["leaf",43,-0.007923453112928832],["leaf",44,-0.023212353669954545],["leaf",45,-0.009557312668765404],["leaf",46,-0.0003913617780554692],["leaf",47,-0.006245005721425928],["leaf",48,0.01226797909166095],["leaf",49,0.0018968767349409493],["leaf",50,0.015159124188034277],["leaf",51,0.005806772830068968],["leaf",52,-0.01175578680491828],["leaf",53,0.017959951243090157],["leaf",54,-0.0037735327045427098],["leaf",55,0.007592691312581139],["leaf",56,-0.009070259417349125],["leaf",57,0.033770772899456666],["leaf",58,0.01377762540353838]],[["split",0,7.0,8,0],["split",1,4.0,24,1],["split",2,8.0,19,1],["split",3,21.0,4,2],["split",4,17.0,0,2],["split",5,26.0,18,2],["split",6,9.0,27,2],["split",7,26.0,6,3],["split",8,26.0,12,3],["split",9,0.0,21,3],["split",10,23.0,18,3],["split",11,25.0,5,3],["split",12,12.0,18,3],["split",13,22.0,24,3],["split",14,23.0,19,3],["leaf",15,-0.02822970666232506],["leaf",16,-0.01172807671754683],["split",17,10.0,25,4],["split",18,2.0,5,4],["split",19,19.0,0,4],["split",20,13.0,2,4],["leaf",21,-0.014035378038317347],["leaf",22,-0.03990512592123333],["split",23,24.0,2,4],["split",24,11.0,2,4],["split",25,22.0,25,4],["split",26,25.0,24,4],["split",27,21.0,17,4],["split",28,10.0,12,4],["split",29,25.0,15,4],["leaf",30,0.027283193555388704],["leaf",35,-0.01616593767669685],["leaf",36,0.0063983968023968016],["leaf",37,0.013103034557685397],["leaf",38,-0.002276823295478586],["leaf",39,-0.021487609628175516],["leaf",40,-0.0032072280496911396],["leaf",41,0.00041408451622018574],["leaf",42,-0.012537155093901214],["leaf",47,-0.026386698031808303],["leaf",48,-0.007611972833476001],["leaf",49,0.014284300428849596],["leaf",50,-0.002114201407416817],["leaf",51,0.0024194349829947405],["leaf",52,-0.012351373095877022],["leaf",53,0.007844507844105538],["leaf",54,0.022340252505524908],["leaf",55,0.005382762334475265],["leaf",56,0.013341978436340695],["leaf",57,-0.008075848690079684],["leaf",58,0.0032719887749454875],["leaf",59,0.006782428222303226],["leaf",60,0.019607767523946197]],[["split",0,7.0,10,0],["split",1,7.0,2,1],["split",2,2.0,4,1],["split",3,2.0,28,2],["split",4,10.0,15,2],["split",5,20.0,0,2],["split",6,3.0,20,2],["split",7,7.0,0,3],["split",8,23.0,25,3],["split",9,20.0,0,3],["split",10,11.0,21,3],["split",11,7.0,30,3],["split",12,21.0,16,3],["split",13,1.0,0,3],["split",14,25.0,29,3],["split",15,0.0,11,4],["split",16,21.0,6,4],["leaf",17,-0.03811726440095558],["split",18,9.0,11,4],["split",19,3.0,24,4],["split",20,25.0,15,4],["split",21,2.0,27,4],["split",22,26.0,18,4],["split",23,19.0,0,4],["leaf",24,0.03351326815793781],["leaf",25,-0.01996997719753888],["leaf",26,0.003175823701117736],["leaf",27,-0.023110679821830456],["split",28,7.0,24,4],["split",29,21.0,2,4],["leaf",30,0.025826512310005125],["leaf",31,-0.006514324108073008],["leaf",32,-0.03285843394598222],["leaf",33,-0.01828347832695107],["leaf",34,-0.005197440226970303],["leaf",37,-0.00902866746039597],["leaf",38,-0.031705394264722304],["leaf",39,-0.016886968874672954],["leaf",40,0.0011954282996460587],["leaf",41,-0.008944172992590716],["leaf",42,0.014331139761118552],["leaf",43,-0.001692894035587525],["leaf",44,-0.017245117703887588],["leaf",45,-0.00027785923815337396],["leaf",46,0.013290436276427682],["leaf",47,0.016439480480756826],["leaf",48,0.004161233793275282],["leaf",57,-0.00238757972514222],["leaf",58,0.005801819355435295],["leaf",59,-0.008911978130396441],["leaf",60,0.008040717624243073]],[["split",0,7.0,10,0],["split",1,7.0,1,1],["split",2,6.0,26,1],["split",3,13.0,2,2],["split",4,21.0,15,2],["split",5,10.0,5,2],["split",6,1.0,4,2],["split",7,1.0,0,3],["split",8,22.0,17,3],["split",9,24.0,11,3],["split",10,26.0,15,3],["split",11,11.0,17,3],["split",12,26.0,26,3],["leaf",13,-0.00749536272123194],["split",14,0.0,2,3],["leaf",15,-0.002797216826370365],["split",16,24.0,15,4],["leaf",17,-0.03938820326165786],["leaf",18,-0.017756072862478038],["split",19,3.0,19,4],["split",20,10.0,17,4],["split",21,12.0,7,4],["split",22,23.0,27,4],["split",23,20.0,0,4],["split",24,13.0,12,4],["split",25,25.0,0,4],["split",26,19.0,0,4],["leaf",29,-0.0031275653279575306],["split",30,4.0,20,4],["leaf",33,-0.02776541177931859],["leaf",34,-0.005013683619208534],["leaf",39,-0.021325540108927635],["leaf",40,-0.008126314686291587],["leaf",41,-0.01110099844274071],["leaf",42,0.0015169896105503393],["leaf",43,-0.003256322376666633],["leaf",44,-0.013951112253906186],["leaf",45,0.00534978916932008],["leaf",46,-0.006393792224516879],["leaf",47,-0.007956552665516743],["leaf",48,-0.026883310733588885],["leaf",49,0.010863942880039409],["leaf",50,-0.0020986274499315313],["leaf",51,-0.01588410402754085],["leaf",52,0.002453302370834387],["leaf",53,0.014682949372003951],["leaf",54,-0.0013014530912779049],["leaf",61,0.0138911560189794],["leaf",62,-0.0027811119286900205]],[["split",0,7.0,13,0],["split",1,7.0,2,1],["split",2,5.0,0,1],["split",3,24.0,17,2],["split",4,26.0,16,2],["split",5,9.0,23,2],["split",6,2.0,13,2],["split",7,4.0,21,3],["split",8,21.0,5,3],["split",9,10.0,16,3],["split",10,24.0,5,3],["leaf",11,0.011915653249295355],["leaf",12,0.026000725255127696],["split",13,7.0,29,3],["split",14,26.0,22,3],["leaf",15,-0.0005603264112278531],["split",16,1.0,2,4],["leaf",17,0.009013210464908468],["split",18,21.0,18,4],["split",19,20.0,0,4],["split",20,23.0,10,4],["split",21,23.0,13,4],["split",22,21.0,17,4],["split",27,20.0,0,4],["split",28,1.0,20,4],["split",29,26.0,5,4],["split",30,10.0,3,4],["leaf",33,-0.012754700541569835],["leaf",34,-0.03036085189638969],["leaf",37,-0.022613245761109367],["leaf",38,-0.004172200271959042],["leaf",39,-0.013463231564403546],["leaf",40,0.0030708627776381754],["leaf",41,0.0032261155975070828],["leaf",42,-0.008795802637266157],["leaf",43,0.00014694102720866313],["leaf",44,-0.016765202642677404],["leaf",45,-0.0013773550767481753],["leaf",46,0.006349625358174938],["leaf",55,0.007368394699671252],["leaf",56,-0.006242204878938373],["leaf",57,0.003991243767474745],["leaf",58,0.02354445024817631],["leaf",59,-0.01004487206944762],["leaf",60,-0.00035390784289513383],["leaf",61,-0.00526816923434664],["leaf",62,0.011426402777813654]],[["split",0,6.0,8,0],["split",1,4.0,19,1],["split",2,12.0,18,1],["split",3,7.0,14,2],["split",4,21.0,20,2],["split",5,23.0,29,2],["split",6,10.0,16,2],["split",7,20.0,0,3],["leaf",8,0.012748932821535747],["split",9,7.0,2,3],["split",10,20.0,0,3],["split",11,21.0,7,3],["leaf",12,-0.015932318324864986],["split",13,7.0,21,3],["split",14,1.0,5,3],["split",15,26.0,4,4],["leaf",16,0.010173127189088478],["split",19,9.0,13,4],["split",20,9.0,23,4],["split",21,19.0,0,4],["leaf",22,0.010948217085425774],["split",23,20.0,0,4],["split",24,4.0,2,4],["split",27,5.0,3,4],["split",28,2.0,2,4],["split",29,0.0,18,4],["split",30,9.0,22,4],["leaf",31,-0.01735174857402401],["leaf",32,-0.002334832572075085],["leaf",39,-0.030671680553532452],["leaf",40,-0.011773133543363665],["leaf",41,-0.007255182239926396],["leaf",42,-0.019520170165132297],["leaf",43,-0.010645448180514865],["leaf",44,0.007872078608664273],["leaf",47,-0.0011010718799547563],["leaf",48,-0.022240926273392383],["leaf",49,0.014692619954630528],["leaf",50,0.0019522994142698636],["leaf",55,0.009261243470518113],["leaf",56,-0.004951111132555707],["leaf",57,0.01729340794468836],["leaf",58,0.004515681644313259],["leaf",59,-0.011089899138689097],["leaf",60,0.0036700594678292188],["leaf",61,0.009572127618835637],["leaf",62,0.017406590827466026]],[["split",0,7.0,6,0],["split",1,3.0,28,1],["split",2,7.0,17,1],["split",3,10.0,5,2],["leaf",4,0.01090902754442057],["split",5,9.0,25,2],["split",6,7.0,30,2],["split",7,17.0,0,3],["split",8,19.0,0,3],["split",11,3.0,9,3],["split",12,5.0,23,3],["split",13,3.0,23,3],["split",14,4.0,10,3],["split",15,7.0,1,4],["leaf",16,-0.041932475705551135],["split",17,2.0,30,4],["split",18,10.0,12,4],["split",23,11.0,26,4],["split",24,1.0,3,4],["split",25,17.0,0,4],["leaf",26,-0.011696005389583384],["split",27,10.0,8,4],["split",28,15.0,0,4],["leaf",29,0.032982559222264105],["split",30,10.0,18,4],["leaf",31,-0.02767207240730066],["leaf",32,-0.011956349336408706],["leaf",35,-0.008785704318936704],["leaf",36,-0.029660465783661782],["leaf",37,-0.011601744167065527],["leaf",38,0.00909846486347877],["leaf",47,-0.011590872688813468],["leaf",48,0.007701131078229164],["leaf",49,-0.012135169993818156],["leaf",50,-0.00018130157657904672],["leaf",51,0.005681694156528692],["leaf",52,0.020362399217773046],["leaf",55,-0.002759040748389285],["leaf",56,0.005102152526075281],["leaf",57,0.008673956442746572],["leaf",58,0.02362416435294269],["leaf",61,0.016576694473887314],["leaf",62,-0.003999275635047352]],[["split",0,6.0,16,0],["split",1,4.0,28,1],["split",2,19.0,0,1],["split",3,25.0,2,2],["split",4,25.0,9,2],["split",5,5.0,11,2],["split",6,21.0,23,2],["split",7,26.0,16,3],["split",8,21.0,1,3],["split",9,23.0,26,3],["split",10,3.0,6,3],["split",11,7.0,26,3],["split",12,21.0,15,3],["split",13,4.0,8,3],["split",14,23.0,13,3],["split",15,12.0,14,4],["leaf",16,0.005113407996933563],["split",17,25.0,23,4],["split",18,7.0,5,4],["split",19,10.0,12,4],["leaf",20,-0.017080591039218523],["split",21,11.0,9,4],["split",22,0.0,23,4],["split",23,12.0,13,4],["split",24,16.0,0,4],["split",25,23.0,25,4],["split",26,7.0,28,4],["split",27,12.0,25,4],["split",28,2.0,12,4],["leaf",29,-0.002309292052869554],["leaf",30,-0.018455207885455807],["leaf",31,-0.012220285173398373],["leaf",32,-0.026947747522777645],["leaf",35,-0.008081708235645878],["leaf",36,-0.025909194007123323],["leaf",37,-0.00533392785855357],["leaf",38,0.000588555798276018],["leaf",39,-0.006525590180714647],["leaf",40,0.012148552062909289],["leaf",43,0.0020894740580854726],["leaf",44,-0.01600237638644215],["leaf",45,-0.02979228550536429],["leaf",46,-0.013488587581760216],["leaf",47,-0.0062099505354346945],["leaf",48,0.008627753737274035],["leaf",49,0.013334876902562144],["leaf",50,0.032861664949534054],["leaf",51,-0.0002169614912148065],["leaf",52,-0.014636744062751625],["leaf",53,0.0027664976711026614],["leaf",54,0.01885268483037391],["leaf",55,-0.012416893905646795],["leaf",56,0.0017429259199109354],["leaf",57,0.013189286232859431],["leaf",58,-0.0034996196206651987]],[["split",0,8.0,17,0],["split",1,4.0,22,1],["split",2,2.0,15,1],["split",3,25.0,13,2],["split",4,20.0,0,2],["split",5,23.0,28,2],["split",6,26.0,16,2],["split",7,10.0,16,3],["split",8,20.0,0,3],["split",9,10.0,4,3],["split",10,25.0,9,3],["split",11,20.0,0,3],["split",12,22.0,16,3],["split",13,20.0,0,3],["split",14,9.0,13,3],["split",15,26.0,18,4],["split",16,26.0,4,4],["split",17,23.0,24,4],["split",18,26.0,15,4],["split",19,6.0,6,4],["split",20,2.0,28,4],["leaf",21,-0.018249434310544144],["split",22,25.0,23,4],["split",23,9.0,23,4],["split",24,25.0,7,4],["leaf",25,-0.00205612236208382],["leaf",26,-0.010904587998787938],["split",27,23.0,14,4],["leaf",28,0.011253374639564367],["split",29,24.0,22,4],["split",30,5.0,16,4],["leaf",31,-0.011946635790259284],["leaf",32,-0.0009814167809094514],["leaf",33,-0.012755048283480886],["leaf",34,0.0024898710492600763],["leaf",35,0.0014062702794657817],["leaf",36,-0.008836208322357323],["leaf",37,0.021948945675555446],["leaf",38,0.0011297782868368723],["leaf",39,-0.028356166897785254],["leaf",40,-0.006870184938267503],["leaf",41,-0.005852472159689452],["leaf",42,-0.01890650247014547],["leaf",45,0.004578216675190627],["leaf",46,0.028054453320634117],["leaf",47,0.007331611709752556],["leaf",48,0.01688614811257037],["leaf",49,-0.020602523444027698],["leaf",50,0.004132658448582145],["leaf",55,-9.479499245219848e-05],["leaf",56,-0.01660926863548689],["leaf",59,-0.0033844444168091925],["leaf",60,0.008570081439123115],["leaf",61,0.011002414274582505],["leaf",62,-0.0007845291296257344]],[["split",0,6.0,19,0],["split",1,7.0,4,1],["split",2,1.0,2,1],["split",3,11.0,23,2],["split",4,24.0,0,2],["leaf",5,-0.010391171920295922],["split",6,7.0,29,2],["split",7,19.0,0,3],["split",8,3.0,4,3],["split",9,10.0,15,3],["split",10,23.0,27,3],["split",13,20.0,0,3],["split",14,25.0,26,3],["split",15,9.0,1,4],["split",16,3.0,9,4],["leaf",17,0.012196550654584313],["split",18,22.0,23,4],["leaf",19,-0.008042837595554148],["leaf",20,-0.025299711684696777],["split",21,26.0,6,4],["split",22,2.0,18,4],["split",27,23.0,27,4],["split",28,25.0,21,4],["split",29,22.0,23,4],["leaf",30,0.02554657171124836],["leaf",31,0.002863550044146961],["leaf",32,-0.015430032795377179],["leaf",33,0.012649684429581278],["leaf",34,-0.011066276480238866],["leaf",37,-0.01404070504884309],["leaf",38,0.012166973455253163],["leaf",43,-0.005983548650009969],["leaf",44,0.0016215275483642527],["leaf",45,-0.022602459755546644],["leaf",46,-0.0022085326976638503],["leaf",55,0.007806208757593761],["leaf",56,-0.010986612680689229],["leaf",57,-0.013229957976927664],["leaf",58,0.016489263124451866],["leaf",59,0.017836034613276725],["leaf",60,-0.005530004496341259]],[["split",0,7.0,15,0],["split",1,7.0,3,1],["split",2,1.0,2,1],["split",3,20.0,0,2],["split",4,3.0,17,2],["split",5,12.0,20,2],["split",6,2.0,3,2],["split",7,10.0,19,3],["split",8,21.0,14,3],["split",9,25.0,28,3],["split",10,10.0,6,3],["split",11,4.0,19,3],["leaf",12,-0.021534563064747745],["split",13,20.0,0,3],["split",14,12.0,28,3],["split",15,3.0,24,4],["split",16,3.0,11,4],["leaf",17,-0.009592045510108602],["leaf",18,0.021051241635432434],["split",19,13.0,4,4],["split",20,21.0,13,4],["split",21,2.0,19,4],["split",22,1.0,9,4],["leaf",23,0.004332180160547136],["leaf",24,-0.011064250446591857],["split",27,9.0,4,4],["leaf",28,-0.0051130424188614194],["split",29,1.0,26,4],["split",30,0.0,12,4],["leaf",31,-0.02216553966830844],["leaf",32,0.00410870188877202],["leaf",33,0.0064531704058351244],["leaf",34,-0.020103821936749238],["leaf",39,-0.0028761744649739274],["leaf",40,-0.01019968145293154],["leaf",41,-0.0062569999464871465],["leaf",42,0.012347567657562881],["leaf",43,-0.003469378510530082],["leaf",44,-0.01703215020458891],["leaf",45,-0.0029253009387298114],["leaf",46,0.005038221370262179],["leaf",55,0.000643642934369944],["leaf",56,0.015730689200726417],["leaf",59,0.0033443559789763278],["leaf",60,-0.0035607623103142824],["leaf",61,-0.0008693805345112072],["leaf",62,0.013526378178133772]],[["split",0,12.0,16,0],["split",1,4.0,8,1],["split",2,23.0,25,1],["split",3,17.0,0,2],["split",4,19.0,0,2],["split",5,13.0,30,2],["split",6,17.0,0,2],["split",7,20.0,0,3],["leaf",8,0.01841924561058805],["split",9,7.0,5,3],["split",10,2.0,7,3],["split",11,7.0,6,3],["split",12,3.0,22,3],["split",13,9.0,9,3],["leaf",14,-0.027291663373646316],["split",15,24.0,20,4],["leaf",16,-0.010364037959435638],["split",19,8.0,0,4],["split",20,26.0,2,4],["leaf",21,-0.011900324814912799],["split",22,10.0,25,4],["leaf",23,-0.011446648462098086],["split",24,3.0,29,4],["leaf",25,0.009552221256732399],["split",26,10.0,21,4],["split",27,23.0,27,4],["split",28,8.0,12,4],["leaf",31,-0.00012039081748312642],["leaf",32,0.010993607103650093],["leaf",39,0.0013753007605400405],["leaf",40,-0.013662589621768301],["leaf",41,-0.012595589078739424],["leaf",42,-0.0018067915810516192],["leaf",45,0.0029079278085489078],["leaf",46,0.021334790841456067],["leaf",49,0.0046790503024355355],["leaf",50,0.015266585735865892],["leaf",53,0.01716208557796602],["leaf",54,0.032704223237556784],["leaf",55,0.0006611730791966128],["leaf",56,-0.018981607775917845],["leaf",57,0.014396947843767341],["leaf",58,0.0009666948073024743]],[["split",0,7.0,13,0],["split",1,4.0,28,1],["split",2,2.0,2,1],["split",3,10.0,12,2],["split",4,25.0,28,2],["split",5,7.0,28,2],["split",6,21.0,13,2],["split",7,20.0,0,3],["split",8,2.0,9,3],["split",9,22.0,22,3],["leaf",10,0.003489336158163457],["split",11,20.0,0,3],["split",12,26.0,13,3],["split",13,20.0,0,3],["split",14,20.0,0,3],["split",15,4.0,3,4],["split",16,21.0,10,4],["split",17,20.0,0,4],["split",18,26.0,2,4],["split",19,7.0,2,4],["split",20,24.0,12,4],["split",23,19.0,0,4],["leaf",24,-0.014586539351560348],["leaf",25,0.02124182271008222],["leaf",26,0.03193480244045735],["split",27,3.0,20,4],["split",28,23.0,22,4],["split",29,1.0,15,4],["split",30,2.0,22,4],["leaf",31,0.013009432222583264],["leaf",32,-0.011660489824015002],["leaf",33,-0.010848686430230426],["leaf",34,0.014658965606718456],["leaf",35,0.011274521105569731],["leaf",36,-0.023708124127574184],["leaf",37,-0.014851137404622725],["leaf",38,-0.001840295238001044],["leaf",39,-0.02613544838508636],["leaf",40,-0.012235194321387019],["leaf",41,-0.014327350616159591],["leaf",42,0.0004626635146493435],["leaf",47,0.014347850901863001],["leaf",48,-0.005957985834551305],["leaf",55,-0.001814944664309083],["leaf",56,0.008019501255589931],["leaf",57,-0.017387702442853592],["leaf",58,0.007167226998331358],["leaf",59,2.842500343832714e-05],["leaf",60,0.007156864163140687],["leaf",61,0.007130675473753079],["leaf",62,0.03745566306680068]],[["split",0,7.0,16,0],["split",1,23.0,29,1],["split",2,4.0,4,1],["split",3,19.0,0,2],["split",4,4.0,26,2],["split",5,7.0,25,2],["split",6,7.0,30,2],["split",7,26.0,21,3],["split",8,21.0,9,3],["split",9,21.0,22,3],["split",10,3.0,6,3],["split",11,17.0,0,3],["split",12,25.0,20,3],["split",13,22.0,25,3],["split",14,8.0,27,3],["split",15,4.0,30,4],["split",16,25.0,29,4],["split",17,11.0,10,4],["split",18,9.0,23,4],["split",19,8.0,4,4],["leaf",20,-0.0195258104867998],["leaf",21,-0.03183155556132006],["leaf",22,-0.011810538387720853],["split",23,25.0,7,4],["leaf",24,0.020453776408150864],["split",25,25.0,7,4],["leaf",26,0.009277583411216374],["split",27,25.0,9,4],["split",28,23.0,18,4],["split",29,21.0,15,4],["leaf",30,-0.0005899640075660743],["leaf",31,-0.00617808253905927],["leaf",32,-0.019402445217849864],["leaf",33,0.002089250138461425],["leaf",34,0.020931208699989753],["leaf",35,0.023372177310351835],["leaf",36,0.004127743973364798],["leaf",37,0.0031794585723982755],["leaf",38,-0.008398025064753538],["leaf",39,0.00961161515190503],["leaf",40,-0.009047261099681332],["leaf",47,-0.0035155844820039563],["leaf",48,0.007857947023408262],["leaf",51,0.012106680344002597],["leaf",52,0.02420470757930101],["leaf",55,-0.0016368677084126354],["leaf",56,0.004733677828837216],["leaf",57,-0.00035491739510734013],["leaf",58,-0.010814181024763612],["leaf",59,0.011241520439904328],["leaf",60,0.028576704846681032]],[["split",0,1.0,3,0],["split",1,13.0,1,1],["split",2,4.0,20,1],["split",3,9.0,11,2],["split",4,25.0,4,2],["split",5,22.0,21,2],["split",6,16.0,0,2],["leaf",7,0.011113460990053272],["leaf",8,-0.00959091357384508],["split",9,10.0,10,3],["split",10,3.0,2,3],["split",11,26.0,25,3],["split",12,20.0,0,3],["split",13,10.0,2,3],["split",14,9.0,24,3],["leaf",19,-0.01702126730143086],["leaf",20,-0.029965951442477975],["leaf",21,0.007859979874926015],["split",22,17.0,0,4],["split",23,7.0,27,4],["split",24,24.0,22,4],["split",25,13.0,9,4],["split",26,21.0,21,4],["split",27,1.0,19,4],["split",28,25.0,26,4],["split",29,5.0,26,4],["leaf",30,0.0010518499748572014],["leaf",45,-0.010762061957876056],["leaf",46,-0.025625118725422627],["leaf",47,0.002662832511468204],["leaf",48,0.012971933353849836],["leaf",49,0.008455360803714628],["leaf",50,0.0176103089447811],["leaf",51,-0.011454291665505427],["leaf",52,0.0033265711709894643],["leaf",53,-0.020822819224972314],["leaf",54,0.008634787194717383],["leaf",55,-0.005558401697669763],["leaf",56,-0.021963908384526398],["leaf",57,-0.001809331658132028],["leaf",58,0.008174645535414454],["leaf",59,-0.009708680155001332],["leaf",60,-0.02157366343545532]],[["split",0,12.0,13,0],["split",1,3.0,23,1],["split",2,25.0,17,1],["split",3,1.0,0,2],["split",4,9.0,29,2],["split",5,20.0,0,2],["split",6,4.0,21,2],["split",7,13.0,2,3],["split",8,21.0,4,3],["split",9,17.0,0,3],["leaf",10,0.01543723633849505],["split",11,8.0,7,3],["split",12,7.0,27,3],["split",13,7.0,24,3],["split",14,10.0,6,3],["leaf",15,-0.011153980826458344],["leaf",16,-0.030150282039679215],["split",17,3.0,1,4],["split",18,3.0,14,4],["split",19,23.0,18,4],["leaf",20,0.014290461499541893],["split",23,3.0,10,4],["split",24,23.0,26,4],["split",25,11.0,5,4],["leaf",26,0.007986262388380022],["split",27,3.0,6,4],["split",28,1.0,25,4],["leaf",29,-0.022786575578857556],["split",30,9.0,18,4],["leaf",35,-0.023313853419868226],["leaf",36,-0.008829035843409385],["leaf",37,-0.0017167692933147448],["leaf",38,-0.007560752219780529],["leaf",39,-0.005216419544648961],["leaf",40,0.006717046870222628],["leaf",47,-0.0012995139303267804],["leaf",48,-0.023066215492898463],["leaf",49,0.0033939134321302076],["leaf",50,-0.008725365679633305],["leaf",51,-0.03166364282929704],["leaf",52,-0.008370135249462844],["leaf",55,-0.013929747688790618],["leaf",56,0.005916049013495316],["leaf",57,0.014490216686157046],["leaf",58,0.0011679484879647492],["leaf",61,0.008616948177601681],["leaf",62,-0.008888308728979895]],[["split",0,2.0,8,0],["split",1,7.0,26,1],["split",2,20.0,0,1],["split",3,20.0,0,2],["split",4,20.0,0,2],["split",5,23.0,23,2],["split",6,2.0,18,2],["split",7,17.0,0,3],["split",8,25.0,14,3],["split",9,21.0,24,3],["leaf",10,-0.005366760520500077],["split",11,26.0,6,3],["split",12,17.0,0,3],["split",13,25.0,18,3],["split",14,21.0,15,3],["split",15,22.0,19,4],["split",16,7.0,15,4],["leaf",17,-0.025713462631124104],["split",18,26.0,18,4],["split",19,25.0,11,4],["split",20,25.0,15,4],["split",23,10.0,11,4],["split",24,10.0,12,4],["split",25,4.0,10,4],["split",26,23.0,27,4],["split",27,11.0,14,4],["split",28,24.0,22,4],["split",29,2.0,23,4],["split",30,23.0,23,4],["leaf",31,0.006119325346415926],["leaf",32,-0.0011258978696995657],["leaf",33,0.023403679345292718],["leaf",34,0.007901261671440164],["leaf",37,-0.014681598162353992],["leaf",38,0.003432539898700694],["leaf",39,0.0050045321249043186],["leaf",40,0.017098126215434547],["leaf",41,0.02712615274550867],["leaf",42,0.018161424298026262],["leaf",47,-0.013755053980739706],["leaf",48,-0.004712534526350823],["leaf",49,-0.003077016498524615],["leaf",50,0.0037750414607387833],["leaf",51,0.002105013489658408],["leaf",52,-0.009427073882411305],["leaf",53,-0.011545733888267608],["leaf",54,-0.03376489089966981],["leaf",55,-0.02184588165738383],["leaf",56,-0.003062558781541596],["leaf",57,-0.0015711952520312596],["leaf",58,0.019049653593627196],["leaf",59,-0.007061498376203532],["leaf",60,0.012313777846356797],["leaf",61,0.019618436931184032],["leaf",62,0.04025830959790759]],[["split",0,4.0,24,0],["split",1,1.0,7,1],["split",2,10.0,0,1],["split",3,3.0,2,2],["split",4,3.0,16,2],["leaf",5,-0.025650281088492433],["split",6,25.0,30,2],["leaf",7,0.013817719224437214],["split",8,22.0,19,3],["split",9,7.0,28,3],["split",10,19.0,0,3],["split",13,2.0,23,3],["leaf",14,0.01834910509442786],["split",17,24.0,5,4],["split",18,24.0,26,4],["split",19,5.0,21,4],["split",20,2.0,12,4],["split",21,0.0,5,4],["split",22,24.0,19,4],["split",27,16.0,0,4],["split",28,15.0,0,4],["leaf",35,-0.010762061031696545],["leaf",36,-0.00031293112528714683],["leaf",37,-0.012419635890683957],["leaf",38,0.0009476819887718253],["leaf",39,-0.0025278050716714875],["leaf",40,0.0037848375038861902],["leaf",41,0.010041574082981704],["leaf",42,-0.0020582631856997063],["leaf",43,-0.0015981673555544803],["leaf",44,0.00798908648541801],["leaf",45,-0.00770223304146565],["leaf",46,0.0009572594264793271],["leaf",55,0.00048152121742060024],["leaf",56,-0.017446599751881075],["leaf",57,-0.006070076948320604],["leaf",58,-0.0218180948979926]],[["split",0,7.0,20,0],["split",1,26.0,22,1],["split",2,6.0,12,1],["split",3,23.0,10,2],["split",4,1.0,3,2],["split",5,1.0,7,2],["split",6,2.0,14,2],["split",7,2.0,4,3],["split",8,20.0,0,3],["split",9,8.0,5,3],["split",10,10.0,15,3],["leaf",11,0.026115049252709074],["leaf",12,0.01198841596724902],["split",13,7.0,22,3],["split",14,20.0,0,3],["split",15,21.0,17,4],["split",16,26.0,1,4],["split",17,19.0,0,4],["split",18,21.0,9,4],["leaf",19,-4.7335711175794165e-05],["leaf",20,-0.014556228028756371],["split",21,20.0,0,4],["split",22,20.0,0,4],["split",27,1.0,19,4],["split",28,20.0,0,4],["split",29,3.0,8,4],["split",30,11.0,14,4],["leaf",31,0.0018286686043156224],["leaf",32,0.01453423525928045],["leaf",33,-0.01758081082772883],["leaf",34,-0.0008875498332032119],["leaf",35,-0.010336569857405435],["leaf",36,-0.0013378862824731273],["leaf",37,-0.010569385614601596],["leaf",38,0.008073334403475388],["leaf",43,-0.002414929073461047],["leaf",44,0.010874875094100659],["leaf",45,0.007841025334546165],["leaf",46,-0.006853236175856084],["leaf",55,-0.007845241857762246],["leaf",56,0.0048236998898891],["leaf",57,0.008056966117667087],["leaf",58,-0.0007565063560157802],["leaf",59,-0.013514087592498579],["leaf",60,0.0015393113742601693],["leaf",61,0.0030260045514100007],["leaf",62,0.02482503354482288]],[["split",0,7.0,16,0],["split",1,10.0,6,1],["split",2,8.0,30,1],["split",3,4.0,21,2],["split",4,25.0,21,2],["split",5,21.0,13,2],["split",6,24.0,9,2],["split",7,13.0,2,3],["split",8,20.0,0,3],["split",9,3.0,0,3],["split",10,10.0,8,3],["split",11,13.0,12,3],["split",12,23.0,12,3],["leaf",13,0.002828375414690309],["split",14,10.0,16,3],["leaf",15,0.014384423167834795],["split",16,26.0,21,4],["split",17,1.0,10,4],["leaf",18,0.00393062877002261],["leaf",19,0.01268857528264228],["split",20,1.0,5,4],["leaf",21,-0.009696479547847857],["split",22,1.0,5,4],["split",23,13.0,9,4],["split",24,2.0,19,4],["split",25,15.0,0,4],["split",26,26.0,19,4],["leaf",29,0.008931522114244231],["leaf",30,0.021923037018073844],["leaf",33,-0.008635192251851553],["leaf",34,0.003237095457674983],["leaf",35,-0.012152380586395487],["leaf",36,-0.026752631118025182],["leaf",41,-0.010729571392148364],["leaf",42,-0.002844017241788664],["leaf",45,-0.004807146010381436],["leaf",46,0.005800821265422263],["leaf",47,0.006616993592731191],["leaf",48,0.018099823811511203],["leaf",49,-0.0011609187717053046],["leaf",50,-0.00932790566876579],["leaf",51,0.005643322702851679],["leaf",52,0.016300296456601516],["leaf",53,-0.0021897648274367323],["leaf",54,0.005804386201399608]],[["split",0,7.0,8,0],["split",1,19.0,0,1],["split",2,25.0,3,1],["split",3,12.0,9,2],["split",4,3.0,5,2],["split",5,21.0,3,2],["split",6,12.0,30,2],["split",7,26.0,1,3],["split",8,10.0,15,3],["leaf",9,0.020638869227294392],["split",10,5.0,26,3],["split",11,13.0,19,3],["split",12,13.0,15,3],["split",13,7.0,18,3],["split",14,9.0,15,3],["split",15,2.0,20,4],["split",16,22.0,9,4],["split",17,0.0,14,4],["split",18,21.0,24,4],["split",21,26.0,14,4],["leaf",22,-0.010767835648925507],["leaf",23,-0.006654930110013283],["leaf",24,-0.024682772258727753],["split",25,10.0,14,4],["split",26,2.0,7,4],["split",27,26.0,8,4],["split",28,6.0,10,4],["leaf",29,0.002322875384325664],["split",30,26.0,21,4],["leaf",31,-0.0070499655402733695],["leaf",32,-0.026986802633828025],["leaf",33,-0.009576848997514199],["leaf",34,0.0003718891333320603],["leaf",35,-0.029451490240918983],["leaf",36,-0.014255052931496788],["leaf",37,-0.00130503960451901],["leaf",38,-0.017180225446467173],["leaf",43,0.013400741549119976],["leaf",44,-0.001636216095780249],["leaf",51,-0.002232110045794073],["leaf",52,0.011704091798483803],["leaf",53,0.004140399058193379],["leaf",54,-0.00731077180214126],["leaf",55,-0.005562107276592896],["leaf",56,0.0013829357551457962],["leaf",57,0.027311104624449283],["leaf",58,0.004046888511131589],["leaf",61,0.012097317959811294],["leaf",62,0.028322731990210015]],[["split",0,12.0,13,0],["split",1,19.0,0,1],["split",2,9.0,10,1],["split",3,7.0,11,2],["split",4,3.0,6,2],["split",5,23.0,17,2],["split",6,7.0,28,2],["split",7,21.0,6,3],["split",8,12.0,5,3],["split",9,0.0,15,3],["split",10,9.0,7,3],["split",11,26.0,14,3],["split",12,24.0,25,3],["split",13,4.0,28,3],["split",14,0.0,13,3],["split",15,24.0,26,4],["split",16,15.0,0,4],["split",17,2.0,18,4],["split",18,4.0,23,4],["leaf",19,0.006466324318596195],["leaf",20,0.019875906335546],["leaf",21,-0.00859369379860402],["split",22,2.0,11,4],["split",23,25.0,5,4],["split",24,7.0,22,4],["split",25,21.0,24,4],["split",26,12.0,22,4],["split",27,11.0,1,4],["leaf",28,-0.01384418895951173],["split",29,23.0,14,4],["split",30,26.0,5,4],["leaf",31,-0.015026553738778717],["leaf",32,0.004956112328168024],["leaf",33,-0.002935364844846037],["leaf",34,-0.012869251862370399],["leaf",35,-0.01696853989421239],["leaf",36,-5.580000984663688e-05],["leaf",37,0.004422082577013633],["leaf",38,-0.0034995914870261744],["leaf",45,-0.008934922517854748],["leaf",46,0.008457330735752477],["leaf",47,-0.010499598803328758],["leaf",48,0.0018511480070494972],["leaf",49,0.004127928448120106],["leaf",50,0.013727636551128075],["leaf",51,-0.004212627450804599],["leaf",52,0.007636693060648812],["leaf",53,-0.01948536464285983],["leaf",54,-0.01287128918490273],["leaf",55,-0.0048745648471438694],["leaf",56,0.004653345640352919],["leaf",59,0.01364904848009147],["leaf",60,-0.0036612911387033003],["leaf",61,0.0059845794877476325],["leaf",62,0.02397572546926832]],[["split",0,7.0,4,0],["split",1,24.0,19,1],["split",2,3.0,21,1],["split",3,3.0,26,2],["split",4,11.0,17,2],["split",5,13.0,4,2],["split",6,17.0,0,2],["split",7,8.0,6,3],["leaf",8,0.005711396852528679],["split",9,22.0,24,3],["split",10,6.0,1,3],["split",11,12.0,2,3],["split",12,10.0,7,3],["split",13,20.0,0,3],["split",14,1.0,9,3],["split",15,4.0,17,4],["leaf",16,-0.024474211818329168],["split",19,24.0,27,4],["leaf",20,0.007310620155010958],["leaf",21,0.016891463686505435],["split",22,8.0,4,4],["leaf",23,-0.004260638975655873],["split",24,10.0,12,4],["split",25,26.0,7,4],["split",26,24.0,30,4],["split",27,19.0,0,4],["split",28,24.0,14,4],["leaf",29,-0.002604705372080053],["split",30,9.0,25,4],["leaf",31,0.0066678620197588925],["leaf",32,-0.011575796716953336],["leaf",39,-0.00545346008099956],["leaf",40,-0.018743433197553797],["leaf",45,-0.009301817824293528],["leaf",46,0.00979254139522296],["leaf",49,0.0069128984719665785],["leaf",50,0.017100104598986313],["leaf",51,-0.010170539545316788],["leaf",52,-0.0033662686583414516],["leaf",53,0.0008369381823360811],["leaf",54,-0.013154343829479257],["leaf",55,0.006530036871761436],["leaf",56,-0.0059309040689730095],["leaf",57,-0.01688776942762479],["leaf",58,0.0023193515226927214],["leaf",61,0.014552873788543613],["leaf",62,0.030016816828094615]],[["split",0,7.0,17,0],["split",1,3.0,29,1],["split",2,13.0,29,1],["split",3,7.0,0,2],["split",4,0.0,15,2],["split",5,13.0,14,2],["split",6,25.0,25,2],["split",7,1.0,15,3],["split",8,24.0,29,3],["split",9,23.0,12,3],["split",10,23.0,9,3],["split",11,24.0,25,3],["split",12,26.0,0,3],["split",13,23.0,25,3],["leaf",14,0.02299967839535701],["split",15,2.0,29,4],["leaf",16,-0.023355528648860147],["split",17,23.0,24,4],["split",18,2.0,24,4],["leaf",19,-0.007950181094529227],["leaf",20,0.005365643359550715],["leaf",21,0.0037224651812196217],["split",22,11.0,15,4],["split",23,1.0,22,4],["leaf",24,0.022649974556711142],["leaf",25,-0.01303813850865959],["split",26,1.0,4,4],["split",27,26.0,27,4],["leaf",28,-0.002399621749002621],["leaf",31,-0.0016728937964171106],["leaf",32,-0.021703354262372666],["leaf",35,-0.0014093469587339058],["leaf",36,-0.008102858224718243],["leaf",37,0.0027699736269052263],["leaf",38,0.02023271827175573],["leaf",45,0.015364774131738855],["leaf",46,0.025599839222615467],["leaf",47,0.008996127544540152],["leaf",48,-0.004370439886992713],["leaf",53,-0.006732825370665276],["leaf",54,0.002682628513059768],["leaf",55,0.006497363580165635],["leaf",56,0.0181190303275543]],[["split",0,4.0,11,0],["split",1,19.0,0,1],["split",2,3.0,1,1],["split",3,2.0,8,2],["split",4,12.0,18,2],["split",5,22.0,26,2],["split",6,20.0,0,2],["split",7,20.0,0,3],["split",8,25.0,19,3],["split",9,12.0,13,3],["split",10,24.0,23,3],["split",11,10.0,6,3],["leaf",12,-0.005766910288343661],["split",13,2.0,13,3],["split",14,21.0,16,3],["split",15,7.0,27,4],["split",16,24.0,16,4],["split",17,26.0,20,4],["split",18,23.0,8,4],["leaf",19,-0.007168587521085651],["leaf",20,0.010288100865573357],["split",21,1.0,14,4],["leaf",22,-0.015411561107099838],["leaf",23,-0.007111060659630788],["split",24,11.0,24,4],["split",27,23.0,21,4],["split",28,21.0,29,4],["split",29,23.0,25,4],["split",30,2.0,16,4],["leaf",31,0.007008041401597477],["leaf",32,0.018616293075832832],["leaf",33,-0.013671567802360738],["leaf",34,0.00391078413296066],["leaf",35,-0.004162758394876957],["leaf",36,0.003194419106277986],["leaf",37,0.012007703976492227],["leaf",38,0.0006925754434397371],["leaf",43,-0.011947662015543797],["leaf",44,-0.0022644869443956316],["leaf",49,0.007402728449327889],["leaf",50,0.03152641620210731],["leaf",55,0.004181302407935292],["leaf",56,-0.00726309315214135],["leaf",57,-0.005575016233344013],["leaf",58,-0.016715511084093655],["leaf",59,-0.010266126171746954],["leaf",60,0.009015661302572068],["leaf",61,0.0007689789602218809],["leaf",62,0.027917975992799816]],[["split",0,7.0,5,0],["split",1,1.0,28,1],["split",2,22.0,21,1],["split",3,2.0,28,2],["split",4,23.0,23,2],["split",5,21.0,0,2],["split",6,8.0,9,2],["split",7,5.0,30,3],["split",8,10.0,19,3],["leaf",9,-0.008943315435557727],["leaf",10,-0.024176931622942],["split",11,24.0,11,3],["split",12,1.0,0,3],["split",13,25.0,23,3],["split",14,10.0,21,3],["split",15,23.0,28,4],["leaf",16,0.011774882969342405],["split",17,0.0,10,4],["leaf",18,-0.023958770449615578],["leaf",23,-0.02023507598831313],["leaf",24,2.261032097213438e-06],["leaf",25,-0.017054780771351905],["split",26,7.0,29,4],["split",27,24.0,23,4],["leaf",28,-0.006496072489698152],["split",29,2.0,4,4],["split",30,0.0,13,4],["leaf",31,-0.0032289328396581762],["leaf",32,0.00859921427030124],["leaf",35,-0.01484352001243605],["leaf",36,-0.002948469643851832],["leaf",53,0.0032943828285638627],["leaf",54,0.013288294451867156],["leaf",55,0.005017981904146916],["leaf",56,0.0200270672409253],["leaf",59,-0.0007945736227440485],["leaf",60,-0.008674074843132028],["leaf",61,-0.006097525189616348],["leaf",62,0.005293531032019307]],[["split",0,7.0,25,0],["split",1,26.0,29,1],["split",2,4.0,10,1],["split",3,21.0,6,2],["split",4,19.0,0,2],["split",5,2.0,7,2],["split",6,5.0,11,2],["split",7,20.0,0,3],["split",8,22.0,21,3],["split",9,24.0,21,3],["leaf",10,-0.0055564075197083555],["split",11,0.0,23,3],["split",12,9.0,22,3],["split",13,9.0,14,3],["split",14,13.0,26,3],["split",15,7.0,1,4],["split",16,23.0,17,4],["split",17,20.0,0,4],["split",18,4.0,2,4],["split",19,12.0,21,4],["leaf",20,0.018002429746702946],["split",23,12.0,23,4],["leaf",24,0.02582437141391912],["split",25,6.0,29,4],["leaf",26,0.015409382140420388],["leaf",27,-0.018474901618932546],["leaf",28,0.00013011215003087733],["split",29,23.0,12,4],["split",30,12.0,26,4],["leaf",31,-0.018411484930572482],["leaf",32,-0.000917529539072608],["leaf",33,-0.03067088216155058],["leaf",34,0.002070243417940485],["leaf",35,0.00018949194583126194],["leaf",36,0.00802143100013094],["leaf",37,0.006075483915909037],["leaf",38,-0.004313036194786172],["leaf",39,0.0015277649133017663],["leaf",40,0.013097116585749469],["leaf",47,0.021401760858697383],["leaf",48,0.005842856268332204],["leaf",51,-0.0020808682560262194],["leaf",52,0.00867177835028727],["leaf",59,0.01028687198196387],["leaf",60,-0.0034826219670922023],["leaf",61,0.01581071968036992],["leaf",62,0.001250917958089901]],[["split",0,2.0,9,0],["split",1,20.0,0,1],["split",2,20.0,0,1],["split",3,22.0,8,2],["split",4,22.0,12,2],["split",5,3.0,27,2],["split",6,21.0,9,2],["split",7,4.0,2,3],["split",8,11.0,26,3],["split",9,24.0,15,3],["split",10,12.0,17,3],["split",11,23.0,25,3],["split",12,5.0,16,3],["split",13,2.0,16,3],["split",14,23.0,21,3],["leaf",15,0.02145404790589075],["split",16,1.0,12,4],["split",17,23.0,12,4],["split",18,9.0,24,4],["leaf",19,-0.0010992143291679842],["leaf",20,0.01925842395300013],["leaf",21,-0.02730303210893358],["split",22,23.0,12,4],["split",23,26.0,0,4],["split",24,17.0,0,4],["split",25,12.0,28,4],["leaf",26,0.01600022253013522],["leaf",27,-0.024266586799185887],["split",28,24.0,9,4],["split",29,11.0,17,4],["split",30,9.0,7,4],["leaf",33,0.002514175171240024],["leaf",34,0.01054926798130103],["leaf",35,0.010364379308896297],["leaf",36,0.002783478219907662],["leaf",37,-0.007687247309359588],["leaf",38,0.009011085587739213],["leaf",45,-0.013515359541757466],["leaf",46,0.001879550338970832],["leaf",47,-0.014552578057115668],["leaf",48,-0.00020135066438103536],["leaf",49,-0.004957488218009752],["leaf",50,-0.024893057557962384],["leaf",51,0.0025373035041197194],["leaf",52,0.017141152862151156],["leaf",57,-0.012909987372952177],["leaf",58,0.001206326205272615],["leaf",59,-0.0024299091866829777],["leaf",60,0.013198262941799908],["leaf",61,0.007816839230808233],["leaf",62,0.03629797596905662]],[["split",0,21.0,2,0],["split",1,2.0,26,1],["split",2,4.0,30,1],["split",3,20.0,0,2],["leaf",4,0.007394798620316797],["split",5,7.0,19,2],["split",6,25.0,15,2],["split",7,0.0,1,3],["leaf",8,-0.026482206812060143],["split",11,24.0,27,3],["split",12,23.0,30,3],["leaf",13,-0.0010954772996327644],["leaf",14,-0.02059038864053337],["leaf",15,-0.020535165985546423],["split",16,0.0,20,4],["split",23,26.0,4,4],["split",24,26.0,4,4],["split",25,22.0,19,4],["leaf",26,-0.014663511189048802],["leaf",33,-0.000660704412145467],["leaf",34,-0.018070217482358143],["leaf",47,-0.006693435575846485],["leaf",48,-0.00043974931832620857],["leaf",49,-0.005690800497152634],["leaf",50,0.007399512899669175],["leaf",51,0.006412062134057875],["leaf",52,0.0006358803418835736]],[["split",0,7.0,12,0],["split",1,23.0,19,1],["split",2,23.0,18,1],["split",3,10.0,27,2],["split",4,17.0,0,2],["split",5,20.0,0,2],["split",6,13.0,5,2],["split",7,2.0,19,3],["split",8,11.0,15,3],["split",9,15.0,0,3],["split",10,5.0,21,3],["split",11,26.0,2,3],["split",12,21.0,17,3],["leaf",13,0.01959265194731106],["split",14,19.0,0,3],["split",15,20.0,0,4],["split",16,1.0,2,4],["split",17,0.0,15,4],["leaf",18,0.001109497020987912],["split",19,2.0,15,4],["split",20,3.0,18,4],["leaf",21,-0.0024572642404123274],["split",22,23.0,26,4],["split",23,13.0,13,4],["split",24,17.0,0,4],["split",25,21.0,10,4],["split",26,5.0,14,4],["split",29,7.0,30,4],["split",30,26.0,9,4],["leaf",31,0.0041315251584489055],["leaf",32,-0.012782293276809624],["leaf",33,-0.017600650028933358],["leaf",34,-0.0032726314102709234],["leaf",35,0.020567362219902952],["leaf",36,0.00503077690763361],["leaf",39,-0.00933217134421501],["leaf",40,0.0002823632155772385],["leaf",41,-0.021597703982770066],["leaf",42,-0.0006079902724953301],["leaf",45,-0.016151751838652147],["leaf",46,-0.03278335954446678],["leaf",47,0.006612302256039291],["leaf",48,-0.008980618687034885],["leaf",49,0.004253520890005778],["leaf",50,0.012030569341931358],["leaf",51,-0.01921460884922233],["leaf",52,-0.0028308890893172216],["leaf",53,0.006990095693263922],["leaf",54,0.025695552379295872],["leaf",59,-0.0022665593708563925],["leaf",60,0.009787666153733483],["leaf",61,-0.023419480516064222],["leaf",62,-0.007339799380251728]],[["split",0,7.0,0,0],["split",1,4.0,27,1],["split",2,7.0,22,1],["leaf",3,-0.0011408811496317105],["split",4,10.0,12,2],["split",5,10.0,0,2],["split",6,23.0,24,2],["leaf",9,-0.024106511998959367],["leaf",10,-0.0031715836692463085],["split",11,13.0,6,3],["split",12,9.0,5,3],["split",13,15.0,0,3],["split",14,4.0,11,3],["leaf",23,-0.003655315884025721],["leaf",24,-0.016150121697070228],["split",25,22.0,8,4],["split",26,21.0,0,4],["split",27,17.0,0,4],["split",28,12.0,21,4],["split",29,9.0,13,4],["split",30,25.0,6,4],["leaf",51,-0.011490253553619639],["leaf",52,-0.0018728363191759538],["leaf",53,-0.011379676993047337],["leaf",54,0.0003222889813859129],["leaf",55,0.0013999518844130961],["leaf",56,0.010328577839064514],["leaf",57,0.0038598349765397],["leaf",58,0.01786181175569604],["leaf",59,0.009007255013459491],["leaf",60,-0.0022725450905284714],["leaf",61,-0.01977967464859583],["leaf",62,-0.007794545566580561]],[["split",0,4.0,4,0],["split",1,17.0,0,1],["split",2,25.0,0,1],["split",3,20.0,0,2],["split",4,4.0,1,2],["split",5,26.0,5,2],["split",6,7.0,0,2],["split",7,9.0,29,3],["split",8,3.0,27,3],["leaf",9,0.02736157460306185],["leaf",10,0.008246019737774685],["split",11,21.0,12,3],["leaf",12,-0.003674518953753115],["split",13,12.0,2,3],["split",14,25.0,18,3],["split",15,4.0,0,4],["leaf",16,0.016130444683413226],["leaf",17,-0.0032795914819565676],["leaf",18,-0.02034818481446495],["leaf",23,-0.020422827197079794],["leaf",24,-0.01684938662554707],["split",27,26.0,7,4],["leaf",28,-0.019665670975894025],["split",29,2.0,30,4],["split",30,6.0,2,4],["leaf",31,0.008897238730441839],["leaf",32,0.002068145077395644],["leaf",55,-0.01315174719174109],["leaf",56,0.003547926075135642],["leaf",59,-0.001262403374413141],["leaf",60,-0.011920213877235904],["leaf",61,0.011444244651927721],["leaf",62,0.0007439425217335867]],[["split",0,26.0,24,0],["split",1,21.0,2,1],["split",2,1.0,3,1],["split",3,20.0,0,2],["split",4,7.0,25,2],["split",5,23.0,28,2],["split",6,3.0,7,2],["split",7,24.0,24,3],["leaf",8,-0.022331109112405063],["split",9,23.0,29,3],["split",10,25.0,29,3],["leaf",11,-0.003343667472438872],["leaf",12,-0.016353323386649878],["leaf",13,-0.009001084841125542],["split",14,25.0,29,3],["split",15,23.0,24,4],["leaf",16,0.004074015559051849],["split",19,21.0,9,4],["split",20,13.0,4,4],["split",21,1.0,28,4],["leaf",22,0.014422380198830015],["split",29,20.0,0,4],["leaf",30,0.016109478836962624],["leaf",31,-0.0053624594877621995],["leaf",32,-0.02239370823354345],["leaf",39,-0.005346710780793147],["leaf",40,-0.00017042980735422695],["leaf",41,-0.00358941077013533],["leaf",42,-0.025821663770558983],["leaf",43,0.0038268541826136037],["leaf",44,-0.00395656677500168],["leaf",59,0.006242462321640982],["leaf",60,-0.004417524936687337]],[["split",0,2.0,7,0],["split",1,20.0,0,1],["split",2,10.0,3,1],["split",3,17.0,0,2],["split",4,21.0,10,2],["split",5,22.0,4,2],["split",6,25.0,27,2],["split",7,22.0,10,3],["split",8,22.0,16,3],["leaf",9,-0.023880411238824426],["split",10,24.0,20,3],["split",11,2.0,18,3],["split",12,1.0,0,3],["split",13,1.0,1,3],["split",14,20.0,0,3],["split",15,9.0,17,4],["split",16,7.0,22,4],["split",17,5.0,9,4],["split",18,24.0,17,4],["split",21,2.0,3,4],["leaf",22,0.007963108230126874],["leaf",23,-0.008066635515825135],["leaf",24,-0.024224556572119728],["leaf",25,0.011924206965352204],["split",26,21.0,2,4],["split",27,4.0,25,4],["split",28,9.0,30,4],["split",29,25.0,28,4],["leaf",30,0.03264485756950032],["leaf",31,0.0026054645233819142],["leaf",32,0.016798424507981892],["leaf",33,-0.0015604127279588889],["leaf",34,0.005307917487939427],["leaf",35,0.015235801672484974],["leaf",36,0.005056637891077055],["leaf",37,0.014877077672504267],["leaf",38,0.030725172454014814],["leaf",43,-0.010910969103717742],["leaf",44,-0.00336655574500413],["leaf",53,-0.02372780250041138],["leaf",54,-0.0056395778471034184],["leaf",55,-0.01720478461892552],["leaf",56,-0.0025559835846865084],["leaf",57,-0.0010845424067784668],["leaf",58,0.01005775807493969],["leaf",59,0.012463874731089158],["leaf",60,-0.001681144161016841]],[["split",0,24.0,13,0],["split",1,7.0,10,1],["split",2,26.0,28,1],["split",3,3.0,25,2],["split",4,25.0,0,2],["split",5,8.0,2,2],["split",6,0.0,26,2],["split",7,1.0,0,3],["split",8,21.0,15,3],["leaf",9,-0.014721192480591167],["split",10,2.0,11,3],["split",11,23.0,21,3],["split",12,10.0,26,3],["split",13,12.0,18,3],["leaf",14,0.018173035427212002],["leaf",15,-0.02205490619131838],["split",16,14.0,0,4],["leaf",17,-0.008639759683448817],["leaf",18,0.012734317400006004],["split",21,20.0,0,4],["split",22,11.0,24,4],["split",23,2.0,18,4],["split",24,24.0,26,4],["split",25,1.0,26,4],["split",26,7.0,28,4],["split",27,10.0,14,4],["split",28,11.0,19,4],["leaf",33,-0.005465900087355552],["leaf",34,-0.016461225589101066],["leaf",43,0.00458159961251805],["leaf",44,-0.008590020223371367],["leaf",45,-0.0037825424338382547],["leaf",46,0.0030019785300116663],["leaf",47,0.02346879918873251],["leaf",48,0.00817822917904003],["leaf",49,0.007464067207023341],["leaf",50,-0.012608985150983664],["leaf",51,-0.0004155063320650713],["leaf",52,-0.006517480078552171],["leaf",53,0.0026284686451695126],["leaf",54,0.014995993342017629],["leaf",55,0.0039823455298738635],["leaf",56,-0.0072928645850769335],["leaf",57,0.013131723874129843],["leaf",58,0.001581456743538373]],[["split",0,22.0,28,0],["split",1,7.0,30,1],["split",2,24.0,12,1],["split",3,9.0,26,2],["split",4,23.0,23,2],["split",5,9.0,24,2],["split",6,26.0,25,2],["split",7,16.0,0,3],["split",8,17.0,0,3],["split",9,24.0,14,3],["leaf",10,0.0015905989910263766],["split",11,22.0,30,3],["leaf",12,-0.0021348484976626797],["split",13,25.0,9,3],["leaf",14,0.008989845130405617],["split",15,3.0,7,4],["split",16,24.0,5,4],["split",17,7.0,25,4],["split",18,10.0,21,4],["leaf",19,0.005633111312499723],["leaf",20,0.02739121197696918],["split",23,0.0,10,4],["leaf",24,-0.029220551595898],["leaf",27,0.001119569871137698],["split",28,23.0,18,4],["leaf",31,0.003566394474537855],["leaf",32,-0.00105244324399691],["leaf",33,-0.016352839087843313],["leaf",34,-0.0013174601439814224],["leaf",35,0.0008146521403668289],["leaf",36,0.013219290885996104],["leaf",37,0.003966478417268344],["leaf",38,0.027895034899015066],["leaf",47,-0.01708720056192089],["leaf",48,-0.005554680215286047],["leaf",57,-0.005806197155838505],["leaf",58,-0.01987253416649692]],[["split",0,10.0,1,0],["split",1,12.0,18,1],["split",2,9.0,30,1],["split",3,3.0,12,2],["split",4,9.0,16,2],["split",5,7.0,28,2],["split",6,21.0,12,2],["split",7,0.0,12,3],["split",8,4.0,20,3],["split",9,24.0,15,3],["leaf",10,-0.009857843388466144],["split",11,26.0,1,3],["split",12,2.0,6,3],["leaf",13,-0.004882706430207853],["split",14,23.0,12,3],["leaf",15,-0.007461848424934295],["leaf",16,-0.027666866890588657],["leaf",17,-0.01010433012497496],["leaf",18,0.0011976162640379108],["leaf",19,-0.0031295967206125047],["leaf",20,0.014141447545447784],["split",23,10.0,25,4],["split",24,1.0,4,4],["split",25,24.0,16,4],["split",26,10.0,14,4],["leaf",29,0.010219409799826525],["leaf",30,0.024459061557073593],["leaf",47,-0.012482205350580962],["leaf",48,0.008880317255671474],["leaf",49,-0.004566501862037894],["leaf",50,-0.0002296110856334314],["leaf",51,0.016917269499751645],["leaf",52,0.002678852752374253],["leaf",53,0.006217324915971265],["leaf",54,-0.0058851479165112925]],[["split",0,7.0,27,0],["split",1,25.0,29,1],["split",2,18.0,0,1],["split",3,1.0,30,2],["split",4,11.0,17,2],["split",5,11.0,3,2],["split",6,25.0,14,2],["split",7,10.0,16,3],["split",8,12.0,11,3],["split",9,10.0,21,3],["split",10,24.0,19,3],["leaf",11,0.013244138404287042],["split",12,25.0,26,3],["leaf",13,-0.016933624012916988],["leaf",14,0.0025351449219739373],["split",15,11.0,28,4],["split",16,20.0,0,4],["leaf",17,-0.006943034612358227],["leaf",18,-0.02159221287184585],["split",19,6.0,11,4],["leaf",20,0.013239842599058865],["split",21,3.0,12,4],["leaf",22,0.024587488258633883],["split",25,13.0,22,4],["split",26,21.0,16,4],["leaf",31,-0.0019138347630726492],["leaf",32,-0.009658007482563909],["leaf",33,0.002013199351947073],["leaf",34,-0.004912100746938866],["leaf",39,-0.014319092986502292],["leaf",40,0.0010313481591417018],["leaf",43,0.012576693800276319],["leaf",44,0.0016209670092442916],["leaf",51,0.010725459059279158],["leaf",52,0.0005083413401065202],["leaf",53,-0.0001541994847351344],["leaf",54,0.019737607495230618]],[["split",0,13.0,27,0],["split",1,1.0,1,1],["split",2,21.0,26,1],["split",3,6.0,5,2],["split",4,25.0,30,2],["split",5,9.0,29,2],["split",6,3.0,21,2],["split",7,26.0,3,3],["split",8,7.0,18,3],["split",9,1.0,27,3],["split",10,21.0,21,3],["split",11,24.0,8,3],["leaf",12,0.01398543068429681],["leaf",13,0.008857422865557364],["leaf",14,0.021445451684053953],["leaf",15,-0.012626457027172005],["split",16,10.0,8,4],["leaf",17,-0.02591986831241177],["leaf",18,-0.0016293533776172064],["split",19,26.0,24,4],["split",20,26.0,25,4],["split",21,22.0,14,4],["leaf",22,0.018136400518710485],["split",23,26.0,12,4],["split",24,4.0,17,4],["leaf",33,0.00951060701198883],["leaf",34,-0.0051783849782010855],["leaf",39,-0.0011251566976463886],["leaf",40,0.0032852300090793958],["leaf",41,-0.003456874575535819],["leaf",42,-0.01965441045616229],["leaf",43,0.005191145587618394],["leaf",44,-0.0026865667991235343],["leaf",47,-0.01820491589651294],["leaf",48,0.0010548602266341445],["leaf",49,0.002093551538321775],["leaf",50,0.015128302433240027]],[["split",0,2.0,1,0],["split",1,7.0,28,1],["split",2,1.0,30,1],["split",3,22.0,5,2],["leaf",4,0.01896562202815239],["split",5,4.0,10,2],["split",6,0.0,7,2],["leaf",7,0.015540913428988412],["split",8,1.0,25,3],["split",11,22.0,12,3],["split",12,23.0,30,3],["leaf",13,0.0023476130716677326],["split",14,7.0,22,3],["split",17,24.0,14,4],["leaf",18,0.010344866472864901],["split",23,9.0,22,4],["split",24,20.0,0,4],["split",25,19.0,0,4],["split",26,9.0,11,4],["leaf",29,-0.019877734300003395],["leaf",30,-0.00649633858163914],["leaf",35,-0.008037433521290887],["leaf",36,0.004242130789189119],["leaf",47,0.002477029138309981],["leaf",48,0.01038839651279989],["leaf",49,0.0018231711136443899],["leaf",50,-0.006964824190859151],["leaf",51,-0.0014149899810766522],["leaf",52,0.0042197388337139605],["leaf",53,-0.0007755456684109006],["leaf",54,-0.024533297111355844]],[["split",0,24.0,24,0],["split",1,4.0,28,1],["split",2,25.0,21,1],["split",3,22.0,30,2],["split",4,3.0,1,2],["split",5,11.0,15,2],["split",6,20.0,0,2],["split",7,26.0,5,3],["split",8,11.0,11,3],["leaf",9,-0.024612811481732978],["split",10,2.0,23,3],["split",11,1.0,28,3],["split",12,17.0,0,3],["split",13,7.0,23,3],["split",14,23.0,16,3],["split",15,7.0,25,4],["split",16,21.0,29,4],["leaf",17,-0.018816911884392683],["leaf",18,-0.007026935180071125],["split",21,10.0,14,4],["split",22,7.0,3,4],["split",23,4.0,3,4],["leaf",24,0.01957646261180013],["split",25,0.0,6,4],["leaf",26,0.009099678843140774],["split",27,7.0,3,4],["split",28,10.0,14,4],["leaf",29,0.010826720329995564],["leaf",30,0.03760971584076073],["leaf",31,-0.007274633385947495],["leaf",32,0.0033510190501063785],["leaf",33,0.000114801940648475],["leaf",34,0.008789747897790174],["leaf",43,0.011553908717301362],["leaf",44,-0.004730467250335635],["leaf",45,-0.0030519645893816263],["leaf",46,-0.017728604582724367],["leaf",47,0.016909531636453416],["leaf",48,0.0014956523654967879],["leaf",51,-0.01713461836452675],["leaf",52,-0.004121129346032667],["leaf",55,-0.007523628731868418],["leaf",56,0.00349965478991333],["leaf",57,0.011836723128407962],["leaf",58,0.023291737746251085]],[["split",0,25.0,1,0],["split",1,1.0,9,1],["split",2,4.0,30,1],["split",3,26.0,7,2],["split",4,24.0,11,2],["split",5,16.0,0,2],["split",6,9.0,14,2],["leaf",7,0.010724148825874295],["leaf",8,-0.009097994600890245],["split",9,25.0,0,3],["split",10,4.0,10,3],["split",11,3.0,0,3],["split",12,3.0,3,3],["leaf",13,-0.016358825149106717],["leaf",14,0.0006013815697123651],["leaf",19,-0.024368735640517517],["leaf",20,-0.010936506348115379],["leaf",21,0.008491579903007451],["split",22,22.0,15,4],["split",23,0.0,18,4],["split",24,9.0,4,4],["split",25,9.0,16,4],["split",26,23.0,27,4],["leaf",45,-0.0007293602894708059],["leaf",46,-0.017814670778259087],["leaf",47,0.018315337913283766],["leaf",48,0.001631594772825001],["leaf",49,-0.0026444849084390934],["leaf",50,0.0015665253853517072],["leaf",51,-0.005643798443326573],["leaf",52,-0.02802911553722341],["leaf",53,-0.0022255573396186635],["leaf",54,0.009573231232596435]],[["split",0,26.0,0,0],["split",1,2.0,15,1],["split",2,23.0,29,1],["split",3,25.0,9,2],["split",4,5.0,24,2],["split",5,21.0,1,2],["split",6,17.0,0,2],["leaf",7,-0.008216267272696726],["leaf",8,0.00462181493940212],["leaf",9,-0.022744556784161228],["leaf",10,-0.008314341986550483],["split",11,24.0,17,3],["split",12,0.0,6,3],["split",13,25.0,21,3],["leaf",14,-0.02819977599230797],["split",23,11.0,24,4],["split",24,9.0,13,4],["split",25,7.0,26,4],["split",26,7.0,0,4],["split",27,26.0,12,4],["split",28,23.0,30,4],["leaf",47,0.005900038461840275],["leaf",48,-0.008799394399688368],["leaf",49,0.00710043935445809],["leaf",50,0.021763776704747616],["leaf",51,-0.004898251606479112],["leaf",52,0.004634934292833409],["leaf",53,-0.008579867742309457],["leaf",54,0.001839714725593526],["leaf",55,0.007625479475450084],["leaf",56,-0.012686458976489465],["leaf",57,-2.7640521499668678e-05],["leaf",58,0.016137381849750995]],[["split",0,24.0,2,0],["split",1,3.0,12,1],["split",2,7.0,26,1],["split",3,25.0,25,2],["split",4,3.0,26,2],["split",5,10.0,1,2],["split",6,4.0,17,2],["split",7,25.0,17,3],["leaf",8,-0.01312257242745775],["split",9,21.0,4,3],["leaf",10,0.001909205113111582],["split",11,11.0,4,3],["split",12,21.0,4,3],["split",13,5.0,15,3],["split",14,12.0,21,3],["split",15,0.0,8,4],["leaf",16,0.014383272782058903],["leaf",19,-0.026024360733295688],["split",20,9.0,20,4],["leaf",23,-0.02119704209161865],["split",24,25.0,22,4],["split",25,11.0,2,4],["split",26,1.0,0,4],["split",27,20.0,0,4],["split",28,11.0,15,4],["split",29,22.0,10,4],["split",30,1.0,20,4],["leaf",31,0.0072539056470333286],["leaf",32,-0.010541452538177221],["leaf",41,-0.01523554755316696],["leaf",42,-0.0024835286874858387],["leaf",49,-0.007868340155619314],["leaf",50,0.0064611356302098156],["leaf",51,0.008190659274835425],["leaf",52,-0.0061174792748606484],["leaf",53,-0.010644837689733268],["leaf",54,0.0009729641676689136],["leaf",55,0.006471710291415411],["leaf",56,-0.009671332241372626],["leaf",57,0.024102996866815538],["leaf",58,0.0098876697826575],["leaf",59,0.01132249876462098],["leaf",60,-0.001775105543523626],["leaf",61,-0.019946511346760336],["leaf",62,-0.0036217026786024855]],[["split",0,13.0,2,0],["split",1,23.0,22,1],["split",2,23.0,11,1],["split",3,2.0,25,2],["split",4,17.0,0,2],["split",5,10.0,14,2],["split",6,20.0,0,2],["split",7,0.0,22,3],["leaf",8,-0.014391509076385922],["split",9,10.0,14,3],["leaf",10,-0.011233670901296634],["split",11,9.0,10,3],["split",12,25.0,26,3],["split",13,4.0,16,3],["split",14,2.0,9,3],["split",15,23.0,12,4],["leaf",16,0.015755008501489403],["split",19,26.0,15,4],["leaf",20,0.0017862282464476353],["split",23,23.0,5,4],["split",24,1.0,13,4],["split",25,20.0,0,4],["split",26,3.0,19,4],["split",27,3.0,30,4],["split",28,19.0,0,4],["split",29,11.0,11,4],["split",30,25.0,25,4],["leaf",31,0.01159162289430997],["leaf",32,-0.00844454669808806],["leaf",39,0.023124492197408617],["leaf",40,0.005972120644331917],["leaf",47,-0.0012230053288906104],["leaf",48,0.007595219278220736],["leaf",49,-0.010065695021463681],["leaf",50,-0.00024361967651927846],["leaf",51,0.0053429185593727386],["leaf",52,-0.005459654734711994],["leaf",53,0.01689322386832773],["leaf",54,0.004557024292377691],["leaf",55,0.00022414457425549931],["leaf",56,0.011249814642027064],["leaf",57,-0.007365344844618565],["leaf",58,0.004269973215045456],["leaf",59,-0.02159177403268273],["leaf",60,-0.0023117091334669817],["leaf",61,0.006657324300312859],["leaf",62,0.036722888882421124]],[["split",0,8.0,7,0],["split",1,15.0,0,1],["split",2,1.0,1,1],["split",3,17.0,0,2],["split",4,22.0,17,2],["split",5,21.0,17,2],["split",6,21.0,5,2],["split",7,13.0,0,3],["split",8,10.0,20,3],["split",9,2.0,20,3],["split",10,1.0,7,3],["leaf",11,-0.002560685635699764],["leaf",12,-0.01996843419438709],["split",13,13.0,13,3],["split",14,19.0,0,3],["split",15,26.0,7,4],["split",16,21.0,4,4],["split",17,26.0,15,4],["leaf",18,0.005971194355701922],["leaf",19,-0.008387255160767762],["leaf",20,-0.029270860345576744],["leaf",21,-0.013489451621333545],["leaf",22,0.002721698756323403],["split",27,7.0,16,4],["split",28,0.0,2,4],["split",29,0.0,9,4],["split",30,21.0,26,4],["leaf",31,0.017902448209820534],["leaf",32,-0.0008599968471555547],["leaf",33,-0.0068334199894632556],["leaf",34,0.0007000404100480943],["leaf",35,-0.012708789167325053],["leaf",36,-0.031228478798728167],["leaf",55,0.0003965640290497299],["leaf",56,0.01564539697848879],["leaf",57,-0.014047562730494643],["leaf",58,-0.002923179650000687],["leaf",59,-0.0013264225110490955],["leaf",60,0.00416211528652427],["leaf",61,-0.000273009877524581],["leaf",62,-0.012351703766790567]],[["split",0,2.0,3,0],["split",1,17.0,0,1],["split",2,26.0,25,1],["split",3,15.0,0,2],["split",4,2.0,1,2],["split",5,5.0,28,2],["split",6,9.0,26,2],["split",7,9.0,3,3],["split",8,12.0,25,3],["leaf",9,0.02406325575776008],["leaf",10,0.013026546093864078],["split",11,26.0,5,3],["split",12,4.0,29,3],["split",13,26.0,28,3],["split",14,4.0,5,3],["leaf",15,-0.013210818711092968],["split",16,14.0,0,4],["leaf",17,0.00564632420346303],["leaf",18,0.016555533595546387],["split",23,4.0,25,4],["split",24,21.0,14,4],["split",25,21.0,25,4],["split",26,3.0,8,4],["split",27,0.0,17,4],["split",28,0.0,24,4],["split",29,11.0,15,4],["split",30,7.0,15,4],["leaf",33,-0.0011269302793533997],["leaf",34,0.011240733868627405],["leaf",47,-0.00419965918657249],["leaf",48,-0.014597474981717672],["leaf",49,-0.004403023966717055],["leaf",50,0.0008987003471155862],["leaf",51,0.005546774103622819],["leaf",52,0.021533552420374715],["leaf",53,0.003967118226883977],["leaf",54,-0.008968869521090043],["leaf",55,0.008575282764702447],["leaf",56,0.002139782652467098],["leaf",57,0.0012018605825190958],["leaf",58,-0.008137536156902487],["leaf",59,-0.0035226863829760256],["leaf",60,0.0066227089136917956],["leaf",61,0.005739119389167523],["leaf",62,0.021805873332460698]],[["split",0,9.0,15,0],["split",1,20.0,0,1],["split",2,24.0,21,1],["split",3,23.0,13,2],["split",4,1.0,9,2],["split",5,3.0,3,2],["split",6,9.0,27,2],["split",7,1.0,24,3],["split",8,3.0,29,3],["split",9,2.0,17,3],["split",10,2.0,17,3],["split",11,11.0,23,3],["split",12,0.0,5,3],["split",13,26.0,6,3],["split",14,5.0,11,3],["split",15,7.0,7,4],["split",16,5.0,19,4],["split",17,2.0,10,4],["leaf",18,0.010337940530829181],["leaf",19,-0.00045550491501404746],["split",20,9.0,6,4],["split",21,12.0,22,4],["split",22,1.0,19,4],["split",23,22.0,22,4],["leaf",24,0.022801892300756302],["split",25,8.0,6,4],["split",26,0.0,25,4],["split",27,4.0,24,4],["split",28,3.0,12,4],["split",29,0.0,20,4],["split",30,4.0,20,4],["leaf",31,-0.009457434961787478],["leaf",32,-0.00024215178300615548],["leaf",33,0.0014251112356805944],["leaf",34,0.01418800072862644],["leaf",35,-0.0009753166002696957],["leaf",36,-0.00788234115714587],["leaf",41,0.008878049457403649],["leaf",42,0.027879638937524792],["leaf",43,-0.018319169864538125],["leaf",44,0.0010662366992010115],["leaf",45,0.019636306567834435],["leaf",46,0.004091106620136847],["leaf",47,0.004222719587094512],["leaf",48,-0.012668625530057601],["leaf",51,0.013063886450372169],["leaf",52,0.0014908564060057557],["leaf",53,-0.0039254898542596325],["leaf",54,0.0034216241013499296],["leaf",55,0.002813716993963285],["leaf",56,-0.014035524944977007],["leaf",57,0.009514188034610318],["leaf",58,0.001077702441722311],["leaf",59,0.008559067442750864],["leaf",60,0.020722165538016143],["leaf",61,-0.0015298402441316149],["leaf",62,0.008931679062070836]],[["split",0,7.0,30,0],["split",1,10.0,8,1],["split",2,3.0,7,1],["split",3,1.0,28,2],["split",4,17.0,0,2],["split",5,25.0,15,2],["split",6,3.0,13,2],["split",7,12.0,28,3],["split",8,3.0,6,3],["split",9,24.0,4,3],["split",10,23.0,27,3],["leaf",11,0.010460720915780663],["leaf",12,-0.006549262109633501],["leaf",13,0.028485159278426053],["leaf",14,0.012346465267926118],["split",15,23.0,28,4],["split",16,1.0,18,4],["leaf",17,-0.024132893039246442],["leaf",18,-0.005907108471730606],["split",19,11.0,12,4],["split",20,4.0,30,4],["split",21,12.0,3,4],["leaf",22,-0.017503678516927714],["leaf",31,-0.0013596996860060413],["leaf",32,0.005456461407056151],["leaf",33,-0.004368158755786663],["leaf",34,-0.020667641096252593],["leaf",39,-0.009593013705410797],["leaf",40,9.535730520283456e-05],["leaf",41,0.0018757244670193922],["leaf",42,-0.0075309659038705935],["leaf",43,0.021971313796762985],["leaf",44,0.005850761454618591]],[["split",0,13.0,1,0],["split",1,12.0,2,1],["split",2,26.0,5,1],["split",3,13.0,0,2],["split",4,5.0,25,2],["split",5,2.0,4,2],["split",6,1.0,13,2],["split",7,21.0,19,3],["split",8,1.0,5,3],["leaf",9,-0.002801466053950135],["split",10,24.0,17,3],["split",11,25.0,9,3],["split",12,1.0,25,3],["split",13,21.0,29,3],["split",14,17.0,0,3],["leaf",15,0.013270723201980411],["leaf",16,-0.0028940115332684205],["leaf",17,0.003088338675039184],["leaf",18,-0.014114160302540523],["leaf",21,0.010276024586696913],["leaf",22,0.029046050635200996],["leaf",23,-0.0014373394846349066],["leaf",24,0.012994985466279831],["split",25,4.0,27,4],["split",26,24.0,3,4],["split",27,7.0,13,4],["split",28,9.0,13,4],["split",29,11.0,23,4],["split",30,0.0,26,4],["leaf",51,-0.0014977775229564076],["leaf",52,-0.013007809479209074],["leaf",53,-0.026741270364397298],["leaf",54,-0.009893550378852835],["leaf",55,-0.003623915360504844],["leaf",56,0.001552328322597074],["leaf",57,-0.018340890437866752],["leaf",58,-0.0032674237795233293],["leaf",59,-0.0002341381489911056],["leaf",60,0.006282931778624259],["leaf",61,0.013368046093971216],["leaf",62,-0.006313792192492221]],[["split",0,9.0,29,0],["split",1,12.0,2,1],["split",2,20.0,0,1],["split",3,19.0,0,2],["split",4,7.0,28,2],["split",5,0.0,26,2],["leaf",6,-0.006028922950064299],["split",7,21.0,17,3],["leaf",8,0.008031341555397814],["split",9,26.0,1,3],["split",10,5.0,22,3],["split",11,6.0,15,3],["leaf",12,0.020949589269686906],["split",15,2.0,27,4],["split",16,4.0,29,4],["split",19,4.0,18,4],["split",20,23.0,19,4],["split",21,9.0,2,4],["leaf",22,0.01910109107654044],["leaf",23,0.016239910697572416],["split",24,11.0,19,4],["leaf",31,-0.007212925761457472],["leaf",32,-0.01831708180024614],["leaf",33,0.0033738920023125307],["leaf",34,-0.012692238531749169],["leaf",39,-0.00040782990695713506],["leaf",40,-0.012512185243241809],["leaf",41,0.0017597502791881865],["leaf",42,-0.0020947131075399862],["leaf",43,-0.007953273105146298],["leaf",44,0.006803667105521388],["leaf",49,0.006546394323989346],["leaf",50,-0.007073130711767863]],[["split",0,1.0,27,0],["split",1,4.0,29,1],["split",2,9.0,2,1],["split",3,5.0,27,2],["split",4,2.0,29,2],["split",5,7.0,23,2],["split",6,12.0,28,2],["split",7,22.0,17,3],["split",8,12.0,14,3],["split",9,6.0,0,3],["split",10,22.0,17,3],["leaf",11,-0.021626047201285466],["leaf",12,-0.006359232038038359],["split",13,10.0,13,3],["split",14,11.0,15,3],["split",15,7.0,23,4],["split",16,3.0,6,4],["split",17,1.0,9,4],["leaf",18,-0.007492631797339223],["leaf",19,0.00907416746769315],["split",20,23.0,18,4],["leaf",21,-0.01940295021402708],["leaf",22,-0.0031746889652096606],["split",27,24.0,19,4],["split",28,22.0,7,4],["leaf",29,0.006334282230722762],["leaf",30,0.015755357495367987],["leaf",31,0.000569466647872105],["leaf",32,0.004726879133925423],["leaf",33,0.003604591868127246],["leaf",34,-0.002476856659081812],["leaf",35,0.004384897199409377],["leaf",36,0.01675208445455029],["leaf",41,0.0032432959706390724],["leaf",42,-0.011259756675907536],["leaf",55,-0.0045259431705219465],["leaf",56,-0.018667296358462836],["leaf",57,-0.008012506874148374],["leaf",58,0.0030423139906423334]],[["split",0,1.0,0,0],["split",1,13.0,3,1],["split",2,9.0,27,1],["split",3,12.0,2,2],["split",4,3.0,14,2],["split",5,7.0,2,2],["split",6,16.0,0,2],["leaf",7,-0.012902516760380185],["leaf",8,0.00984993104299016],["leaf",9,-0.00739024698803251],["leaf",10,-0.02669119203800864],["split",11,22.0,15,3],["split",12,11.0,0,3],["split",13,11.0,29,3],["split",14,4.0,12,3],["split",23,10.0,3,4],["split",24,7.0,1,4],["split",25,4.0,18,4],["split",26,8.0,2,4],["split",27,22.0,16,4],["leaf",28,0.01586169533030012],["leaf",29,0.004694840715462314],["leaf",30,0.020123909948595614],["leaf",47,-0.028591851916679154],["leaf",48,-0.007392345928276875],["leaf",49,0.006088508273126223],["leaf",50,-0.00683528602139864],["leaf",51,-0.0016983342185043267],["leaf",52,-0.018560954199597243],["leaf",53,0.00667387167091029],["leaf",54,-0.0006468940699866052],["leaf",55,0.004476679830698575],["leaf",56,-0.0033646538112289677]],[["split",0,7.0,27,0],["split",1,24.0,5,1],["split",2,4.0,10,1],["split",3,22.0,4,2],["split",4,11.0,21,2],["split",5,19.0,0,2],["split",6,24.0,27,2],["split",7,26.0,8,3],["split",8,13.0,2,3],["split",9,3.0,12,3],["split",10,13.0,13,3],["split",11,15.0,0,3],["leaf",12,-0.00472977862223566],["split",13,9.0,16,3],["leaf",14,0.014025751117624864],["leaf",15,-0.024170536223396277],["leaf",16,-0.007718492479520527],["split",17,4.0,26,4],["split",18,1.0,13,4],["split",19,5.0,20,4],["split",20,16.0,0,4],["split",21,25.0,27,4],["split",22,26.0,4,4],["split",23,12.0,25,4],["leaf",24,0.026491809504978085],["split",27,9.0,12,4],["split",28,2.0,13,4],["leaf",35,-0.005505239337221081],["leaf",36,0.012217457861673553],["leaf",37,-0.010925461338015549],["leaf",38,-0.0008398786462232139],["leaf",39,-0.008741463810876838],["leaf",40,-0.0017724895300150392],["leaf",41,-0.0002163938337019806],["leaf",42,0.00702274821572286],["leaf",43,0.00472812790939363],["leaf",44,0.01788488920779859],["leaf",45,-0.019154524569757598],["leaf",46,0.0012982184414088345],["leaf",47,0.01612271379374597],["leaf",48,0.0062745277198863105],["leaf",55,-0.0002255290997675006],["leaf",56,0.017680145726258867],["leaf",57,0.00010360783084951178],["leaf",58,-0.016887587141384323]],[["split",0,21.0,30,0],["split",1,13.0,2,1],["split",2,7.0,9,1],["split",3,12.0,0,2],["split",4,7.0,2,2],["leaf",5,-0.017373996298069273],["split",6,2.0,13,2],["split",7,24.0,14,3],["split",8,1.0,9,3],["split",9,1.0,5,3],["split",10,1.0,1,3],["leaf",13,-0.01067239013271315],["leaf",14,0.008562522915512261],["leaf",15,0.005625891702752705],["leaf",16,-0.01839753985250798],["split",17,11.0,6,4],["split",18,2.0,14,4],["leaf",19,0.0006409751640266568],["split",20,11.0,13,4],["split",21,2.0,18,4],["split",22,23.0,27,4],["leaf",35,0.027785429871566],["leaf",36,0.009340481715808483],["leaf",37,-0.017701164294094313],["leaf",38,0.007182542638340494],["leaf",41,-0.004414382998304192],["leaf",42,-0.0217877455995],["leaf",43,-0.021315923965096146],["leaf",44,-0.002658033656980172],["leaf",45,0.0005036438901627247],["leaf",46,0.006970260836547391]],[["split",0,6.0,2,0],["split",1,4.0,25,1],["split",2,26.0,2,1],["split",3,7.0,2,2],["split",4,5.0,26,2],["split",5,21.0,29,2],["split",6,8.0,9,2],["split",7,24.0,13,3],["split",8,2.0,19,3],["leaf",9,-0.021031332145137745],["split",10,24.0,3,3],["split",11,20.0,0,3],["leaf",12,-0.019593622539546454],["split",13,26.0,6,3],["split",14,24.0,20,3],["leaf",15,0.01516486058975187],["leaf",16,-0.0037298043498196366],["leaf",17,0.012338659887675671],["leaf",18,0.02680055564540398],["leaf",21,0.01939004916373862],["split",22,24.0,21,4],["split",23,16.0,0,4],["leaf",24,0.006939810010641304],["split",27,22.0,11,4],["split",28,18.0,0,4],["split",29,14.0,0,4],["split",30,10.0,27,4],["leaf",45,-0.006093174787037941],["leaf",46,0.012914034969619839],["leaf",47,-0.010367680928062518],["leaf",48,0.009064521351852303],["leaf",55,-0.01843333834801149],["leaf",56,-0.0037823761872177275],["leaf",57,-0.0028096527750984513],["leaf",58,0.008961382091030621],["leaf",59,-0.0008410218641362097],["leaf",60,0.003832796828661],["leaf",61,0.004385363489409363],["leaf",62,-0.0038912870034578115]],[["split",0,26.0,0,0],["split",1,3.0,6,1],["split",2,7.0,29,1],["split",3,25.0,14,2],["leaf",4,0.0025305109546007263],["split",5,1.0,1,2],["split",6,21.0,14,2],["leaf",7,-0.02428787886949549],["leaf",8,-0.005003030581307569],["split",11,12.0,3,3],["split",12,10.0,5,3],["split",13,3.0,5,3],["split",14,21.0,22,3],["split",23,11.0,12,4],["split",24,24.0,11,4],["split",25,12.0,24,4],["split",26,0.0,29,4],["leaf",27,0.01149043856262378],["split",28,11.0,17,4],["leaf",29,0.02196050227552422],["split",30,0.0,21,4],["leaf",47,0.014873570672899607],["leaf",48,-0.0033986219573798387],["leaf",49,0.00017137036170851834],["leaf",50,-0.017596232284100486],["leaf",51,-0.0011524842312982107],["leaf",52,-0.010576558578305736],["leaf",53,0.00026267436598029903],["leaf",54,0.006364294994191608],["leaf",57,-0.00011617008231186392],["leaf",58,-0.014892348951051439],["leaf",61,-0.0010965088217289596],["leaf",62,0.015092154578710675]],[["split",0,3.0,0,0],["split",1,12.0,5,1],["split",2,21.0,0,1],["leaf",3,0.01982245509651367],["split",4,2.0,16,2],["split",5,7.0,6,2],["split",6,25.0,26,2],["leaf",9,0.011537685893667578],["leaf",10,-0.008556139526539951],["leaf",11,-0.018524487883060235],["split",12,3.0,8,3],["split",13,2.0,30,3],["split",14,20.0,0,3],["leaf",25,0.004248926678557993],["leaf",26,-0.009939252930094244],["split",27,12.0,12,4],["split",28,11.0,16,4],["split",29,1.0,10,4],["split",30,23.0,18,4],["leaf",55,0.0023277980807679395],["leaf",56,-0.0007318340896409419],["leaf",57,0.0007665049486885641],["leaf",58,-0.015519128445936597],["leaf",59,-0.004299018762008718],["leaf",60,0.005753235332962382],["leaf",61,0.006533720710023525],["leaf",62,0.023720842786416573]],[["split",0,7.0,25,0],["split",1,13.0,7,1],["split",2,9.0,10,1],["split",3,2.0,29,2],["split",4,3.0,10,2],["split",5,26.0,4,2],["split",6,24.0,1,2],["split",7,10.0,28,3],["split",8,7.0,1,3],["split",9,9.0,20,3],["split",10,26.0,1,3],["leaf",11,0.008315159090443598],["split",12,23.0,27,3],["leaf",13,-0.008349284322193031],["split",14,22.0,15,3],["split",15,0.0,5,4],["split",16,26.0,7,4],["split",17,9.0,9,4],["split",18,21.0,12,4],["split",19,2.0,8,4],["split",20,3.0,7,4],["leaf",21,-0.014602052415226495],["split",22,22.0,28,4],["split",25,6.0,25,4],["leaf",26,-0.014042791889519101],["split",29,5.0,18,4],["split",30,14.0,0,4],["leaf",31,0.011940357926504169],["leaf",32,0.003191424966367534],["leaf",33,-0.014258183178470276],["leaf",34,0.0004220268983038438],["leaf",35,-0.00924465459482546],["leaf",36,-0.017433378772326893],["leaf",37,-0.00918976520256683],["leaf",38,0.004502413119173007],["leaf",39,0.005183314518153652],["leaf",40,-0.00618422135012213],["leaf",41,-0.018303421025673165],["leaf",42,-0.00282637127209041],["leaf",45,0.0005176992570149354],["leaf",46,-0.006350962667909872],["leaf",51,-0.0073671731234080245],["leaf",52,0.002799931021871517],["leaf",59,0.014789360348063937],["leaf",60,-0.0007694592773921725],["leaf",61,0.0009656152180422507],["leaf",62,0.02130563176694664]],[["split",0,4.0,0,0],["split",1,11.0,21,1],["split",2,23.0,30,1],["split",3,23.0,20,2],["leaf",4,0.020850665937419605],["split",5,1.0,4,2],["split",6,26.0,12,2],["split",7,11.0,11,3],["leaf",8,0.016158185594992613],["split",11,17.0,0,3],["split",12,21.0,0,3],["leaf",13,0.007941706543421932],["split",14,2.0,23,3],["leaf",15,0.013386602299127378],["leaf",16,-0.008378500298488075],["split",23,10.0,13,4],["split",24,10.0,13,4],["split",25,10.0,9,4],["split",26,25.0,29,4],["leaf",29,-0.023141779674449574],["leaf",30,-0.0023202803132444006],["leaf",47,0.01084846509717377],["leaf",48,0.0007424827038008467],["leaf",49,-0.01979853468039037],["leaf",50,-0.001251285663375845],["leaf",51,-0.01812323390529831],["leaf",52,-0.0037329824215413246],["leaf",53,-1.0849395749708614e-05],["leaf",54,0.0064251296041953326]],[["split",0,22.0,11,0],["split",1,17.0,0,1],["split",2,18.0,0,1],["split",3,21.0,0,2],["split",4,2.0,16,2],["split",5,9.0,24,2],["split",6,0.0,15,2],["leaf",7,-0.008458496004595159],["split",8,7.0,29,3],["split",9,12.0,12,3],["split",10,10.0,6,3],["split",11,13.0,4,3],["split",12,8.0,25,3],["split",13,25.0,10,3],["split",14,1.0,18,3],["split",17,1.0,28,4],["split",18,26.0,10,4],["leaf",19,0.01772137657838353],["split",20,13.0,23,4],["leaf",21,-0.023350619901652946],["split",22,9.0,16,4],["split",23,1.0,6,4],["split",24,5.0,29,4],["split",25,10.0,13,4],["split",26,1.0,12,4],["leaf",27,-0.00017739339541025578],["split",28,4.0,12,4],["split",29,10.0,20,4],["split",30,26.0,16,4],["leaf",35,0.0037334663828056413],["leaf",36,-0.004714579364326183],["leaf",37,0.02086640235739807],["leaf",38,0.0011587972426063912],["leaf",41,-0.012378297108294384],["leaf",42,0.009822904147546937],["leaf",45,-0.010875713262496069],["leaf",46,0.0023339714554829476],["leaf",47,0.009834058613055746],["leaf",48,-0.0024332311498748856],["leaf",49,-0.0012784353840154259],["leaf",50,-0.014675925463092515],["leaf",51,-0.013857041220514586],["leaf",52,-0.004682025790110962],["leaf",53,-0.007595041815842564],["leaf",54,0.007817399013364902],["leaf",57,0.0034639675603801707],["leaf",58,0.01855974462626542],["leaf",59,-0.004434196336099701],["leaf",60,-0.016783380402547624],["leaf",61,0.01620701480977921],["leaf",62,-0.0022744098467874694]],[["split",0,10.0,29,0],["split",1,22.0,2,1],["split",2,6.0,17,1],["split",3,25.0,27,2],["split",4,26.0,27,2],["split",5,8.0,4,2],["split",6,25.0,18,2],["split",7,18.0,0,3],["leaf",8,0.007912774069867174],["split",9,7.0,18,3],["split",10,5.0,0,3],["leaf",11,-0.015300629635634508],["split",12,24.0,12,3],["split",13,11.0,12,3],["leaf",14,-0.020904529729585394],["split",15,1.0,8,4],["leaf",16,-0.019034470356374043],["split",19,10.0,17,4],["split",20,11.0,26,4],["leaf",21,0.018308243738513467],["split",22,0.0,20,4],["leaf",25,-0.004508027634011232],["leaf",26,0.009080459982545995],["leaf",27,0.002531631956848128],["leaf",28,-0.016562578967941808],["leaf",31,-0.011853250456188754],["leaf",32,-0.0019034578507231008],["leaf",39,-0.0028181796130867295],["leaf",40,0.0014514103843867138],["leaf",41,0.001030604942292795],["leaf",42,0.00860799050492271],["leaf",45,0.0003301891858966132],["leaf",46,0.008042356126950827]],[["split",0,0.0,29,0],["split",1,2.0,27,1],["split",2,17.0,0,1],["split",3,8.0,11,2],["split",4,7.0,2,2],["split",5,2.0,11,2],["leaf",6,-0.01918878537204962],["split",7,19.0,0,3],["split",8,7.0,15,3],["split",9,26.0,6,3],["split",10,23.0,26,3],["split",11,7.0,22,3],["split",12,7.0,21,3],["split",15,3.0,20,4],["split",16,24.0,25,4],["split",17,23.0,24,4],["split",18,21.0,28,4],["leaf",19,-0.021089904405745086],["split",20,23.0,29,4],["split",21,25.0,10,4],["split",22,7.0,11,4],["leaf",23,-0.02126715551145405],["leaf",24,0.0015161033081849164],["split",25,8.0,8,4],["leaf",26,-0.010090102249156588],["leaf",31,0.0011637725645515344],["leaf",32,0.009033118180563085],["leaf",33,0.01386673015359992],["leaf",34,-0.004787630890140678],["leaf",35,-0.003730745381103144],["leaf",36,-0.015074275815274343],["leaf",37,0.001502233562202235],["leaf",38,-0.007676851361075014],["leaf",41,-0.013019129119668738],["leaf",42,0.003339896593629999],["leaf",43,-0.012649271227021634],["leaf",44,0.0017452189400882904],["leaf",45,0.0030371113899151127],["leaf",46,0.01354382231627276],["leaf",51,-0.0014678592858784853],["leaf",52,0.01450224447296919]],[["split",0,9.0,15,0],["split",1,24.0,17,1],["split",2,3.0,0,1],["split",3,23.0,25,2],["split",4,13.0,1,2],["split",5,9.0,22,2],["split",6,12.0,10,2],["split",7,20.0,0,3],["split",8,20.0,0,3],["leaf",9,0.012102061862998037],["split",10,0.0,27,3],["leaf",11,0.0012330409097710115],["leaf",12,0.01955409676667583],["split",13,13.0,0,3],["split",14,13.0,14,3],["split",15,7.0,6,4],["split",16,1.0,20,4],["split",17,17.0,0,4],["leaf",18,0.005948894695839199],["split",21,25.0,19,4],["split",22,26.0,9,4],["leaf",27,0.011049993197212523],["split",28,7.0,1,4],["split",29,7.0,13,4],["split",30,15.0,0,4],["leaf",31,-0.0058766012762058424],["leaf",32,0.0009317476903729506],["leaf",33,-0.00020054475602115406],["leaf",34,-0.021724568550711722],["leaf",35,-0.00693837850554236],["leaf",36,-0.025154075155082462],["leaf",43,-0.0013623336245393298],["leaf",44,0.006196306958333845],["leaf",45,-0.022614017105004883],["leaf",46,-0.0020504540503464403],["leaf",57,-0.012740850912239025],["leaf",58,-0.0015147640679624883],["leaf",59,0.003001528078255335],["leaf",60,0.013815909714971273],["leaf",61,0.00024130795836901434],["leaf",62,0.009634471297936351]],[["split",0,10.0,4,0],["split",1,17.0,0,1],["split",2,0.0,6,1],["split",3,7.0,0,2],["split",4,3.0,18,2],["split",5,2.0,21,2],["split",6,10.0,29,2],["leaf",7,-0.014502996809730733],["split",8,5.0,25,3],["split",9,23.0,23,3],["leaf",10,-0.00032747821450804644],["split",11,3.0,4,3],["split",12,12.0,15,3],["split",13,12.0,0,3],["split",14,6.0,5,3],["split",17,21.0,4,4],["split",18,24.0,11,4],["leaf",19,-0.008718669853346447],["leaf",20,-0.03251679911468255],["split",23,13.0,5,4],["split",24,24.0,18,4],["split",25,3.0,6,4],["leaf",26,-0.018047798776708906],["leaf",27,0.012085309209431037],["split",28,7.0,0,4],["leaf",29,-0.02135793050012215],["split",30,24.0,6,4],["leaf",35,-0.013042101324889453],["leaf",36,-0.000664730881859344],["leaf",37,0.01861665162811988],["leaf",38,0.004436463615070449],["leaf",47,0.021892053231137958],["leaf",48,0.0022829339653927987],["leaf",49,-0.0048510818493393395],["leaf",50,0.00507564399480923],["leaf",51,0.0008857730807110615],["leaf",52,-0.008060816623140229],["leaf",57,-0.011641393186112476],["leaf",58,0.0018028954918680481],["leaf",61,-0.012399618580339208],["leaf",62,0.0009394775096815391]],[["split",0,24.0,3,0],["split",1,12.0,1,1],["split",2,20.0,0,1],["leaf",3,-0.020753779839346672],["split",4,10.0,12,2],["split",5,2.0,9,2],["split",6,21.0,17,2],["split",9,12.0,14,3],["split",10,6.0,14,3],["split",11,5.0,20,3],["split",12,23.0,21,3],["split",13,23.0,27,3],["split",14,2.0,20,3],["split",19,9.0,13,4],["split",20,23.0,16,4],["split",21,4.0,22,4],["split",22,23.0,21,4],["split",23,5.0,18,4],["split",24,26.0,15,4],["split",25,7.0,13,4],["split",26,13.0,0,4],["split",27,25.0,9,4],["leaf",28,0.025640150313883173],["split",29,24.0,16,4],["split",30,3.0,18,4],["leaf",39,-0.010490452568893305],["leaf",40,0.005981131705373251],["leaf",41,-0.0040145618686177435],["leaf",42,-0.018217034592221577],["leaf",43,0.013678740241617738],["leaf",44,0.0007892448792959679],["leaf",45,0.0020310215354995245],["leaf",46,-0.016566459162717165],["leaf",47,0.003084763888874927],["leaf",48,-0.010403113920187534],["leaf",49,0.018064298286147036],["leaf",50,0.003576005966480646],["leaf",51,-0.0012397795910132673],["leaf",52,0.0032527450067987972],["leaf",53,0.009494020199786023],["leaf",54,-0.00576661589143284],["leaf",55,-0.015152423936766218],["leaf",56,0.0020872072017804467],["leaf",59,-0.007018491914624375],["leaf",60,0.010425253818345248],["leaf",61,0.03778567557559661],["leaf",62,0.015736753064526713]],[["split",0,7.0,12,0],["split",1,3.0,29,1],["split",2,11.0,23,1],["split",3,2.0,1,2],["split",4,22.0,13,2],["split",5,12.0,4,2],["split",6,20.0,0,2],["leaf",7,0.01131792003294077],["split",8,1.0,7,3],["leaf",9,0.01693338066660009],["split",10,13.0,13,3],["leaf",11,-0.009942083203453985],["split",12,11.0,19,3],["split",13,23.0,28,3],["split",14,2.0,9,3],["split",17,9.0,14,4],["split",18,13.0,10,4],["leaf",21,0.0037447293461098418],["leaf",22,-0.010785361523808245],["split",25,26.0,0,4],["split",26,10.0,16,4],["split",27,12.0,9,4],["leaf",28,-0.014430613218254533],["leaf",29,0.002227095930319406],["leaf",30,0.026001543695599115],["leaf",35,-0.0018388083471158304],["leaf",36,-0.010451308377165267],["leaf",37,0.000640937995947926],["leaf",38,-0.005680636103065193],["leaf",51,-0.010225504940015635],["leaf",52,0.002366369876499784],["leaf",53,0.0021579251784665182],["leaf",54,-0.008678277216506648],["leaf",55,0.015425651995208942],["leaf",56,0.0020125598920164053]],[["split",0,23.0,30,0],["split",1,6.0,22,1],["split",2,26.0,18,1],["split",3,7.0,15,2],["split",4,3.0,2,2],["leaf",5,-0.015057788925772881],["split",6,11.0,15,2],["split",7,12.0,24,3],["split",8,5.0,23,3],["leaf",9,-0.013417753524144937],["split",10,7.0,29,3],["leaf",13,-0.008728757639148915],["leaf",14,0.009074172566594414],["split",15,26.0,28,4],["split",16,5.0,3,4],["split",17,1.0,2,4],["split",18,23.0,9,4],["split",21,13.0,30,4],["split",22,22.0,5,4],["leaf",31,0.00015752474397347937],["leaf",32,0.008165177159828289],["leaf",33,0.004718335464532423],["leaf",34,-0.013685259782886624],["leaf",35,0.018331212686931008],["leaf",36,0.0048962947130430775],["leaf",37,-0.014418910347611216],["leaf",38,0.0001005963865000438],["leaf",43,-0.0031030872926646872],["leaf",44,0.0059803716186448],["leaf",45,-0.00777339736124921],["leaf",46,0.009323556139675917]],[["split",0,7.0,27,0],["split",1,13.0,6,1],["split",2,2.0,4,1],["split",3,3.0,23,2],["split",4,17.0,0,2],["split",5,25.0,13,2],["split",6,12.0,30,2],["split",7,20.0,0,3],["leaf",8,0.012766593561357563],["split",9,7.0,3,3],["split",10,23.0,26,3],["leaf",11,0.004162885253078563],["split",12,9.0,15,3],["split",13,6.0,20,3],["leaf",14,-0.011150775313147667],["split",15,4.0,15,4],["split",16,10.0,16,4],["leaf",19,-0.013966983107786012],["split",20,1.0,19,4],["split",21,11.0,27,4],["leaf",22,-0.018066723958809375],["leaf",25,0.02260318951283918],["leaf",26,0.0103980525215588],["leaf",27,0.014659860499594815],["split",28,3.0,4,4],["leaf",31,0.007832516523291776],["leaf",32,-0.0011192530244889122],["leaf",33,0.022524662857086663],["leaf",34,-0.009801201991456402],["leaf",41,-0.0012861329882770088],["leaf",42,-0.004618829700450779],["leaf",43,0.007401146933723066],["leaf",44,-0.009916214498020503],["leaf",57,-0.006639100494027458],["leaf",58,0.004894539121524461]],[["split",0,7.0,30,0],["split",1,0.0,15,1],["split",2,4.0,9,1],["split",3,23.0,30,2],["split",4,1.0,30,2],["leaf",5,-0.003719992570099311],["split",6,24.0,16,2],["split",7,6.0,0,3],["leaf",8,0.00885921512005893],["split",9,5.0,8,3],["leaf",10,-0.010462038593641941],["leaf",13,0.024471085416289298],["leaf",14,0.0072337429392983574],["leaf",15,0.00803520856381444],["split",16,23.0,19,4],["split",19,3.0,12,4],["split",20,26.0,30,4],["leaf",33,-0.0015583004353015868],["leaf",34,-0.005514480191598589],["leaf",39,-0.016509095327937963],["leaf",40,-0.0009915844155950876],["leaf",41,0.0017691234124361055],["leaf",42,0.014335655845435145]],[["split",0,21.0,3,0],["split",1,7.0,27,1],["split",2,23.0,0,1],["split",3,2.0,14,2],["split",4,4.0,15,2],["split",5,22.0,7,2],["split",6,12.0,0,2],["split",7,22.0,5,3],["split",8,3.0,4,3],["leaf",9,-0.005967614159434203],["leaf",10,-0.020149180551167153],["leaf",11,0.01681782922221705],["split",12,22.0,19,3],["split",13,23.0,20,3],["split",14,21.0,27,3],["leaf",15,-0.016137927284609634],["split",16,4.0,15,4],["split",17,1.0,10,4],["split",18,24.0,10,4],["leaf",25,-0.004342624057225464],["leaf",26,0.009995426423062979],["leaf",27,-0.003838663695032309],["leaf",28,0.013531528805143541],["split",29,25.0,4,4],["split",30,20.0,0,4],["leaf",33,-0.010815962804817674],["leaf",34,0.0019308379916477953],["leaf",35,-0.01297053419519564],["leaf",36,0.0031139067168486317],["leaf",37,-0.0009546643889080755],["leaf",38,0.010840964825607752],["leaf",59,-0.002619015122447513],["leaf",60,0.0011767814039196091],["leaf",61,-0.004967421295296748],["leaf",62,0.011436649251713829]],[["split",0,7.0,25,0],["split",1,21.0,17,1],["split",2,1.0,15,1],["split",3,25.0,27,2],["split",4,5.0,30,2],["split",5,2.0,24,2],["split",6,25.0,7,2],["split",7,0.0,1,3],["split",8,0.0,18,3],["split",9,4.0,27,3],["leaf",10,0.009638091250659325],["split",11,9.0,7,3],["leaf",12,0.010280066576131522],["split",13,6.0,26,3],["split",14,21.0,10,3],["split",15,26.0,2,4],["split",16,22.0,24,4],["split",17,11.0,10,4],["split",18,21.0,10,4],["split",19,3.0,1,4],["split",20,23.0,27,4],["split",23,23.0,20,4],["split",24,22.0,6,4],["split",27,2.0,10,4],["leaf",28,0.005274910663721451],["split",29,22.0,17,4],["split",30,2.0,2,4],["leaf",31,0.013031281544464597],["leaf",32,0.00023592990165872285],["leaf",33,-0.0011528710139701562],["leaf",34,-0.0061938675710931455],["leaf",35,0.004290252750321457],["leaf",36,-0.01032440687366947],["leaf",37,-0.02247255633443079],["leaf",38,-0.007155012733896621],["leaf",39,-0.009460908936128408],["leaf",40,0.0009944776663675043],["leaf",41,8.71969238661432e-05],["leaf",42,-0.023317433219218253],["leaf",47,-0.003768755794334091],["leaf",48,-0.014800126310357297],["leaf",49,0.009236897937058864],["leaf",50,-0.0024475317601401762],["leaf",55,0.00542161148529849],["leaf",56,-0.023612338241825508],["leaf",59,0.020226622470798126],["leaf",60,0.006799588860660103],["leaf",61,-0.007208205045996935],["leaf",62,0.00694210866206403]],[["split",0,7.0,30,0],["split",1,8.0,20,1],["split",2,10.0,8,1],["split",3,7.0,8,2],["split",4,3.0,14,2],["leaf",5,0.015971207772148485],["split",6,6.0,25,2],["split",7,9.0,1,3],["split",8,3.0,21,3],["split",9,21.0,10,3],["split",10,17.0,0,3],["leaf",13,-0.0076342678467654634],["leaf",14,0.01063809558196336],["split",15,7.0,2,4],["split",16,25.0,30,4],["split",17,9.0,25,4],["split",18,24.0,27,4],["split",19,25.0,17,4],["split",20,3.0,7,4],["split",21,2.0,14,4],["split",22,26.0,14,4],["leaf",31,0.022380720052587993],["leaf",32,0.0014087581401425909],["leaf",33,-0.0037860417098627124],["leaf",34,0.014904621993798963],["leaf",35,-0.0008657730736738043],["leaf",36,0.006228774910936231],["leaf",37,0.00642674965200382],["leaf",38,-0.007690018670587571],["leaf",39,0.002873391588299459],["leaf",40,-0.011240735755652414],["leaf",41,-0.005000465032947829],["leaf",42,-0.015845652883578917],["leaf",43,-0.005040705018170222],["leaf",44,0.0020101075306474103],["leaf",45,-0.0051311913101349],["leaf",46,0.012101069300665996]],[["split",0,23.0,27,0],["split",1,7.0,17,1],["split",2,17.0,0,1],["split",3,13.0,20,2],["split",4,22.0,20,2],["split",5,20.0,0,2],["split",6,4.0,21,2],["split",7,21.0,28,3],["split",8,3.0,28,3],["split",9,8.0,13,3],["split",10,3.0,8,3],["split",11,16.0,0,3],["split",12,1.0,7,3],["leaf",13,-0.014892791882722049],["leaf",14,-0.030555289399000225],["split",15,18.0,0,4],["split",16,22.0,23,4],["split",17,12.0,19,4],["leaf",18,0.0017459535294175818],["split",19,4.0,21,4],["split",20,1.0,3,4],["split",21,11.0,23,4],["split",22,6.0,27,4],["split",23,1.0,0,4],["split",24,22.0,18,4],["leaf",25,-0.0016486189469130308],["leaf",26,0.022236395772628297],["leaf",31,-0.0005774859490158085],["leaf",32,0.00528823974289572],["leaf",33,-0.012497946410942075],["leaf",34,0.004685598268718258],["leaf",35,-0.01752710276316954],["leaf",36,-0.006370310294197173],["leaf",39,0.0046242590558380646],["leaf",40,0.021978699512231924],["leaf",41,-0.007869966022067524],["leaf",42,0.003133534112013213],["leaf",43,-0.012346374513321209],["leaf",44,0.004322019714556604],["leaf",45,0.004421050239759507],["leaf",46,-0.004988026911658077],["leaf",47,0.0064921506590788465],["leaf",48,-0.00791732052664279],["leaf",49,0.012386007032193232],["leaf",50,-0.0017920649872226566]],[["split",0,4.0,27,0],["split",1,13.0,1,1],["split",2,9.0,26,1],["split",3,24.0,22,2],["split",4,0.0,18,2],["split",5,3.0,3,2],["leaf",6,-0.014418972509154228],["split",7,0.0,12,3],["leaf",8,0.01944961669659608],["split",9,9.0,4,3],["split",10,24.0,24,3],["split",11,2.0,23,3],["split",12,13.0,1,3],["leaf",15,0.010107777175122769],["leaf",16,-0.00046721410681214374],["split",19,25.0,20,4],["split",20,12.0,26,4],["split",21,23.0,26,4],["split",22,25.0,25,4],["split",23,12.0,5,4],["split",24,1.0,5,4],["leaf",25,0.003045903950037654],["split",26,6.0,2,4],["leaf",39,-0.0027587740538780295],["leaf",40,-0.012371786688066351],["leaf",41,0.0006268729300195893],["leaf",42,-0.0048620724192582655],["leaf",43,0.0018959590685235044],["leaf",44,-0.0066560081296688604],["leaf",45,0.0027291051834713543],["leaf",46,0.016825695572604497],["leaf",47,0.018789357908849044],["leaf",48,-0.0026421160869065765],["leaf",49,0.002139910804754503],["leaf",50,-0.00916703034412357],["leaf",53,-0.02246367425450168],["leaf",54,-0.005633625857120423]],[["split",0,13.0,11,0],["split",1,7.0,3,1],["split",2,8.0,22,1],["split",3,4.0,22,2],["split",4,8.0,7,2],["split",5,4.0,20,2],["split",6,4.0,7,2],["split",7,3.0,25,3],["split",8,7.0,1,3],["split",9,7.0,9,3],["split",10,7.0,13,3],["split",11,8.0,17,3],["split",12,21.0,6,3],["split",13,20.0,0,3],["split",14,12.0,19,3],["split",15,10.0,10,4],["leaf",16,-0.023058286914582497],["split",17,10.0,11,4],["split",18,24.0,17,4],["split",19,4.0,11,4],["split",20,4.0,24,4],["split",21,20.0,0,4],["split",22,25.0,21,4],["split",23,5.0,15,4],["split",24,3.0,22,4],["split",25,25.0,13,4],["split",26,7.0,10,4],["split",27,19.0,0,4],["split",28,21.0,14,4],["split",29,2.0,8,4],["split",30,7.0,29,4],["leaf",31,-0.06525363105899949],["leaf",32,-0.03964500686960935],["leaf",35,-0.09543436447116475],["leaf",36,-0.07575926223411684],["leaf",37,-0.07155030606163977],["leaf",38,-0.05340872155144876],["leaf",39,-0.0174613271537114],["leaf",40,-0.04295688142041447],["leaf",41,-0.011325733471468648],["leaf",42,-0.030749583233800906],["leaf",43,-0.025094894321821933],["leaf",44,0.000705861079424434],["leaf",45,-0.0076307785418806996],["leaf",46,0.00755425790654175],["leaf",47,0.012209819542381487],["leaf",48,-0.0005091696262634082],["leaf",49,0.017974964793678275],["leaf",50,0.032337585350971386],["leaf",51,-0.03246456890073172],["leaf",52,-0.013932741147499865],["leaf",53,-0.02036609121187531],["leaf",54,0.0008273427806072001],["leaf",55,0.06203417290247347],["leaf",56,0.03527131980498691],["leaf",57,0.008276245565573766],["leaf",58,0.04290799709301519],["leaf",59,0.03766554330438102],["leaf",60,0.016169319848200324],["leaf",61,0.03575013464562202],["leaf",62,0.04955770623832695]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,8.0,24,1],["split",3,4.0,27,2],["split",4,4.0,14,2],["split",5,4.0,12,2],["split",6,4.0,7,2],["split",7,4.0,15,3],["split",8,7.0,0,3],["split",9,10.0,4,3],["split",10,12.0,3,3],["split",11,12.0,15,3],["split",12,8.0,19,3],["split",13,20.0,0,3],["split",14,5.0,21,3],["leaf",15,-0.0326005591082501],["split",16,23.0,11,4],["split",17,26.0,8,4],["split",18,0.0,23,4],["leaf",19,-0.02653435442844807],["split",20,7.0,8,4],["split",21,7.0,6,4],["split",22,7.0,8,4],["split",23,8.0,21,4],["split",24,20.0,0,4],["split",25,4.0,26,4],["split",26,10.0,8,4],["split",27,12.0,22,4],["split",28,21.0,15,4],["split",29,21.0,7,4],["leaf",30,0.011386256949541542],["leaf",33,-0.04337270086122561],["leaf",34,-0.0637483102239699],["leaf",35,-0.08231615431884921],["leaf",36,-0.09957415323652363],["leaf",37,-0.0740856764708302],["leaf",38,-0.0483176634928082],["leaf",41,-0.011964177502024914],["leaf",42,0.007197674157683334],["leaf",43,-0.0461202277273605],["leaf",44,-0.030383055611029105],["leaf",45,-0.03311989067949963],["leaf",46,-0.015647513153970033],["leaf",47,0.004380066621322131],["leaf",48,0.02816364760499217],["leaf",49,0.028922114687385463],["leaf",50,0.007888623388400864],["leaf",51,0.0005361698044605026],["leaf",52,-0.018148810676730586],["leaf",53,0.0033662178053343594],["leaf",54,0.01985111327752644],["leaf",55,0.04169537970444647],["leaf",56,0.06497700459594466],["leaf",57,0.011826276573888208],["leaf",58,0.039088350769895495],["leaf",59,0.02780943901056024],["leaf",60,0.04174142484211375]],[["split",0,13.0,11,0],["split",1,6.0,5,1],["split",2,13.0,19,1],["split",3,4.0,30,2],["split",4,7.0,4,2],["split",5,4.0,9,2],["split",6,8.0,27,2],["split",7,7.0,2,3],["split",8,7.0,0,3],["split",9,2.0,28,3],["split",10,4.0,20,3],["split",11,8.0,16,3],["split",12,8.0,16,3],["split",13,4.0,19,3],["split",14,4.0,9,3],["split",15,20.0,0,4],["split",16,7.0,8,4],["leaf",17,-0.09354180855401788],["leaf",18,-0.06396295559956407],["split",19,4.0,16,4],["leaf",20,-0.06441253338900292],["split",21,21.0,4,4],["split",22,7.0,6,4],["split",23,5.0,2,4],["split",24,2.0,16,4],["split",25,26.0,19,4],["split",26,12.0,10,4],["split",27,20.0,0,4],["split",28,21.0,11,4],["split",29,13.0,30,4],["split",30,2.0,13,4],["leaf",31,-0.06969676913986016],["leaf",32,-0.03359272221517668],["leaf",33,-0.038441391580234345],["leaf",34,-0.016435801248616762],["leaf",39,-0.01759152539206812],["leaf",40,-0.04489709365726822],["leaf",43,-0.026651474349557877],["leaf",44,-0.008247542205632943],["leaf",45,-0.03861203582586169],["leaf",46,-0.019573542664779182],["leaf",47,0.020865176972424494],["leaf",48,0.0055650572002766486],["leaf",49,0.027373220342362056],["leaf",50,0.009870116459244965],["leaf",51,-0.010590842975898084],["leaf",52,0.0042483240307623245],["leaf",53,-0.009126595869400586],["leaf",54,0.011604218346055814],["leaf",55,0.03272053377617156],["leaf",56,0.010644823724279858],["leaf",57,-0.004149401657919042],["leaf",58,0.013495541262022239],["leaf",59,0.04848718527334284],["leaf",60,0.062467694181139406],["leaf",61,0.04156296447271907],["leaf",62,0.0283995613343638]],[["split",0,6.0,10,0],["split",1,7.0,2,1],["split",2,8.0,22,1],["split",3,20.0,0,2],["split",4,10.0,9,2],["split",5,4.0,15,2],["split",6,4.0,8,2],["split",7,10.0,6,3],["leaf",8,-0.03945602623047338],["split",9,7.0,6,3],["split",10,7.0,10,3],["split",11,6.0,17,3],["split",12,8.0,14,3],["split",13,8.0,29,3],["split",14,1.0,4,3],["split",15,9.0,3,4],["split",16,2.0,26,4],["split",19,23.0,14,4],["split",20,17.0,0,4],["split",21,4.0,25,4],["split",22,21.0,13,4],["split",23,7.0,7,4],["split",24,3.0,8,4],["split",25,23.0,14,4],["split",26,7.0,25,4],["split",27,20.0,0,4],["split",28,12.0,28,4],["leaf",29,0.003357771122567926],["split",30,25.0,3,4],["leaf",31,-0.06301993781314179],["leaf",32,-0.09096026626264529],["leaf",33,-0.05230330382635181],["leaf",34,-0.0713098796134643],["leaf",39,-0.030430119203683017],["leaf",40,-0.04928131001227622],["leaf",41,-0.01689518315762813],["leaf",42,-0.04483210351504347],["leaf",43,-0.01921882752654669],["leaf",44,-0.03404214701486734],["leaf",45,-0.014440229443548258],["leaf",46,0.0008287045528873228],["leaf",47,-0.010416254927384837],["leaf",48,0.010237224699895409],["leaf",49,0.001193938365186886],["leaf",50,0.023211555948738073],["leaf",51,-0.0018644133665058945],["leaf",52,-0.01637324147859676],["leaf",53,-0.00034699058436238637],["leaf",54,0.012794140863755788],["leaf",55,0.04178261951231663],["leaf",56,0.013175504686454202],["leaf",57,0.05042026316309016],["leaf",58,0.06418581907115802],["leaf",61,0.01089670848598499],["leaf",62,0.029767204360261423]],[["split",0,6.0,12,0],["split",1,7.0,3,1],["split",2,8.0,23,1],["split",3,20.0,0,2],["split",4,7.0,5,2],["split",5,4.0,13,2],["split",6,5.0,3,2],["split",7,3.0,26,3],["split",8,0.0,15,3],["split",9,4.0,24,3],["split",10,4.0,15,3],["split",11,20.0,0,3],["split",12,25.0,3,3],["split",13,20.0,0,3],["split",14,4.0,16,3],["split",15,10.0,6,4],["leaf",16,-0.018404704724389646],["leaf",17,-0.04383726041542662],["leaf",18,-0.006764662870323892],["split",19,11.0,17,4],["split",20,10.0,17,4],["split",21,25.0,23,4],["split",22,7.0,14,4],["split",23,2.0,6,4],["split",24,25.0,19,4],["split",25,21.0,13,4],["split",26,7.0,14,4],["split",27,19.0,0,4],["leaf",28,0.02111146482137003],["split",29,20.0,0,4],["split",30,23.0,23,4],["leaf",31,-0.08081165801797677],["leaf",32,-0.05422321112339086],["leaf",39,-0.032892685097205936],["leaf",40,-0.01676440854509134],["leaf",41,-0.04490036024583092],["leaf",42,-0.030773781279050668],["leaf",43,-0.009721671648879713],["leaf",44,0.007698199725139161],["leaf",45,-0.022601139673773248],["leaf",46,-0.010565244171862513],["leaf",47,0.028022519000622685],["leaf",48,0.014067927504010898],["leaf",49,-0.0119401069903858],["leaf",50,0.021977014119608566],["leaf",51,-0.025642930786782814],["leaf",52,-0.004840667316207444],["leaf",53,-0.0044389219341342415],["leaf",54,0.007673229011158494],["leaf",55,0.05421612930234802],["leaf",56,0.02905157578517403],["leaf",59,0.03634389712372295],["leaf",60,0.014186159158760975],["leaf",61,0.02314874558639621],["leaf",62,0.0052714214669730974]],[["split",0,13.0,12,0],["split",1,13.0,7,1],["split",2,13.0,24,1],["split",3,7.0,2,2],["split",4,4.0,20,2],["split",5,5.0,12,2],["split",6,8.0,29,2],["split",7,7.0,0,3],["split",8,7.0,8,3],["split",9,7.0,4,3],["split",10,2.0,29,3],["split",11,6.0,17,3],["split",12,6.0,16,3],["split",13,20.0,0,3],["split",14,20.0,0,3],["split",15,10.0,10,4],["split",16,20.0,0,4],["split",17,10.0,12,4],["split",18,0.0,4,4],["leaf",19,-0.02704029046743477],["split",20,12.0,10,4],["split",21,26.0,3,4],["leaf",22,-0.041120836502532884],["split",23,26.0,19,4],["split",24,20.0,0,4],["split",25,4.0,20,4],["split",26,21.0,7,4],["split",27,4.0,16,4],["split",28,3.0,15,4],["split",29,1.0,18,4],["leaf",30,0.02378446483798031],["leaf",31,-0.07932274708008223],["leaf",32,-0.05604296672773032],["leaf",33,-0.05220516865354442],["leaf",34,-0.024735546172895657],["leaf",35,-0.042892923018767154],["leaf",36,-0.02717788735758821],["leaf",37,-0.028263354588322877],["leaf",38,-0.010812733930360391],["leaf",41,-0.012319488223319271],["leaf",42,0.0009395700997930214],["leaf",43,-0.0354969750354315],["leaf",44,-0.015285839451440852],["leaf",47,0.0032543079324279973],["leaf",48,0.014963446638507035],["leaf",49,0.024868323838888803],["leaf",50,0.008075916807511274],["leaf",51,0.000416808865090822],["leaf",52,-0.014063433351790626],["leaf",53,-0.005346210405986205],["leaf",54,0.011967600735229775],["leaf",55,0.0370491027110959],["leaf",56,0.01578195300311598],["leaf",57,0.03319080385138903],["leaf",58,0.0016224197530017544],["leaf",59,0.03915244952745201],["leaf",60,0.058463300810791666]],[["split",0,6.0,11,0],["split",1,7.0,4,1],["split",2,8.0,21,1],["split",3,7.0,0,2],["split",4,4.0,12,2],["split",5,5.0,14,2],["split",6,12.0,21,2],["split",7,10.0,13,3],["split",8,5.0,15,3],["split",9,7.0,10,3],["split",10,21.0,6,3],["split",11,12.0,17,3],["split",12,4.0,26,3],["split",13,24.0,3,3],["split",14,2.0,11,3],["split",15,3.0,5,4],["leaf",16,-0.05439337836375228],["split",17,10.0,11,4],["split",18,20.0,0,4],["split",19,26.0,15,4],["split",20,22.0,12,4],["split",21,7.0,15,4],["split",22,20.0,0,4],["split",23,20.0,0,4],["split",24,10.0,5,4],["split",25,8.0,15,4],["split",26,7.0,13,4],["leaf",27,0.0028920264763653],["split",28,12.0,9,4],["split",29,20.0,0,4],["split",30,13.0,29,4],["leaf",31,-0.08759558103306378],["leaf",32,-0.06372026100359808],["leaf",35,-0.028200283516123782],["leaf",36,-0.0093318230114132],["leaf",37,-0.04819313535999667],["leaf",38,-0.020012657421942876],["leaf",39,-0.022962107930193768],["leaf",40,0.00022837098854192997],["leaf",41,0.017466258977390774],["leaf",42,-0.0019067170845108658],["leaf",43,-0.03416479020162473],["leaf",44,-0.014203070072209504],["leaf",45,-0.02007295607726213],["leaf",46,0.0009900829182873398],["leaf",47,0.007498166275452738],["leaf",48,-0.015336022271546014],["leaf",49,-0.00026942904290136666],["leaf",50,0.01982645357917832],["leaf",51,-0.007430452738981008],["leaf",52,0.005063563553769045],["leaf",53,-0.028471008228704374],["leaf",54,-0.011163872502824919],["leaf",57,0.004883937182512469],["leaf",58,0.023119848255951973],["leaf",59,0.04374055914498313],["leaf",60,0.011323261605986366],["leaf",61,0.02006920541430537],["leaf",62,0.04121530817036217]],[["split",0,8.0,12,0],["split",1,4.0,22,1],["split",2,6.0,25,1],["split",3,7.0,7,2],["split",4,7.0,4,2],["split",5,4.0,10,2],["split",6,4.0,12,2],["split",7,20.0,0,3],["split",8,7.0,13,3],["split",9,10.0,6,3],["split",10,16.0,0,3],["split",11,20.0,0,3],["split",12,0.0,5,3],["split",13,20.0,0,3],["split",14,21.0,6,3],["split",15,3.0,28,4],["leaf",16,-0.0035785719173951386],["split",17,10.0,12,4],["split",18,25.0,7,4],["split",19,7.0,1,4],["split",20,10.0,18,4],["split",21,7.0,6,4],["leaf",22,-0.03676567579551378],["split",23,2.0,4,4],["split",24,21.0,13,4],["split",25,7.0,25,4],["split",26,6.0,17,4],["split",27,19.0,0,4],["split",28,21.0,18,4],["leaf",29,0.006188700801285526],["split",30,8.0,29,4],["leaf",31,-0.03001383606475142],["leaf",32,0.00765512465299094],["leaf",35,-0.016583200605672192],["leaf",36,-0.002463700190982508],["leaf",37,-0.008808701245219456],["leaf",38,0.010906877894875553],["leaf",39,-0.06931278923354373],["leaf",40,-0.04987595461283837],["leaf",41,-0.044289242501550245],["leaf",42,-0.030465067374278787],["leaf",43,-0.02902776919930722],["leaf",44,-0.015694815261018017],["leaf",47,0.02996956025970505],["leaf",48,0.015983688021377028],["leaf",49,-0.013797234464059864],["leaf",50,0.01081587102227186],["leaf",51,-0.012015417461104216],["leaf",52,0.0058719940072765],["leaf",53,0.0006180515611150418],["leaf",54,0.011764062456202803],["leaf",55,0.04103992056931373],["leaf",56,0.0227081531985046],["leaf",57,0.010095377923148106],["leaf",58,0.030566738091721515],["leaf",61,0.018552577828112118],["leaf",62,0.03859073625324104]],[["split",0,6.0,14,0],["split",1,6.0,8,1],["split",2,6.0,26,1],["split",3,20.0,0,2],["split",4,4.0,20,2],["split",5,4.0,10,2],["split",6,5.0,10,2],["split",7,7.0,3,3],["split",8,25.0,20,3],["split",9,22.0,20,3],["split",10,23.0,15,3],["split",11,10.0,15,3],["split",12,13.0,20,3],["split",13,20.0,0,3],["split",14,0.0,6,3],["split",15,2.0,12,4],["split",16,7.0,7,4],["split",17,21.0,7,4],["leaf",18,0.0119645034359958],["split",19,10.0,8,4],["split",20,2.0,16,4],["split",21,10.0,19,4],["split",22,4.0,25,4],["split",23,4.0,4,4],["split",24,6.0,22,4],["split",25,26.0,19,4],["split",26,2.0,11,4],["split",27,4.0,2,4],["split",28,25.0,15,4],["leaf",29,0.005302361041001625],["split",30,13.0,29,4],["leaf",31,-0.019464517216240856],["leaf",32,-0.050880210985238206],["leaf",33,-0.02945700863240329],["leaf",34,-0.013174254735551455],["leaf",35,-0.04083363074125832],["leaf",36,-0.011264119272987748],["leaf",39,-0.007237267770236268],["leaf",40,0.005254688244685319],["leaf",41,-0.0033960955419354926],["leaf",42,-0.014636489403649556],["leaf",43,-0.017496302997776443],["leaf",44,-2.215072407349156e-05],["leaf",45,-0.016013124690778487],["leaf",46,-0.027940273239572988],["leaf",47,0.018588564422135957],["leaf",48,0.009075844839179038],["leaf",49,0.019560127337491438],["leaf",50,0.032120136345081395],["leaf",51,-0.0029611817235820305],["leaf",52,0.008595190744717396],["leaf",53,0.020409125700337476],["leaf",54,0.0038889324781745775],["leaf",55,0.05039636446241556],["leaf",56,0.035833195601218414],["leaf",57,0.006324314658402323],["leaf",58,0.023138036347573276],["leaf",61,0.018069708890304112],["leaf",62,0.03234402657351627]],[["split",0,8.0,12,0],["split",1,7.0,2,1],["split",2,8.0,26,1],["split",3,20.0,0,2],["split",4,4.0,11,2],["split",5,4.0,16,2],["split",6,4.0,13,2],["split",7,2.0,21,3],["leaf",8,-0.02131119712383241],["split",9,8.0,6,3],["split",10,7.0,9,3],["split",11,13.0,18,3],["split",12,12.0,6,3],["split",13,20.0,0,3],["split",14,7.0,27,3],["split",15,7.0,0,4],["split",16,7.0,1,4],["leaf",19,-0.012179197973173447],["split",20,2.0,7,4],["split",21,20.0,0,4],["split",22,25.0,12,4],["split",23,21.0,6,4],["split",24,20.0,0,4],["split",25,26.0,13,4],["split",26,23.0,19,4],["split",27,19.0,0,4],["split",28,7.0,26,4],["split",29,9.0,19,4],["split",30,26.0,13,4],["leaf",31,-0.048754167512066196],["leaf",32,-0.029496786349040426],["leaf",33,-0.058949951489671484],["leaf",34,-0.04161773882919057],["leaf",41,0.012201205972515023],["leaf",42,0.0008814221135802819],["leaf",43,-0.025568179475975943],["leaf",44,-0.006516876958048984],["leaf",45,-0.018746781455307535],["leaf",46,-0.005284841391237549],["leaf",47,-0.0029325281787089394],["leaf",48,0.011218484575660085],["leaf",49,0.021467731119786105],["leaf",50,0.005225769743656682],["leaf",51,-0.026750963216239983],["leaf",52,-0.010041023263312136],["leaf",53,0.00492665525106937],["leaf",54,-0.006754679767243308],["leaf",55,0.04016847746214164],["leaf",56,0.022692614621609954],["leaf",57,0.008826093014419036],["leaf",58,0.03427841348077465],["leaf",59,0.00809994354175589],["leaf",60,0.02006143875884887],["leaf",61,0.022313359541410397],["leaf",62,0.03303021676813939]],[["split",0,8.0,14,0],["split",1,7.0,4,1],["split",2,8.0,24,1],["split",3,7.0,0,2],["split",4,3.0,20,2],["split",5,12.0,17,2],["split",6,8.0,30,2],["split",7,21.0,9,3],["split",8,20.0,0,3],["split",9,7.0,13,3],["split",10,26.0,8,3],["split",11,5.0,26,3],["split",12,3.0,22,3],["split",13,0.0,8,3],["split",14,0.0,10,3],["leaf",15,-0.06253298637555442],["split",16,1.0,4,4],["split",17,2.0,21,4],["split",18,21.0,16,4],["split",19,4.0,25,4],["split",20,25.0,18,4],["leaf",21,-0.014078340655798487],["split",22,2.0,10,4],["split",23,26.0,18,4],["leaf",24,-0.015697198462775966],["split",25,2.0,8,4],["split",26,23.0,22,4],["split",27,2.0,6,4],["split",28,4.0,13,4],["leaf",29,0.02998627238732546],["split",30,9.0,24,4],["leaf",33,-0.03383539487675967],["leaf",34,-0.05163229364042208],["leaf",35,-0.024018120036271056],["leaf",36,-0.040895769016012364],["leaf",37,-0.026219442455171632],["leaf",38,0.007543126830281266],["leaf",39,-0.013693983928709106],["leaf",40,-0.02613213227585003],["leaf",41,-0.0074091559409054175],["leaf",42,0.004855920763843684],["leaf",45,0.011992792163802958],["leaf",46,-0.0003125125199995693],["leaf",47,-0.0010001720242608153],["leaf",48,0.008676969727001077],["leaf",51,0.017160947055471543],["leaf",52,0.0041271419395076826],["leaf",53,0.026039392779702183],["leaf",54,0.006172372703426671],["leaf",55,0.02392550365078596],["leaf",56,0.0055987816005059554],["leaf",57,0.030118313513262687],["leaf",58,0.016771201172181633],["leaf",61,0.040601616489488616],["leaf",62,0.05639256871126746]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,13.0,22,1],["split",3,20.0,0,2],["split",4,4.0,11,2],["split",5,21.0,12,2],["split",6,5.0,11,2],["split",7,4.0,15,3],["split",8,21.0,17,3],["split",9,24.0,10,3],["split",10,7.0,6,3],["split",11,4.0,13,3],["split",12,12.0,20,3],["split",13,7.0,27,3],["split",14,21.0,21,3],["leaf",15,-0.011040745479129787],["split",16,7.0,0,4],["leaf",17,-0.029562436075503216],["leaf",18,0.004688211806266091],["leaf",19,-0.009113057702379813],["split",20,4.0,5,4],["split",21,10.0,7,4],["split",22,10.0,4,4],["split",23,15.0,0,4],["split",24,13.0,17,4],["split",25,9.0,9,4],["split",26,2.0,2,4],["split",27,17.0,0,4],["split",28,19.0,0,4],["split",29,2.0,18,4],["split",30,22.0,19,4],["leaf",33,-0.0558364581659489],["leaf",34,-0.03803188888203201],["leaf",41,0.013173106186880773],["leaf",42,-0.0019801925681266234],["leaf",43,-0.037506334603713797],["leaf",44,-0.017965146161497223],["leaf",45,-0.021746277125615245],["leaf",46,-0.0093027678272449],["leaf",47,0.0028885024685055237],["leaf",48,0.018178559215300105],["leaf",49,-0.012742531227586255],["leaf",50,-0.001506952434215248],["leaf",51,-0.0028191041096880835],["leaf",52,0.007818518667590868],["leaf",53,0.02842991528506118],["leaf",54,0.013077835460760655],["leaf",55,0.018490753779423535],["leaf",56,0.03519427529844846],["leaf",57,0.042291363020123166],["leaf",58,0.008308531176950821],["leaf",59,0.011756641972645839],["leaf",60,-0.00363590382663093],["leaf",61,0.02727710612213722],["leaf",62,0.00873815527621125]],[["split",0,13.0,13,0],["split",1,4.0,18,1],["split",2,13.0,22,1],["split",3,7.0,5,2],["split",4,7.0,5,2],["split",5,12.0,11,2],["split",6,2.0,10,2],["split",7,3.0,25,3],["split",8,21.0,13,3],["split",9,19.0,0,3],["split",10,20.0,0,3],["split",11,7.0,18,3],["split",12,4.0,14,3],["split",13,20.0,0,3],["split",14,3.0,15,3],["split",15,1.0,10,4],["leaf",16,0.00025485562816234534],["split",17,24.0,4,4],["split",18,20.0,0,4],["split",19,20.0,0,4],["split",20,10.0,10,4],["split",21,10.0,4,4],["split",22,25.0,15,4],["split",23,26.0,22,4],["split",24,26.0,10,4],["split",25,8.0,18,4],["split",26,25.0,14,4],["split",27,7.0,26,4],["split",28,12.0,24,4],["split",29,7.0,27,4],["split",30,21.0,19,4],["leaf",31,-0.03407671493518016],["leaf",32,-0.015823157628109496],["leaf",35,0.005989891719955408],["leaf",36,-0.007093672355320058],["leaf",37,0.002989676151770249],["leaf",38,0.021560511573803995],["leaf",39,-0.04241800712172761],["leaf",40,-0.015987628305847584],["leaf",41,-0.028915582652964934],["leaf",42,-0.008016823444043895],["leaf",43,-0.02896674247050127],["leaf",44,-0.01168365254287721],["leaf",45,-0.018389829569268417],["leaf",46,0.01711823955846175],["leaf",47,-0.01169502329304729],["leaf",48,0.0035225376768403005],["leaf",49,-0.002318846074200707],["leaf",50,0.014019388433752849],["leaf",51,0.006909360012537575],["leaf",52,0.016928363606239602],["leaf",53,-0.0042249905596443435],["leaf",54,0.01020035534197697],["leaf",55,0.023540285855238995],["leaf",56,0.036417163572017824],["leaf",57,-0.006334379696787153],["leaf",58,0.01686434291649069],["leaf",59,-0.007294604587218901],["leaf",60,0.009240493810887734],["leaf",61,0.014668943282799945],["leaf",62,0.0273214840806373]],[["split",0,6.0,11,0],["split",1,7.0,1,1],["split",2,6.0,25,1],["split",3,19.0,0,2],["split",4,4.0,24,2],["split",5,12.0,14,2],["split",6,7.0,27,2],["split",7,4.0,21,3],["leaf",8,-0.015445188208398873],["split",9,17.0,0,3],["split",10,0.0,5,3],["split",11,4.0,22,3],["split",12,23.0,28,3],["split",13,3.0,27,3],["split",14,2.0,12,3],["leaf",15,-0.02000949633154679],["split",16,10.0,3,4],["split",19,7.0,19,4],["split",20,23.0,19,4],["split",21,8.0,6,4],["split",22,1.0,5,4],["split",23,7.0,21,4],["split",24,24.0,22,4],["split",25,10.0,16,4],["split",26,13.0,19,4],["split",27,21.0,13,4],["split",28,12.0,26,4],["split",29,3.0,20,4],["split",30,13.0,29,4],["leaf",33,-0.07024005349869407],["leaf",34,-0.043009198378347435],["leaf",39,-0.007956135881749996],["leaf",40,0.0172764136484129],["leaf",41,-0.01087450393374194],["leaf",42,-0.035269781893488644],["leaf",43,-0.03850680374219521],["leaf",44,-0.02004313624003392],["leaf",45,-0.025304353089859972],["leaf",46,-0.011998932889824161],["leaf",47,-0.0021574528961279225],["leaf",48,0.01164556515847612],["leaf",49,-0.018175548357842637],["leaf",50,-0.0006453761239243701],["leaf",51,0.0054100583685138546],["leaf",52,0.01470657382021961],["leaf",53,-0.01772028487866946],["leaf",54,-0.0035638895313469736],["leaf",55,0.008106567849443826],["leaf",56,0.01967994967306131],["leaf",57,0.020753494221297002],["leaf",58,0.03476258383399228],["leaf",59,0.03215680128697418],["leaf",60,0.05309202795288247],["leaf",61,0.006728389807368706],["leaf",62,0.026921254460323302]],[["split",0,7.0,11,0],["split",1,7.0,2,1],["split",2,8.0,23,1],["split",3,10.0,20,2],["split",4,4.0,15,2],["split",5,21.0,5,2],["split",6,12.0,24,2],["split",7,20.0,0,3],["split",8,23.0,20,3],["split",9,10.0,6,3],["split",10,19.0,0,3],["split",11,20.0,0,3],["split",12,5.0,18,3],["split",13,1.0,7,3],["split",14,8.0,30,3],["split",15,2.0,26,4],["leaf",16,-0.011040300507464183],["leaf",17,-0.003078968983752704],["leaf",18,-0.026273926410249483],["split",19,1.0,13,4],["split",20,2.0,11,4],["split",21,20.0,0,4],["split",22,10.0,14,4],["split",23,23.0,16,4],["leaf",24,-0.025849002908385738],["split",25,10.0,3,4],["split",26,20.0,0,4],["split",27,25.0,12,4],["split",28,4.0,19,4],["split",29,10.0,17,4],["split",30,5.0,3,4],["leaf",31,-0.0314622009893721],["leaf",32,-0.05043338807728932],["leaf",39,-0.02109567929675179],["leaf",40,-0.0075487186492145565],["leaf",41,0.008385768849914915],["leaf",42,-0.0037887230478102505],["leaf",43,-0.020672555422098723],["leaf",44,-0.006987221809649486],["leaf",45,-0.012519201304618763],["leaf",46,0.004939846596542944],["leaf",47,0.0040795387491761115],["leaf",48,-0.012759237251286241],["leaf",51,-0.003340975952873139],["leaf",52,0.01259147230801166],["leaf",53,-0.0024467096350704936],["leaf",54,0.01737490687391512],["leaf",55,-0.010199640233800317],["leaf",56,0.011370332440451172],["leaf",57,0.018630238575974605],["leaf",58,-0.0009077434587211805],["leaf",59,0.017459832096689963],["leaf",60,0.027157663004155487],["leaf",61,0.04352958988054463],["leaf",62,0.029051348651853368]],[["split",0,7.0,9,0],["split",1,13.0,4,1],["split",2,8.0,24,1],["split",3,7.0,0,2],["split",4,10.0,18,2],["split",5,5.0,18,2],["split",6,12.0,26,2],["split",7,24.0,7,3],["split",8,24.0,21,3],["split",9,20.0,0,3],["split",10,21.0,4,3],["split",11,8.0,15,3],["split",12,10.0,6,3],["split",13,22.0,7,3],["split",14,2.0,0,3],["leaf",15,-0.057951586666330705],["split",16,26.0,7,4],["split",17,19.0,0,4],["split",18,3.0,4,4],["split",19,7.0,3,4],["leaf",20,0.0007919997843867917],["leaf",21,-0.029432557320019623],["split",22,1.0,11,4],["split",23,10.0,20,4],["split",24,20.0,0,4],["split",25,25.0,8,4],["split",26,26.0,11,4],["split",27,2.0,4,4],["split",28,20.0,0,4],["leaf",29,0.039626534298556615],["split",30,19.0,0,4],["leaf",33,-0.04601624621706839],["leaf",34,-0.03134977933869591],["leaf",35,-0.03134434324298039],["leaf",36,-0.0031313078610366693],["leaf",37,-0.0007897308511328195],["leaf",38,-0.01513206816857675],["leaf",39,-0.03902725308744369],["leaf",40,-0.016659856902983727],["leaf",45,-0.012666099620266534],["leaf",46,0.0024888857939134082],["leaf",47,-0.0029122546047018714],["leaf",48,0.010710574699400741],["leaf",49,0.012988415369099516],["leaf",50,-0.00211489690706051],["leaf",51,-0.020239821014336096],["leaf",52,-0.006914430868079901],["leaf",53,-0.006420770929170966],["leaf",54,0.0044852358378193544],["leaf",55,0.03883647109320372],["leaf",56,0.017604478892808743],["leaf",57,0.01453475341598197],["leaf",58,0.000471749380790777],["leaf",61,0.027550732769032384],["leaf",62,0.007850477328188813]],[["split",0,8.0,14,0],["split",1,7.0,6,1],["split",2,6.0,28,1],["split",3,7.0,0,2],["split",4,21.0,11,2],["split",5,12.0,16,2],["split",6,6.0,30,2],["split",7,4.0,25,3],["split",8,20.0,0,3],["split",9,5.0,28,3],["split",10,7.0,19,3],["split",11,5.0,17,3],["split",12,25.0,7,3],["split",13,20.0,0,3],["split",14,11.0,14,3],["leaf",15,-0.01831135405629089],["split",16,21.0,12,4],["split",17,19.0,0,4],["split",18,21.0,16,4],["split",19,25.0,2,4],["leaf",20,-0.02974443961209678],["split",21,4.0,28,4],["split",22,1.0,12,4],["split",23,20.0,0,4],["split",24,13.0,15,4],["split",25,22.0,23,4],["split",26,5.0,2,4],["split",27,9.0,19,4],["leaf",28,0.006932951252617658],["leaf",29,0.042336197750021114],["split",30,11.0,23,4],["leaf",33,-0.05899342176983644],["leaf",34,-0.03966092599531002],["leaf",35,-0.024473777094910675],["leaf",36,-0.005246578190508962],["leaf",37,-0.022719139002137404],["leaf",38,0.016994949842483],["leaf",39,-0.028509982417433896],["leaf",40,-0.006765749126840151],["leaf",43,-0.001912528872107285],["leaf",44,-0.01761625967730518],["leaf",45,0.016842953437384622],["leaf",46,0.002714393315500559],["leaf",47,0.008821556310528769],["leaf",48,-0.008687066676560554],["leaf",49,0.011306235278778196],["leaf",50,-0.006330126695715519],["leaf",51,0.009050970123554165],["leaf",52,-0.00655105139671677],["leaf",53,0.023426703957824833],["leaf",54,0.013206112913301772],["leaf",55,0.019985395537769002],["leaf",56,0.0306508817345657],["leaf",61,0.016347520886515844],["leaf",62,0.03464135238009451]],[["split",0,13.0,9,0],["split",1,7.0,5,1],["split",2,8.0,20,1],["split",3,4.0,21,2],["split",4,23.0,28,2],["split",5,5.0,8,2],["split",6,0.0,7,2],["split",7,3.0,7,3],["split",8,15.0,0,3],["split",9,4.0,16,3],["split",10,4.0,22,3],["split",11,20.0,0,3],["split",12,25.0,0,3],["split",13,2.0,3,3],["split",14,7.0,30,3],["leaf",15,0.006491238395220082],["split",16,4.0,7,4],["split",17,12.0,10,4],["split",18,22.0,17,4],["split",19,0.0,5,4],["split",20,26.0,1,4],["leaf",21,-0.010621979968960797],["leaf",22,-0.030347872521730608],["split",23,6.0,16,4],["leaf",24,-0.010487412420658676],["leaf",25,-0.026567323821728692],["split",26,24.0,4,4],["split",27,22.0,17,4],["split",28,22.0,26,4],["split",29,3.0,21,4],["split",30,2.0,5,4],["leaf",33,0.007307720100430954],["leaf",34,-0.020534997042190476],["leaf",35,-0.02097432756590299],["leaf",36,-0.037110973022125966],["leaf",37,-0.04720111370354423],["leaf",38,-0.029168803413573575],["leaf",39,-0.0080526907447383],["leaf",40,0.005561664548109889],["leaf",41,-0.022752177049015363],["leaf",42,-0.006895100788220267],["leaf",47,0.006840922484226432],["leaf",48,0.01811431326363026],["leaf",53,-0.00891745205209994],["leaf",54,0.0012459998314826369],["leaf",55,0.025669014203483987],["leaf",56,0.004740040986216137],["leaf",57,0.0037253587834594646],["leaf",58,-0.015085185986385658],["leaf",59,0.010726996086555743],["leaf",60,0.02084842519965462],["leaf",61,0.04997294501605529],["leaf",62,0.024466779112887492]],[["split",0,7.0,8,0],["split",1,7.0,1,1],["split",2,4.0,8,1],["split",3,3.0,19,2],["split",4,10.0,6,2],["split",5,13.0,22,2],["split",6,6.0,18,2],["split",7,2.0,29,3],["leaf",8,-0.012776989014817355],["split",9,24.0,11,3],["split",10,6.0,8,3],["split",11,3.0,24,3],["split",12,7.0,25,3],["split",13,21.0,7,3],["split",14,21.0,2,3],["split",15,8.0,1,4],["leaf",16,-0.05228797552565713],["split",19,22.0,10,4],["split",20,4.0,25,4],["split",21,22.0,10,4],["split",22,10.0,21,4],["split",23,2.0,14,4],["split",24,1.0,9,4],["split",25,17.0,0,4],["split",26,2.0,6,4],["split",27,9.0,15,4],["split",28,5.0,25,4],["split",29,7.0,27,4],["split",30,5.0,15,4],["leaf",31,-0.036374992883219985],["leaf",32,-0.01732219884868367],["leaf",39,-0.03825233442939771],["leaf",40,-0.02408616976156683],["leaf",41,-0.010544946290402694],["leaf",42,-0.025913120558868775],["leaf",43,-0.02206886245013481],["leaf",44,-0.008689718023754274],["leaf",45,-0.009101982085847577],["leaf",46,0.006055733566680269],["leaf",47,0.007168392247003681],["leaf",48,-0.006958811526213424],["leaf",49,-0.0015615646942238898],["leaf",50,0.01990352786932478],["leaf",51,0.012269010959355292],["leaf",52,0.03240598066511605],["leaf",53,0.03289104764811698],["leaf",54,0.02002666078719469],["leaf",55,-0.015498358405336468],["leaf",56,-0.003948245326568302],["leaf",57,0.002382504747764576],["leaf",58,-0.008373592478045382],["leaf",59,-0.016469778194738223],["leaf",60,-7.651415327671048e-05],["leaf",61,0.012015740127038661],["leaf",62,0.003751817923597411]],[["split",0,7.0,12,0],["split",1,3.0,22,1],["split",2,6.0,21,1],["split",3,21.0,10,2],["split",4,7.0,6,2],["split",5,26.0,17,2],["split",6,8.0,30,2],["split",7,23.0,24,3],["split",8,2.0,10,3],["split",9,3.0,29,3],["split",10,10.0,19,3],["split",11,2.0,10,3],["split",12,23.0,28,3],["split",13,20.0,0,3],["split",14,24.0,14,3],["split",15,7.0,7,4],["split",16,24.0,7,4],["split",17,5.0,22,4],["split",18,25.0,11,4],["split",19,10.0,18,4],["leaf",20,0.007606148514469227],["split",21,7.0,8,4],["split",22,22.0,9,4],["split",23,20.0,0,4],["split",24,21.0,4,4],["split",25,25.0,15,4],["leaf",26,-0.008270391075679015],["split",27,19.0,0,4],["split",28,25.0,17,4],["leaf",29,0.016468679085602227],["split",30,1.0,21,4],["leaf",31,-0.025285736169177984],["leaf",32,-0.01346699299823292],["leaf",33,-0.021525671615028538],["leaf",34,-0.03854771349646877],["leaf",35,0.007634141238275045],["leaf",36,-0.007568411431874146],["leaf",37,-0.022423631812704344],["leaf",38,-0.010280725055585686],["leaf",39,-0.02128474033529757],["leaf",40,-0.003953617219604139],["leaf",43,-0.012527570693344441],["leaf",44,0.0016470060914371178],["leaf",45,0.02185831308569225],["leaf",46,0.005341675324290415],["leaf",47,0.007844016360586447],["leaf",48,-0.01607804289676667],["leaf",49,-0.014323575462062022],["leaf",50,-0.0034953648612183574],["leaf",51,0.004469839774681268],["leaf",52,0.015707431177069308],["leaf",55,0.01503882432759982],["leaf",56,0.0033260371901831843],["leaf",57,-0.011199781095202525],["leaf",58,0.014244160944780645],["leaf",61,0.025658082468666178],["leaf",62,0.04248253863041657]],[["split",0,8.0,9,0],["split",1,7.0,1,1],["split",2,12.0,18,1],["split",3,2.0,15,2],["split",4,7.0,18,2],["split",5,21.0,14,2],["split",6,4.0,3,2],["leaf",7,-0.007257548269108563],["split",8,19.0,0,3],["split",9,20.0,0,3],["leaf",10,0.012293805845695711],["split",11,3.0,10,3],["split",12,12.0,5,3],["split",13,20.0,0,3],["split",14,24.0,9,3],["split",17,11.0,23,4],["leaf",18,-0.013525473501862681],["split",19,2.0,15,4],["split",20,2.0,16,4],["split",23,25.0,4,4],["split",24,20.0,0,4],["split",25,0.0,15,4],["split",26,25.0,29,4],["split",27,2.0,3,4],["leaf",28,0.0025901085777191367],["split",29,23.0,26,4],["split",30,4.0,25,4],["leaf",35,-0.03944394934245408],["leaf",36,-0.023965417012801592],["leaf",39,-0.0024560700285825208],["leaf",40,-0.015329785603961834],["leaf",41,-0.01559378402777256],["leaf",42,0.009631409079160944],["leaf",47,-0.02291600685011736],["leaf",48,-0.008609262328577933],["leaf",49,0.0021638795725913585],["leaf",50,-0.017731278505070714],["leaf",51,-0.01751716539581811],["leaf",52,-0.0016629191608992207],["leaf",53,0.004719619184020861],["leaf",54,0.025570324261598262],["leaf",55,0.029608959057081732],["leaf",56,0.015264567410548838],["leaf",59,0.005109959001246557],["leaf",60,-0.015583528470231384],["leaf",61,0.011465727338944257],["leaf",62,-0.007981827813466002]],[["split",0,7.0,9,0],["split",1,4.0,14,1],["split",2,13.0,21,1],["split",3,26.0,22,2],["split",4,20.0,0,2],["split",5,5.0,9,2],["split",6,2.0,6,2],["split",7,3.0,28,3],["split",8,9.0,11,3],["split",9,10.0,6,3],["split",10,25.0,17,3],["split",11,23.0,26,3],["split",12,23.0,22,3],["split",13,20.0,0,3],["split",14,3.0,17,3],["split",15,23.0,21,4],["leaf",16,0.0054358634389077825],["leaf",17,-0.00678630011218205],["split",18,12.0,13,4],["split",19,17.0,0,4],["split",20,23.0,7,4],["split",21,21.0,9,4],["leaf",22,0.013711622153027934],["split",23,10.0,21,4],["leaf",24,-0.010211875576195089],["split",25,26.0,8,4],["split",26,17.0,0,4],["split",27,3.0,26,4],["split",28,25.0,15,4],["split",29,20.0,0,4],["split",30,1.0,7,4],["leaf",31,-0.005291358155690097],["leaf",32,-0.02227020663345931],["leaf",37,0.0016328153928851037],["leaf",38,0.01877939557445412],["leaf",39,-0.024296041323675552],["leaf",40,-0.0513718914684732],["leaf",41,-0.006300656062703694],["leaf",42,-0.01758936885508452],["leaf",43,-0.02759777087337204],["leaf",44,0.0026105645428046056],["leaf",47,0.0052363030092285896],["leaf",48,0.014957217569652901],["leaf",51,-0.003565252403983582],["leaf",52,0.0037711681201747925],["leaf",53,-0.002770545986166258],["leaf",54,-0.03533927479232965],["leaf",55,0.019740338212660885],["leaf",56,0.032997679615354054],["leaf",57,-0.014431553336181248],["leaf",58,0.008015247752620464],["leaf",59,-0.0002459472746483053],["leaf",60,0.014979473347810452],["leaf",61,-0.001873304794137039],["leaf",62,0.015503640260361985]],[["split",0,6.0,8,0],["split",1,7.0,1,1],["split",2,13.0,26,1],["split",3,2.0,24,2],["split",4,7.0,18,2],["split",5,26.0,0,2],["split",6,9.0,27,2],["split",7,0.0,16,3],["split",8,4.0,25,3],["split",9,22.0,1,3],["leaf",10,0.014631082047754642],["split",11,12.0,15,3],["split",12,2.0,9,3],["split",13,22.0,29,3],["split",14,24.0,13,3],["leaf",15,-0.024245618889029343],["leaf",16,-0.005292378868165707],["leaf",17,-0.013367081343652502],["split",18,10.0,9,4],["leaf",19,-0.024144994127401787],["split",20,20.0,0,4],["leaf",23,-0.026306757229310865],["leaf",24,-0.007391510790530197],["split",25,20.0,0,4],["split",26,3.0,19,4],["split",27,7.0,22,4],["leaf",28,-0.008443411123216678],["leaf",29,0.01124045128041815],["split",30,8.0,29,4],["leaf",37,-0.04956137637448126],["leaf",38,-0.02719449416533787],["leaf",41,-0.011848916811061129],["leaf",42,0.0025308622123055],["leaf",51,0.009780295897786095],["leaf",52,-0.005580806484595174],["leaf",53,-0.0027612198376362066],["leaf",54,0.005083276290001998],["leaf",55,0.0012106150248790387],["leaf",56,0.015096104651049389],["leaf",61,0.023047827599342235],["leaf",62,0.03535330896184395]],[["split",0,7.0,12,0],["split",1,7.0,2,1],["split",2,4.0,12,1],["split",3,22.0,15,2],["split",4,3.0,24,2],["split",5,1.0,7,2],["split",6,26.0,1,2],["split",7,9.0,22,3],["split",8,12.0,1,3],["split",9,15.0,0,3],["split",10,1.0,4,3],["split",11,21.0,9,3],["split",12,24.0,11,3],["split",13,10.0,10,3],["split",14,5.0,23,3],["split",15,21.0,8,4],["leaf",16,-0.04124097829938961],["split",17,10.0,9,4],["split",18,9.0,17,4],["split",19,21.0,16,4],["split",20,5.0,23,4],["leaf",21,-0.012186657167957308],["split",22,7.0,8,4],["leaf",23,-0.004750013246309518],["split",24,11.0,23,4],["split",25,2.0,9,4],["split",26,26.0,28,4],["leaf",27,-0.026162162187249476],["split",28,24.0,11,4],["split",29,23.0,22,4],["split",30,20.0,0,4],["leaf",31,-0.013347002996462808],["leaf",32,-0.03027160451969212],["leaf",35,-0.04448836140709812],["leaf",36,-0.013329307187638525],["leaf",37,-0.012653501672998198],["leaf",38,0.0065794374837455805],["leaf",39,-0.0106695497624272],["leaf",40,-0.003017293676045218],["leaf",41,-0.012762925031404138],["leaf",42,-0.025191370969516964],["leaf",45,-0.003367861621263356],["leaf",46,0.008545616524734333],["leaf",49,0.00959607389596509],["leaf",50,-0.0014061287191443278],["leaf",51,0.013275645906332586],["leaf",52,0.0018185497955373335],["leaf",53,0.015346400131957129],["leaf",54,0.02699816731835509],["leaf",57,0.0034797722693502855],["leaf",58,-0.015175281180000067],["leaf",59,0.007001223593602291],["leaf",60,-0.0011280686716157366],["leaf",61,-0.008482450779891116],["leaf",62,0.010119757258785436]],[["split",0,8.0,8,0],["split",1,7.0,3,1],["split",2,6.0,21,1],["split",3,20.0,0,2],["split",4,21.0,8,2],["split",5,21.0,2,2],["split",6,5.0,5,2],["split",7,3.0,25,3],["split",8,25.0,13,3],["split",9,13.0,6,3],["split",10,20.0,0,3],["split",11,11.0,21,3],["split",12,12.0,10,3],["split",13,20.0,0,3],["split",14,7.0,27,3],["split",15,10.0,12,4],["leaf",16,0.005878587762158547],["leaf",17,-0.017904741196790974],["leaf",18,0.008858092526150036],["split",19,23.0,21,4],["split",20,5.0,21,4],["split",21,23.0,22,4],["split",22,2.0,18,4],["split",23,23.0,12,4],["leaf",24,-0.002500578460683475],["split",25,3.0,3,4],["split",26,10.0,4,4],["split",27,19.0,0,4],["split",28,25.0,16,4],["split",29,22.0,25,4],["split",30,9.0,6,4],["leaf",31,-0.03258562831922886],["leaf",32,-0.014107725580538749],["leaf",39,-0.014020132918884926],["leaf",40,-0.0008920204072677808],["leaf",41,-0.016318386490023262],["leaf",42,-0.031026080804580466],["leaf",43,-0.002973675565293405],["leaf",44,-0.013843624076129832],["leaf",45,-0.004426940242653138],["leaf",46,0.02214612410611987],["leaf",47,-0.010650400611600724],["leaf",48,-0.024437546742927627],["leaf",51,-0.022239209916187483],["leaf",52,-0.0030813421968834726],["leaf",53,-0.006921020156349379],["leaf",54,0.005658025713948037],["leaf",55,0.022748818039405636],["leaf",56,0.008160773612573656],["leaf",57,-0.012434684962514288],["leaf",58,0.014457556290560667],["leaf",59,0.005937104629145619],["leaf",60,-0.0070192673155054204],["leaf",61,0.004674629540244661],["leaf",62,0.017782253698944712]],[["split",0,7.0,8,0],["split",1,7.0,2,1],["split",2,12.0,24,1],["split",3,3.0,1,2],["split",4,1.0,9,2],["split",5,21.0,4,2],["split",6,19.0,0,2],["leaf",7,-0.002223629678858819],["split",8,4.0,19,3],["split",9,20.0,0,3],["split",10,21.0,2,3],["split",11,2.0,10,3],["split",12,13.0,21,3],["split",13,7.0,29,3],["split",14,9.0,14,3],["leaf",17,-0.005508532259968268],["split",18,20.0,0,4],["split",19,2.0,25,4],["leaf",20,0.004401507624681509],["leaf",21,-0.02052289832725739],["split",22,4.0,4,4],["split",23,7.0,17,4],["split",24,0.0,2,4],["split",25,26.0,16,4],["split",26,21.0,12,4],["split",27,3.0,17,4],["split",28,25.0,12,4],["leaf",29,-0.006452541573548189],["split",30,2.0,9,4],["leaf",37,-0.02920215435743699],["leaf",38,-0.0067959495212070635],["leaf",39,-0.010247364366206907],["leaf",40,-0.023894758286540013],["leaf",45,0.013155115672470994],["leaf",46,-0.0034976996658459287],["leaf",47,-0.007864905427682096],["leaf",48,0.00934403953564879],["leaf",49,-0.01872915187607786],["leaf",50,-0.007039144300296634],["leaf",51,-0.0017256131565728783],["leaf",52,0.0059719507747989465],["leaf",53,0.0027928590987423493],["leaf",54,0.012283561290013723],["leaf",55,0.004964843385884329],["leaf",56,0.015372493177502906],["leaf",57,0.01189383482736809],["leaf",58,0.0301943636872626],["leaf",61,0.015198731877092177],["leaf",62,0.0005418475234246981]],[["split",0,7.0,8,0],["split",1,2.0,27,1],["split",2,12.0,22,1],["split",3,10.0,2,2],["split",4,20.0,0,2],["split",5,2.0,8,2],["split",6,1.0,3,2],["split",7,1.0,16,3],["split",8,25.0,29,3],["split",9,3.0,23,3],["leaf",10,0.008538597972722448],["split",11,20.0,0,3],["split",12,9.0,5,3],["leaf",13,-0.011094076400644413],["split",14,9.0,27,3],["leaf",15,-0.01233947341264693],["leaf",16,-0.029192343550179058],["split",17,26.0,1,4],["leaf",18,0.009816920648709489],["split",19,23.0,27,4],["leaf",20,-0.009639395123088377],["split",23,7.0,25,4],["split",24,25.0,13,4],["split",25,5.0,24,4],["split",26,12.0,15,4],["split",29,16.0,0,4],["split",30,25.0,23,4],["leaf",35,-0.01973030328149487],["leaf",36,-0.005269291369601395],["leaf",39,-0.023078501163845737],["leaf",40,-0.04263194145589783],["leaf",47,0.005890772298083461],["leaf",48,0.01959638536713144],["leaf",49,-0.025319974530510832],["leaf",50,0.0012124845160306182],["leaf",51,-0.011235610285431009],["leaf",52,0.004067769720184416],["leaf",53,-0.0034444955411417205],["leaf",54,0.0047424057656901],["leaf",59,0.007488787283326115],["leaf",60,0.017824226355413822],["leaf",61,0.016327589338445733],["leaf",62,0.03613338440042105]],[["split",0,7.0,15,0],["split",1,4.0,18,1],["split",2,6.0,22,1],["split",3,3.0,27,2],["split",4,24.0,17,2],["split",5,25.0,4,2],["split",6,25.0,29,2],["split",7,10.0,5,3],["split",8,20.0,0,3],["split",9,7.0,3,3],["split",10,13.0,0,3],["split",11,2.0,8,3],["split",12,21.0,2,3],["split",13,22.0,22,3],["split",14,0.0,20,3],["split",15,2.0,22,4],["split",16,26.0,10,4],["split",17,4.0,1,4],["leaf",18,-0.014917559424925642],["split",19,2.0,29,4],["split",20,26.0,12,4],["leaf",21,0.008422955654214978],["split",22,6.0,8,4],["leaf",23,0.0036257504919036425],["split",24,26.0,3,4],["split",25,10.0,14,4],["split",26,26.0,12,4],["split",27,9.0,21,4],["split",28,9.0,24,4],["leaf",29,0.017154902960129958],["leaf",30,0.03592078208956685],["leaf",31,-0.018107208676679552],["leaf",32,-0.0039034062233495375],["leaf",33,-0.012895812251226614],["leaf",34,0.0006074980058767793],["leaf",35,0.018857191172696672],["leaf",36,0.0039426805161768145],["leaf",39,-0.017126847571066545],["leaf",40,-0.03552337197360637],["leaf",41,-0.01591093107734048],["leaf",42,-0.007059522599723071],["leaf",45,-0.01248826659914526],["leaf",46,0.00013405339688717314],["leaf",49,-0.02627141572246145],["leaf",50,-0.008894742480156106],["leaf",51,-0.019030705212538464],["leaf",52,0.004882595862506802],["leaf",53,0.0025544908289028526],["leaf",54,0.01005460801005799],["leaf",55,0.009256394921789913],["leaf",56,0.01811745032188208],["leaf",57,-0.0020722915301133694],["leaf",58,0.015429458541351007]],[["split",0,4.0,18,0],["split",1,12.0,28,1],["split",2,4.0,29,1],["split",3,21.0,4,2],["split",4,4.0,0,2],["split",5,10.0,2,2],["split",6,10.0,8,2],["split",7,20.0,0,3],["split",8,22.0,11,3],["leaf",9,0.03188850548667761],["split",10,21.0,14,3],["split",11,25.0,13,3],["split",12,8.0,13,3],["split",13,7.0,3,3],["split",14,1.0,2,3],["split",15,6.0,19,4],["leaf",16,-0.02125498002163487],["split",17,12.0,13,4],["split",18,26.0,26,4],["split",21,7.0,28,4],["split",22,26.0,17,4],["split",23,25.0,6,4],["split",24,7.0,6,4],["split",25,21.0,16,4],["split",26,26.0,1,4],["leaf",27,-0.040219241634424155],["leaf",28,-0.01707449394520403],["leaf",29,-0.02462336525867205],["split",30,24.0,10,4],["leaf",31,-0.0074513032299041515],["leaf",32,0.0042621741476959385],["leaf",35,0.0012981165122096382],["leaf",36,0.010193914005230191],["leaf",37,0.0005095640706809269],["leaf",38,0.007252272058005604],["leaf",43,-0.0010369424778474833],["leaf",44,0.01318394711909893],["leaf",45,0.006417616926304446],["leaf",46,0.02016951254498121],["leaf",47,-0.016458731742002766],["leaf",48,-0.02899036839630136],["leaf",49,-0.01550635605075739],["leaf",50,-0.0037943597698448716],["leaf",51,-0.0094645370462793],["leaf",52,-0.002340829096013113],["leaf",53,-0.011985219614139854],["leaf",54,0.001364843258080687],["leaf",61,-0.017220145751452257],["leaf",62,0.003070457156800684]],[["split",0,6.0,11,0],["split",1,21.0,19,1],["split",2,7.0,23,1],["split",3,7.0,1,2],["split",4,20.0,0,2],["split",5,10.0,12,2],["split",6,2.0,9,2],["split",7,1.0,1,3],["split",8,2.0,30,3],["split",9,10.0,3,3],["split",10,2.0,18,3],["split",11,26.0,20,3],["split",12,17.0,0,3],["split",13,22.0,22,3],["split",14,26.0,13,3],["leaf",15,-0.015149904538011641],["split",16,9.0,14,4],["split",17,24.0,13,4],["leaf",18,-0.02313160342190515],["split",19,21.0,26,4],["split",20,21.0,30,4],["leaf",21,-0.0003905178747683785],["leaf",22,0.028391101694131407],["split",23,10.0,1,4],["split",24,23.0,19,4],["split",25,24.0,15,4],["split",26,26.0,21,4],["split",27,7.0,29,4],["split",28,25.0,17,4],["split",29,22.0,23,4],["split",30,23.0,14,4],["leaf",33,-0.045353522060194114],["leaf",34,-0.019462375880546487],["leaf",35,-0.01074514539497397],["leaf",36,-0.0038830794604020986],["leaf",39,-0.00809247052798606],["leaf",40,-0.025841600897534303],["leaf",41,-0.0007362004848480067],["leaf",42,-0.015045161067114789],["leaf",47,-0.021692521121311318],["leaf",48,-0.006695378254814324],["leaf",49,0.003795081886357383],["leaf",50,-0.00576989423319033],["leaf",51,-0.0021881406717140484],["leaf",52,0.006240426052984887],["leaf",53,0.010706533942394736],["leaf",54,0.023991296656439898],["leaf",55,0.014321341589869184],["leaf",56,0.02619146684673226],["leaf",57,-0.0035848974264758326],["leaf",58,0.011025794834158439],["leaf",59,0.0019220109002450693],["leaf",60,-0.010312939665965625],["leaf",61,0.01917439849549351],["leaf",62,0.0037551133348458195]],[["split",0,7.0,12,0],["split",1,7.0,3,1],["split",2,7.0,25,1],["split",3,13.0,0,2],["split",4,25.0,22,2],["split",5,26.0,8,2],["split",6,13.0,29,2],["split",7,22.0,8,3],["split",8,2.0,19,3],["split",9,26.0,24,3],["split",10,23.0,18,3],["split",11,13.0,13,3],["split",12,9.0,17,3],["split",13,25.0,7,3],["split",14,0.0,11,3],["leaf",15,0.012755983609468571],["split",16,4.0,29,4],["split",17,22.0,10,4],["split",18,20.0,0,4],["split",19,12.0,2,4],["split",20,23.0,24,4],["split",21,2.0,22,4],["split",22,4.0,24,4],["split",23,22.0,21,4],["split",24,2.0,7,4],["split",25,1.0,4,4],["split",26,17.0,0,4],["split",27,8.0,26,4],["split",28,2.0,16,4],["leaf",29,0.008001291506650897],["split",30,9.0,10,4],["leaf",33,0.0025793323830913314],["leaf",34,-0.021032229456886077],["leaf",35,-0.017622661676847222],["leaf",36,-0.0013216427516394957],["leaf",37,-0.025861614539797847],["leaf",38,0.005979900053402544],["leaf",39,-0.018192566377490015],["leaf",40,-0.006894828637581009],["leaf",41,0.0029804244259232014],["leaf",42,-0.008977805604594762],["leaf",43,0.008861813461493825],["leaf",44,-0.0009233672504309888],["leaf",45,-0.0010972031434579618],["leaf",46,-0.0164747690198116],["leaf",47,0.006672288862151466],["leaf",48,-0.014045496231021082],["leaf",49,-0.00036635087881720904],["leaf",50,-0.013349997464775701],["leaf",51,-0.0115808893115634],["leaf",52,0.001261361202422641],["leaf",53,0.006769844318168531],["leaf",54,0.01823337221542411],["leaf",55,-0.003308869435382822],["leaf",56,0.008976642219444537],["leaf",57,0.012381869900005137],["leaf",58,0.0014666881061942878],["leaf",61,0.01410983502226344],["leaf",62,0.02702033066871695]],[["split",0,8.0,8,0],["split",1,7.0,4,1],["split",2,5.0,15,1],["split",3,3.0,27,2],["split",4,17.0,0,2],["split",5,20.0,0,2],["split",6,21.0,15,2],["split",7,20.0,0,3],["leaf",8,0.012858363277457575],["split",9,22.0,19,3],["split",10,23.0,21,3],["split",11,7.0,23,3],["split",12,21.0,13,3],["split",13,7.0,6,3],["split",14,20.0,0,3],["split",15,3.0,4,4],["split",16,25.0,16,4],["split",19,21.0,12,4],["split",20,24.0,8,4],["leaf",21,-0.009202916553978202],["leaf",22,-0.027593131712117847],["split",23,1.0,4,4],["split",24,9.0,25,4],["split",25,22.0,14,4],["split",26,21.0,25,4],["leaf",27,-0.018437614342172556],["split",28,8.0,9,4],["split",29,24.0,3,4],["split",30,3.0,9,4],["leaf",31,-0.003750546630412857],["leaf",32,-0.021038092206066956],["leaf",33,-0.014742097515426116],["leaf",34,0.019042835078237137],["leaf",39,-0.0038798689302722796],["leaf",40,0.010058985052943597],["leaf",41,-0.017965824970220638],["leaf",42,5.449543767464211e-05],["leaf",47,-0.014415652653178202],["leaf",48,0.005858750459201014],["leaf",49,0.01151541811100315],["leaf",50,0.023425121727165425],["leaf",51,-0.006577936668837372],["leaf",52,-0.033156001463740696],["leaf",53,-0.0032171213525548177],["leaf",54,0.01778545935306775],["leaf",57,0.008760943448635617],["leaf",58,-0.004687077488807921],["leaf",59,-0.012936612797038734],["leaf",60,0.0007433904057528712],["leaf",61,0.03283131159150967],["leaf",62,0.007678698853682369]],[["split",0,7.0,15,0],["split",1,10.0,2,1],["split",2,3.0,21,1],["split",3,6.0,1,2],["split",4,1.0,4,2],["split",5,7.0,29,2],["split",6,19.0,0,2],["leaf",7,-0.031266708982987626],["split",8,11.0,12,3],["split",9,20.0,0,3],["split",10,21.0,3,3],["split",11,22.0,22,3],["split",12,2.0,12,3],["split",13,20.0,0,3],["split",14,6.0,26,3],["split",17,1.0,10,4],["split",18,1.0,5,4],["split",19,19.0,0,4],["leaf",20,0.0048643765054624],["split",21,22.0,12,4],["split",22,26.0,16,4],["split",23,7.0,22,4],["split",24,2.0,4,4],["split",25,4.0,14,4],["split",26,21.0,10,4],["split",27,2.0,20,4],["split",28,3.0,26,4],["leaf",29,-0.009590907458766482],["leaf",30,0.003925719323821487],["leaf",35,-0.010166612142755216],["leaf",36,-0.030739159348391356],["leaf",37,-0.017553851819521436],["leaf",38,-0.003831522032559455],["leaf",39,-0.016280395124013403],["leaf",40,0.0006556940253511512],["leaf",43,-0.02041365990591645],["leaf",44,-0.0057808512611875066],["leaf",45,-0.004250504120735801],["leaf",46,0.0024544377802549738],["leaf",47,-0.0015321831672178519],["leaf",48,0.005013638350188517],["leaf",49,0.005374684055616438],["leaf",50,-0.0063359570620265485],["leaf",51,0.026865846883395553],["leaf",52,0.00114825825346578],["leaf",53,-0.01592201127414322],["leaf",54,0.007344607604534538],["leaf",55,0.01618362538950444],["leaf",56,0.004318030334792812],["leaf",57,0.012179126592933855],["leaf",58,-0.010571858305003177]],[["split",0,7.0,8,0],["split",1,5.0,2,1],["split",2,2.0,1,1],["leaf",3,0.012270459386672332],["split",4,20.0,0,2],["split",5,20.0,0,2],["split",6,1.0,8,2],["split",9,10.0,4,3],["split",10,25.0,11,3],["split",11,19.0,0,3],["leaf",12,-0.002155482064244965],["split",13,21.0,3,3],["split",14,5.0,5,3],["split",19,17.0,0,4],["split",20,19.0,0,4],["leaf",21,-0.01727678750571189],["split",22,2.0,15,4],["split",23,24.0,17,4],["leaf",24,0.002138901009681795],["split",27,2.0,18,4],["split",28,23.0,29,4],["split",29,20.0,0,4],["split",30,7.0,30,4],["leaf",39,-0.016592946509627667],["leaf",40,-0.0341876353801487],["leaf",41,-0.010340460763621932],["leaf",42,0.0018038847468591402],["leaf",45,-0.007259860272523239],["leaf",46,0.0222872225659374],["leaf",47,0.013967686806656821],["leaf",48,0.02419319905531782],["leaf",55,-0.026322664586169043],["leaf",56,-0.007803379386724084],["leaf",57,0.0011120757024930127],["leaf",58,-0.017110208869807835],["leaf",59,0.0111705355323807],["leaf",60,-0.0066625922857769596],["leaf",61,0.002238737289947141],["leaf",62,0.013629351412212066]],[["split",0,7.0,12,0],["split",1,10.0,16,1],["split",2,24.0,12,1],["split",3,20.0,0,2],["split",4,2.0,26,2],["split",5,1.0,3,2],["split",6,21.0,5,2],["split",7,4.0,5,3],["split",8,21.0,13,3],["split",9,24.0,15,3],["split",10,11.0,8,3],["split",11,21.0,8,3],["split",12,20.0,0,3],["split",13,26.0,7,3],["split",14,9.0,23,3],["leaf",15,0.006224986091746875],["split",16,2.0,30,4],["leaf",17,-0.015467654412377588],["split",18,25.0,22,4],["split",19,26.0,2,4],["split",20,9.0,5,4],["leaf",21,0.00023911281758240473],["split",22,9.0,15,4],["leaf",23,-0.026290114685639143],["leaf",24,-0.007058286198134215],["split",25,2.0,4,4],["split",26,21.0,13,4],["split",27,11.0,13,4],["split",28,25.0,12,4],["split",29,20.0,0,4],["split",30,23.0,19,4],["leaf",33,-0.011805005187519943],["leaf",34,-0.03009186229358293],["leaf",37,0.0018811151446707546],["leaf",38,0.03177973241466774],["leaf",39,-0.021783721325551276],["leaf",40,-0.0006255684359842222],["leaf",41,-0.004798282663710248],["leaf",42,0.007888188926537828],["leaf",45,-0.022647594451235473],["leaf",46,-0.006967722147869944],["leaf",51,0.01230026177053394],["leaf",52,0.000454186624458421],["leaf",53,-0.023291472942135412],["leaf",54,0.004021364929970117],["leaf",55,-0.020289048648904105],["leaf",56,-0.0028845130270917074],["leaf",57,-0.006937291333929054],["leaf",58,0.007597986858769872],["leaf",59,0.004000307088072837],["leaf",60,0.01577206885901154],["leaf",61,0.008981711588644923],["leaf",62,0.023554761146700368]],[["split",0,7.0,9,0],["split",1,7.0,0,1],["split",2,23.0,30,1],["split",3,9.0,6,2],["split",4,21.0,2,2],["split",5,21.0,9,2],["leaf",6,-0.01942906679591336],["leaf",7,-0.008103429460378564],["split",8,11.0,18,3],["split",9,26.0,5,3],["split",10,24.0,17,3],["split",11,20.0,0,3],["split",12,13.0,28,3],["leaf",17,-0.013062640070228893],["leaf",18,-0.03565572097598051],["leaf",19,-0.031234584340228015],["split",20,3.0,12,4],["split",21,11.0,22,4],["split",22,15.0,0,4],["split",23,2.0,10,4],["split",24,26.0,11,4],["split",25,23.0,23,4],["split",26,17.0,0,4],["leaf",41,-0.01572110675254404],["leaf",42,-0.003634391180752871],["leaf",43,-0.01217551343546909],["leaf",44,0.0008227290924829668],["leaf",45,0.0005458663278255665],["leaf",46,-0.010286542083514648],["leaf",47,0.007430098032554688],["leaf",48,-0.002637769652007376],["leaf",49,-0.002692535266521667],["leaf",50,-0.02244577105555442],["leaf",51,0.005458417612484583],["leaf",52,-0.0016720679943687795],["leaf",53,0.010588945292329512],["leaf",54,0.025484993893656268]],[["split",0,7.0,10,0],["split",1,23.0,27,1],["split",2,10.0,5,1],["split",3,26.0,20,2],["split",4,17.0,0,2],["split",5,26.0,7,2],["split",6,12.0,28,2],["split",7,0.0,2,3],["split",8,24.0,13,3],["split",9,20.0,0,3],["leaf",10,-0.03738874203605168],["split",11,8.0,13,3],["split",12,19.0,0,3],["split",13,12.0,9,3],["split",14,5.0,14,3],["split",15,21.0,7,4],["split",16,26.0,1,4],["split",17,3.0,14,4],["split",18,5.0,21,4],["split",19,3.0,21,4],["leaf",20,0.008003729161219353],["leaf",23,-0.0013319663805357435],["split",24,7.0,26,4],["split",25,7.0,21,4],["leaf",26,-0.016832486860913248],["split",27,22.0,20,4],["split",28,26.0,12,4],["split",29,20.0,0,4],["leaf",30,-0.0018203786802468086],["leaf",31,-0.0247910662891301],["leaf",32,-0.008023289519905151],["leaf",33,-0.017050883457698562],["leaf",34,-0.0054330185142431615],["leaf",35,0.010493225691676638],["leaf",36,-0.006790570444733283],["leaf",37,0.008942311922215734],["leaf",38,-0.0008512838170287568],["leaf",39,-0.01893566244240809],["leaf",40,0.002583095665993023],["leaf",49,-0.023027224869280025],["leaf",50,-0.008666461210691718],["leaf",51,-0.004776948005808468],["leaf",52,0.005014760627771669],["leaf",55,-8.924032743249398e-05],["leaf",56,-0.013006784566176826],["leaf",57,0.00014865873394994674],["leaf",58,0.005939116415251133],["leaf",59,0.01784398336976403],["leaf",60,0.0004315656757529862]],[["split",0,4.0,18,0],["split",1,7.0,20,1],["split",2,21.0,25,1],["split",3,3.0,20,2],["split",4,13.0,30,2],["split",5,10.0,8,2],["split",6,20.0,0,2],["split",7,1.0,26,3],["split",8,20.0,0,3],["split",9,22.0,29,3],["split",10,11.0,17,3],["split",11,0.0,1,3],["split",12,16.0,0,3],["split",13,2.0,13,3],["leaf",14,0.02235705944494131],["split",15,10.0,1,4],["split",16,5.0,11,4],["split",17,5.0,0,4],["split",18,21.0,10,4],["split",19,23.0,21,4],["leaf",20,-0.007404347642206111],["split",21,11.0,10,4],["leaf",22,0.006679377091573814],["leaf",23,0.005313377105477822],["split",24,3.0,1,4],["split",25,1.0,2,4],["split",26,26.0,8,4],["split",27,26.0,10,4],["split",28,22.0,6,4],["leaf",31,-0.013528040953061479],["leaf",32,-0.001207213400566836],["leaf",33,-0.0006024914997322032],["leaf",34,-0.02012902323561541],["leaf",35,0.014215613005544268],["leaf",36,0.0033775199467518556],["leaf",37,-0.022056941348277147],["leaf",38,0.0015690798113804569],["leaf",39,0.008553031184032679],["leaf",40,0.00274602984709801],["leaf",43,0.012632321242093068],["leaf",44,0.029008131750993975],["leaf",49,-0.031351206445996566],["leaf",50,-0.011377090786669602],["leaf",51,-0.011491333061919788],["leaf",52,-0.0017593243086753032],["leaf",53,-0.0249806718729942],["leaf",54,-0.005266342868225783],["leaf",55,0.01691720555079828],["leaf",56,-0.002169208791050752],["leaf",57,-0.01768134604207627],["leaf",58,-0.0037838310346679656]],[["split",0,7.0,10,0],["split",1,21.0,5,1],["split",2,3.0,11,1],["split",3,20.0,0,2],["split",4,7.0,0,2],["split",5,7.0,24,2],["split",6,9.0,3,2],["split",7,17.0,0,3],["leaf",8,-0.02438622842352984],["split",9,2.0,27,3],["split",10,14.0,0,3],["split",11,6.0,17,3],["split",12,21.0,3,3],["split",13,7.0,18,3],["split",14,21.0,10,3],["split",15,9.0,8,4],["leaf",16,-0.024337538321054567],["leaf",19,-0.0007979644815726436],["leaf",20,-0.02716654496434247],["split",21,15.0,0,4],["split",22,1.0,4,4],["split",23,20.0,0,4],["split",24,25.0,7,4],["split",25,25.0,13,4],["split",26,4.0,17,4],["split",27,3.0,21,4],["split",28,23.0,17,4],["split",29,20.0,0,4],["split",30,10.0,16,4],["leaf",31,0.0030522138013343125],["leaf",32,-0.010257199761683084],["leaf",43,0.001314242485427394],["leaf",44,-0.007367460807124537],["leaf",45,-0.001158082614751989],["leaf",46,-0.013465883739698647],["leaf",47,-0.00469751335601891],["leaf",48,0.01085429835274483],["leaf",49,-0.02490732605623609],["leaf",50,-0.009405463486943492],["leaf",51,-0.021476240646610564],["leaf",52,0.00478845268686145],["leaf",53,0.01145418192299267],["leaf",54,-0.0018694542845332793],["leaf",55,-0.015830540036171344],["leaf",56,-0.0025812913511786863],["leaf",57,0.006940036105463183],["leaf",58,-0.0037772455773951285],["leaf",59,0.00501799944317373],["leaf",60,-0.016794459422804785],["leaf",61,0.003651841346974386],["leaf",62,0.01160692853691494]],[["split",0,4.0,20,0],["split",1,12.0,20,1],["split",2,7.0,6,1],["split",3,26.0,0,2],["split",4,20.0,0,2],["split",5,20.0,0,2],["split",6,23.0,14,2],["leaf",7,-0.01593574214036822],["split",8,25.0,26,3],["split",9,3.0,14,3],["split",10,23.0,13,3],["split",11,10.0,6,3],["split",12,25.0,11,3],["split",13,5.0,27,3],["split",14,13.0,4,3],["split",17,4.0,0,4],["split",18,10.0,7,4],["split",19,8.0,25,4],["split",20,6.0,16,4],["split",21,10.0,21,4],["split",22,2.0,14,4],["split",23,1.0,20,4],["split",24,5.0,23,4],["leaf",25,-0.018410066333269724],["leaf",26,0.013385652486657691],["split",27,26.0,2,4],["split",28,21.0,14,4],["leaf",29,0.008497250025795912],["split",30,10.0,6,4],["leaf",35,0.01243241087811208],["leaf",36,-0.0009447332690380182],["leaf",37,-0.005238843451115705],["leaf",38,0.010463026891319224],["leaf",39,0.005360906457474809],["leaf",40,-0.004642872769784214],["leaf",41,-0.0001534442425601851],["leaf",42,0.011239952390766014],["leaf",43,-0.02009908089153714],["leaf",44,-0.001804164125189433],["leaf",45,-0.0005393014340075992],["leaf",46,0.021994037284621418],["leaf",47,-0.014282572278551706],["leaf",48,-0.033792800762784306],["leaf",49,-0.018226283688402057],["leaf",50,-0.006011677164602025],["leaf",55,-0.0067133969747068005],["leaf",56,0.009577050665344398],["leaf",57,-0.015070301373521902],["leaf",58,-0.0061191786938798],["leaf",61,-0.013352697503210718],["leaf",62,-0.0029010255943798515]],[["split",0,8.0,22,0],["split",1,4.0,26,1],["split",2,21.0,15,1],["split",3,21.0,0,2],["split",4,11.0,28,2],["split",5,20.0,0,2],["split",6,17.0,0,2],["split",7,5.0,19,3],["split",8,10.0,10,3],["split",9,9.0,19,3],["leaf",10,0.004172901931214381],["split",11,11.0,28,3],["split",12,23.0,12,3],["split",13,7.0,30,3],["split",14,10.0,15,3],["leaf",15,-0.02101809201152956],["leaf",16,0.0007424100601862038],["split",17,20.0,0,4],["split",18,0.0,15,4],["split",19,1.0,0,4],["split",20,11.0,10,4],["split",23,4.0,21,4],["leaf",24,-0.010093305336672609],["leaf",25,-0.022405020958631016],["leaf",26,-0.0014754865340738184],["split",27,9.0,9,4],["leaf",28,0.020352275545783748],["leaf",29,0.005092553646984693],["leaf",30,0.025465726518271163],["leaf",35,-0.005392783037258979],["leaf",36,0.0056787848954837905],["leaf",37,-0.0016717015464377035],["leaf",38,0.003119252412076863],["leaf",39,0.007738307683475773],["leaf",40,-0.007615468268054823],["leaf",41,-0.005618839221243269],["leaf",42,-0.020055444718213966],["leaf",47,0.0071211466982942255],["leaf",48,-0.011072287773833137],["leaf",55,-0.0009099174431473324],["leaf",56,0.008789251799675997]],[["split",0,7.0,25,0],["split",1,4.0,27,1],["split",2,19.0,0,1],["split",3,22.0,25,2],["split",4,7.0,1,2],["split",5,2.0,10,2],["split",6,23.0,23,2],["split",7,3.0,26,3],["split",8,26.0,29,3],["split",9,22.0,20,3],["split",10,23.0,28,3],["split",11,20.0,0,3],["split",12,20.0,0,3],["split",13,2.0,11,3],["leaf",14,-0.021971093358471284],["split",15,20.0,0,4],["split",16,19.0,0,4],["split",17,23.0,8,4],["leaf",18,0.008823734722726823],["split",19,0.0,19,4],["leaf",20,0.0011701595904583587],["split",21,5.0,28,4],["leaf",22,-0.01829318416889596],["split",23,6.0,22,4],["leaf",24,0.0015584598243733663],["split",25,23.0,19,4],["leaf",26,0.017708507403291766],["leaf",27,0.003389777466778364],["leaf",28,-0.00945113552033538],["leaf",31,-0.0011139799919277923],["leaf",32,0.006244915492183434],["leaf",33,0.008100730668703825],["leaf",34,-0.009740091194928562],["leaf",35,0.0001727429071191248],["leaf",36,-0.009124732244031196],["leaf",39,-0.02667040379183381],["leaf",40,-0.009345101860099745],["leaf",43,-0.007416589362677216],["leaf",44,0.004229973884468163],["leaf",47,0.03143687549606511],["leaf",48,0.013933988647709441],["leaf",51,0.006401737971077402],["leaf",52,-0.007171611888997033]],[["split",0,7.0,27,0],["split",1,26.0,6,1],["split",2,4.0,17,1],["split",3,9.0,4,2],["split",4,23.0,29,2],["split",5,22.0,19,2],["split",6,3.0,3,2],["split",7,3.0,11,3],["split",8,10.0,18,3],["split",9,7.0,6,3],["split",10,2.0,15,3],["split",11,2.0,6,3],["split",12,2.0,7,3],["leaf",13,0.010859569831076315],["split",14,3.0,8,3],["split",15,21.0,13,4],["leaf",16,-0.008532689703624434],["split",17,20.0,0,4],["split",18,19.0,0,4],["split",19,2.0,27,4],["split",20,20.0,0,4],["leaf",21,-0.024206289432537498],["split",22,4.0,27,4],["split",23,5.0,5,4],["split",24,24.0,17,4],["split",25,10.0,16,4],["split",26,24.0,17,4],["leaf",29,-0.014206152152714466],["leaf",30,0.003958592822934583],["leaf",31,-0.015064231604453976],["leaf",32,-0.02831874499277289],["leaf",35,-0.010933109263603216],["leaf",36,0.004152810243314833],["leaf",37,-0.0032689023380438405],["leaf",38,0.013461459417426094],["leaf",39,0.00021472507087988554],["leaf",40,-0.011168160214045932],["leaf",41,0.0033526334369947458],["leaf",42,-0.0034959975246825438],["leaf",45,0.002026353958671173],["leaf",46,-0.018310857207530364],["leaf",47,0.02813019027174772],["leaf",48,0.017003957792047105],["leaf",49,0.015301364316714047],["leaf",50,0.004900985499153537],["leaf",51,0.008304948902454149],["leaf",52,0.02437132512326964],["leaf",53,-0.014789750058824517],["leaf",54,0.005160187455313407]],[["split",0,10.0,7,0],["split",1,7.0,11,1],["split",2,1.0,0,1],["split",3,3.0,27,2],["split",4,25.0,16,2],["split",5,3.0,7,2],["split",6,2.0,2,2],["split",7,20.0,0,3],["leaf",8,0.010409241137855933],["split",9,3.0,8,3],["split",10,1.0,12,3],["leaf",11,-0.004469428703329008],["leaf",12,-0.028400614470834026],["split",13,20.0,0,3],["split",14,26.0,5,3],["split",15,10.0,2,4],["leaf",16,0.006218097017619361],["split",19,1.0,19,4],["split",20,6.0,22,4],["split",21,23.0,21,4],["split",22,24.0,24,4],["split",27,23.0,7,4],["leaf",28,-0.006911244739363056],["split",29,19.0,0,4],["split",30,25.0,30,4],["leaf",31,-0.0221444820521167],["leaf",32,-0.00843679156642186],["leaf",39,-0.006182192104004114],["leaf",40,-0.022749346209899553],["leaf",41,0.0014833536218496891],["leaf",42,-0.009042239975049396],["leaf",43,0.003445664504039272],["leaf",44,-0.010465848537844498],["leaf",45,0.00355469713094994],["leaf",46,0.018403295947575848],["leaf",55,0.004089576924264708],["leaf",56,0.015294677572668468],["leaf",59,-0.006063306172226447],["leaf",60,0.008421558073177427],["leaf",61,0.0015343119425159748],["leaf",62,0.0173577011746844]],[["split",0,7.0,9,0],["split",1,19.0,0,1],["split",2,1.0,1,1],["split",3,21.0,18,2],["split",4,9.0,13,2],["split",5,3.0,6,2],["split",6,22.0,17,2],["split",7,23.0,19,3],["split",8,20.0,0,3],["split",9,25.0,19,3],["split",10,3.0,13,3],["leaf",11,0.004833472942913629],["split",12,13.0,15,3],["split",13,26.0,22,3],["split",14,26.0,8,3],["split",15,0.0,5,4],["split",16,3.0,26,4],["split",17,3.0,26,4],["split",18,1.0,16,4],["leaf",19,-0.0061052730689121855],["leaf",20,0.004106857093637669],["leaf",21,0.02377952405298042],["leaf",22,0.003461192688424758],["leaf",25,-0.008175651181327064],["leaf",26,-0.02673960624231573],["split",27,20.0,0,4],["split",28,11.0,5,4],["split",29,0.0,3,4],["split",30,17.0,0,4],["leaf",31,-0.00947168598630986],["leaf",32,-0.0006014530831705007],["leaf",33,-0.01628818585739705],["leaf",34,0.011003341475882592],["leaf",35,-0.003914687253717314],["leaf",36,0.012418792211747232],["leaf",37,0.02281635839161241],["leaf",38,0.0024750608438367755],["leaf",55,0.0020130016110596017],["leaf",56,0.011033099287190746],["leaf",57,0.01610590217911416],["leaf",58,0.007270819730255929],["leaf",59,0.005509425162609561],["leaf",60,-0.008785246396959896],["leaf",61,0.0005000792129548482],["leaf",62,0.010262457610433004]],[["split",0,12.0,14,0],["split",1,21.0,8,1],["split",2,24.0,7,1],["split",3,19.0,0,2],["split",4,20.0,0,2],["split",5,9.0,29,2],["split",6,1.0,12,2],["split",7,7.0,21,3],["split",8,26.0,11,3],["split",9,21.0,30,3],["split",10,2.0,15,3],["split",11,11.0,7,3],["leaf",12,0.012965489743718361],["split",13,17.0,0,3],["split",14,26.0,1,3],["split",15,4.0,15,4],["split",16,26.0,6,4],["leaf",17,0.013223998885118876],["leaf",18,-0.0017573163784481835],["split",19,1.0,29,4],["leaf",20,-0.018205455050434045],["split",21,25.0,17,4],["split",22,23.0,22,4],["split",23,4.0,15,4],["split",24,26.0,2,4],["split",27,13.0,14,4],["split",28,3.0,17,4],["leaf",29,-0.010919267445308389],["split",30,9.0,30,4],["leaf",31,-0.0034033158688484715],["leaf",32,-0.012056623996583707],["leaf",33,0.01035528877863166],["leaf",34,-0.0029890429622907093],["leaf",39,-0.0030222999049788484],["leaf",40,0.013462003273518241],["leaf",43,-0.0173787350637327],["leaf",44,-0.00396385893502483],["leaf",45,0.006455384938147096],["leaf",46,0.030441513085141397],["leaf",47,-0.004989922468863269],["leaf",48,-0.02401315636476704],["leaf",49,-0.016186671656791093],["leaf",50,0.0012065085879362555],["leaf",55,0.00984325384442997],["leaf",56,-0.0013273166068888138],["leaf",57,-0.019279382789003204],["leaf",58,-4.612319996126971e-06],["leaf",61,0.005973680315316014],["leaf",62,0.01987336073077954]],[["split",0,24.0,5,0],["split",1,22.0,29,1],["split",2,10.0,7,1],["split",3,7.0,29,2],["leaf",4,-0.022260552103139986],["split",5,7.0,19,2],["split",6,6.0,26,2],["split",7,5.0,2,3],["leaf",8,0.00905687560274358],["split",11,25.0,11,3],["split",12,5.0,13,3],["split",13,2.0,8,3],["split",14,19.0,0,3],["leaf",15,0.008787919868417994],["split",16,0.0,2,4],["split",23,23.0,20,4],["split",24,6.0,2,4],["split",25,2.0,5,4],["split",26,25.0,10,4],["split",27,20.0,0,4],["split",28,26.0,0,4],["split",29,25.0,15,4],["split",30,25.0,15,4],["leaf",33,-0.016351240457265954],["leaf",34,-0.004852214845506526],["leaf",47,-0.006573838736180623],["leaf",48,-0.01733230068958],["leaf",49,0.008461180407776742],["leaf",50,-0.004143213428818693],["leaf",51,0.012668139149997605],["leaf",52,0.0020061386704288706],["leaf",53,-0.014012219083946868],["leaf",54,0.002279580967074139],["leaf",55,0.007839405130687675],["leaf",56,-0.007416256165564786],["leaf",57,-0.010790880277576461],["leaf",58,0.001467899218203985],["leaf",59,0.007102612025212511],["leaf",60,0.015483984548046132],["leaf",61,-0.00019592887178446715],["leaf",62,-0.007311969676579153]],[["split",0,6.0,13,0],["split",1,7.0,1,1],["split",2,20.0,0,1],["split",3,8.0,0,2],["split",4,20.0,0,2],["split",5,23.0,26,2],["split",6,21.0,24,2],["split",7,1.0,4,3],["split",8,25.0,24,3],["split",9,23.0,27,3],["split",10,2.0,20,3],["split",11,3.0,21,3],["split",12,2.0,9,3],["split",13,25.0,12,3],["split",14,26.0,16,3],["leaf",15,0.007811588360646251],["leaf",16,-0.010781822204151274],["split",17,4.0,26,4],["leaf",18,-0.006034971853438092],["split",19,26.0,23,4],["split",20,6.0,1,4],["split",21,25.0,18,4],["split",22,21.0,14,4],["split",23,24.0,8,4],["split",24,9.0,30,4],["split",25,4.0,12,4],["split",26,0.0,12,4],["split",27,9.0,18,4],["split",28,22.0,12,4],["leaf",29,0.02655894096034905],["leaf",30,0.007652041351834132],["leaf",35,-0.011636675520315047],["leaf",36,-0.025286123722320763],["leaf",39,-0.003201150087253448],["leaf",40,0.006139254390479827],["leaf",41,0.0088167226989337],["leaf",42,-0.017468336001270936],["leaf",43,-0.017022046986105332],["leaf",44,0.00451354077201471],["leaf",45,0.005893051897048706],["leaf",46,0.029159441402773743],["leaf",47,-0.004249411564137358],["leaf",48,0.0048963614145885554],["leaf",49,0.006549983847716021],["leaf",50,0.018704818179878555],["leaf",51,0.013130264102647045],["leaf",52,-0.011250699476279727],["leaf",53,-0.024965160224433854],["leaf",54,-0.010894445470933609],["leaf",55,-0.012380120335695478],["leaf",56,-0.026347353357539936],["leaf",57,0.0060721362662147305],["leaf",58,-0.010193838961831646]],[["split",0,2.0,3,0],["split",1,20.0,0,1],["split",2,10.0,2,1],["split",3,9.0,21,2],["split",4,22.0,18,2],["split",5,25.0,2,2],["split",6,1.0,1,2],["split",7,8.0,29,3],["split",8,0.0,25,3],["leaf",9,0.0006099066037523578],["leaf",10,-0.02292187698267931],["leaf",11,-0.023704379074605618],["split",12,11.0,17,3],["split",13,10.0,8,3],["split",14,21.0,1,3],["split",15,10.0,4,4],["leaf",16,0.01541734243292764],["split",17,3.0,14,4],["leaf",18,0.024938081020590426],["split",25,25.0,23,4],["split",26,13.0,17,4],["split",27,11.0,16,4],["split",28,0.0,10,4],["split",29,5.0,13,4],["split",30,11.0,27,4],["leaf",31,0.015391838133031749],["leaf",32,0.00137948383459147],["leaf",35,-0.0010647115313395321],["leaf",36,0.013949610085661621],["leaf",51,-0.013401275895787754],["leaf",52,-0.00037702270600513936],["leaf",53,-0.00521950561710133],["leaf",54,0.0060422409886296356],["leaf",55,0.007096492655264339],["leaf",56,-0.00860459446570155],["leaf",57,-0.006040863404790759],["leaf",58,-0.020487401813286073],["leaf",59,-0.020965834263123075],["leaf",60,-0.0039014610121538666],["leaf",61,-0.0006031974570359687],["leaf",62,0.005860770461202917]],[["split",0,10.0,2,0],["split",1,7.0,1,1],["split",2,7.0,12,1],["leaf",3,-0.022097293268440312],["split",4,2.0,22,2],["split",5,24.0,16,2],["split",6,3.0,6,2],["split",9,4.0,6,3],["split",10,0.0,13,3],["split",11,2.0,28,3],["split",12,5.0,21,3],["split",13,20.0,0,3],["split",14,17.0,0,3],["leaf",19,0.0006134002968103656],["split",20,24.0,9,4],["leaf",21,0.0075279870272982035],["leaf",22,-0.0026084764188719738],["split",23,12.0,8,4],["split",24,7.0,2,4],["split",25,12.0,19,4],["split",26,9.0,4,4],["split",27,16.0,0,4],["split",28,2.0,15,4],["split",29,21.0,20,4],["split",30,23.0,23,4],["leaf",41,-0.0034220385305780573],["leaf",42,-0.015904414518405116],["leaf",47,0.001974294193517398],["leaf",48,-0.005370935519194612],["leaf",49,-0.02148846653994262],["leaf",50,-0.0017221854268171298],["leaf",51,-0.00349538674227787],["leaf",52,0.006605166842596899],["leaf",53,0.019216886564633422],["leaf",54,0.002222925420949551],["leaf",55,-0.0014179580125405465],["leaf",56,-0.0151646622498515],["leaf",57,0.001109402672914472],["leaf",58,0.029377683810641746],["leaf",59,0.0017933829136965085],["leaf",60,0.006179816478826629],["leaf",61,0.013646005900472507],["leaf",62,-0.008048482314186799]],[["split",0,4.0,21,0],["split",1,25.0,20,1],["split",2,2.0,18,1],["split",3,21.0,6,2],["split",4,7.0,27,2],["split",5,10.0,23,2],["split",6,20.0,0,2],["split",7,20.0,0,3],["split",8,6.0,9,3],["split",9,8.0,3,3],["split",10,24.0,5,3],["split",11,12.0,2,3],["split",12,12.0,5,3],["split",13,23.0,11,3],["split",14,23.0,17,3],["split",15,24.0,8,4],["split",16,13.0,17,4],["split",17,4.0,7,4],["split",18,1.0,29,4],["leaf",19,0.015158249898766143],["split",20,22.0,16,4],["leaf",21,-0.00421275619716456],["split",22,10.0,24,4],["leaf",23,0.008438742944076986],["split",24,7.0,4,4],["leaf",25,-0.0035286099658133996],["split",26,10.0,28,4],["split",27,1.0,7,4],["split",28,17.0,0,4],["leaf",29,-0.00920754747024735],["split",30,9.0,12,4],["leaf",31,-0.009919774465845162],["leaf",32,0.0005937920674622183],["leaf",33,-0.010076484192583616],["leaf",34,-0.023992493767545267],["leaf",35,0.004453515860236848],["leaf",36,-0.008545501026317598],["leaf",37,0.00278015865350112],["leaf",38,-0.00853489648702511],["leaf",41,0.004854445014086899],["leaf",42,-5.011346001745128e-05],["leaf",45,0.020643302191195406],["leaf",46,0.0028997110626126977],["leaf",49,-0.018729362252636304],["leaf",50,-0.003941488146174308],["leaf",53,0.018903146591316644],["leaf",54,0.004818047358834623],["leaf",55,-0.013558550679693223],["leaf",56,0.0006737789109795666],["leaf",57,-0.010258076134202255],["leaf",58,-0.02359094754490099],["leaf",61,0.004681155833449443],["leaf",62,0.01868612884651715]],[["split",0,7.0,30,0],["split",1,21.0,10,1],["split",2,24.0,14,1],["split",3,20.0,0,2],["split",4,10.0,14,2],["leaf",5,0.003356086560035997],["split",6,25.0,14,2],["split",7,11.0,27,3],["split",8,23.0,24,3],["split",9,10.0,0,3],["split",10,25.0,27,3],["leaf",13,0.011775288938739732],["leaf",14,0.027357989287372472],["split",15,23.0,28,4],["split",16,9.0,24,4],["split",17,22.0,7,4],["leaf",18,0.006743847616918406],["split",19,4.0,18,4],["split",20,8.0,0,4],["split",21,1.0,1,4],["split",22,20.0,0,4],["leaf",31,0.0007959657576547273],["leaf",32,-0.01223838046878571],["leaf",33,-0.014903946897034429],["leaf",34,0.005988823020797785],["leaf",35,0.0004158947704616724],["leaf",36,-0.024405697468629084],["leaf",39,-0.002935848832466476],["leaf",40,-0.023803046359159943],["leaf",41,0.011032840214115495],["leaf",42,-0.0013191299060263597],["leaf",43,-0.011367437586102801],["leaf",44,0.0030362947349209892],["leaf",45,0.007322803085392306],["leaf",46,0.027960105833131414]],[["split",0,1.0,9,0],["split",1,4.0,29,1],["split",2,26.0,8,1],["split",3,3.0,1,2],["split",4,6.0,2,2],["split",5,7.0,14,2],["split",6,25.0,23,2],["split",7,9.0,14,3],["split",8,20.0,0,3],["split",9,25.0,11,3],["split",10,9.0,10,3],["split",11,3.0,11,3],["split",12,9.0,17,3],["split",13,20.0,0,3],["split",14,20.0,0,3],["leaf",15,-0.0027091529587770077],["leaf",16,0.022082270104726586],["split",17,3.0,30,4],["split",18,2.0,14,4],["leaf",19,0.009036506678285015],["leaf",20,-0.011700483910134084],["leaf",21,-0.034243342859021295],["leaf",22,-0.017042357502091914],["split",23,24.0,26,4],["split",24,7.0,6,4],["split",25,10.0,5,4],["split",26,0.0,12,4],["split",27,9.0,25,4],["split",28,25.0,10,4],["split",29,23.0,25,4],["split",30,4.0,13,4],["leaf",35,-0.004672164020430971],["leaf",36,0.01362286412771922],["leaf",37,-0.011724722280586018],["leaf",38,0.013063915558206926],["leaf",47,-0.006057160041807545],["leaf",48,0.012772287812356965],["leaf",49,-0.024034568370309824],["leaf",50,-0.009444792168099774],["leaf",51,-0.008156349651426066],["leaf",52,0.0065501439852429805],["leaf",53,0.003514628737023299],["leaf",54,-0.01219851370022994],["leaf",55,0.001127821986926528],["leaf",56,0.011016820587208705],["leaf",57,-0.021627033851620937],["leaf",58,-0.001082664424877804],["leaf",59,0.0082513167702259],["leaf",60,-0.004963967493898127],["leaf",61,0.008661455089825004],["leaf",62,0.03288387506860748]],[["split",0,21.0,5,0],["split",1,20.0,0,1],["split",2,25.0,27,1],["split",3,7.0,15,2],["split",4,7.0,22,2],["split",5,2.0,21,2],["split",6,20.0,0,2],["split",7,8.0,0,3],["split",8,13.0,13,3],["split",9,13.0,10,3],["leaf",10,-0.004136258459626907],["split",11,14.0,0,3],["split",12,15.0,0,3],["split",13,2.0,2,3],["split",14,2.0,16,3],["leaf",15,0.006850588272933076],["split",16,9.0,16,4],["leaf",17,0.014916504630106684],["split",18,2.0,4,4],["leaf",19,-0.013122649832548862],["leaf",20,-0.03307386535804558],["split",23,21.0,28,4],["split",24,0.0,8,4],["split",25,20.0,0,4],["split",26,0.0,13,4],["leaf",27,0.016047328089561358],["split",28,24.0,20,4],["leaf",29,0.015851041022200925],["leaf",30,0.03835675905172641],["leaf",33,-0.015071300560369788],["leaf",34,-0.005200856029704416],["leaf",37,0.010209910195945525],["leaf",38,-0.004320684389777347],["leaf",47,0.0017838675174582574],["leaf",48,-0.006958179007380256],["leaf",49,0.0002125594699585942],["leaf",50,0.01055939329975139],["leaf",51,-0.0038926460671227883],["leaf",52,0.013495863700923623],["leaf",53,-0.021304656683382644],["leaf",54,-0.007226981709778079],["leaf",57,-0.004824266955073112],["leaf",58,0.007048718491560758]],[["split",0,7.0,8,0],["split",1,12.0,16,1],["split",2,15.0,0,1],["split",3,3.0,26,2],["split",4,3.0,17,2],["split",5,21.0,8,2],["split",6,7.0,27,2],["split",7,11.0,10,3],["split",8,10.0,13,3],["leaf",9,-0.029508900464856148],["split",10,1.0,14,3],["split",11,22.0,22,3],["split",12,1.0,1,3],["split",13,26.0,17,3],["split",14,0.0,13,3],["split",15,25.0,12,4],["split",16,7.0,0,4],["leaf",17,0.019106565617830536],["leaf",18,-0.0007763500499878593],["leaf",21,-0.018927151610012898],["leaf",22,0.006817729579091208],["split",23,25.0,9,4],["split",24,6.0,19,4],["split",25,9.0,9,4],["split",26,12.0,29,4],["split",27,6.0,20,4],["split",28,2.0,9,4],["leaf",29,0.008155874210453743],["leaf",30,0.029601895587661985],["leaf",31,-0.005284346002083992],["leaf",32,0.004445500317252729],["leaf",33,-0.017893113014030862],["leaf",34,-0.004879273621542924],["leaf",47,-0.006026514280431876],["leaf",48,0.0026096234492613397],["leaf",49,-0.0017310128001815637],["leaf",50,-0.014271722049702288],["leaf",51,-0.0190268648746128],["leaf",52,0.0007775908898703973],["leaf",53,0.0021943144296523156],["leaf",54,0.009192861145028318],["leaf",55,0.005022577008797765],["leaf",56,-0.01343077940665433],["leaf",57,0.01591836906147333],["leaf",58,0.007855386463783849]],[["split",0,2.0,2,0],["split",1,20.0,0,1],["split",2,23.0,29,1],["split",3,0.0,3,2],["leaf",4,-0.015832042760048758],["split",5,3.0,4,2],["split",6,20.0,0,2],["leaf",7,0.025500629306530864],["split",8,11.0,6,3],["split",11,9.0,9,3],["split",12,7.0,0,3],["split",13,13.0,1,3],["leaf",14,0.023085260971370702],["split",17,9.0,17,4],["split",18,24.0,12,4],["split",23,13.0,5,4],["split",24,25.0,22,4],["leaf",25,-0.019330374224548367],["split",26,26.0,1,4],["leaf",27,0.003137376940746587],["split",28,25.0,9,4],["leaf",35,0.010997702618598247],["leaf",36,0.022712885155099016],["leaf",37,-0.0007910885797837089],["leaf",38,0.009262625251985068],["leaf",47,0.007240637353647444],["leaf",48,-0.014760163060797176],["leaf",49,0.0026630384306105797],["leaf",50,0.014749221376907776],["leaf",53,-0.009726788650797404],["leaf",54,-0.00019301112711349132],["leaf",57,-0.005081824116079546],["leaf",58,-0.022861831164533244]],[["split",0,7.0,21,0],["split",1,25.0,13,1],["split",2,3.0,18,1],["split",3,7.0,0,2],["split",4,18.0,0,2],["split",5,13.0,15,2],["split",6,15.0,0,2],["leaf",7,-0.017830242393421614],["split",8,20.0,0,3],["split",9,20.0,0,3],["split",10,3.0,6,3],["split",11,25.0,19,3],["split",12,3.0,5,3],["split",13,13.0,21,3],["split",14,11.0,15,3],["split",17,6.0,1,4],["split",18,2.0,25,4],["split",19,2.0,27,4],["split",20,23.0,24,4],["leaf",21,0.02385626367162937],["split",22,2.0,20,4],["leaf",23,0.0022244101220950854],["leaf",24,0.0184653043807834],["split",25,20.0,0,4],["split",26,1.0,17,4],["leaf",27,0.015454841302460222],["split",28,10.0,8,4],["leaf",29,0.012474020601078077],["leaf",30,0.022026464362724377],["leaf",35,0.009686419487913893],["leaf",36,-0.0028438053785088146],["leaf",37,-0.016817243022415617],["leaf",38,0.006060181805289676],["leaf",39,-0.0014225910086219774],["leaf",40,-0.010644301329409534],["leaf",41,0.003985628969888601],["leaf",42,0.02380789431748422],["leaf",45,-0.0015582227580779018],["leaf",46,0.009257806589590344],["leaf",51,-0.00863907827444048],["leaf",52,0.012731750124024855],["leaf",53,-0.0032934297589925026],["leaf",54,0.004976717590668077],["leaf",57,-0.004235089333553873],["leaf",58,0.006306901419091488]],[["split",0,4.0,9,0],["split",1,20.0,0,1],["split",2,26.0,28,1],["split",3,19.0,0,2],["split",4,21.0,16,2],["split",5,7.0,1,2],["split",6,24.0,21,2],["split",7,3.0,24,3],["split",8,7.0,26,3],["split",9,23.0,16,3],["split",10,11.0,20,3],["split",11,19.0,0,3],["split",12,13.0,1,3],["split",13,5.0,23,3],["leaf",14,0.015792713787204368],["split",15,7.0,28,4],["split",16,23.0,23,4],["split",17,6.0,13,4],["leaf",18,0.008126002431210916],["leaf",19,-0.019042619646227464],["leaf",20,-0.0017516457800683598],["leaf",21,-0.0021540826338477754],["leaf",22,0.009505156107107938],["split",23,25.0,26,4],["leaf",24,0.011248453030947677],["split",25,2.0,23,4],["split",26,14.0,0,4],["split",27,9.0,18,4],["leaf",28,0.012577461937627207],["leaf",31,0.0007168658005077806],["leaf",32,0.013858028005186178],["leaf",33,0.008265147705724955],["leaf",34,0.02022007851273371],["leaf",35,0.005342232786007032],["leaf",36,-0.009138911561575693],["leaf",47,-0.015890466499215935],["leaf",48,0.004069879499334382],["leaf",51,0.004998826860867353],["leaf",52,0.020482058790264837],["leaf",53,-0.000988434974772024],["leaf",54,-0.007275217841073821],["leaf",55,-0.008650541999766111],["leaf",56,0.0035886887273743977]],[["split",0,4.0,16,0],["split",1,7.0,26,1],["split",2,7.0,9,1],["split",3,13.0,2,2],["split",4,2.0,4,2],["split",5,19.0,0,2],["split",6,23.0,21,2],["leaf",7,0.018746870515801744],["split",8,26.0,22,3],["split",9,7.0,30,3],["split",10,3.0,15,3],["split",11,12.0,17,3],["split",12,3.0,4,3],["split",13,16.0,0,3],["split",14,20.0,0,3],["split",17,1.0,3,4],["split",18,4.0,2,4],["split",19,9.0,8,4],["leaf",20,0.022404369222788146],["split",21,26.0,18,4],["split",22,26.0,11,4],["split",23,21.0,19,4],["leaf",24,-0.01974840941749616],["leaf",25,0.017872929873388232],["split",26,22.0,13,4],["split",27,11.0,2,4],["split",28,22.0,13,4],["split",29,10.0,16,4],["leaf",30,0.014480581992715255],["leaf",35,-0.013690961027230967],["leaf",36,-0.00032660991463384534],["leaf",37,0.00860624486482357],["leaf",38,0.0011479427791102816],["leaf",39,-0.0006466206670421477],["leaf",40,0.01275721090442164],["leaf",43,0.004224061547790462],["leaf",44,-0.013899967551066492],["leaf",45,0.015553978257195447],["leaf",46,0.004999838274660604],["leaf",47,-0.009594966727977576],["leaf",48,-0.0012911085510212321],["leaf",53,-0.008907946614876717],["leaf",54,0.007720186805597072],["leaf",55,-0.008221232065362223],["leaf",56,0.0036711498800010346],["leaf",57,-0.014324510642474227],["leaf",58,-0.0009904178401236518],["leaf",59,-0.004589136899949867],["leaf",60,-0.019607656221621975]],[["split",0,7.0,4,0],["split",1,19.0,0,1],["split",2,26.0,0,1],["split",3,4.0,29,2],["split",4,9.0,15,2],["split",5,10.0,19,2],["split",6,9.0,27,2],["split",7,5.0,24,3],["split",8,3.0,3,3],["leaf",9,-0.0024810181004310985],["leaf",10,0.014717548521169419],["split",11,11.0,15,3],["leaf",12,7.0087652248929e-05],["split",13,9.0,26,3],["split",14,17.0,0,3],["split",15,4.0,13,4],["split",16,0.0,16,4],["leaf",17,-0.003183933313322089],["split",18,2.0,29,4],["leaf",23,-0.012675998394602274],["leaf",24,-0.02217556453542878],["split",27,23.0,25,4],["split",28,26.0,17,4],["split",29,22.0,8,4],["split",30,11.0,14,4],["leaf",31,0.005046243221060629],["leaf",32,-0.015209527245157223],["leaf",33,-0.005972284088305338],["leaf",34,0.008368267038915012],["leaf",37,-0.011039718222810854],["leaf",38,-0.032652967265407416],["leaf",55,0.0014399132633407376],["leaf",56,-0.0034044393078950424],["leaf",57,-0.014213820490547143],["leaf",58,-0.0033394738539042],["leaf",59,0.012889570299299711],["leaf",60,-0.0011354209202767797],["leaf",61,0.026525298028563845],["leaf",62,0.013240549504264048]],[["split",0,7.0,18,0],["split",1,21.0,2,1],["split",2,25.0,8,1],["split",3,25.0,11,2],["split",4,26.0,0,2],["split",5,23.0,11,2],["split",6,22.0,6,2],["split",7,22.0,10,3],["split",8,11.0,18,3],["leaf",9,-0.01868968663747939],["split",10,10.0,6,3],["split",11,10.0,24,3],["split",12,18.0,0,3],["split",13,6.0,15,3],["split",14,10.0,1,3],["leaf",15,-0.014613677453530396],["leaf",16,0.006240634345834546],["split",17,13.0,5,4],["leaf",18,-0.0252070212355633],["split",21,20.0,0,4],["split",22,21.0,17,4],["split",23,11.0,7,4],["leaf",24,0.014359102645873636],["split",25,13.0,15,4],["leaf",26,-0.017062035589794116],["leaf",27,0.025072888278116292],["split",28,19.0,0,4],["leaf",29,0.011521032017364501],["split",30,3.0,3,4],["leaf",35,-0.0011530803076948261],["leaf",36,-0.013780161945549672],["leaf",43,-0.0076142666764116695],["leaf",44,0.008331808457557092],["leaf",45,-0.0025008694510642786],["leaf",46,0.0021355577329040774],["leaf",47,0.010759949093937859],["leaf",48,-0.005309585573114521],["leaf",51,0.0072143481369667094],["leaf",52,-0.0044262869808442806],["leaf",57,0.008551319077017014],["leaf",58,-0.004298219169451395],["leaf",61,-0.006852995719992507],["leaf",62,0.0026363572025431864]],[["split",0,7.0,20,0],["split",1,13.0,1,1],["split",2,9.0,8,1],["split",3,22.0,14,2],["split",4,10.0,0,2],["split",5,2.0,1,2],["split",6,21.0,26,2],["split",7,11.0,9,3],["split",8,0.0,20,3],["split",9,13.0,7,3],["split",10,1.0,0,3],["leaf",11,0.012882579212278047],["split",12,10.0,23,3],["split",13,25.0,28,3],["split",14,6.0,21,3],["leaf",15,0.010364317607763353],["split",16,4.0,26,4],["split",17,2.0,21,4],["leaf",18,0.004204672251489531],["leaf",19,-0.0019262497455674436],["leaf",20,-0.021439763618741805],["leaf",21,-0.019397075101910664],["split",22,12.0,24,4],["split",25,1.0,25,4],["split",26,0.0,12,4],["split",27,15.0,0,4],["split",28,9.0,17,4],["leaf",29,0.00947101472302872],["split",30,3.0,11,4],["leaf",33,0.003292551003058859],["leaf",34,-0.012319560710692441],["leaf",35,0.029166880297038372],["leaf",36,0.010447221480810192],["leaf",45,-0.001493205446366014],["leaf",46,0.003895475030880231],["leaf",51,0.0034725635705371615],["leaf",52,-0.006690746099726915],["leaf",53,-0.015305234253571906],["leaf",54,-0.000790050996107439],["leaf",55,0.0047042263059438715],["leaf",56,0.013861066785735508],["leaf",57,0.009262223690496821],["leaf",58,0.0245855703274601],["leaf",61,-0.013608560750545952],["leaf",62,-0.0004629752517690033]],[["split",0,7.0,29,0],["split",1,26.0,21,1],["split",2,2.0,11,1],["split",3,6.0,12,2],["split",4,1.0,28,2],["split",5,5.0,5,2],["split",6,4.0,18,2],["split",7,19.0,0,3],["split",8,11.0,12,3],["split",9,12.0,14,3],["leaf",10,-0.007108711279437846],["leaf",11,0.024278843649804396],["split",12,6.0,25,3],["split",13,9.0,13,3],["leaf",14,0.010014609713110939],["split",15,9.0,11,4],["split",16,3.0,7,4],["split",17,23.0,26,4],["split",18,7.0,25,4],["split",19,4.0,22,4],["split",20,13.0,18,4],["leaf",25,0.020541355706252363],["split",26,22.0,15,4],["leaf",27,-0.017859474322692907],["leaf",28,-0.004509930234475051],["leaf",31,0.0015524069984626195],["leaf",32,-0.003600393827830851],["leaf",33,0.018787306721120153],["leaf",34,-0.0004208835587294237],["leaf",35,-0.0068183267590017724],["leaf",36,-0.02032909468616352],["leaf",37,-0.0037891217231226164],["leaf",38,0.003291921920519954],["leaf",39,-0.0024898960706913513],["leaf",40,0.004911216216002506],["leaf",41,0.007379438890840675],["leaf",42,0.0023868676328664138],["leaf",53,0.002860360525124291],["leaf",54,0.00010373926338795608]],[["split",0,25.0,30,0],["split",1,4.0,27,1],["split",2,21.0,8,1],["split",3,9.0,6,2],["split",4,2.0,25,2],["leaf",5,0.0002703936728739266],["split",6,2.0,16,2],["split",7,21.0,1,3],["split",8,7.0,27,3],["split",9,5.0,30,3],["split",10,7.0,0,3],["leaf",13,0.004017655363150744],["leaf",14,0.024180554805928768],["leaf",15,-0.01911086115045585],["split",16,11.0,18,4],["split",17,6.0,20,4],["split",18,15.0,0,4],["split",19,21.0,13,4],["leaf",20,0.011745067378793857],["leaf",21,-0.023438049854295564],["split",22,24.0,5,4],["leaf",33,-0.005313637425102646],["leaf",34,0.0016486022274290755],["leaf",35,0.0008957703716593531],["leaf",36,-0.00387164623834086],["leaf",37,0.0031928415860523994],["leaf",38,0.016298202048674406],["leaf",39,-0.00804123929446506],["leaf",40,0.0008945697077632226],["leaf",45,-0.02218613129079525],["leaf",46,-0.00591138639675131]],[["split",0,7.0,28,0],["split",1,4.0,4,1],["split",2,24.0,21,1],["split",3,23.0,18,2],["split",4,1.0,28,2],["split",5,2.0,17,2],["split",6,26.0,6,2],["split",7,19.0,0,3],["split",8,13.0,27,3],["split",9,13.0,2,3],["split",10,10.0,13,3],["split",11,10.0,10,3],["leaf",12,-0.0007773284068229232],["leaf",13,-0.01300669993583959],["split",14,11.0,16,3],["split",15,20.0,0,4],["leaf",16,-0.01241793446297266],["split",17,23.0,21,4],["leaf",18,8.120565955121723e-06],["split",19,21.0,14,4],["split",20,23.0,30,4],["split",21,24.0,19,4],["split",22,3.0,16,4],["split",23,8.0,26,4],["split",24,21.0,7,4],["leaf",29,0.01382973421522805],["leaf",30,-0.0032346850189279453],["leaf",31,0.006546667868439653],["leaf",32,-0.007212867618653989],["leaf",35,0.003048589168325306],["leaf",36,0.01612032353585567],["leaf",39,-0.0021668812429220677],["leaf",40,0.010651746578726796],["leaf",41,-0.0004768052706664941],["leaf",42,-0.014826582007342328],["leaf",43,-0.007973754800560529],["leaf",44,-0.02085560780740277],["leaf",45,-0.0047293889873101696],["leaf",46,0.008399196117283606],["leaf",47,0.020404828728326385],["leaf",48,-0.005357639144327909],["leaf",49,0.006929266612706967],["leaf",50,0.022944201987597713]],[["split",0,7.0,30,0],["split",1,26.0,0,1],["split",2,10.0,14,1],["split",3,8.0,5,2],["split",4,10.0,6,2],["leaf",5,0.020493968052335748],["split",6,9.0,14,2],["leaf",7,-0.0005788894645080948],["split",8,10.0,13,3],["split",9,17.0,0,3],["split",10,1.0,27,3],["leaf",13,0.013203193450324413],["leaf",14,-0.005717248978770788],["leaf",17,-0.022572529419070193],["leaf",18,-0.009239630470379403],["split",19,1.0,28,4],["split",20,2.0,18,4],["split",21,24.0,28,4],["split",22,21.0,27,4],["leaf",39,0.0003777337465584977],["leaf",40,-0.011777467088878843],["leaf",41,0.0017392128474450986],["leaf",42,-0.022536526910389824],["leaf",43,0.0012125028449173818],["leaf",44,0.006824407305518202],["leaf",45,0.005137871460039227],["leaf",46,0.017375077291534948]],[["split",0,24.0,20,0],["split",1,7.0,23,1],["split",2,25.0,28,1],["split",3,23.0,29,2],["split",4,22.0,19,2],["split",5,13.0,30,2],["split",6,20.0,0,2],["split",7,1.0,8,3],["split",8,0.0,22,3],["split",9,24.0,14,3],["split",10,25.0,11,3],["split",11,9.0,25,3],["leaf",12,0.01384202583151466],["split",13,2.0,10,3],["leaf",14,0.03027834736791003],["split",15,7.0,0,4],["split",16,20.0,0,4],["split",17,10.0,8,4],["leaf",18,-0.024665082734819953],["split",19,10.0,4,4],["split",20,26.0,21,4],["split",21,7.0,28,4],["split",22,7.0,25,4],["split",23,25.0,0,4],["split",24,22.0,21,4],["leaf",27,0.02033742322578115],["split",28,9.0,17,4],["leaf",31,-0.01831493235638064],["leaf",32,-0.005332028338392271],["leaf",33,0.0007728193739695152],["leaf",34,-0.009554187935550344],["leaf",35,-0.0035328171851362103],["leaf",36,-0.01766832504827348],["leaf",39,-0.0069293512066889605],["leaf",40,0.006566893701924312],["leaf",41,0.016982286610844263],["leaf",42,0.0012246861244999355],["leaf",43,0.0020680030920991904],["leaf",44,0.013989051630811307],["leaf",45,-0.01306930568460744],["leaf",46,-0.0027217785731119483],["leaf",47,-0.0143255664474329],["leaf",48,0.0015620821089362147],["leaf",49,0.009881580235179793],["leaf",50,-0.0005264777432178938],["leaf",57,-0.00786773995984898],["leaf",58,0.00899853275574287]],[["split",0,7.0,18,0],["split",1,8.0,8,1],["split",2,19.0,0,1],["split",3,2.0,4,2],["split",4,12.0,4,2],["split",5,1.0,1,2],["split",6,8.0,23,2],["leaf",7,0.016205433619486852],["split",8,19.0,0,3],["split",9,26.0,21,3],["split",10,1.0,8,3],["leaf",11,-0.00855630509433901],["split",12,12.0,6,3],["split",13,25.0,9,3],["split",14,9.0,8,3],["split",17,25.0,30,4],["split",18,10.0,18,4],["split",19,4.0,21,4],["leaf",20,-7.185931715583999e-06],["split",21,20.0,0,4],["split",22,17.0,0,4],["leaf",25,-0.007892144868718112],["split",26,10.0,7,4],["leaf",27,0.011205411534600022],["split",28,11.0,17,4],["leaf",29,-0.019164384522821414],["split",30,9.0,20,4],["leaf",35,-0.0023401253120426207],["leaf",36,0.013252815790958937],["leaf",37,0.0035824719001302896],["leaf",38,0.013793577306459739],["leaf",39,-0.009763573836456796],["leaf",40,-0.02195825197640675],["leaf",43,-0.008544142271976888],["leaf",44,0.0038584905628845393],["leaf",45,-0.0027411747059922606],["leaf",46,0.005170652710982217],["leaf",53,0.00041893271116366757],["leaf",54,0.005439404174799712],["leaf",57,-0.007092681243767761],["leaf",58,0.005000046885134502],["leaf",61,0.0009159858037789518],["leaf",62,-0.010073519428307148]],[["split",0,26.0,1,0],["split",1,5.0,9,1],["split",2,7.0,1,1],["leaf",3,0.010912020507349093],["split",4,21.0,28,2],["split",5,26.0,9,2],["split",6,7.0,15,2],["split",9,1.0,25,3],["leaf",10,-0.023147918955125215],["split",11,11.0,14,3],["split",12,23.0,28,3],["split",13,11.0,20,3],["split",14,13.0,7,3],["split",19,25.0,3,4],["leaf",20,-0.020702637937752785],["leaf",23,0.009041486956868176],["leaf",24,-0.007461450476659988],["split",25,7.0,0,4],["leaf",26,-0.020951398433782912],["split",27,26.0,23,4],["split",28,21.0,29,4],["leaf",29,0.017196272356683664],["split",30,22.0,21,4],["leaf",39,-0.016903644856428042],["leaf",40,0.002490855920764158],["leaf",51,0.0003528719849225653],["leaf",52,-0.01613487355143748],["leaf",55,-0.00368947873523502],["leaf",56,0.0022072946679394983],["leaf",57,0.0016286870440295352],["leaf",58,0.014081811673878914],["leaf",61,0.003579120151902265],["leaf",62,-0.0011610599577587413]],[["split",0,9.0,29,0],["split",1,7.0,3,1],["split",2,11.0,17,1],["split",3,12.0,7,2],["split",4,22.0,21,2],["split",5,5.0,9,2],["split",6,0.0,23,2],["split",7,23.0,18,3],["split",8,2.0,26,3],["split",9,13.0,4,3],["split",10,5.0,26,3],["split",11,4.0,2,3],["split",12,7.0,12,3],["split",13,22.0,15,3],["leaf",14,0.00535491683792524],["split",15,22.0,11,4],["split",16,22.0,19,4],["split",17,21.0,15,4],["leaf",18,-0.024791198105911884],["split",19,9.0,5,4],["split",20,24.0,2,4],["split",21,26.0,27,4],["split",22,10.0,12,4],["leaf",23,0.013165003657280355],["leaf",24,0.000541374324010774],["leaf",25,0.014326460077910826],["leaf",26,0.024228982322897535],["leaf",27,0.0017181841052342523],["leaf",28,-0.015080335637720037],["leaf",31,-0.019100866376394434],["leaf",32,-0.007122061092970003],["leaf",33,-0.0028602240497620315],["leaf",34,0.014044467315791205],["leaf",35,0.00034246476001312693],["leaf",36,-0.01643624986556848],["leaf",39,0.0001272471689778936],["leaf",40,0.012306808057804156],["leaf",41,-0.008257961606985206],["leaf",42,0.0009712487864158619],["leaf",43,-0.005280937911607261],["leaf",44,0.0037820125934501093],["leaf",45,0.013261383931962216],["leaf",46,-0.0017030211122249994]],[["split",0,9.0,7,0],["split",1,13.0,1,1],["split",2,1.0,0,1],["split",3,11.0,15,2],["split",4,22.0,6,2],["split",5,8.0,3,2],["split",6,7.0,0,2],["leaf",7,0.02130999428714245],["leaf",8,-0.0030790109313599324],["split",9,5.0,7,3],["split",10,7.0,2,3],["leaf",11,-0.003060730691673542],["leaf",12,-0.02540786659775381],["split",13,25.0,16,3],["split",14,13.0,12,3],["leaf",19,0.00349237350159137],["split",20,8.0,20,4],["leaf",21,-0.013368087230166449],["split",22,4.0,26,4],["leaf",27,-0.018039847022178736],["leaf",28,0.0019177269537926517],["split",29,1.0,20,4],["split",30,1.0,13,4],["leaf",41,-0.01628064062022345],["leaf",42,-0.004152125321969212],["leaf",45,-0.0020569350972211263],["leaf",46,0.007319680700716484],["leaf",59,0.0061140314570190465],["leaf",60,-0.0007580429870969966],["leaf",61,-0.002125666754989863],["leaf",62,0.0022710537295627616]],[["split",0,24.0,29,0],["split",1,1.0,30,1],["split",2,20.0,0,1],["split",3,5.0,28,2],["split",4,25.0,22,2],["split",5,7.0,18,2],["leaf",6,0.022283845594561907],["split",7,4.0,24,3],["split",8,14.0,0,3],["split",9,23.0,23,3],["leaf",10,-0.02087799096621127],["split",11,10.0,12,3],["split",12,10.0,20,3],["split",15,22.0,30,4],["split",16,12.0,10,4],["split",17,9.0,17,4],["leaf",18,-0.00663987087424044],["leaf",19,-0.0025529340398155525],["leaf",20,-0.013947273698738698],["leaf",23,-0.009973883627949027],["split",24,11.0,17,4],["leaf",25,0.01833014639992375],["leaf",26,-9.830737862089011e-05],["leaf",31,0.0007093790923204246],["leaf",32,-0.01023570248194371],["leaf",33,-0.0017976560578395228],["leaf",34,-0.009991026048804771],["leaf",35,0.0036870343441962813],["leaf",36,0.014865222009460746],["leaf",49,-0.002510939189837035],["leaf",50,0.021301193778130202]],[["split",0,4.0,29,0],["split",1,7.0,19,1],["split",2,25.0,10,1],["split",3,26.0,2,2],["split",4,6.0,12,2],["split",5,10.0,11,2],["split",6,21.0,12,2],["split",7,12.0,2,3],["split",8,21.0,15,3],["split",9,1.0,5,3],["split",10,18.0,0,3],["leaf",11,0.010129982836364834],["leaf",12,-0.008726432023290064],["split",13,25.0,21,3],["split",14,9.0,15,3],["leaf",15,0.004541059404215982],["split",16,8.0,5,4],["split",17,20.0,0,4],["split",18,25.0,30,4],["leaf",19,0.021671115247289822],["leaf",20,0.009523115566890067],["split",21,25.0,26,4],["split",22,4.0,20,4],["leaf",27,-0.012076727856331303],["leaf",28,0.0004748397460949792],["leaf",29,-0.011046217555621435],["leaf",30,-0.02419524681553128],["leaf",33,-0.021771331313610896],["leaf",34,-0.005268621516110663],["leaf",35,-0.00037446246894453917],["leaf",36,-0.009943039272312075],["leaf",37,0.001181956940252228],["leaf",38,0.022587751232950922],["leaf",43,0.001537256440534568],["leaf",44,0.010774761880248339],["leaf",45,-0.002081971291376177],["leaf",46,-0.020592142129887452]],[["split",0,2.0,4,0],["split",1,17.0,0,1],["split",2,7.0,0,1],["split",3,1.0,29,2],["split",4,10.0,14,2],["split",5,26.0,4,2],["split",6,13.0,1,2],["split",7,22.0,9,3],["leaf",8,0.015059713223539507],["leaf",9,0.005370135012094282],["leaf",10,0.02148127422496621],["leaf",11,0.008518414094062813],["split",12,3.0,11,3],["split",13,1.0,23,3],["split",14,3.0,2,3],["split",15,11.0,12,4],["split",16,20.0,0,4],["leaf",25,-0.02388872930199206],["leaf",26,-0.004324816181525638],["split",27,2.0,20,4],["leaf",28,0.02232873047052915],["split",29,25.0,27,4],["split",30,25.0,0,4],["leaf",31,-0.0025801820074247625],["leaf",32,0.008411341230969203],["leaf",33,-0.0012573067394854203],["leaf",34,-0.01110081939141131],["leaf",55,-0.004209293278000049],["leaf",56,0.011285024560151053],["leaf",59,-0.005582533931553173],["leaf",60,-0.020176077131861956],["leaf",61,-0.010542182041983029],["leaf",62,-0.0004000404820705705]],[["split",0,10.0,0,0],["split",1,13.0,4,1],["split",2,7.0,30,1],["leaf",3,0.006168462092247477],["split",4,21.0,20,2],["split",5,0.0,30,2],["split",6,23.0,26,2],["split",9,21.0,10,3],["leaf",10,-0.022563275598687746],["split",11,10.0,30,3],["split",12,11.0,14,3],["split",13,1.0,23,3],["leaf",14,-0.00017443750784249516],["leaf",19,-0.016728893032658815],["leaf",20,-0.001698109769284267],["split",23,7.0,16,4],["split",24,23.0,16,4],["leaf",25,-0.015550566902676099],["split",26,23.0,17,4],["leaf",27,0.01929094380168392],["leaf",28,0.008809816395550571],["leaf",47,-0.00045297080490541567],["leaf",48,0.002066798308181951],["leaf",49,0.002482052355024957],["leaf",50,-0.02130315677342646],["leaf",53,0.0037158929479649125],["leaf",54,-0.009553995474997702]],[["split",0,1.0,23,0],["split",1,4.0,23,1],["split",2,23.0,27,1],["split",3,24.0,20,2],["split",4,20.0,0,2],["split",5,20.0,0,2],["split",6,8.0,23,2],["split",7,3.0,4,3],["split",8,13.0,29,3],["split",9,25.0,16,3],["split",10,2.0,20,3],["split",11,24.0,4,3],["split",12,25.0,19,3],["split",13,11.0,18,3],["leaf",14,0.0018469619400502237],["split",15,3.0,2,4],["split",16,7.0,2,4],["split",17,5.0,11,4],["leaf",18,0.012699786420075243],["split",19,5.0,27,4],["split",20,24.0,10,4],["leaf",21,-0.0064275956123758865],["split",22,23.0,21,4],["split",23,2.0,15,4],["split",24,26.0,5,4],["split",25,5.0,17,4],["leaf",26,-0.0019911394477745412],["leaf",27,-0.009597322127362487],["leaf",28,-0.024151310258998113],["leaf",31,0.003037431872038834],["leaf",32,0.01816866583019743],["leaf",33,0.015134284490795036],["leaf",34,-0.0018500985675712549],["leaf",35,-0.0013365050007234911],["leaf",36,0.005158844052957247],["leaf",39,-0.005188353017883804],["leaf",40,0.005384005365873842],["leaf",41,-0.016455553210066264],["leaf",42,-0.005738825495312356],["leaf",45,0.008180839966646127],["leaf",46,0.02776664669796923],["leaf",47,0.012292579568504975],["leaf",48,-0.0032281454161090712],["leaf",49,-0.008718938140735768],["leaf",50,-0.0003444906006641434],["leaf",51,-0.010624419499175603],["leaf",52,-0.025570908624717083]],[["split",0,7.0,15,0],["split",1,8.0,22,1],["split",2,1.0,1,1],["split",3,10.0,14,2],["leaf",4,-0.013445433923409947],["split",5,24.0,17,2],["split",6,25.0,10,2],["split",7,20.0,0,3],["split",8,17.0,0,3],["leaf",11,-0.02083104307769033],["leaf",12,0.000789653653021229],["split",13,3.0,22,3],["split",14,26.0,3,3],["split",15,19.0,0,4],["split",16,2.0,12,4],["split",17,1.0,19,4],["split",18,4.0,24,4],["split",27,14.0,0,4],["split",28,9.0,28,4],["split",29,3.0,4,4],["split",30,22.0,29,4],["leaf",31,-0.0059774684191573405],["leaf",32,0.0034022642289924386],["leaf",33,-0.015503434049304184],["leaf",34,0.014272212309215885],["leaf",35,0.0015802480329381694],["leaf",36,-0.004140149174311664],["leaf",37,0.012675394097192785],["leaf",38,-0.004850455244370016],["leaf",55,-0.004065543169663625],["leaf",56,0.008575638948665277],["leaf",57,0.002600693546623732],["leaf",58,0.019839290203994567],["leaf",59,0.0202891342989179],["leaf",60,0.006826708566937135],["leaf",61,0.0036077030145298824],["leaf",62,-0.0071872483249759735]],[["split",0,1.0,30,0],["split",1,4.0,30,1],["split",2,6.0,23,1],["split",3,21.0,13,2],["split",4,26.0,12,2],["split",5,10.0,11,2],["leaf",6,0.001206312695835503],["split",7,20.0,0,3],["split",8,8.0,0,3],["split",9,24.0,15,3],["leaf",10,-0.016040850073073196],["leaf",11,0.004681921219105699],["leaf",12,-0.01939381088156079],["split",15,9.0,0,4],["split",16,2.0,18,4],["leaf",17,0.012978016560949074],["split",18,25.0,21,4],["leaf",19,-0.00514894112766249],["leaf",20,0.004762742654684338],["leaf",31,0.013582522150787263],["leaf",32,8.591826879969417e-05],["leaf",33,-0.016249466687483644],["leaf",34,0.006260935650420197],["leaf",37,0.00042122966335381926],["leaf",38,0.00419118383735531]],[["split",0,11.0,20,0],["split",1,7.0,28,1],["split",2,25.0,27,1],["split",3,26.0,1,2],["split",4,1.0,29,2],["split",5,23.0,16,2],["split",6,20.0,0,2],["split",7,3.0,10,3],["split",8,20.0,0,3],["split",9,2.0,5,3],["leaf",10,-0.0019640343223129592],["split",11,20.0,0,3],["split",12,17.0,0,3],["split",13,26.0,5,3],["leaf",14,0.02876717797542452],["split",15,21.0,20,4],["leaf",16,-0.019873194431700294],["split",17,23.0,25,4],["split",18,25.0,23,4],["leaf",19,0.018104156184017457],["split",20,10.0,18,4],["split",23,3.0,5,4],["split",24,9.0,19,4],["split",25,1.0,24,4],["split",26,6.0,11,4],["leaf",27,0.016074757542032355],["split",28,5.0,12,4],["leaf",31,0.0023002434538816592],["leaf",32,-0.0153803967404522],["leaf",35,0.0009978441178918197],["leaf",36,-0.004814251950683319],["leaf",37,-0.010152961192513705],["leaf",38,0.009983496617021772],["leaf",41,0.009666301077153619],["leaf",42,-0.0028239259244552304],["leaf",47,-0.003388648886986572],["leaf",48,0.00707348248057334],["leaf",49,0.0017924000346951896],["leaf",50,-0.013218960010331831],["leaf",51,0.0020999493609784384],["leaf",52,-0.008226694985394693],["leaf",53,-0.020898474074568928],["leaf",54,-7.47521115702368e-05],["leaf",57,0.010278231871916463],["leaf",58,-0.001422384163665705]],[["split",0,21.0,30,0],["split",1,10.0,3,1],["split",2,25.0,27,1],["split",3,22.0,11,2],["split",4,13.0,3,2],["split",5,24.0,13,2],["leaf",6,0.006413230709317673],["split",7,7.0,14,3],["split",8,1.0,26,3],["split",9,22.0,27,3],["split",10,23.0,23,3],["leaf",11,0.0020782717471866732],["split",12,0.0,26,3],["split",15,9.0,12,4],["split",16,22.0,5,4],["split",17,24.0,5,4],["leaf",18,-0.019265924763053305],["split",19,20.0,0,4],["leaf",20,0.019444429795814885],["split",21,17.0,0,4],["split",22,20.0,0,4],["leaf",25,-0.012329335641324426],["leaf",26,-0.02613123353958358],["leaf",31,0.0033219352752430634],["leaf",32,-0.013353210066174546],["leaf",33,0.001867233889787187],["leaf",34,0.014823771344491933],["leaf",35,0.004246731543184285],["leaf",36,-0.007247760137260563],["leaf",39,0.006637530732550655],["leaf",40,-0.005769785204517712],["leaf",43,0.000917860953602678],["leaf",44,0.007723826915170046],["leaf",45,-0.005613985503354418],["leaf",46,0.013097993044820441]],[["split",0,24.0,26,0],["split",1,7.0,28,1],["split",2,20.0,0,1],["split",3,10.0,0,2],["split",4,4.0,3,2],["split",5,8.0,2,2],["split",6,2.0,18,2],["split",7,22.0,12,3],["split",8,3.0,19,3],["leaf",9,0.016614942814728882],["split",10,20.0,0,3],["split",11,24.0,29,3],["split",12,2.0,20,3],["leaf",13,0.004492392068816644],["leaf",14,0.026821615264528594],["leaf",15,-0.0006953913466534258],["leaf",16,-0.01533746985001092],["split",17,19.0,0,4],["split",18,21.0,9,4],["split",21,6.0,23,4],["leaf",22,0.018737854662391255],["leaf",23,0.023027940693283237],["leaf",24,0.0051870822354591496],["split",25,0.0,11,4],["split",26,8.0,14,4],["leaf",35,-0.0024012378710323743],["leaf",36,0.004222156136534594],["leaf",37,-0.002670606548131578],["leaf",38,0.0030391334294414254],["leaf",43,0.009476463473495947],["leaf",44,-0.0034977026010051086],["leaf",51,-0.002694222212058163],["leaf",52,0.006421225649703199],["leaf",53,-0.0005259679846051096],["leaf",54,-0.013653885983312514]],[["split",0,24.0,24,0],["split",1,10.0,8,1],["split",2,5.0,26,1],["split",3,22.0,27,2],["split",4,1.0,6,2],["split",5,11.0,5,2],["split",6,22.0,22,2],["split",7,0.0,1,3],["split",8,23.0,22,3],["split",9,17.0,0,3],["split",10,7.0,29,3],["split",11,5.0,14,3],["split",12,10.0,21,3],["split",13,6.0,6,3],["leaf",14,0.02429881906275852],["leaf",15,0.008734186966617638],["split",16,26.0,6,4],["leaf",17,-0.0092559510163588],["leaf",18,-0.020767648048871986],["split",19,6.0,4,4],["leaf",20,-0.020415291933147058],["split",21,9.0,4,4],["split",22,24.0,4,4],["split",23,21.0,16,4],["split",24,23.0,15,4],["split",25,11.0,28,4],["split",26,3.0,7,4],["split",27,7.0,2,4],["leaf",28,-0.0051794408484313135],["leaf",33,-0.011199715047713932],["leaf",34,-0.0020579765488537094],["leaf",39,0.00761410939426818],["leaf",40,-0.006004951530087969],["leaf",43,-0.004827514342946884],["leaf",44,0.0010840790200103362],["leaf",45,-0.004806518906504907],["leaf",46,0.014303966587656921],["leaf",47,0.007455374110048464],["leaf",48,-0.00696851182694751],["leaf",49,-0.01935118455760522],["leaf",50,-0.006756076747034273],["leaf",51,0.005899253269318527],["leaf",52,-0.005081132304243921],["leaf",53,-0.01332242077388463],["leaf",54,0.0007729822581316282],["leaf",55,0.0005115817861057465],["leaf",56,0.017500116213495964]],[["split",0,25.0,3,0],["split",1,21.0,0,1],["split",2,5.0,20,1],["leaf",3,-0.022049005610112575],["split",4,15.0,0,2],["split",5,3.0,28,2],["split",6,2.0,12,2],["split",9,5.0,27,3],["split",10,21.0,15,3],["split",11,0.0,27,3],["split",12,17.0,0,3],["split",13,26.0,3,3],["split",14,20.0,0,3],["split",19,21.0,27,4],["split",20,26.0,7,4],["leaf",21,0.0007424571300073323],["leaf",22,0.01631536834630399],["split",23,23.0,30,4],["split",24,11.0,24,4],["split",25,20.0,0,4],["leaf",26,0.016222980056627874],["leaf",27,0.022303622568015772],["split",28,1.0,22,4],["split",29,17.0,0,4],["split",30,21.0,14,4],["leaf",39,-0.0035798047921087356],["leaf",40,-0.014893749589242505],["leaf",41,0.0003813012460259436],["leaf",42,0.012019940104800748],["leaf",47,0.0010381090994137385],["leaf",48,-0.013192126596268935],["leaf",49,-0.008350563777242795],["leaf",50,0.003674932923731199],["leaf",51,0.004778800664794775],["leaf",52,-0.008469117027724668],["leaf",57,0.0024995211875468324],["leaf",58,0.015744544867182582],["leaf",59,0.0024613402897328564],["leaf",60,-0.006548685078082171],["leaf",61,0.00011259211640867118],["leaf",62,0.020202215703742175]],[["split",0,14.0,0,0],["split",1,9.0,12,1],["split",2,11.0,1,1],["split",3,7.0,0,2],["split",4,1.0,20,2],["leaf",5,0.01804617182075525],["split",6,6.0,3,2],["leaf",7,-0.013960223875365736],["split",8,13.0,1,3],["split",9,25.0,11,3],["split",10,11.0,25,3],["split",13,0.0,13,3],["split",14,13.0,17,3],["split",17,26.0,7,4],["split",18,8.0,30,4],["split",19,26.0,25,4],["split",20,20.0,0,4],["split",21,3.0,9,4],["split",22,3.0,6,4],["leaf",27,0.0061344500944135],["leaf",28,0.01869149299150602],["split",29,23.0,17,4],["split",30,10.0,19,4],["leaf",35,0.011353441463558874],["leaf",36,0.0057877118647236],["leaf",37,-0.002090175558315676],["leaf",38,-0.014535164441586649],["leaf",39,-0.005955620489797449],["leaf",40,0.005247219255064829],["leaf",41,-0.00013039753122191667],["leaf",42,0.010051136878348514],["leaf",43,-0.002142050974237021],["leaf",44,0.0050537630925423845],["leaf",45,0.025135194751161206],["leaf",46,0.005826289340244492],["leaf",59,0.0017502461393405894],["leaf",60,-0.011589313996609113],["leaf",61,0.010772922985556163],["leaf",62,0.0016225151550390388]],[["split",0,7.0,4,0],["split",1,9.0,2,1],["split",2,24.0,2,1],["split",3,24.0,13,2],["split",4,3.0,11,2],["split",5,13.0,11,2],["split",6,21.0,30,2],["leaf",7,0.019254225673440973],["leaf",8,-0.0019616125566337323],["split",9,23.0,7,3],["split",10,20.0,0,3],["split",11,0.0,17,3],["split",12,9.0,23,3],["split",13,10.0,0,3],["split",14,8.0,21,3],["leaf",19,-0.01560477726021573],["split",20,13.0,3,4],["split",21,4.0,21,4],["leaf",22,0.009228948887798741],["split",23,12.0,7,4],["leaf",24,-0.004597863819549154],["split",25,25.0,16,4],["leaf",26,-0.0174418727428534],["split",27,4.0,11,4],["split",28,1.0,24,4],["leaf",29,-0.013408665541666805],["leaf",30,0.000706190372711177],["leaf",41,-0.0013754595570825315],["leaf",42,0.014776442582828031],["leaf",43,-0.0069980730173465205],["leaf",44,-0.01996169150766979],["leaf",47,0.011013463347630694],["leaf",48,-0.00038424880837316676],["leaf",51,-0.012176084283123709],["leaf",52,-0.0001886378897180417],["leaf",55,-0.01468338788367759],["leaf",56,-0.002321529665804042],["leaf",57,0.002374357498544796],["leaf",58,-0.0011739629813896292]],[["split",0,12.0,30,0],["split",1,23.0,30,1],["split",2,10.0,15,1],["split",3,18.0,0,2],["split",4,2.0,18,2],["leaf",5,-0.004433940738799412],["split",6,9.0,19,2],["split",7,1.0,11,3],["split",8,25.0,12,3],["leaf",9,-0.019955020776927887],["split",10,1.0,2,3],["leaf",13,0.006096544559816814],["leaf",14,0.024758419825077517],["split",15,17.0,0,4],["split",16,19.0,0,4],["split",17,0.0,14,4],["split",18,13.0,3,4],["leaf",21,0.009331424324670146],["leaf",22,-0.012158722960527224],["leaf",31,-0.0017604485628479694],["leaf",32,-0.008763091390347272],["leaf",33,0.001546395045908309],["leaf",34,-0.0044939500363609325],["leaf",35,-1.9633340995549404e-05],["leaf",36,-0.007902443230271167],["leaf",37,0.025224190330454668],["leaf",38,0.005363451641986573]],[["split",0,7.0,29,0],["split",1,3.0,29,1],["split",2,11.0,26,1],["split",3,26.0,25,2],["split",4,20.0,0,2],["split",5,2.0,12,2],["leaf",6,0.02010459996245659],["split",7,4.0,29,3],["split",8,1.0,4,3],["split",9,9.0,8,3],["leaf",10,-0.021083855782887215],["split",11,25.0,12,3],["split",12,25.0,17,3],["split",15,22.0,28,4],["split",16,9.0,6,4],["split",17,24.0,15,4],["split",18,0.0,27,4],["leaf",19,0.024749565085626646],["split",20,23.0,15,4],["leaf",23,0.000812646147596977],["split",24,1.0,26,4],["leaf",25,0.006077309697560019],["leaf",26,-0.01298709318199447],["leaf",31,-0.0012168910918275977],["leaf",32,-0.006883615479865648],["leaf",33,-0.01751719263800656],["leaf",34,-0.004249460562184288],["leaf",35,-0.0010973140846254358],["leaf",36,-0.01091562580829453],["leaf",37,0.0026804538215675897],["leaf",38,0.012336201808149497],["leaf",41,0.0017455633047679573],["leaf",42,0.01688315016751609],["leaf",49,0.007878577303784747],["leaf",50,0.022580263977511332]],[["split",0,3.0,17,0],["split",1,23.0,24,1],["split",2,2.0,30,1],["split",3,16.0,0,2],["split",4,17.0,0,2],["split",5,26.0,26,2],["split",6,13.0,9,2],["split",7,5.0,20,3],["split",8,4.0,20,3],["split",9,20.0,0,3],["split",10,23.0,27,3],["split",11,10.0,14,3],["split",12,11.0,17,3],["leaf",13,-0.005838535498252138],["leaf",14,-0.01622940287534952],["split",15,7.0,14,4],["split",16,11.0,20,4],["split",17,22.0,22,4],["split",18,26.0,4,4],["split",19,0.0,2,4],["split",20,2.0,16,4],["leaf",21,-0.011135213361601632],["leaf",22,-0.03188707276646649],["split",23,2.0,24,4],["split",24,26.0,21,4],["split",25,7.0,12,4],["split",26,1.0,22,4],["leaf",31,-0.008500438358821567],["leaf",32,-0.0005038681947095734],["leaf",33,0.001330881678328091],["leaf",34,0.008160184366556618],["leaf",35,0.006434108999717041],["leaf",36,-0.011601080753605223],["leaf",37,-0.025866140094916112],["leaf",38,-0.01072755640702751],["leaf",39,-0.023451412340854044],["leaf",40,-0.0037579877758096643],["leaf",41,-0.0073004507122660205],["leaf",42,0.022374145277244732],["leaf",47,-0.003936196039005952],["leaf",48,0.005182210145427104],["leaf",49,0.00464623583335803],["leaf",50,-0.0014624614513328803],["leaf",51,0.01301587294974025],["leaf",52,0.005776112728435537],["leaf",53,0.0033152745382911204],["leaf",54,-0.005904525146652964]],[["split",0,7.0,3,0],["split",1,1.0,1,1],["split",2,23.0,8,1],["split",3,11.0,8,2],["split",4,26.0,7,2],["split",5,20.0,0,2],["split",6,13.0,3,2],["leaf",7,0.018954973061633622],["split",8,10.0,5,3],["split",9,0.0,17,3],["split",10,22.0,21,3],["split",11,1.0,12,3],["split",12,11.0,21,3],["split",13,26.0,7,3],["split",14,26.0,6,3],["leaf",17,0.011238583209834802],["leaf",18,-0.00906705815008067],["split",19,23.0,22,4],["leaf",20,-0.023388883215185473],["split",21,24.0,20,4],["split",22,26.0,15,4],["split",23,2.0,25,4],["split",24,17.0,0,4],["split",25,10.0,15,4],["leaf",26,0.009067422832313423],["split",27,1.0,8,4],["split",28,25.0,13,4],["split",29,1.0,15,4],["split",30,1.0,0,4],["leaf",39,-0.01714887531447234],["leaf",40,0.006697861288876749],["leaf",43,-0.010921736718101854],["leaf",44,0.0026920342942651997],["leaf",45,-0.0013728996609604016],["leaf",46,0.011095319978558095],["leaf",47,0.0014150937134293912],["leaf",48,-0.012568195026744607],["leaf",49,0.004658865480428486],["leaf",50,0.012617013078828185],["leaf",51,-0.01784867849194367],["leaf",52,-0.003337502498872434],["leaf",55,0.010092628440089355],["leaf",56,0.024169739735909596],["leaf",57,0.009389669899172351],["leaf",58,-0.007731962652759342],["leaf",59,0.0012072378581824371],["leaf",60,-0.010211967133192184],["leaf",61,-0.017053803557318826],["leaf",62,0.0002905005161609172]],[["split",0,3.0,7,0],["split",1,12.0,5,1],["split",2,12.0,0,1],["split",3,24.0,8,2],["split",4,20.0,0,2],["leaf",5,-0.01217246689074147],["split",6,7.0,25,2],["split",7,26.0,5,3],["split",8,26.0,10,3],["split",9,25.0,25,3],["split",10,25.0,22,3],["split",13,8.0,22,3],["split",14,4.0,18,3],["leaf",15,-0.008669783352480231],["split",16,24.0,3,4],["split",17,5.0,25,4],["split",18,1.0,7,4],["split",19,4.0,18,4],["split",20,21.0,4,4],["split",21,23.0,17,4],["leaf",22,0.028836301235661027],["split",27,11.0,27,4],["split",28,22.0,19,4],["split",29,2.0,2,4],["split",30,22.0,15,4],["leaf",33,0.018199813785793866],["leaf",34,-0.00559234830283748],["leaf",35,0.005161911151672489],["leaf",36,0.019204291739903845],["leaf",37,0.012638775146532884],["leaf",38,-0.0044131748336601615],["leaf",39,0.004792674661030387],["leaf",40,-0.004306914196275607],["leaf",41,0.003719478508693736],["leaf",42,-0.014778841687650503],["leaf",43,-0.00610609920820655],["leaf",44,0.011787885067313179],["leaf",55,0.0004448414459559848],["leaf",56,-0.005528196365760156],["leaf",57,-0.009982693702892006],["leaf",58,-0.002709090810753058],["leaf",59,-0.004429249904243825],["leaf",60,0.005024544462746735],["leaf",61,-0.0021116643731734757],["leaf",62,-0.013609160511409151]],[["split",0,7.0,28,0],["split",1,7.0,0,1],["split",2,2.0,11,1],["split",3,10.0,3,2],["split",4,8.0,2,2],["split",5,7.0,30,2],["split",6,0.0,7,2],["leaf",7,0.0040275263775416004],["split",8,9.0,14,3],["split",9,20.0,0,3],["split",10,23.0,30,3],["split",11,0.0,15,3],["split",12,3.0,9,3],["leaf",13,0.016839074460805983],["split",14,4.0,12,3],["leaf",17,-0.02209166598937444],["leaf",18,-0.006482459554627278],["split",19,5.0,27,4],["leaf",20,-0.006732955132509428],["split",21,0.0,27,4],["split",22,9.0,7,4],["split",23,9.0,12,4],["split",24,2.0,5,4],["leaf",25,0.030283438915029503],["leaf",26,0.009665267611741094],["leaf",29,0.004542485444280457],["split",30,21.0,19,4],["leaf",39,0.0015737885787725251],["leaf",40,0.014503394861262255],["leaf",43,-0.0005837690662474348],["leaf",44,0.004568109329223094],["leaf",45,-0.015570117295052517],["leaf",46,-0.010794887676790127],["leaf",47,0.002825708618876961],["leaf",48,0.021125570701119457],["leaf",49,-0.0020228474702136566],["leaf",50,0.007475889011805724],["leaf",61,-0.014526480429172288],["leaf",62,-0.003283658503152975]],[["split",0,11.0,13,0],["split",1,25.0,28,1],["split",2,23.0,23,1],["split",3,26.0,1,2],["split",4,10.0,10,2],["split",5,26.0,10,2],["split",6,17.0,0,2],["split",7,1.0,6,3],["split",8,26.0,25,3],["leaf",9,-0.021350668205904526],["split",10,1.0,17,3],["split",11,6.0,1,3],["split",12,13.0,3,3],["split",13,26.0,9,3],["split",14,22.0,16,3],["leaf",15,0.019776023022651978],["split",16,7.0,20,4],["split",17,23.0,16,4],["split",18,23.0,5,4],["leaf",21,-0.010742408208280152],["leaf",22,0.0016684562000385031],["leaf",23,-0.01680739871227948],["split",24,1.0,2,4],["leaf",25,0.013778161564354865],["split",26,0.0,27,4],["split",27,25.0,5,4],["split",28,10.0,27,4],["leaf",29,-0.015047203332670213],["leaf",30,0.0029402584059813465],["leaf",33,-0.00498083635372476],["leaf",34,0.015422534941637537],["leaf",35,0.00018085046014492675],["leaf",36,-0.006240354381550251],["leaf",37,-0.006277464306843852],["leaf",38,0.006412433813719722],["leaf",49,-0.01609421097697416],["leaf",50,-0.0013134341963556978],["leaf",53,0.0003655892900714823],["leaf",54,0.008745819994878038],["leaf",55,0.031061995776713668],["leaf",56,0.007578230240655498],["leaf",57,0.00615579071895656],["leaf",58,-0.009373336984368189]],[["split",0,25.0,30,0],["split",1,24.0,30,1],["split",2,12.0,10,1],["split",3,26.0,2,2],["split",4,21.0,12,2],["leaf",5,-0.00016878654604563176],["split",6,2.0,15,2],["split",7,7.0,28,3],["split",8,24.0,3,3],["leaf",9,-0.020737222499430517],["split",10,0.0,24,3],["leaf",13,0.0069845124253841845],["leaf",14,0.020816353381061296],["split",15,15.0,0,4],["leaf",16,0.017392126458496797],["split",17,23.0,28,4],["split",18,11.0,30,4],["leaf",21,-0.010957176201506377],["leaf",22,0.005255997018265767],["leaf",31,0.003893423033484953],["leaf",32,-0.01045494004654627],["leaf",35,-0.002692979274493035],["leaf",36,-0.01795010091218651],["leaf",37,0.00042794432643138966],["leaf",38,-0.0076747012952517275]],[["split",0,12.0,30,0],["split",1,7.0,0,1],["split",2,1.0,26,1],["split",3,9.0,8,2],["split",4,23.0,29,2],["split",5,13.0,27,2],["leaf",6,0.018246612383233583],["leaf",7,0.003399762738377762],["split",8,24.0,11,3],["split",9,24.0,5,3],["split",10,26.0,11,3],["leaf",11,0.012786135530768999],["leaf",12,0.00038071335379865137],["leaf",17,-0.006102492685093912],["leaf",18,-0.018691876497222717],["split",19,3.0,6,4],["split",20,21.0,30,4],["leaf",21,0.007702139846325979],["split",22,13.0,6,4],["leaf",39,0.0038035894115330043],["leaf",40,-0.005718449665660093],["leaf",41,0.0009519474326924636],["leaf",42,-0.0071047310481075505],["leaf",45,-0.00386815915600705],["leaf",46,-0.01748575831140907]],[["split",0,10.0,3,0],["split",1,21.0,29,1],["split",2,24.0,25,1],["split",3,7.0,16,2],["leaf",4,-0.019087173078696947],["split",5,6.0,7,2],["split",6,25.0,30,2],["split",7,22.0,24,3],["split",8,12.0,17,3],["split",11,26.0,27,3],["split",12,3.0,11,3],["split",13,25.0,3,3],["leaf",14,0.020931257436163385],["split",15,22.0,11,4],["leaf",16,0.005750859283128206],["split",17,9.0,8,4],["split",18,8.0,26,4],["split",23,3.0,13,4],["leaf",24,0.018313807238344228],["split",25,1.0,15,4],["split",26,17.0,0,4],["split",27,22.0,11,4],["split",28,4.0,29,4],["leaf",31,-0.004375285148585274],["leaf",32,-0.014652306079051354],["leaf",35,0.0025770837209922715],["leaf",36,0.014031998033351952],["leaf",37,0.0023667215478134704],["leaf",38,-0.01243407078064941],["leaf",47,0.004892053480468909],["leaf",48,-0.003576181424347779],["leaf",51,-0.0006327406923730737],["leaf",52,-0.006402775504123654],["leaf",53,-0.0012147883378277912],["leaf",54,0.0076618549383077085],["leaf",55,0.004021553826231215],["leaf",56,0.018288567789340345],["leaf",57,0.002187078791823874],["leaf",58,-0.013833262362665952]],[["split",0,26.0,0,0],["split",1,5.0,24,1],["split",2,25.0,0,1],["split",3,24.0,20,2],["leaf",4,-0.0024029410517825978],["split",5,13.0,21,2],["split",6,23.0,2,2],["leaf",7,-0.008000220433880852],["leaf",8,-0.022664264665243374],["split",11,26.0,7,3],["leaf",12,-0.009680145294292217],["split",13,5.0,10,3],["split",14,14.0,0,3],["leaf",23,0.026463772102885044],["leaf",24,0.004342318292734015],["split",27,0.0,11,4],["split",28,22.0,23,4],["split",29,0.0,30,4],["split",30,2.0,26,4],["leaf",55,0.006161351117637069],["leaf",56,-0.004978557172395301],["leaf",57,0.010943995049898268],["leaf",58,-0.001108138193710596],["leaf",59,0.00026218556214414304],["leaf",60,0.007971358977119874],["leaf",61,-0.001111828078643732],["leaf",62,-0.014261179272531168]],[["split",0,7.0,1,0],["split",1,19.0,0,1],["split",2,13.0,1,1],["split",3,21.0,27,2],["leaf",4,0.005282482375701532],["split",5,3.0,1,2],["split",6,12.0,17,2],["split",7,26.0,20,3],["leaf",8,-0.019155593794365107],["leaf",11,0.02031827554144306],["split",12,21.0,16,3],["split",13,0.0,27,3],["split",14,26.0,8,3],["split",15,9.0,23,4],["leaf",16,-0.017936654623822414],["leaf",25,0.009904070759419478],["leaf",26,-0.0030651279708144596],["split",27,7.0,18,4],["split",28,12.0,14,4],["split",29,23.0,8,4],["split",30,19.0,0,4],["leaf",31,-0.006811522249039827],["leaf",32,0.012957521351999905],["leaf",55,-0.0012061771602427274],["leaf",56,0.003016555164459463],["leaf",57,0.0038755287613075487],["leaf",58,0.01875032083703391],["leaf",59,0.0005984598560601812],["leaf",60,-0.01103821545889144],["leaf",61,0.00028118290656192594],["leaf",62,-0.007264511315683736]],[["split",0,22.0,30,0],["split",1,7.0,16,1],["split",2,13.0,15,1],["split",3,12.0,0,2],["split",4,8.0,21,2],["split",5,12.0,8,2],["split",6,23.0,15,2],["split",7,24.0,10,3],["split",8,6.0,1,3],["split",9,4.0,16,3],["split",10,0.0,4,3],["leaf",11,0.00857390799966102],["leaf",12,-0.008874722322159582],["leaf",13,-0.02506121522100392],["leaf",14,-0.009750169524229557],["leaf",15,0.00337176301849677],["split",16,3.0,6,4],["split",17,25.0,8,4],["split",18,1.0,2,4],["split",19,12.0,23,4],["split",20,24.0,2,4],["split",21,2.0,5,4],["split",22,11.0,9,4],["leaf",33,-0.027826331428856637],["leaf",34,-0.006898499563572075],["leaf",35,0.02087833409033176],["leaf",36,0.0012369436272825136],["leaf",37,-0.010494990330232037],["leaf",38,-0.0017246285377053338],["leaf",39,0.009672473693375164],["leaf",40,0.0014133135655682195],["leaf",41,-0.013881595709323638],["leaf",42,0.0019047663059379247],["leaf",43,-0.015163404768477941],["leaf",44,-0.002509445425629495],["leaf",45,-0.0031534098417924147],["leaf",46,0.00248965949878802]],[["split",0,0.0,30,0],["split",1,22.0,18,1],["split",2,5.0,9,1],["split",3,10.0,30,2],["split",4,8.0,1,2],["leaf",5,-0.021603144725928267],["split",6,21.0,25,2],["split",7,7.0,0,3],["split",8,3.0,13,3],["split",9,21.0,17,3],["split",10,21.0,26,3],["leaf",13,0.0040843987493312615],["leaf",14,-0.009247399559642294],["split",15,11.0,13,4],["split",16,3.0,1,4],["leaf",17,-0.02130597524784146],["leaf",18,0.0017894107518077013],["leaf",19,-0.002753001943127654],["leaf",20,0.019959863052928815],["split",21,20.0,0,4],["split",22,26.0,25,4],["leaf",31,0.0010412899490885043],["leaf",32,-0.015327874472894595],["leaf",33,0.0092168287901042],["leaf",34,0.0010183619382511352],["leaf",43,-0.0025780775312878386],["leaf",44,-0.009356146850228881],["leaf",45,0.0008424360411369858],["leaf",46,0.013047465476659996]],[["split",0,26.0,21,0],["split",1,22.0,30,1],["split",2,4.0,24,1],["split",3,12.0,9,2],["split",4,11.0,19,2],["split",5,19.0,0,2],["split",6,22.0,15,2],["split",7,24.0,24,3],["split",8,7.0,8,3],["leaf",9,0.0011552245596981403],["leaf",10,0.015266197044141314],["split",11,7.0,8,3],["split",12,13.0,11,3],["split",13,23.0,25,3],["split",14,10.0,12,3],["split",15,0.0,21,4],["split",16,23.0,25,4],["split",17,3.0,4,4],["split",18,26.0,13,4],["split",23,25.0,8,4],["split",24,20.0,0,4],["leaf",25,0.00925743311832845],["split",26,11.0,10,4],["leaf",27,0.007705392755053558],["leaf",28,-0.007113235103589577],["leaf",29,0.019348940465084077],["leaf",30,0.005485087100633255],["leaf",31,0.001024615049195459],["leaf",32,-0.0062494559742153144],["leaf",33,0.002326463455955288],["leaf",34,0.01566202086554424],["leaf",35,-0.022015843555830226],["leaf",36,-0.006740146747071748],["leaf",37,-0.0020505088006057587],["leaf",38,0.0017930191763217147],["leaf",47,0.010609942777812433],["leaf",48,-0.005118711267642151],["leaf",49,0.006158171484214151],["leaf",50,-0.0025208240287901023],["leaf",53,-0.00035801788319969976],["leaf",54,-0.0119629223460737]],[["split",0,8.0,11,0],["split",1,6.0,6,1],["split",2,8.0,22,1],["split",3,7.0,3,2],["split",4,7.0,7,2],["split",5,5.0,15,2],["split",6,5.0,10,2],["split",7,7.0,0,3],["split",8,20.0,0,3],["split",9,1.0,6,3],["split",10,20.0,0,3],["split",11,13.0,16,3],["split",12,8.0,18,3],["split",13,20.0,0,3],["split",14,0.0,9,3],["split",15,4.0,27,4],["split",16,20.0,0,4],["split",17,17.0,0,4],["split",18,2.0,20,4],["split",19,26.0,21,4],["split",20,26.0,10,4],["split",21,4.0,16,4],["leaf",22,0.001343747983990742],["split",23,2.0,4,4],["split",24,20.0,0,4],["split",25,12.0,5,4],["split",26,2.0,7,4],["split",27,8.0,28,4],["split",28,25.0,21,4],["split",29,4.0,20,4],["split",30,5.0,20,4],["leaf",31,-0.07175658812683525],["leaf",32,-0.09717024324489787],["leaf",33,-0.06738493344502008],["leaf",34,-0.031697301551265246],["leaf",35,-0.0330356228799859],["leaf",36,-0.04882017975395024],["leaf",37,-0.024671855996108574],["leaf",38,0.00019271291325792148],["leaf",39,-0.053073280154326524],["leaf",40,-0.04434651801639103],["leaf",41,-0.04667943649589325],["leaf",42,-0.022143294908980933],["leaf",43,-0.008842289842790866],["leaf",44,-0.02209353875629416],["leaf",47,0.019624812912994175],["leaf",48,0.0055794458854114826],["leaf",49,0.02642030212508971],["leaf",50,0.004474707117389874],["leaf",51,-0.028699062543076782],["leaf",52,-0.004794550197598718],["leaf",53,0.026204890227466273],["leaf",54,0.005873263571609086],["leaf",55,0.046983543947024885],["leaf",56,0.06355740998530669],["leaf",57,0.018112907028297945],["leaf",58,0.04633510306701707],["leaf",59,0.024973110872645338],["leaf",60,0.003347138462212348],["leaf",61,0.038613150995996784],["leaf",62,0.02062398541110371]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,8.0,24,1],["split",3,7.0,1,2],["split",4,7.0,6,2],["split",5,4.0,15,2],["split",6,4.0,8,2],["split",7,4.0,22,3],["split",8,3.0,23,3],["split",9,10.0,6,3],["split",10,4.0,18,3],["split",11,8.0,18,3],["split",12,8.0,17,3],["split",13,20.0,0,3],["split",14,6.0,29,3],["leaf",15,-0.05445223812648055],["split",16,3.0,4,4],["split",17,10.0,4,4],["leaf",18,-0.02633123470583307],["split",19,21.0,12,4],["split",20,20.0,0,4],["split",21,21.0,5,4],["split",22,7.0,13,4],["split",23,4.0,8,4],["split",24,19.0,0,4],["split",25,7.0,9,4],["split",26,23.0,13,4],["split",27,8.0,29,4],["split",28,25.0,17,4],["split",29,24.0,14,4],["split",30,4.0,13,4],["leaf",33,-0.06794287140671976],["leaf",34,-0.08849083761029354],["leaf",35,-0.06597183200574913],["leaf",36,-0.04797697954243157],["leaf",39,-0.06834415798721863],["leaf",40,-0.04794148285152692],["leaf",41,-0.034056439989836325],["leaf",42,-0.012216900992847507],["leaf",43,-0.022323312642465277],["leaf",44,-0.005303943326468954],["leaf",45,-0.02909889066925602],["leaf",46,-0.012108938017095268],["leaf",47,0.01782138427509789],["leaf",48,0.003971273466185016],["leaf",49,0.029730064655073017],["leaf",50,0.011788264881297073],["leaf",51,-0.021967089092351465],["leaf",52,-0.00499499937137291],["leaf",53,0.015435878340766544],["leaf",54,0.0014316279982333534],["leaf",55,0.05200105184257924],["leaf",56,0.06685798994897374],["leaf",57,0.00819445531317704],["leaf",58,0.040554421310888246],["leaf",59,0.022443946605938743],["leaf",60,0.03456185055956947],["leaf",61,0.05795242520825473],["leaf",62,0.03674625668381258]],[["split",0,6.0,9,0],["split",1,7.0,3,1],["split",2,8.0,21,1],["split",3,7.0,0,2],["split",4,7.0,9,2],["split",5,4.0,15,2],["split",6,5.0,7,2],["split",7,10.0,13,3],["split",8,20.0,0,3],["split",9,21.0,2,3],["split",10,4.0,22,3],["split",11,12.0,15,3],["split",12,8.0,14,3],["split",13,20.0,0,3],["split",14,21.0,8,3],["split",15,5.0,28,4],["leaf",16,-0.06242897403073104],["split",17,10.0,7,4],["leaf",18,-0.026185261931061226],["leaf",19,-0.05156442738708629],["split",20,10.0,17,4],["split",21,24.0,25,4],["split",22,22.0,10,4],["split",23,21.0,5,4],["split",24,21.0,3,4],["split",25,23.0,25,4],["split",26,25.0,10,4],["split",27,6.0,25,4],["split",28,25.0,18,4],["split",29,4.0,16,4],["split",30,6.0,26,4],["leaf",31,-0.07938194022882356],["leaf",32,-0.09676077697095561],["leaf",35,-0.07468468383526683],["leaf",36,-0.04988307214830064],["leaf",41,-0.03570014352906997],["leaf",42,-0.019291500379349937],["leaf",43,-0.0076551977267995275],["leaf",44,0.009606631756933996],["leaf",45,-0.011440477148617243],["leaf",46,-0.028723584950222637],["leaf",47,-0.009218603529419223],["leaf",48,0.0058099913623081],["leaf",49,-0.002348240311153255],["leaf",50,0.021473989585034648],["leaf",51,-0.009405606423902797],["leaf",52,-0.02498306059027141],["leaf",53,-0.005485496586689178],["leaf",54,0.005905757948349059],["leaf",55,0.036007044750955175],["leaf",56,0.05146408741724784],["leaf",57,0.005239260929719221],["leaf",58,0.03703705571555104],["leaf",59,0.02125502980823015],["leaf",60,0.0007384409194807815],["leaf",61,0.025560780424160705],["leaf",62,0.040799347289334216]],[["split",0,6.0,13,0],["split",1,7.0,4,1],["split",2,8.0,24,1],["split",3,4.0,28,2],["split",4,7.0,8,2],["split",5,4.0,13,2],["split",6,4.0,11,2],["split",7,20.0,0,3],["split",8,2.0,26,3],["split",9,6.0,10,3],["split",10,4.0,24,3],["split",11,8.0,17,3],["split",12,12.0,10,3],["split",13,20.0,0,3],["split",14,21.0,5,3],["split",15,3.0,27,4],["leaf",16,-0.011650036480226753],["split",17,7.0,0,4],["split",18,4.0,30,4],["split",19,4.0,17,4],["split",20,23.0,14,4],["split",21,25.0,1,4],["split",22,12.0,6,4],["split",23,20.0,0,4],["split",24,19.0,0,4],["split",25,3.0,8,4],["split",26,21.0,7,4],["split",27,2.0,5,4],["split",28,21.0,12,4],["leaf",29,0.009196525590799379],["split",30,13.0,27,4],["leaf",31,-0.05343813605221283],["leaf",32,-0.01475163367130468],["leaf",35,-0.07129151048451247],["leaf",36,-0.046452504144188556],["leaf",37,-0.06787246388882713],["leaf",38,-0.08637201639273628],["leaf",39,-0.018498331423671178],["leaf",40,-0.03489352449141038],["leaf",41,-0.0050049064915065776],["leaf",42,-0.016549090440371544],["leaf",43,-0.03139139825079485],["leaf",44,-0.0037119390562600275],["leaf",45,-0.033036952461503316],["leaf",46,-0.014558206588051473],["leaf",47,0.014866804122800902],["leaf",48,-0.008455533546759984],["leaf",49,0.027203928379317194],["leaf",50,0.01363928423564245],["leaf",51,-0.019343288505325157],["leaf",52,-0.005061280360169886],["leaf",53,-0.0032452737971213888],["leaf",54,0.011630380843389073],["leaf",55,0.05879032025005729],["leaf",56,0.042377567197478636],["leaf",57,-0.0025252720046830884],["leaf",58,0.03727906548609405],["leaf",61,0.022491175636929765],["leaf",62,0.03833740893325945]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,6.0,22,1],["split",3,7.0,1,2],["split",4,4.0,19,2],["split",5,4.0,11,2],["split",6,13.0,28,2],["split",7,2.0,29,3],["split",8,20.0,0,3],["split",9,7.0,10,3],["split",10,7.0,10,3],["split",11,8.0,18,3],["split",12,13.0,18,3],["split",13,4.0,16,3],["split",14,20.0,0,3],["split",15,9.0,5,4],["split",16,11.0,14,4],["split",17,10.0,8,4],["leaf",18,-0.016824312167639348],["split",19,26.0,12,4],["split",20,26.0,20,4],["split",21,10.0,6,4],["split",22,3.0,6,4],["split",23,20.0,0,4],["split",24,0.0,17,4],["split",25,5.0,17,4],["split",26,10.0,7,4],["split",27,7.0,30,4],["split",28,2.0,6,4],["split",29,2.0,13,4],["leaf",30,0.02277034650538104],["leaf",31,-0.04202661722503139],["leaf",32,-0.06163228290727387],["leaf",33,-0.08575514099248507],["leaf",34,-0.07517866308825408],["leaf",35,-0.06070287956972318],["leaf",36,-0.039522648594310046],["leaf",39,-0.025737120713604],["leaf",40,-0.008804140380536011],["leaf",41,-0.004654812982151205],["leaf",42,0.010004352885056121],["leaf",43,-0.04663945892271235],["leaf",44,-0.027686797827350063],["leaf",45,-0.024363734451350018],["leaf",46,-0.010925375072431992],["leaf",47,0.013620472216344163],["leaf",48,-0.004591085592475583],["leaf",49,0.01659452079912071],["leaf",50,0.03367252402958735],["leaf",51,0.0020825720376984433],["leaf",52,-0.010079057675813758],["leaf",53,-0.0026820847265890567],["leaf",54,0.012253128232627745],["leaf",55,0.03008439843835058],["leaf",56,0.05392271786342886],["leaf",57,0.030800846770623645],["leaf",58,0.008831323100842716],["leaf",59,0.05479157440181446],["leaf",60,0.03602202568177643]],[["split",0,13.0,12,0],["split",1,7.0,5,1],["split",2,13.0,23,1],["split",3,7.0,2,2],["split",4,7.0,19,2],["split",5,5.0,14,2],["split",6,5.0,8,2],["split",7,4.0,20,3],["split",8,20.0,0,3],["split",9,4.0,25,3],["split",10,6.0,9,3],["split",11,20.0,0,3],["split",12,10.0,8,3],["split",13,20.0,0,3],["split",14,23.0,20,3],["split",15,4.0,14,4],["split",16,15.0,0,4],["split",17,3.0,20,4],["leaf",18,-0.003980504232784339],["split",19,7.0,8,4],["split",20,0.0,15,4],["leaf",21,0.018169145248487208],["leaf",22,0.0006150795287372735],["split",23,4.0,4,4],["split",24,25.0,15,4],["split",25,6.0,15,4],["split",26,12.0,10,4],["split",27,1.0,5,4],["split",28,25.0,13,4],["split",29,8.0,27,4],["split",30,7.0,30,4],["leaf",31,-0.02385155578561867],["leaf",32,-0.040138209550624235],["leaf",33,-0.05630423698059658],["leaf",34,-0.07754948583016788],["leaf",35,-0.04601698107634045],["leaf",36,-0.026584896462532304],["leaf",39,-0.018877437232675463],["leaf",40,-0.00662513866532278],["leaf",41,-0.032538555328160385],["leaf",42,-0.018801637931913636],["leaf",47,0.025931980416880673],["leaf",48,0.01286993860346277],["leaf",49,-0.018207962656136473],["leaf",50,0.016194563326999867],["leaf",51,-0.01868623648543918],["leaf",52,-0.005195343656802328],["leaf",53,-0.007242107747234195],["leaf",54,0.006900511188300824],["leaf",55,0.018363778456775844],["leaf",56,0.04584459875291221],["leaf",57,-0.0013374012851139969],["leaf",58,0.03478560284521435],["leaf",59,0.02462444385464928],["leaf",60,0.03835050636306746],["leaf",61,0.010700320109627492],["leaf",62,0.0350925302408628]],[["split",0,6.0,11,0],["split",1,7.0,3,1],["split",2,6.0,24,1],["split",3,4.0,18,2],["split",4,7.0,12,2],["split",5,13.0,16,2],["split",6,5.0,5,2],["split",7,22.0,11,3],["split",8,2.0,29,3],["split",9,4.0,20,3],["split",10,17.0,0,3],["split",11,4.0,22,3],["split",12,26.0,20,3],["split",13,20.0,0,3],["split",14,12.0,20,3],["leaf",15,-0.03372098466475488],["leaf",16,-0.014186468372107738],["split",17,7.0,1,4],["split",18,0.0,17,4],["split",19,8.0,7,4],["split",20,2.0,20,4],["split",21,25.0,4,4],["leaf",22,-0.024289920714666496],["split",23,10.0,2,4],["split",24,26.0,12,4],["split",25,12.0,15,4],["split",26,20.0,0,4],["split",27,19.0,0,4],["leaf",28,0.013490574153202182],["split",29,25.0,8,4],["split",30,10.0,12,4],["leaf",35,-0.05809425156400235],["leaf",36,-0.03707892068434509],["leaf",37,-0.07218879431540955],["leaf",38,-0.054175417849264586],["leaf",39,-0.020474435537203503],["leaf",40,-0.006729947779981827],["leaf",41,-0.021207758536521026],["leaf",42,-0.03313288861221568],["leaf",43,-0.016097379879213076],["leaf",44,0.002495231214120736],["leaf",47,-0.012778919988751504],["leaf",48,0.005036157829701768],["leaf",49,-0.019948872932187545],["leaf",50,-0.008676582489624543],["leaf",51,2.085193103257995e-06],["leaf",52,0.014153108979126625],["leaf",53,0.024221274921492394],["leaf",54,0.00524060211074067],["leaf",55,0.04978547247188976],["leaf",56,0.026268028246985715],["leaf",59,0.003108531791788428],["leaf",60,0.022804742641357775],["leaf",61,0.023809359038442342],["leaf",62,0.03605057246360007]],[["split",0,13.0,10,0],["split",1,7.0,1,1],["split",2,6.0,21,1],["split",3,10.0,8,2],["split",4,7.0,11,2],["split",5,12.0,9,2],["split",6,13.0,26,2],["split",7,4.0,25,3],["split",8,0.0,25,3],["split",9,3.0,25,3],["split",10,7.0,17,3],["split",11,4.0,19,3],["split",12,7.0,6,3],["split",13,25.0,20,3],["split",14,5.0,12,3],["leaf",15,-0.046344730021905825],["split",16,12.0,2,4],["split",17,24.0,15,4],["leaf",18,-0.025787589563051806],["split",19,20.0,0,4],["split",20,2.0,25,4],["split",21,23.0,22,4],["split",22,10.0,9,4],["split",23,6.0,17,4],["split",24,23.0,15,4],["leaf",25,-0.022355735088145413],["split",26,3.0,24,4],["split",27,20.0,0,4],["split",28,10.0,7,4],["split",29,20.0,0,4],["split",30,1.0,19,4],["leaf",33,-0.0689549907652875],["leaf",34,-0.09690607161787512],["leaf",35,-0.05837104082758083],["leaf",36,-0.04291035824036054],["leaf",39,-0.02991702328086916],["leaf",40,-0.011112075659658016],["leaf",41,-0.00026048745124297257],["leaf",42,-0.018885102513060914],["leaf",43,-0.007862548482392788],["leaf",44,-0.02531912740274749],["leaf",45,0.011631922235186179],["leaf",46,-0.0070638904481929235],["leaf",47,-0.0073599590347636955],["leaf",48,0.009365818414346276],["leaf",49,-0.010533528021093058],["leaf",50,-0.025649733411953056],["leaf",53,0.0027507352910273574],["leaf",54,0.01426166883887585],["leaf",55,0.01762962082991067],["leaf",56,-0.005945571416495883],["leaf",57,0.013979736759910433],["leaf",58,0.0319884757154],["leaf",59,0.040547492286621106],["leaf",60,0.01698648671505574],["leaf",61,0.012299921172919339],["leaf",62,0.02719286301427206]],[["split",0,8.0,12,0],["split",1,7.0,4,1],["split",2,13.0,22,1],["split",3,3.0,27,2],["split",4,4.0,15,2],["split",5,4.0,18,2],["split",6,4.0,12,2],["split",7,10.0,10,3],["leaf",8,-0.010458062862170634],["split",9,25.0,4,3],["split",10,21.0,15,3],["split",11,12.0,18,3],["split",12,25.0,5,3],["split",13,13.0,30,3],["split",14,10.0,7,3],["split",15,20.0,0,4],["split",16,2.0,8,4],["leaf",19,-0.020831191295336768],["split",20,21.0,14,4],["split",21,18.0,0,4],["split",22,20.0,0,4],["split",23,10.0,2,4],["split",24,26.0,16,4],["split",25,26.0,9,4],["split",26,20.0,0,4],["split",27,20.0,0,4],["split",28,4.0,5,4],["split",29,25.0,10,4],["split",30,5.0,20,4],["leaf",31,-0.05456909618002422],["leaf",32,-0.018547123041617798],["leaf",33,-0.015203916255611441],["leaf",34,-0.03732839309262533],["leaf",41,-0.008616488815280711],["leaf",42,0.003268765706890183],["leaf",43,-0.02532645027753731],["leaf",44,-0.008869340164609581],["leaf",45,-0.015282093428108769],["leaf",46,0.011516097064712325],["leaf",47,-0.015348132302238433],["leaf",48,0.00742507425803837],["leaf",49,0.008822191210823506],["leaf",50,0.02288420193631459],["leaf",51,-0.021837066241683505],["leaf",52,-0.007553148811691732],["leaf",53,-0.005012747934582057],["leaf",54,0.012575240074835707],["leaf",55,0.030914387534767943],["leaf",56,0.0097763838407188],["leaf",57,0.05539596091459713],["leaf",58,0.03038580236215364],["leaf",59,-0.010760265648224182],["leaf",60,0.010769085895674524],["leaf",61,0.021713324862136774],["leaf",62,0.007482542846911483]],[["split",0,7.0,10,0],["split",1,7.0,4,1],["split",2,8.0,23,1],["split",3,4.0,20,2],["split",4,3.0,22,2],["split",5,5.0,15,2],["split",6,6.0,30,2],["split",7,3.0,23,3],["split",8,7.0,0,3],["split",9,23.0,15,3],["split",10,12.0,6,3],["split",11,2.0,3,3],["split",12,26.0,7,3],["split",13,4.0,15,3],["split",14,12.0,25,3],["split",15,10.0,11,4],["leaf",16,-0.004816869952345649],["split",17,10.0,12,4],["split",18,21.0,24,4],["split",19,26.0,5,4],["split",20,20.0,0,4],["leaf",21,-0.016175943000027945],["split",22,2.0,27,4],["split",23,1.0,14,4],["split",24,9.0,22,4],["split",25,24.0,27,4],["split",26,23.0,19,4],["split",27,20.0,0,4],["split",28,25.0,7,4],["leaf",29,0.02588283323814226],["split",30,4.0,2,4],["leaf",31,-0.03638567198204936],["leaf",32,-0.009216530983296285],["leaf",35,-0.0621701160175253],["leaf",36,-0.04505595951980965],["leaf",37,-0.03927498995572784],["leaf",38,-0.025304438534164576],["leaf",39,-0.025204368496572777],["leaf",40,-0.007796773029768559],["leaf",41,-0.026825566907290276],["leaf",42,-0.004185661777536649],["leaf",45,0.00470001556929555],["leaf",46,-0.011363457328277116],["leaf",47,0.010404485837942075],["leaf",48,0.0303643891414151],["leaf",49,0.004399754908704222],["leaf",50,0.01621474689492076],["leaf",51,-0.012116463377192629],["leaf",52,0.008615485488290301],["leaf",53,0.006855302815293965],["leaf",54,-0.004808097474833009],["leaf",55,0.028147990811245367],["leaf",56,0.012747486019400997],["leaf",57,0.0018619140540494213],["leaf",58,0.01805730629716433],["leaf",61,0.05598586977900645],["leaf",62,0.040484017503640175]],[["split",0,13.0,11,0],["split",1,7.0,3,1],["split",2,6.0,21,1],["split",3,2.0,22,2],["split",4,21.0,6,2],["split",5,4.0,18,2],["split",6,4.0,10,2],["split",7,3.0,20,3],["split",8,20.0,0,3],["split",9,7.0,6,3],["split",10,20.0,0,3],["split",11,21.0,12,3],["split",12,26.0,10,3],["split",13,20.0,0,3],["split",14,2.0,8,3],["split",15,22.0,8,4],["leaf",16,-0.009386097380561141],["split",17,4.0,25,4],["leaf",18,-0.003700267628321867],["split",19,0.0,5,4],["split",20,10.0,14,4],["split",21,4.0,8,4],["split",22,2.0,16,4],["split",23,20.0,0,4],["split",24,12.0,27,4],["split",25,7.0,15,4],["split",26,13.0,17,4],["split",27,2.0,6,4],["split",28,25.0,15,4],["split",29,1.0,12,4],["split",30,26.0,11,4],["leaf",31,-0.042034311158831245],["leaf",32,-0.028896443731917274],["leaf",35,-0.04065309433043536],["leaf",36,-0.06022849077783028],["leaf",39,-0.04660080929127782],["leaf",40,-0.028361471105697325],["leaf",41,-0.024574411510122577],["leaf",42,-0.013228122399469593],["leaf",43,0.00746460763909326],["leaf",44,-0.0169962103779182],["leaf",45,-0.00936117350075479],["leaf",46,0.017220271509114482],["leaf",47,0.003281902954943145],["leaf",48,-0.022698811740612196],["leaf",49,0.009121911957356853],["leaf",50,0.02372341600243558],["leaf",51,-0.029456355052865035],["leaf",52,-0.007068516220934445],["leaf",53,-0.006566219615144997],["leaf",54,0.009027458989318662],["leaf",55,0.03867359640172532],["leaf",56,0.025550049724543396],["leaf",57,-0.0012490758333380569],["leaf",58,0.022596946361421438],["leaf",59,0.011727801947194862],["leaf",60,0.025740004256648658],["leaf",61,-0.0008431981313146622],["leaf",62,0.014727992453175591]],[["split",0,8.0,10,0],["split",1,5.0,14,1],["split",2,8.0,24,1],["split",3,23.0,11,2],["split",4,20.0,0,2],["split",5,4.0,23,2],["split",6,6.0,29,2],["split",7,7.0,6,3],["split",8,3.0,26,3],["split",9,7.0,3,3],["split",10,25.0,16,3],["split",11,12.0,15,3],["split",12,25.0,3,3],["split",13,2.0,8,3],["split",14,4.0,8,3],["leaf",15,-0.0072884904746182725],["leaf",16,0.010138247791256055],["split",17,7.0,8,4],["leaf",18,0.0015447195079223875],["split",19,19.0,0,4],["split",20,7.0,8,4],["split",21,5.0,24,4],["split",22,1.0,9,4],["split",23,21.0,9,4],["split",24,1.0,7,4],["leaf",25,-0.02353558018768659],["split",26,20.0,0,4],["split",27,7.0,30,4],["split",28,5.0,3,4],["split",29,12.0,29,4],["split",30,1.0,17,4],["leaf",35,-0.02884087025018325],["leaf",36,0.00014944943530540735],["leaf",39,-0.04478509315689519],["leaf",40,-0.018232510870027162],["leaf",41,-0.025804465813211754],["leaf",42,-0.012234113844855211],["leaf",43,-0.009335556521511976],["leaf",44,-0.038500711289496004],["leaf",45,0.021710500495141852],["leaf",46,-0.0017340737139199562],["leaf",47,-0.004801470282280489],["leaf",48,0.004720217637616343],["leaf",49,-0.00020586814997644538],["leaf",50,0.014687871422358471],["leaf",53,-0.011055874207173352],["leaf",54,0.006136753957341295],["leaf",55,0.023423398268609695],["leaf",56,0.03900204426852205],["leaf",57,0.027445502936742215],["leaf",58,0.012513199741019013],["leaf",59,0.03686770785163688],["leaf",60,0.04958191020273091],["leaf",61,0.015758661555853464],["leaf",62,0.03125857008893931]],[["split",0,6.0,14,0],["split",1,7.0,4,1],["split",2,13.0,24,1],["split",3,7.0,1,2],["split",4,5.0,13,2],["split",5,5.0,17,2],["split",6,6.0,30,2],["split",7,20.0,0,3],["split",8,21.0,18,3],["split",9,12.0,12,3],["split",10,20.0,0,3],["split",11,20.0,0,3],["split",12,12.0,8,3],["split",13,21.0,8,3],["split",14,2.0,5,3],["split",15,10.0,22,4],["leaf",16,-0.020196565974687386],["split",17,14.0,0,4],["split",18,26.0,11,4],["split",19,10.0,14,4],["split",20,25.0,16,4],["split",21,10.0,11,4],["split",22,25.0,11,4],["split",23,2.0,6,4],["split",24,21.0,17,4],["leaf",25,-0.01400028046983597],["split",26,0.0,4,4],["split",27,20.0,0,4],["split",28,12.0,17,4],["leaf",29,0.04745665365543085],["split",30,3.0,24,4],["leaf",31,-0.05109473951689689],["leaf",32,-0.024568448173270932],["leaf",35,-0.027521163577833],["leaf",36,-0.05002028221064268],["leaf",37,-0.023977314422991624],["leaf",38,-0.010618724611525774],["leaf",39,-0.01440407280261973],["leaf",40,0.0005771118788917701],["leaf",41,-0.0028630154138816755],["leaf",42,0.01232960120928883],["leaf",43,-0.01955434281548607],["leaf",44,-0.009345611928425245],["leaf",45,-0.022646929281292093],["leaf",46,0.01687058856352064],["leaf",47,0.020930550078197645],["leaf",48,0.009206374071078232],["leaf",49,-0.012495581454847826],["leaf",50,0.01244718041654436],["leaf",53,-0.01125652005124711],["leaf",54,0.004533524321987001],["leaf",55,0.013326430133604196],["leaf",56,-0.009641843457905358],["leaf",57,0.010981428089075363],["leaf",58,0.02373838339332186],["leaf",61,0.02191398770311417],["leaf",62,0.0352220139322834]],[["split",0,7.0,9,0],["split",1,4.0,15,1],["split",2,13.0,21,1],["split",3,15.0,0,2],["split",4,7.0,5,2],["split",5,4.0,2,2],["split",6,8.0,29,2],["split",7,22.0,26,3],["leaf",8,-0.021343044888445604],["split",9,20.0,0,3],["split",10,24.0,22,3],["split",11,12.0,19,3],["split",12,21.0,1,3],["split",13,2.0,12,3],["split",14,7.0,29,3],["split",15,25.0,23,4],["leaf",16,-0.015443940490174182],["split",19,10.0,10,4],["split",20,25.0,16,4],["split",21,16.0,0,4],["split",22,10.0,19,4],["leaf",23,0.011952624253621843],["split",24,0.0,19,4],["split",25,12.0,17,4],["split",26,10.0,6,4],["split",27,20.0,0,4],["split",28,0.0,6,4],["split",29,3.0,26,4],["leaf",30,0.04254731519899642],["leaf",31,-0.0055450215680022475],["leaf",32,0.00925416056148493],["leaf",39,-0.04430225108175831],["leaf",40,-0.026432203205495386],["leaf",41,-0.026317045663386014],["leaf",42,0.007091210154941655],["leaf",43,-0.016410822414567497],["leaf",44,-0.03246220167442985],["leaf",45,-0.00994423033883988],["leaf",46,0.005417020256463297],["leaf",49,0.03505887176780753],["leaf",50,0.024569876194553263],["leaf",51,-0.023915379834784915],["leaf",52,-0.00018168892155113453],["leaf",53,-0.00667675389340335],["leaf",54,0.0051474076733468346],["leaf",55,0.024418757949481326],["leaf",56,-0.003457316286338319],["leaf",57,-0.0011834384660522478],["leaf",58,0.014046652919460357],["leaf",59,0.020976146272124706],["leaf",60,0.03635138590734779]],[["split",0,8.0,9,0],["split",1,7.0,7,1],["split",2,8.0,19,1],["split",3,2.0,27,2],["split",4,17.0,0,2],["split",5,21.0,5,2],["split",6,12.0,27,2],["split",7,19.0,0,3],["split",8,20.0,0,3],["split",9,7.0,16,3],["leaf",10,-0.025351911937927178],["split",11,20.0,0,3],["split",12,23.0,29,3],["split",13,4.0,20,3],["split",14,4.0,4,3],["split",15,4.0,13,4],["split",16,26.0,15,4],["split",17,7.0,2,4],["leaf",18,-0.0009114376965236242],["split",19,22.0,4,4],["split",20,1.0,7,4],["split",23,4.0,22,4],["leaf",24,-0.0343287004363979],["split",25,7.0,8,4],["leaf",26,-0.01990571200027934],["split",27,20.0,0,4],["split",28,25.0,6,4],["split",29,11.0,9,4],["split",30,9.0,13,4],["leaf",31,-0.007868582085881587],["leaf",32,-0.02481828451017016],["leaf",33,-0.0008776240518133319],["leaf",34,-0.018512148098714258],["leaf",35,-0.0464737631689182],["leaf",36,-0.027331801131059125],["leaf",39,0.0038876014275839262],["leaf",40,-0.010571484197107373],["leaf",41,-0.0035806072938789897],["leaf",42,0.012762291006844067],["leaf",47,-0.0021624960571875324],["leaf",48,-0.020225479016606124],["leaf",51,-0.004291405225697163],["leaf",52,0.006374099884105164],["leaf",55,0.01651894952193683],["leaf",56,0.0015698011733868626],["leaf",57,-0.012990108828562639],["leaf",58,0.004626417267391239],["leaf",59,0.021943413760361186],["leaf",60,0.03975884217016659],["leaf",61,0.012690173621834385],["leaf",62,0.02419818285483646]],[["split",0,7.0,9,0],["split",1,7.0,2,1],["split",2,4.0,16,1],["split",3,20.0,0,2],["split",4,4.0,7,2],["split",5,12.0,27,2],["split",6,13.0,18,2],["split",7,2.0,27,3],["leaf",8,-0.006446024625103654],["split",9,24.0,11,3],["split",10,21.0,14,3],["split",11,1.0,2,3],["split",12,7.0,27,3],["split",13,26.0,26,3],["split",14,23.0,23,3],["split",15,0.0,25,4],["split",16,7.0,0,4],["leaf",19,-0.012127809973913838],["split",20,25.0,14,4],["split",21,18.0,0,4],["split",22,20.0,0,4],["split",23,7.0,21,4],["split",24,4.0,4,4],["split",25,9.0,15,4],["split",26,25.0,16,4],["split",27,3.0,20,4],["leaf",28,0.007897908598248383],["split",29,5.0,19,4],["split",30,22.0,19,4],["leaf",31,-0.03143868128916759],["leaf",32,-0.013425150121082079],["leaf",33,-0.05799685002123793],["leaf",34,-0.03934277735513232],["leaf",41,3.984745392300873e-05],["leaf",42,0.011760569417624253],["leaf",43,-0.024765160645498628],["leaf",44,-0.006412043820643288],["leaf",45,-0.015363693568629018],["leaf",46,0.014982408716930509],["leaf",47,-0.027239474717162926],["leaf",48,0.00014121154366553306],["leaf",49,0.01818135879231506],["leaf",50,0.00849853113744787],["leaf",51,0.011617610245647794],["leaf",52,0.026272556417925877],["leaf",53,0.025211267306176916],["leaf",54,0.038798375058435905],["leaf",55,-0.00891088384389432],["leaf",56,0.004773249999570789],["leaf",59,0.01584627563491966],["leaf",60,0.0024822539006088638],["leaf",61,-0.001826346046740028],["leaf",62,-0.018064852481979473]],[["split",0,13.0,14,0],["split",1,7.0,4,1],["split",2,8.0,26,1],["split",3,3.0,28,2],["split",4,0.0,5,2],["split",5,4.0,14,2],["split",6,19.0,0,2],["split",7,7.0,1,3],["leaf",8,0.0043133357483342545],["split",9,4.0,20,3],["split",10,10.0,11,3],["split",11,24.0,10,3],["split",12,21.0,4,3],["split",13,4.0,1,3],["split",14,7.0,27,3],["split",15,15.0,0,4],["split",16,24.0,20,4],["split",19,25.0,8,4],["split",20,26.0,7,4],["split",21,20.0,0,4],["split",22,21.0,11,4],["split",23,20.0,0,4],["split",24,7.0,26,4],["split",25,25.0,19,4],["split",26,7.0,28,4],["split",27,12.0,27,4],["split",28,1.0,11,4],["leaf",29,-0.0015116823688554892],["leaf",30,0.01693795419462613],["leaf",31,-0.03216268345113767],["leaf",32,-0.058967523309059744],["leaf",33,-0.025396244982121986],["leaf",34,-0.012306848696875799],["leaf",39,-0.01728901013903056],["leaf",40,-0.0030637917094904457],["leaf",41,-0.026707025944018453],["leaf",42,-0.013545688383276005],["leaf",43,-0.012553667927136622],["leaf",44,0.011310629324255589],["leaf",45,-0.005315411859871802],["leaf",46,0.003552131315860669],["leaf",47,0.008602167455820828],["leaf",48,-0.013764310287038364],["leaf",49,0.013102362899686705],["leaf",50,0.024860315334980166],["leaf",51,-0.01954024383010226],["leaf",52,0.00816943170804997],["leaf",53,0.0033213210158656034],["leaf",54,0.017285801948786556],["leaf",55,0.029615952373948505],["leaf",56,0.0419191854713352],["leaf",57,0.010489979185114785],["leaf",58,0.023620713826057294]],[["split",0,7.0,8,0],["split",1,4.0,21,1],["split",2,8.0,20,1],["split",3,0.0,4,2],["split",4,7.0,0,2],["split",5,4.0,23,2],["split",6,5.0,8,2],["split",7,9.0,13,3],["split",8,25.0,11,3],["split",9,24.0,20,3],["split",10,10.0,6,3],["split",11,10.0,15,3],["split",12,1.0,3,3],["split",13,7.0,30,3],["split",14,0.0,2,3],["leaf",15,-0.03260800823677804],["leaf",16,-0.012917471195481547],["split",17,21.0,12,4],["split",18,18.0,0,4],["split",19,25.0,12,4],["leaf",20,-0.02926359625143251],["split",21,21.0,8,4],["split",22,19.0,0,4],["split",23,25.0,9,4],["split",24,3.0,28,4],["leaf",25,-0.018935244061981688],["split",26,26.0,9,4],["split",27,17.0,0,4],["leaf",28,0.03884000731576864],["split",29,22.0,16,4],["split",30,3.0,20,4],["leaf",35,-0.002777307484138507],["leaf",36,-0.018582644367133933],["leaf",37,-0.0047869501086341285],["leaf",38,0.015286759718968344],["leaf",39,-0.039637244884682565],["leaf",40,-0.053670369842122354],["leaf",43,-0.03986506956788498],["leaf",44,-0.027765157685093366],["leaf",45,-0.01931214697436515],["leaf",46,0.0024018249648650714],["leaf",47,-0.007954183351471673],["leaf",48,0.0023610985729421705],["leaf",49,0.005380812367955841],["leaf",50,0.017931336906288113],["leaf",53,-0.012896345771891972],["leaf",54,0.0019736693503666237],["leaf",55,0.01505590187728265],["leaf",56,0.02837106212318886],["leaf",59,0.001249593592087449],["leaf",60,-0.015866094950363695],["leaf",61,0.00811355611521939],["leaf",62,0.017554518236932975]],[["split",0,6.0,9,0],["split",1,7.0,0,1],["split",2,12.0,19,1],["split",3,24.0,18,2],["split",4,20.0,0,2],["split",5,21.0,9,2],["split",6,13.0,28,2],["split",7,10.0,9,3],["leaf",8,-0.024742884707489844],["split",9,19.0,0,3],["split",10,21.0,12,3],["split",11,20.0,0,3],["split",12,7.0,15,3],["split",13,10.0,12,3],["split",14,19.0,0,3],["leaf",15,-0.04879565532968671],["leaf",16,-0.036776628060598025],["split",19,2.0,13,4],["split",20,0.0,12,4],["split",21,2.0,21,4],["split",22,25.0,14,4],["split",23,12.0,8,4],["split",24,22.0,17,4],["split",25,10.0,7,4],["split",26,0.0,7,4],["split",27,7.0,29,4],["split",28,2.0,7,4],["split",29,17.0,0,4],["leaf",30,0.0031431266593218224],["leaf",39,-0.005888445208453892],["leaf",40,-0.02315195857117911],["leaf",41,0.008660601383956336],["leaf",42,-0.007687218570979503],["leaf",43,-0.026350823698663244],["leaf",44,-0.0011597673297689007],["leaf",45,-0.006520738204126243],["leaf",46,0.019350717776133394],["leaf",47,-0.011342139263832453],["leaf",48,0.0016822243492107453],["leaf",49,-0.007685191127870123],["leaf",50,-0.03034593983670119],["leaf",51,-0.011589174004366062],["leaf",52,0.0027050297269827156],["leaf",53,-0.0006685248457630228],["leaf",54,0.011702294107256882],["leaf",55,0.0008606674053320783],["leaf",56,0.01672661584056639],["leaf",57,0.020264417909758796],["leaf",58,0.01056029814024221],["leaf",59,0.02225860607842107],["leaf",60,0.03756848566800012]],[["split",0,7.0,8,0],["split",1,7.0,1,1],["split",2,6.0,22,1],["split",3,25.0,15,2],["split",4,4.0,15,2],["split",5,12.0,9,2],["split",6,7.0,25,2],["split",7,23.0,23,3],["split",8,10.0,9,3],["split",9,26.0,22,3],["split",10,10.0,7,3],["split",11,26.0,25,3],["split",12,24.0,10,3],["split",13,3.0,22,3],["split",14,2.0,1,3],["leaf",15,-0.04439614437704885],["split",16,26.0,8,4],["split",17,0.0,17,4],["split",18,21.0,16,4],["split",19,21.0,21,4],["split",20,23.0,20,4],["split",21,25.0,19,4],["split",22,3.0,4,4],["split",23,13.0,5,4],["leaf",24,0.00900887760764849],["split",25,22.0,26,4],["split",26,10.0,4,4],["split",27,0.0,16,4],["split",28,20.0,0,4],["split",29,10.0,22,4],["split",30,21.0,9,4],["leaf",33,-0.02068953983152026],["leaf",34,-0.04056794479045495],["leaf",35,-0.03732820737649588],["leaf",36,-0.02848692072125738],["leaf",37,-0.022926872488266872],["leaf",38,0.00029839598655094897],["leaf",39,-0.0017929473067133065],["leaf",40,-0.016611117825023898],["leaf",41,0.012040501752494503],["leaf",42,-0.004775653638362981],["leaf",43,-0.028933589775060806],["leaf",44,-0.015592016855228251],["leaf",45,-0.0008743226690930406],["leaf",46,-0.013143669598816073],["leaf",47,0.00403908021957521],["leaf",48,-0.009099774844737778],["leaf",51,0.0006712268634062197],["leaf",52,-0.012747525164644818],["leaf",53,-0.003933164804929772],["leaf",54,0.00778455542218318],["leaf",55,-0.0023346714555524312],["leaf",56,0.009887078740300589],["leaf",57,0.018524159207904255],["leaf",58,-0.0002794203671607778],["leaf",59,0.02357245272481619],["leaf",60,0.042706617285128555],["leaf",61,0.00957170317231461],["leaf",62,0.020245480387229446]],[["split",0,8.0,11,0],["split",1,7.0,3,1],["split",2,12.0,19,1],["split",3,20.0,0,2],["split",4,7.0,13,2],["split",5,1.0,3,2],["split",6,4.0,12,2],["split",7,2.0,17,3],["split",8,25.0,17,3],["split",9,10.0,7,3],["split",10,4.0,12,3],["split",11,9.0,11,3],["split",12,7.0,15,3],["split",13,7.0,28,3],["split",14,9.0,26,3],["split",15,3.0,8,4],["split",16,4.0,17,4],["leaf",17,-0.02071528404855355],["leaf",18,0.015100528742213315],["split",19,25.0,23,4],["split",20,2.0,13,4],["leaf",21,0.013005912782501577],["split",22,10.0,4,4],["leaf",23,-0.020620899196217988],["leaf",24,-0.006649652462164474],["split",25,3.0,17,4],["split",26,3.0,19,4],["split",27,10.0,14,4],["split",28,2.0,12,4],["split",29,13.0,18,4],["split",30,13.0,22,4],["leaf",31,-0.0016747025038903666],["leaf",32,-0.02124138008080404],["leaf",33,-0.009139344727887709],["leaf",34,-0.031950642429028474],["leaf",39,-0.0240168065251867],["leaf",40,-0.00497345937335522],["leaf",41,0.0012110154891611076],["leaf",42,-0.008986431375600047],["leaf",45,-0.021735470921167475],["leaf",46,0.0009923918942938707],["leaf",51,-0.011365959118140131],["leaf",52,0.0009488537021485889],["leaf",53,0.0036481008269689803],["leaf",54,0.014002577672903823],["leaf",55,0.00781787460692742],["leaf",56,0.018477235265413536],["leaf",57,0.03372059829842089],["leaf",58,0.01129355547023078],["leaf",59,-0.0015665920571911651],["leaf",60,0.00788579055645341],["leaf",61,0.011692568280768867],["leaf",62,0.02340707972667612]],[["split",0,6.0,13,0],["split",1,7.0,2,1],["split",2,12.0,24,1],["split",3,7.0,0,2],["split",4,26.0,0,2],["split",5,20.0,0,2],["split",6,5.0,4,2],["split",7,24.0,23,3],["split",8,3.0,11,3],["leaf",9,-0.026184264540131535],["split",10,10.0,3,3],["split",11,5.0,17,3],["split",12,21.0,9,3],["split",13,19.0,0,3],["split",14,1.0,9,3],["split",15,9.0,13,4],["leaf",16,-0.017462615197344437],["split",17,0.0,25,4],["split",18,4.0,19,4],["split",21,7.0,16,4],["split",22,23.0,29,4],["split",23,15.0,0,4],["split",24,26.0,14,4],["split",25,24.0,12,4],["split",26,2.0,20,4],["split",27,21.0,14,4],["leaf",28,0.003987680593776095],["split",29,22.0,12,4],["split",30,26.0,2,4],["leaf",31,-0.04350552379517256],["leaf",32,-0.03008920959925313],["leaf",35,-0.01463227698749112],["leaf",36,0.004249416662711462],["leaf",37,-0.015456591518320578],["leaf",38,-0.03162641501580259],["leaf",43,-0.01912973328137931],["leaf",44,0.002992420407111388],["leaf",45,-0.0038168230391605884],["leaf",46,-0.018536492211743028],["leaf",47,0.006547022723245683],["leaf",48,0.016178892615974643],["leaf",49,-0.007984436938646979],["leaf",50,0.004980561573114955],["leaf",51,-0.03274360733843562],["leaf",52,-0.011661178083196439],["leaf",53,-0.005258418259290865],["leaf",54,0.01718355135841702],["leaf",55,0.015104516173685736],["leaf",56,0.028368508337399567],["leaf",59,0.00909602085113355],["leaf",60,-0.007984781036380515],["leaf",61,-0.0069958674890045185],["leaf",62,0.013684310736438688]],[["split",0,6.0,9,0],["split",1,20.0,0,1],["split",2,6.0,22,1],["split",3,4.0,27,2],["split",4,25.0,11,2],["split",5,4.0,21,2],["split",6,5.0,7,2],["split",7,4.0,4,3],["split",8,2.0,29,3],["split",9,21.0,8,3],["split",10,2.0,14,3],["split",11,21.0,3,3],["split",12,10.0,7,3],["split",13,20.0,0,3],["split",14,24.0,3,3],["leaf",15,0.01076788466672374],["split",16,26.0,1,4],["split",17,24.0,25,4],["split",18,26.0,13,4],["leaf",19,-0.026978445086024573],["leaf",20,-0.004052399000731859],["leaf",21,-0.003997236994941834],["split",22,25.0,22,4],["split",23,7.0,16,4],["split",24,10.0,16,4],["split",25,7.0,16,4],["split",26,21.0,12,4],["split",27,2.0,10,4],["split",28,21.0,18,4],["split",29,9.0,16,4],["split",30,2.0,3,4],["leaf",33,-0.026173014862840443],["leaf",34,-0.009973464415583192],["leaf",35,-0.02420263313395698],["leaf",36,-0.0020631830796386933],["leaf",37,-0.023603574492663152],["leaf",38,-0.044319178820737604],["leaf",45,0.003914926178228725],["leaf",46,0.02250938626478437],["leaf",47,-0.017050670892224982],["leaf",48,-0.0008068091280893961],["leaf",49,0.0005585861557621087],["leaf",50,0.009334697771516012],["leaf",51,-0.02833893856469685],["leaf",52,-0.0034678566339340984],["leaf",53,-0.01062357707122086],["leaf",54,-0.0014375620913506682],["leaf",55,0.022246298291588463],["leaf",56,0.010443849712322349],["leaf",57,-0.007177647455523585],["leaf",58,0.019552602967558598],["leaf",59,-0.015938011708754477],["leaf",60,0.011225005022900242],["leaf",61,0.019745519330160048],["leaf",62,0.007443481618570318]],[["split",0,13.0,12,0],["split",1,7.0,0,1],["split",2,8.0,23,1],["split",3,4.0,28,2],["split",4,7.0,6,2],["split",5,4.0,25,2],["split",6,7.0,30,2],["leaf",7,-0.021155292489321176],["split",8,21.0,13,3],["split",9,25.0,2,3],["split",10,3.0,2,3],["split",11,12.0,21,3],["split",12,26.0,18,3],["split",13,1.0,4,3],["split",14,23.0,20,3],["leaf",17,-0.05339930239238055],["leaf",18,-0.03530459180392021],["split",19,9.0,13,4],["split",20,20.0,0,4],["split",21,2.0,15,4],["split",22,10.0,12,4],["split",23,25.0,18,4],["split",24,23.0,25,4],["split",25,23.0,17,4],["leaf",26,-0.00341502130146314],["leaf",27,-0.008618301341540647],["split",28,17.0,0,4],["leaf",29,0.04174628194357304],["leaf",30,0.018006300098302064],["leaf",39,-0.019808297627995373],["leaf",40,-0.03417299321018086],["leaf",41,-0.01556264854567366],["leaf",42,-0.0028352015708297843],["leaf",43,-0.0038188591017125663],["leaf",44,-0.020517095183040082],["leaf",45,-0.0076152896291300145],["leaf",46,-0.00013166154201283786],["leaf",47,-0.0007374057241590368],["leaf",48,0.00668654632791304],["leaf",49,0.012004371446719666],["leaf",50,-0.0044190896147364225],["leaf",51,-0.0119759917423699],["leaf",52,-0.02720540111316539],["leaf",57,0.010286232322290884],["leaf",58,0.024740630942053362]],[["split",0,7.0,11,0],["split",1,4.0,20,1],["split",2,4.0,11,1],["split",3,10.0,9,2],["split",4,25.0,29,2],["split",5,20.0,0,2],["split",6,21.0,11,2],["split",7,1.0,23,3],["split",8,25.0,26,3],["split",9,7.0,5,3],["split",10,1.0,9,3],["split",11,7.0,23,3],["split",12,21.0,9,3],["split",13,4.0,24,3],["split",14,7.0,29,3],["split",15,25.0,14,4],["leaf",16,-0.027165796523658213],["split",17,13.0,6,4],["split",18,9.0,18,4],["split",19,17.0,0,4],["split",20,1.0,11,4],["leaf",21,-0.00771137098915636],["leaf",22,0.01610741999020483],["split",23,3.0,20,4],["split",24,8.0,27,4],["leaf",25,-0.018592932778493186],["split",26,2.0,13,4],["split",27,7.0,20,4],["split",28,0.0,13,4],["split",29,26.0,23,4],["split",30,26.0,8,4],["leaf",31,0.003940503964605381],["leaf",32,-0.013526471322012403],["leaf",35,-0.0076006133762589675],["leaf",36,0.0028478652476249524],["leaf",37,-3.66337483130938e-05],["leaf",38,0.02711512833236705],["leaf",39,-0.017797988062772885],["leaf",40,-0.03598645197489831],["leaf",41,-0.015548183086524496],["leaf",42,-0.003886258314253289],["leaf",47,0.001649886082414661],["leaf",48,0.011776542229836751],["leaf",49,0.011171362630942427],["leaf",50,0.02175252874557394],["leaf",53,-0.0007326637317706258],["leaf",54,0.01898832314423907],["leaf",55,-0.0076385344439058615],["leaf",56,0.003718905581990588],["leaf",57,-0.008600235305714913],["leaf",58,-0.027383240931151],["leaf",59,0.000780254930742668],["leaf",60,0.01221202171916691],["leaf",61,0.007383681737979398],["leaf",62,0.02748717119324057]],[["split",0,7.0,10,0],["split",1,4.0,20,1],["split",2,4.0,15,1],["split",3,26.0,9,2],["split",4,10.0,3,2],["split",5,7.0,25,2],["split",6,22.0,24,2],["split",7,26.0,2,3],["split",8,12.0,23,3],["split",9,7.0,1,3],["split",10,19.0,0,3],["split",11,3.0,26,3],["split",12,19.0,0,3],["split",13,7.0,23,3],["split",14,23.0,22,3],["leaf",15,-0.024265241158613803],["split",16,1.0,19,4],["split",17,7.0,7,4],["leaf",18,0.010207142051063656],["leaf",19,-0.0451015029532263],["split",20,21.0,13,4],["split",21,26.0,21,4],["split",22,3.0,7,4],["split",23,10.0,1,4],["split",24,20.0,0,4],["split",25,25.0,28,4],["split",26,6.0,27,4],["split",27,26.0,6,4],["split",28,23.0,27,4],["split",29,8.0,17,4],["split",30,9.0,13,4],["leaf",33,-0.006296278669602416],["leaf",34,-0.02046706799869231],["leaf",35,-0.006015154837426913],["leaf",36,0.0016278494351865732],["leaf",41,-0.02761018496977976],["leaf",42,-0.010761417380715607],["leaf",43,-0.016785398981837403],["leaf",44,-0.005584503805825324],["leaf",45,0.010891296640069804],["leaf",46,-0.012966783230833372],["leaf",47,-0.008371619077778088],["leaf",48,0.004784713176007913],["leaf",49,0.014409677750309427],["leaf",50,-0.009057727770840487],["leaf",51,0.01625644103066596],["leaf",52,0.033264774345842636],["leaf",53,-0.0099551830612445],["leaf",54,0.006851537678264651],["leaf",55,-0.010040085226988965],["leaf",56,0.001654627026188752],["leaf",57,0.011037144143609201],["leaf",58,-0.010281164794859398],["leaf",59,0.0008436173742799239],["leaf",60,-0.011335260250184246],["leaf",61,-0.02886432224947427],["leaf",62,-0.014266919075973914]],[["split",0,8.0,14,0],["split",1,7.0,2,1],["split",2,20.0,0,1],["split",3,10.0,23,2],["split",4,7.0,20,2],["split",5,2.0,9,2],["split",6,21.0,23,2],["split",7,17.0,0,3],["split",8,23.0,19,3],["split",9,19.0,0,3],["split",10,26.0,5,3],["split",11,17.0,0,3],["split",12,26.0,13,3],["split",13,25.0,12,3],["split",14,2.0,11,3],["split",15,20.0,0,4],["leaf",16,-0.037970217093689064],["leaf",17,0.008058330812703995],["leaf",18,-0.009016397919038592],["split",19,21.0,13,4],["split",20,21.0,13,4],["leaf",21,-0.003437423331061139],["split",22,21.0,16,4],["split",23,7.0,26,4],["split",24,23.0,20,4],["split",25,23.0,25,4],["split",26,4.0,1,4],["split",27,2.0,18,4],["split",28,2.0,18,4],["leaf",29,0.004411453220173586],["leaf",30,0.0265896342840339],["leaf",31,-0.02154696091472437],["leaf",32,0.0030954918260078005],["leaf",39,-0.011540132810247052],["leaf",40,-0.0035320534004850824],["leaf",41,-0.0037627487550379573],["leaf",42,0.00721736455020382],["leaf",45,0.019202395185901843],["leaf",46,0.00608564182607662],["leaf",47,0.007544799244428831],["leaf",48,0.017637633459342394],["leaf",49,0.02688338687310244],["leaf",50,0.015024883779176906],["leaf",51,-0.0009578305387077614],["leaf",52,-0.023535783112232995],["leaf",53,0.022347799257481153],["leaf",54,0.006843394051129169],["leaf",55,-0.025868179380721678],["leaf",56,-0.002029866926801213],["leaf",57,-0.00693410380181671],["leaf",58,0.014606583867870084]],[["split",0,7.0,6,0],["split",1,21.0,17,1],["split",2,12.0,15,1],["split",3,7.0,2,2],["split",4,20.0,0,2],["split",5,4.0,19,2],["split",6,8.0,27,2],["split",7,12.0,7,3],["split",8,10.0,15,3],["split",9,13.0,1,3],["leaf",10,0.010580508472823137],["split",11,7.0,26,3],["split",12,26.0,1,3],["split",13,1.0,3,3],["split",14,5.0,2,3],["split",15,2.0,23,4],["leaf",16,-0.033084944164960245],["split",17,2.0,18,4],["split",18,25.0,11,4],["split",19,26.0,6,4],["split",20,1.0,14,4],["split",23,9.0,28,4],["split",24,2.0,11,4],["leaf",25,-0.01926323777050619],["split",26,12.0,3,4],["split",27,4.0,13,4],["split",28,7.0,30,4],["split",29,1.0,15,4],["split",30,9.0,18,4],["leaf",31,-0.010501016038701629],["leaf",32,-0.024664583907441646],["leaf",35,-0.025919954295463735],["leaf",36,-0.012641156939937946],["leaf",37,0.00025235103294800283],["leaf",38,-0.011720469126669544],["leaf",39,0.008587541756826089],["leaf",40,-0.00964069383655399],["leaf",41,-0.02348961900909528],["leaf",42,-0.0037868099671048887],["leaf",47,-0.0020101149887035937],["leaf",48,0.008177753101063551],["leaf",49,0.019188909128504572],["leaf",50,0.006616638975240296],["leaf",53,-0.01402049889062337],["leaf",54,-0.003215421420974267],["leaf",55,-0.01765925977876431],["leaf",56,-0.003490536486653636],["leaf",57,0.005941909935948935],["leaf",58,0.027104611285264705],["leaf",59,0.014333859277942744],["leaf",60,0.027778315873787374],["leaf",61,0.007687431537517916],["leaf",62,0.017180467313986696]],[["split",0,12.0,15,0],["split",1,4.0,22,1],["split",2,8.0,18,1],["split",3,21.0,5,2],["split",4,7.0,0,2],["split",5,23.0,26,2],["split",6,5.0,17,2],["split",7,9.0,14,3],["split",8,2.0,2,3],["split",9,12.0,2,3],["split",10,24.0,7,3],["split",11,21.0,2,3],["split",12,4.0,11,3],["split",13,19.0,0,3],["split",14,26.0,3,3],["split",15,1.0,18,4],["split",16,7.0,12,4],["leaf",17,0.013156159684283212],["split",18,0.0,28,4],["split",19,2.0,25,4],["leaf",20,-0.03394748012399566],["split",21,0.0,2,4],["split",22,21.0,16,4],["leaf",23,-0.01139274330795989],["split",24,11.0,29,4],["leaf",25,0.0020787064660640793],["split",26,10.0,9,4],["split",27,7.0,27,4],["split",28,23.0,22,4],["leaf",29,-0.020936837211289118],["split",30,23.0,15,4],["leaf",31,-0.018966596243819894],["leaf",32,-0.004988921704956734],["leaf",33,-0.009608370563945605],["leaf",34,0.0022023352594360045],["leaf",37,0.0002483175464634542],["leaf",38,-0.011787101900478105],["leaf",39,-0.0028863990081551176],["leaf",40,-0.02632443161809346],["leaf",43,-0.02661575172206947],["leaf",44,-0.013378001542410046],["leaf",45,-0.012240703382362248],["leaf",46,-0.0019242930236879213],["leaf",49,0.0027390263824892936],["leaf",50,0.01931544117667186],["leaf",53,-0.02714135721543441],["leaf",54,-0.011054189398943837],["leaf",55,0.010084069304950193],["leaf",56,0.01788877825314719],["leaf",57,0.004441949825374888],["leaf",58,-0.012293617750301413],["leaf",61,0.01250429798602797],["leaf",62,-0.003276898546303997]],[["split",0,6.0,8,0],["split",1,4.0,29,1],["split",2,8.0,26,1],["split",3,20.0,0,2],["split",4,23.0,29,2],["split",5,21.0,11,2],["split",6,1.0,7,2],["split",7,3.0,27,3],["split",8,21.0,9,3],["split",9,22.0,15,3],["leaf",10,-0.03179244988572527],["split",11,20.0,0,3],["split",12,23.0,29,3],["split",13,11.0,12,3],["split",14,7.0,28,3],["split",15,10.0,3,4],["leaf",16,0.015282566822194572],["leaf",17,-0.01785731746370982],["split",18,2.0,16,4],["split",19,11.0,11,4],["split",20,26.0,8,4],["split",23,2.0,17,4],["split",24,25.0,8,4],["split",25,2.0,11,4],["leaf",26,-0.017076997782850756],["leaf",27,-0.012462236113282288],["leaf",28,0.00211336157629538],["split",29,17.0,0,4],["split",30,24.0,14,4],["leaf",31,-0.021218074475584433],["leaf",32,-0.007090861635068939],["leaf",37,-0.003887033566191496],["leaf",38,0.0216177172806387],["leaf",39,-0.013357654991325657],["leaf",40,-0.02893201138290204],["leaf",41,-0.019387512962052243],["leaf",42,0.00028455558701828823],["leaf",47,0.002825825801503819],["leaf",48,-0.007934116381996007],["leaf",49,-0.026579260287354026],["leaf",50,-0.008102707401328936],["leaf",51,0.007960116151374305],["leaf",52,0.0028548560549699897],["leaf",59,0.007542499821543445],["leaf",60,0.02243544631914392],["leaf",61,0.012979769306783746],["leaf",62,0.02721679649363021]],[["split",0,7.0,15,0],["split",1,10.0,15,1],["split",2,2.0,3,1],["split",3,17.0,0,2],["split",4,26.0,17,2],["split",5,20.0,0,2],["split",6,1.0,0,2],["split",7,4.0,30,3],["split",8,23.0,25,3],["split",9,22.0,4,3],["split",10,1.0,5,3],["split",11,7.0,28,3],["leaf",12,-0.01311980525437421],["leaf",13,-0.02009827191433518],["split",14,26.0,27,3],["split",15,0.0,22,4],["split",16,9.0,13,4],["split",17,26.0,18,4],["leaf",18,-0.04738692612100362],["split",19,24.0,10,4],["split",20,23.0,3,4],["split",21,24.0,15,4],["split",22,20.0,0,4],["split",23,26.0,22,4],["split",24,9.0,15,4],["split",29,4.0,24,4],["split",30,23.0,10,4],["leaf",31,-0.009413526308943343],["leaf",32,-0.00116264731270261],["leaf",33,-0.03366089135505257],["leaf",34,-0.011840108408257223],["leaf",35,-0.0201271400961725],["leaf",36,0.007428385255073579],["leaf",39,-0.025234213458528443],["leaf",40,-0.011293659881927615],["leaf",41,0.005861269168189351],["leaf",42,-0.006704084910356327],["leaf",43,-0.015699843178678556],["leaf",44,-0.0035616114241909655],["leaf",45,0.009006669898591672],["leaf",46,-0.007771690781033386],["leaf",47,0.007501674061766805],["leaf",48,0.01974311556062222],["leaf",49,0.01478758728119324],["leaf",50,0.03324636767272891],["leaf",59,0.0038183065887576573],["leaf",60,-0.005464090066737346],["leaf",61,0.0043936176088672525],["leaf",62,0.015334971021452376]],[["split",0,6.0,15,0],["split",1,7.0,2,1],["split",2,26.0,9,1],["split",3,4.0,14,2],["split",4,1.0,8,2],["split",5,7.0,18,2],["split",6,19.0,0,2],["leaf",7,0.008200683070937308],["split",8,22.0,11,3],["split",9,17.0,0,3],["split",10,4.0,4,3],["split",11,23.0,7,3],["split",12,23.0,11,3],["split",13,21.0,7,3],["split",14,10.0,11,3],["split",17,26.0,4,4],["split",18,10.0,4,4],["split",19,20.0,0,4],["split",20,11.0,11,4],["split",21,12.0,17,4],["split",22,23.0,27,4],["leaf",23,-0.006412181477269416],["leaf",24,-0.02550139169599512],["split",25,13.0,18,4],["split",26,20.0,0,4],["split",27,20.0,0,4],["split",28,1.0,3,4],["split",29,8.0,24,4],["split",30,24.0,16,4],["leaf",35,-0.03434966654551875],["leaf",36,-0.018756771513259322],["leaf",37,-0.024953210115672225],["leaf",38,-0.0061635768996993355],["leaf",39,-0.006518069953815848],["leaf",40,0.009968761962694197],["leaf",41,-0.00639973660007517],["leaf",42,-0.03234986208313355],["leaf",43,0.004028650407914494],["leaf",44,0.01590984219307184],["leaf",45,-0.000840655178511164],["leaf",46,-0.012628534226033975],["leaf",51,-0.006660093656416289],["leaf",52,0.009678572332560236],["leaf",53,-0.006557036064987836],["leaf",54,0.013965385199045955],["leaf",55,0.004902356573960686],["leaf",56,-0.015603499003639288],["leaf",57,-0.009798096310139993],["leaf",58,0.011573716891257812],["leaf",59,-0.0030260240568121474],["leaf",60,-0.016770335899918973],["leaf",61,-0.004647657396118394],["leaf",62,0.005997961067170259]],[["split",0,7.0,6,0],["split",1,2.0,27,1],["split",2,6.0,30,1],["split",3,10.0,8,2],["split",4,20.0,0,2],["split",5,22.0,22,2],["split",6,2.0,3,2],["split",7,9.0,15,3],["split",8,21.0,2,3],["split",9,19.0,0,3],["leaf",10,0.007005119604500134],["split",11,4.0,25,3],["split",12,15.0,0,3],["leaf",13,0.032477205072286235],["split",14,9.0,18,3],["split",15,26.0,11,4],["split",16,26.0,15,4],["leaf",17,-0.019251276249048134],["split",18,6.0,2,4],["split",19,18.0,0,4],["leaf",20,-0.003079943989137312],["split",23,9.0,3,4],["split",24,24.0,27,4],["split",25,26.0,8,4],["split",26,0.0,8,4],["leaf",29,0.007898106668071675],["leaf",30,0.019481857390116514],["leaf",31,0.0009102130334339988],["leaf",32,-0.016640648171982582],["leaf",33,-0.026947052028041773],["leaf",34,-0.011462761892355263],["leaf",37,0.004102523303820885],["leaf",38,-0.0051751691298346555],["leaf",39,-0.028176327994889856],["leaf",40,-0.008759939709971692],["leaf",47,-0.002919104793072154],["leaf",48,0.005469899936717277],["leaf",49,-0.0021064603283952594],["leaf",50,-0.01994248635718907],["leaf",51,-0.01095756059741466],["leaf",52,-0.0019400591005757073],["leaf",53,-0.0074404281201459485],["leaf",54,0.014846867328883129]],[["split",0,7.0,7,0],["split",1,15.0,0,1],["split",2,26.0,19,1],["split",3,10.0,4,2],["split",4,0.0,9,2],["split",5,2.0,6,2],["split",6,1.0,5,2],["split",7,4.0,15,3],["split",8,2.0,9,3],["leaf",9,-0.029531667665903007],["split",10,2.0,20,3],["split",11,20.0,0,3],["split",12,25.0,4,3],["split",13,4.0,20,3],["split",14,17.0,0,3],["leaf",15,-0.0007156320014986712],["split",16,25.0,7,4],["split",17,10.0,16,4],["split",18,20.0,0,4],["leaf",21,0.0018884733570621458],["leaf",22,-0.01964584714755757],["split",23,9.0,13,4],["split",24,11.0,15,4],["split",25,9.0,26,4],["split",26,20.0,0,4],["split",27,22.0,14,4],["leaf",28,-0.015929627444628818],["split",29,21.0,2,4],["split",30,23.0,22,4],["leaf",33,-0.0333116912010142],["leaf",34,-0.015757641559001574],["leaf",35,-0.003837793176063219],["leaf",36,0.01322910256343013],["leaf",37,-0.00730305540916825],["leaf",38,0.004978706379369502],["leaf",47,0.0020133904563195398],["leaf",48,0.01384226756799333],["leaf",49,-0.02074195296794675],["leaf",50,-0.0020012313697689173],["leaf",51,-0.005986358845303796],["leaf",52,-0.02263283515094066],["leaf",53,-0.0019516958458564958],["leaf",54,0.007586440364629408],["leaf",55,0.007400553711421438],["leaf",56,-0.0033934768519461827],["leaf",59,-0.008748492672405893],["leaf",60,0.007029998254671063],["leaf",61,0.02378218436134448],["leaf",62,0.00419869059285278]],[["split",0,7.0,12,0],["split",1,3.0,30,1],["split",2,5.0,7,1],["split",3,7.0,3,2],["leaf",4,0.015219730137074674],["split",5,9.0,30,2],["split",6,3.0,25,2],["split",7,10.0,4,3],["split",8,25.0,29,3],["split",11,7.0,26,3],["leaf",12,0.022262690697394928],["split",13,7.0,22,3],["split",14,25.0,14,3],["split",15,9.0,5,4],["split",16,22.0,17,4],["split",17,15.0,0,4],["split",18,10.0,14,4],["split",23,21.0,10,4],["split",24,25.0,6,4],["split",27,13.0,12,4],["split",28,2.0,9,4],["leaf",29,0.005000143825154537],["split",30,4.0,11,4],["leaf",31,-0.012805506628797984],["leaf",32,-0.026767482582715364],["leaf",33,-0.012850720181124283],["leaf",34,0.0010514981791774716],["leaf",35,-0.002389449426651229],["leaf",36,-0.012159075862360742],["leaf",37,-0.0020958121596965145],["leaf",38,0.017149934366630034],["leaf",47,-0.003393494883556604],["leaf",48,0.009159063699977248],["leaf",49,-0.000751581905993464],["leaf",50,0.017009322133179836],["leaf",55,0.003268091610180281],["leaf",56,-0.004104574709746575],["leaf",57,0.00985092082013396],["leaf",58,0.0012978179031364872],["leaf",61,0.013461832596910825],["leaf",62,0.022229329555277278]],[["split",0,7.0,13,0],["split",1,4.0,7,1],["split",2,22.0,20,1],["split",3,3.0,25,2],["split",4,21.0,3,2],["split",5,7.0,26,2],["split",6,25.0,3,2],["split",7,2.0,7,3],["split",8,23.0,4,3],["split",9,10.0,9,3],["split",10,20.0,0,3],["split",11,3.0,20,3],["split",12,2.0,22,3],["split",13,26.0,9,3],["split",14,12.0,8,3],["leaf",15,0.006256266366597129],["leaf",16,-0.009429203562748161],["leaf",17,-0.00042098183561016446],["split",18,26.0,25,4],["split",19,1.0,8,4],["split",20,7.0,2,4],["split",21,19.0,0,4],["split",22,25.0,20,4],["split",23,0.0,29,4],["split",24,19.0,0,4],["split",25,1.0,3,4],["leaf",26,-0.0015854243096727237],["leaf",27,-0.0033256880293185574],["leaf",28,-0.017181905016714093],["leaf",29,-0.012210740485206423],["split",30,3.0,9,4],["leaf",37,0.004630936946424666],["leaf",38,0.015842508269989727],["leaf",39,-0.014617955986296893],["leaf",40,-0.032097481534335726],["leaf",41,-0.02191980034542132],["leaf",42,-0.005731923799764022],["leaf",43,-0.009055048262102544],["leaf",44,0.0022736781306256274],["leaf",45,-0.0027601822116703877],["leaf",46,0.020835105376340115],["leaf",47,0.0011294362129443442],["leaf",48,-0.012823404927962998],["leaf",49,0.010703885542058702],["leaf",50,-0.004611203497727477],["leaf",51,-0.0028468160070973985],["leaf",52,0.014368602454347637],["leaf",61,-0.003716061778489828],["leaf",62,0.004201381372039905]],[["split",0,7.0,7,0],["split",1,19.0,0,1],["split",2,21.0,9,1],["split",3,4.0,13,2],["split",4,13.0,2,2],["split",5,2.0,10,2],["split",6,12.0,27,2],["split",7,24.0,7,3],["split",8,17.0,0,3],["leaf",9,0.016496870336800386],["split",10,10.0,10,3],["split",11,20.0,0,3],["split",12,3.0,18,3],["split",13,24.0,9,3],["split",14,16.0,0,3],["leaf",15,-0.008702612781542215],["split",16,2.0,15,4],["split",17,0.0,22,4],["split",18,23.0,25,4],["leaf",21,0.009368347150202901],["split",22,11.0,15,4],["split",23,10.0,7,4],["leaf",24,-0.013532839049310018],["split",25,0.0,1,4],["split",26,26.0,10,4],["split",27,4.0,15,4],["split",28,13.0,10,4],["split",29,10.0,9,4],["leaf",30,0.020679200814829102],["leaf",33,0.013490186885501177],["leaf",34,0.0002994522720441288],["leaf",35,-0.013035729330248063],["leaf",36,0.0009280549146906673],["leaf",37,-0.009218044241570657],["leaf",38,-0.045850899426708086],["leaf",45,0.00271693008388687],["leaf",46,-0.010351758658575632],["leaf",47,-0.002234199390131528],["leaf",48,0.007800918002532739],["leaf",51,0.002233085162634261],["leaf",52,-0.010948984792543099],["leaf",53,-0.01518703572470076],["leaf",54,0.0016529082246953604],["leaf",55,0.0022279662311799247],["leaf",56,-0.006000159258372374],["leaf",57,0.011291156608941743],["leaf",58,0.0038860694502332264],["leaf",59,-0.0005021584349045961],["leaf",60,0.012385918170315065]],[["split",0,7.0,15,0],["split",1,10.0,19,1],["split",2,0.0,2,1],["split",3,26.0,30,2],["split",4,1.0,2,2],["split",5,2.0,15,2],["split",6,2.0,12,2],["split",7,3.0,29,3],["leaf",8,0.016428154529364598],["leaf",9,-0.01306811682761546],["split",10,17.0,0,3],["split",11,10.0,9,3],["split",12,26.0,6,3],["split",13,4.0,3,3],["split",14,20.0,0,3],["split",15,23.0,24,4],["leaf",16,0.009051807052191435],["split",21,9.0,7,4],["split",22,1.0,13,4],["leaf",23,-0.009389001218040422],["split",24,11.0,15,4],["leaf",25,-0.0267173691132776],["leaf",26,-0.0019971660197772014],["split",27,7.0,25,4],["split",28,7.0,30,4],["split",29,23.0,27,4],["split",30,25.0,21,4],["leaf",31,-0.005470645232532579],["leaf",32,-0.013465424402764544],["leaf",43,-0.008574490412109738],["leaf",44,0.0030952702972825933],["leaf",45,-0.00011261966827790536],["leaf",46,0.019674546018084424],["leaf",49,-0.0014858530829592654],["leaf",50,0.009249786581924263],["leaf",55,0.01114101458465443],["leaf",56,0.021838011707685023],["leaf",57,0.003728224745808147],["leaf",58,0.019644641681753557],["leaf",59,0.0015534161456728904],["leaf",60,-0.014599454461126695],["leaf",61,-0.0001063702592487695],["leaf",62,0.029157937985608624]],[["split",0,8.0,15,0],["split",1,4.0,15,1],["split",2,12.0,29,1],["split",3,10.0,24,2],["split",4,25.0,27,2],["split",5,21.0,6,2],["split",6,9.0,27,2],["split",7,21.0,9,3],["split",8,7.0,11,3],["split",9,10.0,7,3],["split",10,24.0,16,3],["split",11,7.0,29,3],["split",12,25.0,30,3],["split",13,0.0,18,3],["leaf",14,0.02415719675530975],["split",15,26.0,11,4],["split",16,18.0,0,4],["split",17,2.0,10,4],["leaf",18,0.0015882493011358649],["split",19,20.0,0,4],["split",20,19.0,0,4],["split",21,5.0,22,4],["split",22,2.0,25,4],["split",23,26.0,2,4],["leaf",24,0.00934763016193295],["split",25,1.0,1,4],["leaf",26,0.024687794595591967],["split",27,25.0,10,4],["split",28,3.0,19,4],["leaf",31,-0.01265681230295579],["leaf",32,-0.003942738491853029],["leaf",33,-0.0004144983855725612],["leaf",34,0.014071878550238483],["leaf",35,0.007392723074018619],["leaf",36,0.021067832305924693],["leaf",39,-0.014049632248326659],["leaf",40,0.005360759084271669],["leaf",41,-0.006835067440195511],["leaf",42,0.003028633607730545],["leaf",43,-0.01569633921189258],["leaf",44,-0.0003558811497325533],["leaf",45,0.004508907975722721],["leaf",46,0.023346012537882994],["leaf",47,-0.022018622499452415],["leaf",48,-0.0039868467402105565],["leaf",51,-0.01602588623363117],["leaf",52,0.002871646831433961],["leaf",55,-0.0057501985442055484],["leaf",56,0.007239933381344265],["leaf",57,0.006944126284565547],["leaf",58,0.017874119703068018]],[["split",0,8.0,15,0],["split",1,3.0,25,1],["split",2,12.0,16,1],["split",3,10.0,2,2],["split",4,12.0,23,2],["split",5,8.0,26,2],["split",6,9.0,27,2],["split",7,23.0,25,3],["split",8,1.0,0,3],["split",9,13.0,11,3],["leaf",10,-0.00708672789694697],["split",11,1.0,25,3],["split",12,5.0,9,3],["split",13,22.0,19,3],["split",14,17.0,0,3],["split",15,25.0,18,4],["split",16,7.0,2,4],["split",17,24.0,16,4],["split",18,25.0,1,4],["split",19,21.0,19,4],["split",20,22.0,17,4],["split",23,5.0,25,4],["split",24,3.0,9,4],["leaf",25,0.00173286200323578],["leaf",26,0.01535988738177961],["split",27,7.0,29,4],["split",28,26.0,26,4],["split",29,22.0,8,4],["leaf",30,0.02619159570720747],["leaf",31,-0.01244128736784277],["leaf",32,0.002129663029799285],["leaf",33,-0.03336233859442521],["leaf",34,-0.01732432975302618],["leaf",35,-0.013864129106103315],["leaf",36,-0.02627045584607169],["leaf",37,-0.014738449864569458],["leaf",38,-0.0024774624070378498],["leaf",39,0.017963944739149324],["leaf",40,-0.00048150490784480465],["leaf",41,0.00847772932193509],["leaf",42,-0.007498243086311526],["leaf",47,-0.00033958685987143584],["leaf",48,-0.014484623717272683],["leaf",49,-0.020469841079626013],["leaf",50,-0.0043407261620857455],["leaf",55,0.005284013418533222],["leaf",56,0.014845883196267493],["leaf",57,-0.0017662000084808901],["leaf",58,0.007790934167521084],["leaf",59,0.01952731602615922],["leaf",60,0.007318210132851491]],[["split",0,4.0,16,0],["split",1,7.0,21,1],["split",2,24.0,15,1],["split",3,4.0,1,2],["split",4,25.0,12,2],["split",5,21.0,5,2],["split",6,20.0,0,2],["split",7,8.0,21,3],["split",8,24.0,23,3],["split",9,24.0,9,3],["split",10,9.0,27,3],["split",11,22.0,5,3],["split",12,7.0,0,3],["split",13,23.0,29,3],["split",14,25.0,23,3],["split",15,12.0,15,4],["leaf",16,0.001117851077748015],["split",17,10.0,15,4],["split",18,25.0,27,4],["split",19,7.0,26,4],["split",20,5.0,2,4],["split",21,23.0,16,4],["split",22,24.0,15,4],["leaf",23,-0.024536846013615433],["split",24,25.0,6,4],["leaf",25,-0.02253731574060449],["split",26,11.0,17,4],["split",27,9.0,16,4],["split",28,21.0,19,4],["split",29,2.0,17,4],["leaf",30,0.030912103532464517],["leaf",31,0.002471503074979896],["leaf",32,0.01922361527023729],["leaf",35,-0.00635048529070208],["leaf",36,0.0015021780182172288],["leaf",37,0.002457558812513904],["leaf",38,0.015507315722902522],["leaf",39,-0.012146273696281231],["leaf",40,0.0033629051185965217],["leaf",41,0.015798307420622146],["leaf",42,0.0034847056342763834],["leaf",43,0.012482488496848371],["leaf",44,0.005253275739374657],["leaf",45,0.011954371230913243],["leaf",46,0.02851471719038112],["leaf",49,-0.02137274857045212],["leaf",50,-0.006798704324746326],["leaf",53,-0.0066022965945891215],["leaf",54,0.0011690676112713259],["leaf",55,-0.004150018394819667],["leaf",56,0.004397063215180066],["leaf",57,-0.007213481187886844],["leaf",58,-0.02508433284206621],["leaf",59,-0.014551072644464317],["leaf",60,0.01077341662230022]],[["split",0,4.0,14,0],["split",1,2.0,2,1],["split",2,10.0,5,1],["split",3,22.0,3,2],["split",4,0.0,0,2],["split",5,20.0,0,2],["split",6,7.0,6,2],["leaf",7,0.025547971216057418],["split",8,12.0,22,3],["leaf",9,-0.0162203699642417],["split",10,9.0,26,3],["split",11,7.0,14,3],["leaf",12,0.0055743219248508115],["split",13,6.0,0,3],["split",14,21.0,1,3],["split",17,23.0,8,4],["split",18,8.0,21,4],["split",21,7.0,23,4],["split",22,20.0,0,4],["split",23,17.0,0,4],["split",24,21.0,21,4],["split",27,2.0,21,4],["split",28,16.0,0,4],["split",29,25.0,9,4],["split",30,6.0,3,4],["leaf",35,-0.008325199002538336],["leaf",36,0.007915848017092672],["leaf",37,0.021041231112078507],["leaf",38,0.00914085730793232],["leaf",43,-0.0004260015939600541],["leaf",44,0.005542521018680875],["leaf",45,0.011227023479480869],["leaf",46,-0.003271875582961877],["leaf",47,-0.013687296289884618],["leaf",48,-0.033490425410604284],["leaf",49,-0.011301379297991195],["leaf",50,0.007842107426118412],["leaf",55,-0.006912189504498596],["leaf",56,0.012198575471502908],["leaf",57,-0.005775093537905252],["leaf",58,-0.017352074475836116],["leaf",59,-0.02320688553673224],["leaf",60,-0.005825724415977557],["leaf",61,0.016826804757864446],["leaf",62,0.0006805543626891705]],[["split",0,5.0,17,0],["split",1,22.0,19,1],["split",2,23.0,28,1],["split",3,8.0,15,2],["split",4,12.0,26,2],["split",5,2.0,21,2],["split",6,17.0,0,2],["split",7,25.0,6,3],["split",8,1.0,2,3],["split",9,3.0,3,3],["split",10,20.0,0,3],["split",11,19.0,0,3],["split",12,20.0,0,3],["split",13,2.0,20,3],["leaf",14,-0.03022687400377509],["split",15,24.0,14,4],["split",16,7.0,8,4],["leaf",17,-0.01077691291086563],["split",18,20.0,0,4],["leaf",19,-0.020514557330728527],["split",20,23.0,17,4],["split",21,10.0,21,4],["leaf",22,-0.010425208646403056],["split",23,3.0,22,4],["split",24,22.0,6,4],["split",25,1.0,2,4],["split",26,21.0,17,4],["split",27,26.0,13,4],["split",28,21.0,6,4],["leaf",31,-0.015967072340425884],["leaf",32,0.002745135386573997],["leaf",33,-0.002499527983032743],["leaf",34,0.006455052585145354],["leaf",37,0.009497173261088314],["leaf",38,-7.17685862550302e-05],["leaf",41,0.0010964748442917822],["leaf",42,-0.006347722675493519],["leaf",43,0.005560490924603175],["leaf",44,0.01715378121777536],["leaf",47,-0.0008451728951864448],["leaf",48,0.013600454642504058],["leaf",49,-6.782662147989452e-05],["leaf",50,0.015419348010644755],["leaf",51,-0.017538673114019552],["leaf",52,-0.005188961005456211],["leaf",53,-0.006308147840166422],["leaf",54,0.023055864266908626],["leaf",55,-0.006549065175911181],["leaf",56,-0.0288730124803036],["leaf",57,-0.015543607681291597],["leaf",58,0.002290018359248449]],[["split",0,7.0,16,0],["split",1,26.0,24,1],["split",2,10.0,9,1],["split",3,7.0,0,2],["split",4,23.0,28,2],["split",5,2.0,7,2],["split",6,0.0,9,2],["split",7,11.0,21,3],["split",8,8.0,1,3],["split",9,25.0,25,3],["split",10,13.0,6,3],["split",11,7.0,23,3],["split",12,17.0,0,3],["split",13,21.0,5,3],["split",14,7.0,25,3],["split",15,1.0,2,4],["leaf",16,-0.027991929378898245],["split",17,25.0,26,4],["split",18,26.0,1,4],["split",19,11.0,28,4],["split",20,11.0,15,4],["leaf",21,-0.0023447322795994686],["leaf",22,-0.02148597035381165],["split",23,22.0,14,4],["split",24,3.0,10,4],["split",25,2.0,27,4],["leaf",26,-0.013728590315615347],["split",27,10.0,14,4],["split",28,6.0,17,4],["split",29,9.0,29,4],["split",30,12.0,14,4],["leaf",31,0.008524238324120102],["leaf",32,-0.01659253993026211],["leaf",35,0.0014133480175872433],["leaf",36,0.016219467527385772],["leaf",37,-0.017854168544268603],["leaf",38,-0.0036264850546375432],["leaf",39,0.0035581785665829173],["leaf",40,-0.01017609394211404],["leaf",41,0.0038541711071704],["leaf",42,0.016544279815814817],["leaf",47,-0.0004141645296330085],["leaf",48,-0.011787516395938889],["leaf",49,0.024706820310150587],["leaf",50,0.0030195777233716255],["leaf",51,-0.0036321139472414122],["leaf",52,0.01088953549808379],["leaf",55,0.007886239861510786],["leaf",56,-0.008029579688048948],["leaf",57,0.012518102656630519],["leaf",58,0.0010528862469856128],["leaf",59,0.0032661137268622056],["leaf",60,0.01634705241573836],["leaf",61,0.02339722801861598],["leaf",62,0.007658961958031799]],[["split",0,7.0,13,0],["split",1,10.0,2,1],["split",2,2.0,1,1],["split",3,3.0,12,2],["split",4,26.0,1,2],["split",5,9.0,11,2],["split",6,3.0,13,2],["split",7,7.0,3,3],["split",8,23.0,28,3],["split",9,6.0,4,3],["split",10,1.0,6,3],["split",11,26.0,17,3],["split",12,23.0,13,3],["split",13,0.0,18,3],["split",14,26.0,8,3],["leaf",15,-0.0274473809252718],["leaf",16,-0.010804132017266863],["split",17,8.0,7,4],["leaf",18,0.006625326838332622],["split",19,23.0,17,4],["leaf",20,-0.02646930390286003],["split",21,20.0,0,4],["split",22,21.0,12,4],["leaf",23,0.01025659427679224],["leaf",24,-0.0030661973114138054],["split",25,10.0,20,4],["split",26,26.0,21,4],["split",27,22.0,20,4],["split",28,2.0,22,4],["split",29,10.0,5,4],["split",30,19.0,0,4],["leaf",35,-0.0209871541260585],["leaf",36,-0.008372557666990289],["leaf",39,-0.018163477190019404],["leaf",40,0.0037640119687035057],["leaf",43,-0.009733538346350778],["leaf",44,0.009945323865723341],["leaf",45,-0.0032644242803634457],["leaf",46,0.002641837759686305],["leaf",51,0.0006873292286163445],["leaf",52,0.016996983019802212],["leaf",53,0.012017470108976074],["leaf",54,0.028289938582227645],["leaf",55,0.005469389295234997],["leaf",56,-0.0026525558900535686],["leaf",57,-0.0072814843399116765],["leaf",58,0.006061902438494178],["leaf",59,-0.018933070224635935],["leaf",60,0.0004621910402845745],["leaf",61,0.007609889460921509],["leaf",62,-0.0028065783955533528]],[["split",0,7.0,18,0],["split",1,7.0,0,1],["split",2,9.0,15,1],["split",3,23.0,28,2],["split",4,26.0,21,2],["split",5,2.0,7,2],["split",6,22.0,7,2],["split",7,9.0,17,3],["leaf",8,-0.0031821566012614445],["split",9,13.0,1,3],["split",10,1.0,2,3],["split",11,8.0,21,3],["split",12,26.0,8,3],["split",13,4.0,5,3],["split",14,26.0,22,3],["leaf",15,-0.0318825941203793],["leaf",16,-0.0077051808134984655],["split",19,21.0,25,4],["split",20,4.0,20,4],["leaf",21,-0.01089966300193291],["split",22,1.0,26,4],["split",23,5.0,11,4],["split",24,4.0,15,4],["split",25,3.0,6,4],["split",26,5.0,24,4],["leaf",27,0.02618478999881279],["split",28,26.0,3,4],["split",29,13.0,25,4],["split",30,1.0,9,4],["leaf",39,-0.0007580877405896296],["leaf",40,0.019984051230031864],["leaf",41,-0.002176309237062582],["leaf",42,-0.008079944499147191],["leaf",45,0.0031105707503224223],["leaf",46,-0.009020096284707391],["leaf",47,0.023258198521671165],["leaf",48,0.013499573211096015],["leaf",49,0.005187795445958921],["leaf",50,-0.008240687430696565],["leaf",51,-0.014542336727735755],["leaf",52,-0.0012630738354564878],["leaf",53,-0.0005578311150239562],["leaf",54,0.016594861280200615],["leaf",57,-0.007661620628036743],["leaf",58,0.01340214148888984],["leaf",59,0.005223172211889395],["leaf",60,-0.00425577067397668],["leaf",61,0.0006184760141511814],["leaf",62,0.013234048247599895]],[["split",0,7.0,19,0],["split",1,10.0,15,1],["split",2,9.0,29,1],["split",3,17.0,0,2],["split",4,17.0,0,2],["split",5,15.0,0,2],["split",6,24.0,20,2],["split",7,12.0,17,3],["split",8,23.0,22,3],["split",9,24.0,5,3],["split",10,2.0,7,3],["split",11,25.0,28,3],["split",12,11.0,21,3],["leaf",13,0.011404216691063125],["leaf",14,0.024005818261321005],["split",15,3.0,1,4],["split",16,26.0,14,4],["split",17,1.0,11,4],["split",18,12.0,5,4],["split",19,1.0,11,4],["split",20,25.0,1,4],["leaf",21,0.025081238592730874],["split",22,13.0,16,4],["split",23,13.0,21,4],["split",24,1.0,9,4],["split",25,2.0,12,4],["split",26,10.0,14,4],["leaf",31,-0.01218792517056736],["leaf",32,8.426371651544344e-05],["leaf",33,-0.01613025347876587],["leaf",34,-0.004510908501927546],["leaf",35,-0.016243571989163964],["leaf",36,0.0061690353197379096],["leaf",37,-0.0087895636964752],["leaf",38,-0.03785143359178514],["leaf",39,-0.015165529194423995],["leaf",40,-0.00042452506711442446],["leaf",41,-0.012705765022052442],["leaf",42,0.0017732936136351431],["leaf",45,-0.0005171526243928295],["leaf",46,0.017020115784149886],["leaf",47,0.004909688725410804],["leaf",48,-0.0011574453299824942],["leaf",49,-0.0044428827528788465],["leaf",50,0.01713581756898581],["leaf",51,0.02083383747324406],["leaf",52,0.007842816355566683],["leaf",53,0.005239388368295346],["leaf",54,-0.004362566104457613]],[["split",0,2.0,1,0],["split",1,20.0,0,1],["split",2,21.0,10,1],["split",3,19.0,0,2],["leaf",4,-0.010484393591554968],["split",5,20.0,0,2],["split",6,7.0,13,2],["split",7,23.0,19,3],["leaf",8,-0.0043277461817261085],["split",11,3.0,28,3],["split",12,2.0,20,3],["split",13,20.0,0,3],["split",14,19.0,0,3],["split",15,5.0,3,4],["leaf",16,0.021942212596217182],["split",23,2.0,23,4],["split",24,5.0,4,4],["split",25,22.0,16,4],["split",26,23.0,16,4],["split",27,4.0,4,4],["split",28,2.0,16,4],["split",29,25.0,30,4],["split",30,23.0,18,4],["leaf",31,0.01713523528770552],["leaf",32,0.004876240773920866],["leaf",47,-0.0013708988485725457],["leaf",48,-0.007677045195551535],["leaf",49,0.0009102788679584408],["leaf",50,0.019594061572031047],["leaf",51,-0.01323364395178874],["leaf",52,-0.03850445192025402],["leaf",53,-0.00872764901437175],["leaf",54,0.006611221395077996],["leaf",55,0.00618385547939379],["leaf",56,-0.004393645669016656],["leaf",57,-0.00829216368001769],["leaf",58,0.016195805230903915],["leaf",59,0.0033260299899694596],["leaf",60,0.019300384308545332],["leaf",61,0.0011851629251340516],["leaf",62,-0.01387316729998115]],[["split",0,4.0,27,0],["split",1,26.0,25,1],["split",2,7.0,1,1],["split",3,7.0,20,2],["split",4,1.0,2,2],["split",5,23.0,23,2],["split",6,3.0,17,2],["split",7,15.0,0,3],["split",8,25.0,6,3],["leaf",9,-0.010481812537559365],["split",10,9.0,20,3],["leaf",11,-0.026009525571601966],["split",12,22.0,13,3],["split",13,21.0,18,3],["leaf",14,0.004161949666897535],["split",15,21.0,4,4],["split",16,7.0,4,4],["split",17,10.0,10,4],["split",18,13.0,12,4],["split",21,24.0,16,4],["split",22,12.0,29,4],["leaf",25,0.00028224321865001265],["leaf",26,-0.013874700998368851],["split",27,1.0,13,4],["split",28,1.0,5,4],["leaf",31,-0.007013236947529052],["leaf",32,-0.00014836400107879477],["leaf",33,-0.025993498060518642],["leaf",34,-0.005827993604495637],["leaf",35,-0.013234999513773165],["leaf",36,0.0016767045808085842],["leaf",37,0.025739680908816273],["leaf",38,0.0030025503696882367],["leaf",43,-0.0019279403296291635],["leaf",44,0.006978285743667805],["leaf",45,0.0060992787690086455],["leaf",46,0.02022631108206065],["leaf",55,-0.00839296317826827],["leaf",56,-0.01801716036499523],["leaf",57,0.00738266543920294],["leaf",58,-0.005122086703550273]],[["split",0,7.0,14,0],["split",1,15.0,0,1],["split",2,21.0,2,1],["split",3,25.0,28,2],["split",4,23.0,17,2],["split",5,26.0,2,2],["split",6,22.0,9,2],["split",7,4.0,29,3],["split",8,24.0,25,3],["split",9,26.0,19,3],["split",10,9.0,22,3],["leaf",11,-0.014033127348134493],["split",12,23.0,8,3],["split",13,11.0,26,3],["split",14,0.0,5,3],["split",15,21.0,13,4],["split",16,25.0,10,4],["split",17,22.0,10,4],["leaf",18,0.019470983521304677],["split",19,4.0,16,4],["leaf",20,0.010536510954162068],["split",21,11.0,20,4],["leaf",22,-0.006582358691443434],["leaf",25,0.008255757119168544],["split",26,2.0,15,4],["split",27,25.0,9,4],["split",28,5.0,14,4],["split",29,24.0,4,4],["split",30,1.0,18,4],["leaf",31,-0.005103985311359279],["leaf",32,0.000681518589289935],["leaf",33,-0.002646165535034326],["leaf",34,-0.015665931425838293],["leaf",35,0.012729936441073869],["leaf",36,-0.0019665522289474767],["leaf",39,-0.017639031107713487],["leaf",40,-0.0004333535253912363],["leaf",43,-0.017160451153305167],["leaf",44,-0.02892846129268143],["leaf",53,0.000660467820228192],["leaf",54,-0.01371020665157768],["leaf",55,0.01264358824429689],["leaf",56,0.003954893839678701],["leaf",57,0.00920184118422751],["leaf",58,0.030330651309824784],["leaf",59,-0.012759417815279126],["leaf",60,-0.0015746286395431778],["leaf",61,0.0009882260150319807],["leaf",62,0.006075900410994221]],[["split",0,7.0,28,0],["split",1,26.0,24,1],["split",2,14.0,0,1],["split",3,26.0,1,2],["split",4,1.0,3,2],["split",5,18.0,0,2],["leaf",6,0.021527558119663413],["split",7,13.0,3,3],["split",8,7.0,0,3],["split",9,0.0,17,3],["split",10,24.0,20,3],["split",11,22.0,16,3],["leaf",12,-0.009528781918399872],["split",15,24.0,17,4],["split",16,21.0,3,4],["split",17,24.0,21,4],["split",18,13.0,0,4],["leaf",19,-0.0017642322570310335],["leaf",20,-0.0205579904325213],["split",21,20.0,0,4],["split",22,9.0,4,4],["split",23,25.0,6,4],["split",24,9.0,14,4],["leaf",31,-0.00825361255538172],["leaf",32,0.012373267285810856],["leaf",33,-0.0008710460738959337],["leaf",34,-0.020158431770848497],["leaf",35,-0.02101733455135271],["leaf",36,0.0020683891758820567],["leaf",37,0.01171679869230992],["leaf",38,-0.001724666074253844],["leaf",43,0.005070702215085889],["leaf",44,-0.007708192399424169],["leaf",45,-0.005205869633651246],["leaf",46,0.011546841329715028],["leaf",47,0.0002923222940361241],["leaf",48,0.019645608245701218],["leaf",49,-0.004068476434103947],["leaf",50,0.008399092202596246]],[["split",0,7.0,2,0],["split",1,15.0,0,1],["split",2,6.0,26,1],["split",3,24.0,11,2],["leaf",4,-0.02428942711204088],["split",5,21.0,7,2],["split",6,19.0,0,2],["split",7,26.0,14,3],["split",8,2.0,27,3],["split",11,22.0,16,3],["split",12,1.0,0,3],["split",13,7.0,30,3],["split",14,9.0,21,3],["split",15,24.0,3,4],["leaf",16,-0.0038352379298792925],["split",17,1.0,6,4],["split",18,0.0,17,4],["split",23,0.0,0,4],["split",24,20.0,0,4],["leaf",25,-0.013175487245771857],["split",26,20.0,0,4],["split",27,24.0,3,4],["leaf",28,0.024036773873378332],["leaf",29,-0.013331275040101353],["leaf",30,0.0038224938136032496],["leaf",31,-0.00671626200311842],["leaf",32,-0.028649083602785987],["leaf",35,0.014598863805280913],["leaf",36,-0.0026786713633023813],["leaf",37,-0.01629341285774469],["leaf",38,-0.004918077275942241],["leaf",47,-0.017097746551489983],["leaf",48,0.0016655355755910652],["leaf",49,-0.004027927639180965],["leaf",50,-0.024148462668540312],["leaf",53,0.0006605778477831829],["leaf",54,0.006568546091427237],["leaf",55,-0.005519388751328203],["leaf",56,0.007774295654492172]],[["split",0,7.0,26,0],["split",1,10.0,2,1],["split",2,19.0,0,1],["split",3,12.0,2,2],["split",4,7.0,0,2],["split",5,9.0,18,2],["split",6,8.0,26,2],["leaf",7,0.003999121703661615],["split",8,5.0,26,3],["split",9,26.0,5,3],["split",10,26.0,6,3],["split",11,23.0,23,3],["split",12,10.0,5,3],["leaf",13,0.004305433752745663],["leaf",14,-0.012601826894821014],["split",17,4.0,7,4],["leaf",18,0.0017289618036983837],["leaf",19,-0.02011717060559112],["split",20,3.0,7,4],["split",21,6.0,0,4],["split",22,23.0,28,4],["split",23,7.0,30,4],["split",24,10.0,4,4],["leaf",25,-0.0003582235738869201],["split",26,12.0,22,4],["leaf",35,-0.0035619867520453904],["leaf",36,-0.01580727182407922],["leaf",41,0.004595855997999834],["leaf",42,-0.015982860554041092],["leaf",43,0.012488696432601832],["leaf",44,-0.005409855796738062],["leaf",45,0.0010327152048105047],["leaf",46,-0.008487475797869922],["leaf",47,0.0049032678293410336],["leaf",48,0.023208727867853024],["leaf",49,0.009531762139876334],["leaf",50,-0.010269861931804085],["leaf",53,0.020149703370440697],["leaf",54,0.009475273966511811]],[["split",0,23.0,29,0],["split",1,7.0,6,1],["split",2,20.0,0,1],["split",3,3.0,3,2],["split",4,26.0,0,2],["split",5,17.0,0,2],["leaf",6,0.007968442100577653],["split",7,21.0,24,3],["split",8,21.0,1,3],["split",9,25.0,6,3],["split",10,24.0,16,3],["split",11,13.0,4,3],["leaf",12,-0.031047790924590875],["split",15,5.0,27,4],["leaf",16,0.019882698270015584],["leaf",17,-0.024146815738612114],["split",18,12.0,18,4],["leaf",19,-0.025951611027557536],["leaf",20,0.001352147503378365],["split",21,22.0,25,4],["split",22,25.0,23,4],["split",23,9.0,10,4],["split",24,2.0,17,4],["leaf",31,-0.007254658081410191],["leaf",32,0.007579211048275305],["leaf",37,-0.00523906162526669],["leaf",38,-0.016213846039586317],["leaf",43,0.0013040259924615518],["leaf",44,-0.005930600648198281],["leaf",45,0.0022726821771148686],["leaf",46,0.009575016446169616],["leaf",47,0.011049772979515966],["leaf",48,-0.00546630864969658],["leaf",49,-0.007423664023526332],["leaf",50,-0.024540064742977322]],[["split",0,4.0,18,0],["split",1,20.0,0,1],["split",2,26.0,25,1],["split",3,1.0,12,2],["split",4,25.0,22,2],["split",5,3.0,0,2],["split",6,7.0,3,2],["split",7,8.0,4,3],["split",8,9.0,26,3],["split",9,23.0,19,3],["split",10,2.0,13,3],["split",11,12.0,5,3],["split",12,21.0,4,3],["leaf",13,-0.01018391051057868],["split",14,12.0,18,3],["leaf",15,0.012603473590165644],["split",16,26.0,4,4],["split",17,2.0,11,4],["split",18,25.0,20,4],["split",19,11.0,9,4],["split",20,2.0,18,4],["leaf",21,0.0006811661738448102],["leaf",22,0.02850699754456011],["leaf",23,0.01209028554955335],["leaf",24,0.00020879340687687074],["split",25,20.0,0,4],["split",26,7.0,27,4],["split",29,2.0,18,4],["leaf",30,-0.0017623192523992506],["leaf",33,-0.012215371576559537],["leaf",34,-0.0012380710256955174],["leaf",35,0.005723426090119543],["leaf",36,-9.358952245382773e-05],["leaf",37,0.0060880818834862325],["leaf",38,0.019330694711552736],["leaf",39,-0.026213535533831826],["leaf",40,-0.012088704565245368],["leaf",41,-0.006093103461241157],["leaf",42,0.012470132662035184],["leaf",51,-0.007557004170960054],["leaf",52,-0.022985911727179828],["leaf",53,-0.004339926722322313],["leaf",54,0.006179405194583613],["leaf",59,-0.0006775346487454803],["leaf",60,0.01641254556563997]],[["split",0,7.0,1,0],["split",1,19.0,0,1],["split",2,10.0,1,1],["split",3,5.0,26,2],["leaf",4,0.007888117659383726],["split",5,17.0,0,2],["split",6,21.0,15,2],["split",7,23.0,21,3],["split",8,2.0,23,3],["split",11,0.0,13,3],["leaf",12,-0.02111568536729127],["split",13,20.0,0,3],["split",14,24.0,2,3],["leaf",15,-0.008506906046761738],["leaf",16,-0.025757956316163882],["leaf",17,0.00494075041055391],["split",18,10.0,10,4],["split",23,7.0,21,4],["split",24,3.0,17,4],["split",27,9.0,30,4],["split",28,23.0,17,4],["split",29,3.0,9,4],["split",30,23.0,27,4],["leaf",37,-0.01575751934624548],["leaf",38,-0.0022740627680475378],["leaf",47,-0.0050937853429630815],["leaf",48,0.016035155060451483],["leaf",49,-0.01588731736955785],["leaf",50,-0.0010577980097561932],["leaf",55,0.0006831747945133644],["leaf",56,0.015801039551448515],["leaf",57,-0.015548154811569093],["leaf",58,0.0032909658069792723],["leaf",59,-0.01128920346409355],["leaf",60,0.0020381676984314507],["leaf",61,0.004868972811688137],["leaf",62,-0.001963906962895047]],[["split",0,26.0,1,0],["split",1,7.0,26,1],["split",2,10.0,3,1],["split",3,11.0,17,2],["split",4,24.0,13,2],["split",5,4.0,23,2],["split",6,1.0,8,2],["split",7,1.0,21,3],["split",8,10.0,18,3],["leaf",9,-0.001594006413024098],["leaf",10,0.007987639764991974],["split",11,25.0,21,3],["split",12,8.0,1,3],["split",13,20.0,0,3],["split",14,17.0,0,3],["split",15,22.0,10,4],["leaf",16,-0.01725836428264343],["leaf",17,-0.03049109468515042],["leaf",18,-0.008030526808550029],["split",23,20.0,0,4],["split",24,5.0,14,4],["leaf",25,0.0019172529848754945],["split",26,24.0,9,4],["split",27,2.0,29,4],["split",28,10.0,19,4],["split",29,25.0,19,4],["split",30,2.0,22,4],["leaf",31,-0.011574000093600562],["leaf",32,0.004305338163047361],["leaf",47,-0.004395671660909589],["leaf",48,-0.017604100927197303],["leaf",49,-0.00035346544498442026],["leaf",50,0.012809387016096468],["leaf",53,-0.0068989639394773945],["leaf",54,-0.02355878306438471],["leaf",55,-0.0022028566442574607],["leaf",56,-0.013162742903424635],["leaf",57,0.017769604286591776],["leaf",58,-0.012452389043833197],["leaf",59,-0.00044426861385899754],["leaf",60,0.0038309551936498948],["leaf",61,0.011391060292264363],["leaf",62,-0.0013956448461198902]],[["split",0,24.0,22,0],["split",1,5.0,8,1],["split",2,12.0,7,1],["split",3,20.0,0,2],["split",4,23.0,29,2],["split",5,7.0,2,2],["split",6,25.0,21,2],["split",7,9.0,9,3],["split",8,22.0,22,3],["split",9,19.0,0,3],["split",10,13.0,2,3],["split",11,0.0,20,3],["split",12,8.0,9,3],["split",13,10.0,8,3],["split",14,26.0,10,3],["split",15,8.0,12,4],["split",16,19.0,0,4],["split",17,3.0,25,4],["leaf",18,-0.022311867765684668],["split",19,26.0,10,4],["split",20,5.0,17,4],["leaf",21,-0.002044503921234821],["split",22,4.0,21,4],["leaf",23,-0.008946999426226237],["leaf",24,0.01981288907627923],["split",25,26.0,15,4],["split",26,9.0,15,4],["split",27,0.0,20,4],["split",28,21.0,28,4],["split",29,12.0,16,4],["split",30,20.0,0,4],["leaf",31,0.015009096474799663],["leaf",32,-0.0028699512185229757],["leaf",33,0.009260563473840152],["leaf",34,-0.0023470697338440444],["leaf",35,0.009624694454478853],["leaf",36,-0.014621450406935093],["leaf",39,-0.0060285492567208766],["leaf",40,-0.0009826464738127354],["leaf",41,-0.0028062653831851664],["leaf",42,0.00691112524748761],["leaf",45,-0.006002289684771863],["leaf",46,-0.028102679088921025],["leaf",51,0.010523785060048264],["leaf",52,0.026575096829991707],["leaf",53,-0.0007442199544646868],["leaf",54,0.007857702754062999],["leaf",55,-0.00046924477073854754],["leaf",56,-0.013629887210112627],["leaf",57,0.00346129188498196],["leaf",58,-0.009920844379370011],["leaf",59,-0.011744365313986285],["leaf",60,0.004579632017814535],["leaf",61,0.00618855547045455],["leaf",62,0.022480831584744437]],[["split",0,7.0,29,0],["split",1,3.0,24,1],["split",2,11.0,3,1],["split",3,11.0,18,2],["split",4,20.0,0,2],["leaf",5,-0.010049618937756187],["split",6,2.0,11,2],["split",7,10.0,1,3],["split",8,24.0,29,3],["split",9,9.0,28,3],["split",10,3.0,28,3],["split",13,3.0,10,3],["split",14,0.0,14,3],["split",15,23.0,26,4],["split",16,12.0,1,4],["split",17,5.0,29,4],["split",18,25.0,20,4],["split",19,4.0,1,4],["split",20,19.0,0,4],["split",21,25.0,17,4],["leaf",22,-0.017084514920645413],["split",27,24.0,13,4],["leaf",28,0.008747203411964041],["leaf",29,-0.009235192401146858],["leaf",30,0.011130529065011193],["leaf",31,-0.005721360804438194],["leaf",32,-0.024522690540430915],["leaf",33,0.00815741258045107],["leaf",34,-0.0020549426584780908],["leaf",35,0.00020675322074259258],["leaf",36,0.009312892603223454],["leaf",37,0.0004913494236557863],["leaf",38,0.026333095296590312],["leaf",39,0.01102418930981105],["leaf",40,0.0014660755055415824],["leaf",41,0.016717636440596345],["leaf",42,-0.001923699528466601],["leaf",43,-0.007494781947008965],["leaf",44,0.014378128371150618],["leaf",55,0.028299049955223747],["leaf",56,0.014886774090046658]],[["split",0,10.0,10,0],["split",1,7.0,24,1],["split",2,7.0,14,1],["split",3,20.0,0,2],["split",4,14.0,0,2],["split",5,20.0,0,2],["split",6,0.0,5,2],["split",7,4.0,1,3],["split",8,23.0,24,3],["split",9,22.0,20,3],["leaf",10,0.013211007341867248],["split",11,2.0,19,3],["split",12,21.0,21,3],["split",13,2.0,3,3],["split",14,13.0,12,3],["leaf",15,0.012998881867811243],["split",16,7.0,0,4],["split",17,2.0,14,4],["leaf",18,0.023695576965837923],["split",19,25.0,7,4],["split",20,10.0,5,4],["split",23,5.0,22,4],["split",24,19.0,0,4],["split",25,1.0,19,4],["leaf",26,0.007568504707942176],["leaf",27,0.008298186643661483],["split",28,10.0,13,4],["split",29,25.0,20,4],["split",30,25.0,1,4],["leaf",33,-0.022428638692698125],["leaf",34,-0.006365546216525093],["leaf",35,-0.015487708840373034],["leaf",36,0.0011441060653858547],["leaf",39,-0.005022928295376536],["leaf",40,0.0076970588121634854],["leaf",41,-0.0025156105556702443],["leaf",42,-0.011568909528840477],["leaf",47,0.0011152836138014894],["leaf",48,0.010324457109152198],["leaf",49,-0.005476815507305143],["leaf",50,0.0062564670212042275],["leaf",51,-0.006291881439236837],["leaf",52,-0.026674653790863402],["leaf",57,0.005935497025229231],["leaf",58,-0.006105674990239401],["leaf",59,0.006717239414758757],["leaf",60,0.022722258898102157],["leaf",61,0.013248286974568356],["leaf",62,0.0036055174360812565]],[["split",0,23.0,26,0],["split",1,7.0,28,1],["split",2,20.0,0,1],["split",3,17.0,0,2],["split",4,24.0,5,2],["split",5,3.0,27,2],["split",6,1.0,3,2],["split",7,21.0,7,3],["split",8,1.0,9,3],["split",9,3.0,8,3],["split",10,20.0,0,3],["split",11,17.0,0,3],["leaf",12,0.014284727405939002],["leaf",13,-0.007245065732532097],["leaf",14,0.019629136204349473],["split",15,20.0,0,4],["split",16,25.0,30,4],["split",17,10.0,14,4],["split",18,3.0,28,4],["leaf",19,0.008649487287528153],["leaf",20,-0.010802915845818457],["split",21,23.0,21,4],["leaf",22,0.022996056429071324],["split",23,2.0,7,4],["split",24,23.0,29,4],["leaf",31,-0.0011898923834137906],["leaf",32,-0.016024878030575816],["leaf",33,0.0005040289344049008],["leaf",34,0.011949011438150742],["leaf",35,-0.011990344480342278],["leaf",36,0.003994228858090812],["leaf",37,0.007400438106566878],["leaf",38,0.022395284787081398],["leaf",43,0.010991766702831843],["leaf",44,-0.0003692107630894377],["leaf",47,0.007873583118811023],["leaf",48,-0.007944255059972635],["leaf",49,-0.018583830157652623],["leaf",50,-0.0384185013846545]],[["split",0,7.0,13,0],["split",1,3.0,28,1],["split",2,25.0,28,1],["split",3,17.0,0,2],["split",4,7.0,4,2],["split",5,15.0,0,2],["split",6,2.0,11,2],["split",7,6.0,9,3],["split",8,23.0,16,3],["leaf",9,0.019735781998375387],["split",10,12.0,16,3],["split",11,22.0,28,3],["split",12,5.0,8,3],["split",13,4.0,12,3],["split",14,9.0,14,3],["split",15,19.0,0,4],["split",16,22.0,3,4],["split",17,9.0,20,4],["split",18,6.0,5,4],["split",21,10.0,20,4],["split",22,22.0,18,4],["split",23,25.0,2,4],["split",24,6.0,28,4],["split",25,7.0,25,4],["split",26,26.0,17,4],["split",27,4.0,5,4],["leaf",28,0.0232912043710451],["leaf",29,-0.0012793206818336293],["leaf",30,0.011140187932571792],["leaf",31,-0.0003272102487153684],["leaf",32,0.0073562199652820424],["leaf",33,-0.017490125745099474],["leaf",34,-0.0035084416874201795],["leaf",35,-0.004638137226634063],["leaf",36,0.01362925152295584],["leaf",37,-0.02456749991422644],["leaf",38,-0.007883754580312569],["leaf",43,0.002429849866438487],["leaf",44,-0.007644688248517013],["leaf",45,0.01600151843519653],["leaf",46,-0.0007224214511285022],["leaf",47,-0.004830767148811263],["leaf",48,0.002916383479841693],["leaf",49,-0.008581271911135576],["leaf",50,0.0052197666170260715],["leaf",51,0.009872466402395958],["leaf",52,0.02424354211790353],["leaf",53,-0.001046469677923713],["leaf",54,0.01048381857916209],["leaf",55,0.015313041695353927],["leaf",56,0.004426622116203918]],[["split",0,7.0,7,0],["split",1,24.0,24,1],["split",2,26.0,20,1],["split",3,2.0,27,2],["split",4,15.0,0,2],["split",5,7.0,28,2],["split",6,10.0,0,2],["split",7,12.0,1,3],["split",8,4.0,26,3],["split",9,10.0,4,3],["leaf",10,-0.011933565169000905],["split",11,3.0,2,3],["split",12,2.0,8,3],["leaf",13,-0.008554249441525873],["split",14,19.0,0,3],["split",15,5.0,28,4],["split",16,10.0,20,4],["split",17,4.0,19,4],["split",18,2.0,30,4],["leaf",19,-0.005705218121474699],["split",20,10.0,12,4],["split",23,11.0,24,4],["split",24,6.0,8,4],["split",25,22.0,6,4],["split",26,0.0,8,4],["split",29,15.0,0,4],["split",30,10.0,9,4],["leaf",31,-0.0022354798940446684],["leaf",32,0.017269315449222243],["leaf",33,-0.00814027915875219],["leaf",34,0.000271194340036202],["leaf",35,-0.011576391122239784],["leaf",36,0.006474447072224619],["leaf",37,-0.024420106693644864],["leaf",38,-0.008492416735953906],["leaf",41,0.014707656857008521],["leaf",42,0.003819802211844567],["leaf",47,-0.012438932685898173],["leaf",48,0.004180277165005382],["leaf",49,0.00736594002602713],["leaf",50,-0.001735908842287486],["leaf",51,0.023034449947427912],["leaf",52,0.008255097086624348],["leaf",53,-0.009990303287949118],["leaf",54,0.001598315985529834],["leaf",59,0.004022804300171119],["leaf",60,0.010622839368737308],["leaf",61,0.008665789390916914],["leaf",62,-0.005218041482565249]],[["split",0,26.0,3,0],["split",1,23.0,9,1],["split",2,1.0,1,1],["split",3,2.0,18,2],["split",4,12.0,19,2],["split",5,26.0,14,2],["split",6,10.0,6,2],["split",7,26.0,0,3],["split",8,24.0,14,3],["split",9,5.0,23,3],["split",10,0.0,18,3],["split",11,25.0,21,3],["split",12,22.0,19,3],["split",13,20.0,0,3],["split",14,20.0,0,3],["leaf",15,-0.01563408915092526],["split",16,25.0,18,4],["leaf",17,-0.009916498641404904],["leaf",18,-0.029400275632378497],["split",19,7.0,22,4],["split",20,24.0,6,4],["leaf",21,-0.005761416147806544],["leaf",22,-0.02352991343465163],["split",23,22.0,16,4],["leaf",24,-0.010720663575879431],["split",25,12.0,8,4],["leaf",26,-0.006052505998345798],["split",27,7.0,5,4],["split",28,13.0,10,4],["split",29,2.0,24,4],["split",30,23.0,25,4],["leaf",33,0.0037301386670733428],["leaf",34,-0.010399030226371973],["leaf",39,-0.010924023079199293],["leaf",40,0.005372460745628505],["leaf",41,-0.0051077049437167335],["leaf",42,0.01637335781951264],["leaf",47,0.015749202304168285],["leaf",48,0.0022649565202584126],["leaf",51,-0.028381390505142132],["leaf",52,-0.010781034171981368],["leaf",55,-0.01281984336317773],["leaf",56,-0.0018160987570001149],["leaf",57,0.028507294477334404],["leaf",58,-0.007386266894367787],["leaf",59,0.004261907128818006],["leaf",60,-0.002865901361744522],["leaf",61,-0.0055705772250753275],["leaf",62,0.023117043587319294]],[["split",0,25.0,12,0],["split",1,10.0,10,1],["split",2,24.0,24,1],["split",3,22.0,29,2],["split",4,6.0,1,2],["split",5,7.0,13,2],["split",6,10.0,14,2],["split",7,2.0,13,3],["leaf",8,-0.020141809170004316],["leaf",9,0.014865427065767899],["split",10,1.0,9,3],["split",11,20.0,0,3],["split",12,8.0,13,3],["split",13,23.0,24,3],["split",14,23.0,25,3],["split",15,23.0,7,4],["split",16,4.0,23,4],["split",21,4.0,5,4],["split",22,17.0,0,4],["split",23,2.0,15,4],["split",24,4.0,13,4],["split",25,21.0,24,4],["split",26,24.0,14,4],["split",27,26.0,18,4],["split",28,4.0,23,4],["split",29,0.0,12,4],["leaf",30,-0.013815851381796657],["leaf",31,0.0035314686341056615],["leaf",32,-0.014219849856313334],["leaf",33,0.003233039453924645],["leaf",34,-0.007550061317909089],["leaf",43,0.007757212179124744],["leaf",44,-0.010059671679222688],["leaf",45,-0.00038969028120593857],["leaf",46,0.00968078722265904],["leaf",47,0.002791578376269202],["leaf",48,-0.007295656680547605],["leaf",49,-0.009486923851803972],["leaf",50,0.014010036024839483],["leaf",51,0.007418895550598216],["leaf",52,0.023235411888971273],["leaf",53,-0.0004172907684238585],["leaf",54,0.005148093518094534],["leaf",55,0.0026601168203543328],["leaf",56,0.012914110661415297],["leaf",57,0.013801856167711747],["leaf",58,0.03236862167834705],["leaf",59,-0.00395483836795877],["leaf",60,0.009240922541697139]],[["split",0,23.0,25,0],["split",1,24.0,1,1],["split",2,20.0,0,1],["split",3,21.0,1,2],["split",4,26.0,6,2],["split",5,17.0,0,2],["split",6,24.0,17,2],["leaf",7,-0.019474526010353526],["split",8,9.0,18,3],["split",9,22.0,5,3],["split",10,17.0,0,3],["split",11,3.0,3,3],["split",12,22.0,22,3],["leaf",13,0.0026167568138576376],["leaf",14,0.025372544393368418],["split",17,2.0,21,4],["split",18,13.0,12,4],["split",19,9.0,11,4],["split",20,4.0,11,4],["split",21,22.0,10,4],["split",22,10.0,26,4],["split",23,1.0,20,4],["split",24,10.0,1,4],["split",25,24.0,17,4],["leaf",26,-0.0026189307764957217],["leaf",35,0.0036429521697574742],["leaf",36,-0.009518063940084037],["leaf",37,0.0003633928165446258],["leaf",38,-0.02050580121157724],["leaf",39,0.0008377492744650925],["leaf",40,-0.014605366306290911],["leaf",41,0.006365849183851857],["leaf",42,-0.002988932114554249],["leaf",43,0.004671736405071182],["leaf",44,-0.0008890362703436902],["leaf",45,0.005514464057523572],["leaf",46,0.01765646313203465],["leaf",47,0.01093870127170092],["leaf",48,-0.012356252544433417],["leaf",49,0.003128299161981395],["leaf",50,-0.010571224812180882],["leaf",51,-0.036992257611774236],["leaf",52,-0.015624057753221676]],[["split",0,13.0,1,0],["split",1,12.0,1,1],["split",2,21.0,0,1],["split",3,10.0,5,2],["split",4,22.0,12,2],["split",5,22.0,23,2],["split",6,4.0,25,2],["leaf",7,-0.014883452165903937],["split",8,24.0,12,3],["split",9,0.0,16,3],["split",10,0.0,17,3],["split",11,25.0,12,3],["leaf",12,0.0006794011108818504],["split",13,9.0,3,3],["split",14,21.0,8,3],["leaf",17,0.01583741326105762],["leaf",18,-0.003913327713981715],["leaf",19,0.010960309972372783],["leaf",20,0.027765257360938357],["leaf",21,0.018714276933758205],["leaf",22,-0.0016250382547796055],["leaf",23,-0.02328438981878003],["leaf",24,-0.007924288177367005],["split",27,2.0,25,4],["split",28,7.0,30,4],["split",29,23.0,26,4],["split",30,20.0,0,4],["leaf",55,-0.0026304201975365558],["leaf",56,-0.012359971269961701],["leaf",57,0.0011922192665209082],["leaf",58,0.010109411024067486],["leaf",59,0.0002839121186135577],["leaf",60,0.016943973149710495],["leaf",61,-0.009360987860914579],["leaf",62,0.00678649500600582]],[["split",0,1.0,13,0],["split",1,4.0,29,1],["split",2,7.0,0,1],["split",3,3.0,2,2],["split",4,3.0,15,2],["leaf",5,-0.016422536551194244],["split",6,25.0,28,2],["split",7,2.0,22,3],["split",8,11.0,29,3],["split",9,26.0,4,3],["leaf",10,-0.025964410517506923],["split",13,20.0,0,3],["split",14,10.0,6,3],["split",15,10.0,21,4],["leaf",16,-0.007596668927974985],["split",17,2.0,25,4],["split",18,22.0,12,4],["leaf",19,-0.01937829987515971],["split",20,3.0,5,4],["split",27,25.0,22,4],["split",28,26.0,14,4],["leaf",29,0.015862516382660594],["split",30,7.0,22,4],["leaf",31,0.00491435489152162],["leaf",32,0.02236813438709615],["leaf",35,-0.0038865633378118167],["leaf",36,0.0014404324452754968],["leaf",37,-0.006486274785149543],["leaf",38,-0.020340114414938566],["leaf",41,0.006172911411402623],["leaf",42,-0.011951082009673184],["leaf",55,0.002494042983074923],["leaf",56,-0.003836800573655227],["leaf",57,0.0025345227389542827],["leaf",58,-0.012199326874253963],["leaf",61,0.000612427931467039],["leaf",62,0.013452887869893966]],[["split",0,2.0,3,0],["split",1,20.0,0,1],["split",2,7.0,27,1],["split",3,23.0,8,2],["split",4,25.0,15,2],["split",5,1.0,30,2],["split",6,22.0,18,2],["split",7,26.0,10,3],["split",8,7.0,29,3],["leaf",9,-0.018124828852081616],["leaf",10,-0.0015315088530972417],["split",11,21.0,29,3],["split",12,3.0,6,3],["split",13,20.0,0,3],["split",14,21.0,21,3],["leaf",15,0.02089906990502362],["split",16,0.0,14,4],["split",17,9.0,10,4],["leaf",18,0.01886594395044477],["split",23,21.0,20,4],["split",24,23.0,15,4],["leaf",25,-0.0033840467977919856],["leaf",26,-0.021822961682264747],["split",27,23.0,18,4],["leaf",28,0.02214036906470495],["split",29,11.0,14,4],["leaf",30,0.009978293491443909],["leaf",33,0.0061524384144508205],["leaf",34,0.015145067233587415],["leaf",35,-0.0027974904050681298],["leaf",36,0.006397987848674291],["leaf",47,-0.000679716476303903],["leaf",48,0.0036805704603376744],["leaf",49,0.0013884476365364732],["leaf",50,-0.011847103130677416],["leaf",55,0.012152619443211882],["leaf",56,0.00124671244302894],["leaf",59,0.002984744473748228],["leaf",60,-0.010118707398665496]],[["split",0,12.0,0,0],["split",1,13.0,0,1],["split",2,2.0,30,1],["leaf",3,0.0018817197052859178],["split",4,26.0,13,2],["split",5,23.0,30,2],["split",6,1.0,5,2],["leaf",9,-0.02335099655414661],["leaf",10,-0.008800757748769575],["split",11,5.0,28,3],["split",12,6.0,14,3],["split",13,23.0,27,3],["leaf",14,-0.00015176093927241866],["split",23,8.0,1,4],["split",24,25.0,9,4],["split",25,21.0,17,4],["leaf",26,-0.026346350767334836],["leaf",27,-0.02772309970428046],["leaf",28,-0.004039678444421598],["leaf",47,-0.009961758939584028],["leaf",48,0.0013914416033226993],["leaf",49,0.01591992679996994],["leaf",50,0.004091618356218073],["leaf",51,0.0081843109863907],["leaf",52,-0.01488888886108027]],[["split",0,7.0,29,0],["split",1,22.0,28,1],["split",2,11.0,27,1],["split",3,10.0,26,2],["split",4,23.0,3,2],["split",5,21.0,14,2],["leaf",6,-0.0046227804207151415],["split",7,3.0,29,3],["split",8,23.0,18,3],["leaf",9,0.007766841474383678],["split",10,8.0,2,3],["split",11,22.0,12,3],["split",12,25.0,9,3],["split",15,7.0,24,4],["split",16,23.0,14,4],["split",17,1.0,28,4],["split",18,4.0,9,4],["leaf",21,0.004146886739008403],["split",22,15.0,0,4],["leaf",23,0.009712930420939733],["leaf",24,-0.002657441208322831],["leaf",25,0.0261805995320373],["split",26,2.0,8,4],["leaf",31,-0.0006323530351671621],["leaf",32,0.0048272238110627815],["leaf",33,0.0016368001581285106],["leaf",34,0.015440974476709739],["leaf",35,0.00037328136073608336],["leaf",36,-0.014661050117975194],["leaf",37,0.0005610333481096261],["leaf",38,-0.013846144386548357],["leaf",45,-0.00970956442977589],["leaf",46,0.0027570644438554767],["leaf",53,0.02044857584960803],["leaf",54,0.00431642613168539]],[["split",0,26.0,7,0],["split",1,2.0,27,1],["split",2,24.0,29,1],["split",3,6.0,11,2],["split",4,9.0,16,2],["split",5,2.0,14,2],["split",6,9.0,24,2],["split",7,5.0,22,3],["split",8,10.0,18,3],["leaf",9,-0.022349630200338956],["leaf",10,-0.0005180119719831094],["split",11,20.0,0,3],["split",12,13.0,16,3],["split",13,5.0,12,3],["leaf",14,-0.006782784177355915],["split",15,25.0,6,4],["split",16,21.0,28,4],["split",17,11.0,5,4],["split",18,20.0,0,4],["split",23,19.0,0,4],["split",24,25.0,25,4],["split",25,17.0,0,4],["split",26,3.0,7,4],["leaf",27,-0.0011754288944358683],["split",28,25.0,19,4],["leaf",31,-0.015240068613748051],["leaf",32,-0.0012025624050308375],["leaf",33,0.011005432872063303],["leaf",34,-0.0022824228430309308],["leaf",35,-0.02007647112148731],["leaf",36,-0.006560747349448539],["leaf",37,-0.002558864344689757],["leaf",38,0.011378172797100228],["leaf",47,0.005367526737990037],["leaf",48,-0.000333082348367665],["leaf",49,-0.009850357314290991],["leaf",50,0.008051776625760964],["leaf",51,0.002348084533609592],["leaf",52,-0.007472643325819529],["leaf",53,-0.016662600626306638],["leaf",54,-0.0032498223080048043],["leaf",57,0.013979788195277643],["leaf",58,0.027808191035493948]],[["split",0,7.0,6,0],["split",1,20.0,0,1],["split",2,22.0,30,1],["split",3,10.0,12,2],["split",4,10.0,16,2],["split",5,1.0,0,2],["split",6,25.0,21,2],["split",7,3.0,25,3],["split",8,19.0,0,3],["split",9,25.0,18,3],["leaf",10,-0.007330942008277446],["leaf",11,-0.011276701984845488],["split",12,7.0,29,3],["split",13,10.0,17,3],["leaf",14,-0.020208338708064495],["split",15,10.0,2,4],["leaf",16,0.008620949587591964],["split",17,25.0,20,4],["split",18,13.0,4,4],["leaf",19,0.004757629624834715],["leaf",20,0.02491200194658383],["split",25,13.0,10,4],["split",26,2.0,12,4],["leaf",27,-0.013260372653458993],["leaf",28,0.0012480941678647956],["leaf",31,-0.021085861245782784],["leaf",32,-0.006907253042630129],["leaf",35,0.001608188329546393],["leaf",36,-0.00923567479275175],["leaf",37,0.02059306727141695],["leaf",38,-0.0021520281568013517],["leaf",51,0.0056785610483281205],["leaf",52,0.00040324248650678885],["leaf",53,0.013432751798166206],["leaf",54,0.001657464011555405]],[["split",0,3.0,17,0],["split",1,22.0,30,1],["split",2,21.0,3,1],["split",3,7.0,1,2],["split",4,12.0,11,2],["split",5,3.0,21,2],["split",6,19.0,0,2],["split",7,5.0,25,3],["split",8,24.0,3,3],["leaf",9,-0.006344392747678381],["leaf",10,-0.01889552708292788],["leaf",11,-0.01064164692585686],["split",12,6.0,21,3],["split",13,10.0,13,3],["split",14,4.0,8,3],["leaf",15,-0.015124149481869603],["split",16,3.0,8,4],["split",17,23.0,18,4],["split",18,7.0,30,4],["leaf",25,0.0038350183246903652],["leaf",26,-0.009938861342424539],["split",27,7.0,25,4],["split",28,1.0,2,4],["split",29,10.0,12,4],["split",30,13.0,19,4],["leaf",33,-5.937931861724137e-05],["leaf",34,-0.015282149771925348],["leaf",35,-0.0014230299845168414],["leaf",36,-0.01074719316523482],["leaf",37,-0.0005920660902952757],["leaf",38,0.009310461931561383],["leaf",55,-0.0009503210860816214],["leaf",56,0.008898669485446186],["leaf",57,-0.012438065720714611],["leaf",58,0.005689881626796686],["leaf",59,-0.0011949533216832787],["leaf",60,-0.011691048000218885],["leaf",61,-0.004890243121080337],["leaf",62,0.008755834651381862]],[["split",0,3.0,29,0],["split",1,25.0,13,1],["split",2,20.0,0,1],["split",3,22.0,1,2],["split",4,10.0,29,2],["split",5,17.0,0,2],["leaf",6,-0.007940345797255067],["split",7,23.0,4,3],["split",8,3.0,0,3],["split",9,20.0,0,3],["split",10,23.0,21,3],["split",11,2.0,4,3],["leaf",12,0.02013472851991085],["leaf",15,-0.016354380253092514],["split",16,13.0,11,4],["leaf",17,0.014697250490467747],["split",18,0.0,18,4],["split",19,6.0,2,4],["split",20,2.0,21,4],["split",21,4.0,12,4],["leaf",22,-0.021244849807561878],["leaf",23,0.0149481883193208],["split",24,22.0,17,4],["leaf",33,0.0010168408279347173],["leaf",34,-0.011272399015226475],["leaf",37,-0.0029312435426026465],["leaf",38,0.0017049606977735064],["leaf",39,0.00879188199279572],["leaf",40,0.0007872540666179045],["leaf",41,-0.0004050220856902415],["leaf",42,0.024504563519653008],["leaf",43,0.009016345086207632],["leaf",44,-0.006130013440370229],["leaf",49,-0.00017845000714916746],["leaf",50,0.00889921726824892]],[["split",0,7.0,13,0],["split",1,26.0,0,1],["split",2,24.0,8,1],["leaf",3,-0.021396165788293033],["split",4,1.0,30,2],["split",5,0.0,3,2],["split",6,12.0,22,2],["split",9,23.0,6,3],["leaf",10,-0.017144658691700017],["split",11,25.0,14,3],["split",12,18.0,0,3],["split",13,0.0,8,3],["split",14,1.0,27,3],["split",19,1.0,26,4],["split",20,12.0,8,4],["split",23,26.0,6,4],["leaf",24,0.016884832348371823],["split",25,22.0,25,4],["split",26,21.0,17,4],["split",27,4.0,22,4],["split",28,25.0,4,4],["split",29,0.0,3,4],["split",30,3.0,17,4],["leaf",39,0.0030437688466630943],["leaf",40,-0.011095506752081628],["leaf",41,-0.0011398290754743759],["leaf",42,-0.005928357992924684],["leaf",47,-0.007813710041524253],["leaf",48,0.00463383962039823],["leaf",51,6.549143002248469e-05],["leaf",52,-0.009400600812663976],["leaf",53,-0.006571848057510281],["leaf",54,-0.018463640221521575],["leaf",55,-0.006118612167553581],["leaf",56,0.0039251484537367156],["leaf",57,-0.006831457974570162],["leaf",58,0.003910567114955125],["leaf",59,0.016797117250121594],["leaf",60,0.005674156049663982],["leaf",61,-0.006462569004583459],["leaf",62,0.008265682126165668]],[["split",0,6.0,0,0],["split",1,22.0,16,1],["split",2,4.0,28,1],["split",3,2.0,20,2],["leaf",4,0.019867503701877103],["split",5,26.0,0,2],["split",6,22.0,28,2],["leaf",7,-0.008998612671885804],["leaf",8,0.007050614339615037],["split",11,3.0,3,3],["split",12,7.0,23,3],["split",13,25.0,10,3],["leaf",14,0.011767256282375337],["leaf",23,-0.022705795552307007],["split",24,4.0,16,4],["split",25,5.0,25,4],["split",26,13.0,13,4],["split",27,4.0,29,4],["split",28,10.0,19,4],["leaf",49,0.0021901748287272873],["leaf",50,-0.014180081117118095],["leaf",51,-0.0012600583471394796],["leaf",52,0.004733504936075277],["leaf",53,0.02641878808997505],["leaf",54,0.0022851509988654354],["leaf",55,0.005394181035910129],["leaf",56,-0.007662964424433568],["leaf",57,-0.009095835024707252],["leaf",58,-0.022097646567568494]],[["split",0,3.0,27,0],["split",1,17.0,0,1],["split",2,20.0,0,1],["split",3,1.0,25,2],["split",4,23.0,25,2],["split",5,9.0,8,2],["split",6,24.0,15,2],["split",7,23.0,27,3],["split",8,24.0,27,3],["split",9,22.0,2,3],["split",10,9.0,18,3],["split",11,5.0,4,3],["split",12,12.0,29,3],["leaf",13,-0.01945782535911155],["leaf",14,-0.004115282195065776],["split",15,0.0,27,4],["split",16,20.0,0,4],["split",17,3.0,11,4],["split",18,25.0,14,4],["leaf",19,-0.02106186033914986],["split",20,12.0,22,4],["split",21,0.0,15,4],["leaf",22,-0.008798502052522788],["leaf",23,0.0077616619249028995],["leaf",24,0.023531471751791502],["split",25,17.0,0,4],["leaf",26,0.01694056250171149],["leaf",31,-0.000889730741889153],["leaf",32,0.005049634746827791],["leaf",33,0.004974787962773685],["leaf",34,0.02255188112196948],["leaf",35,-0.00733833435609862],["leaf",36,-0.0002747182684153706],["leaf",37,-0.01962662027145073],["leaf",38,-0.006451023001085711],["leaf",41,0.0038666132601980095],["leaf",42,-0.0069318119087598135],["leaf",43,-0.015680081919780972],["leaf",44,-0.040014757527714016],["leaf",51,-0.0001414419151635796],["leaf",52,0.011156485574397357]],[["split",0,24.0,15,0],["split",1,20.0,0,1],["split",2,25.0,27,1],["split",3,2.0,28,2],["split",4,2.0,16,2],["split",5,21.0,30,2],["split",6,24.0,25,2],["split",7,23.0,25,3],["split",8,0.0,16,3],["split",9,25.0,20,3],["split",10,21.0,15,3],["split",11,6.0,4,3],["leaf",12,-0.009844047384109126],["split",13,6.0,7,3],["split",14,7.0,9,3],["split",15,7.0,8,4],["split",16,24.0,4,4],["split",17,4.0,21,4],["split",18,7.0,6,4],["split",19,8.0,24,4],["leaf",20,-0.0015543969322369938],["split",21,12.0,11,4],["leaf",22,0.014796143490899277],["split",23,7.0,6,4],["split",24,5.0,1,4],["leaf",27,0.012132761870310373],["split",28,1.0,13,4],["leaf",29,0.0016922274107637667],["split",30,3.0,15,4],["leaf",31,-0.003813475999007033],["leaf",32,0.0027444440224553097],["leaf",33,0.0033961355157284623],["leaf",34,-0.009555590602785437],["leaf",35,-0.007116768294389819],["leaf",36,-0.024879158833823824],["leaf",37,-0.008267134199420267],["leaf",38,0.00329947480263678],["leaf",39,-0.024968284093419026],["leaf",40,-0.010081955868342828],["leaf",43,-0.01295524804706472],["leaf",44,0.0019948359878377594],["leaf",47,0.00347218132805883],["leaf",48,0.018796515560669247],["leaf",49,-0.00547624463072413],["leaf",50,0.0008916946892652156],["leaf",57,-0.012118551350684514],["leaf",58,0.0025676278364301416],["leaf",61,0.02657928679846856],["leaf",62,0.007383735986306585]],[["split",0,7.0,2,0],["split",1,3.0,23,1],["split",2,23.0,25,1],["split",3,10.0,2,2],["leaf",4,0.004982545580424007],["split",5,17.0,0,2],["split",6,20.0,0,2],["leaf",7,-0.017420238905609942],["split",8,1.0,7,3],["split",11,15.0,0,3],["split",12,26.0,7,3],["split",13,17.0,0,3],["split",14,2.0,17,3],["split",17,2.0,19,4],["split",18,3.0,8,4],["split",23,5.0,14,4],["split",24,25.0,18,4],["split",25,11.0,10,4],["split",26,1.0,15,4],["split",27,6.0,3,4],["split",28,23.0,28,4],["leaf",29,-0.006443695718634123],["leaf",30,0.026485056863624186],["leaf",35,0.013879012298905981],["leaf",36,-0.0034791799150223797],["leaf",37,-0.004106831066328272],["leaf",38,-0.015476334297639735],["leaf",47,-0.0019930453260819606],["leaf",48,0.0019195321916980433],["leaf",49,0.008485905429958352],["leaf",50,-0.002285817038141961],["leaf",51,-0.013516528867846348],["leaf",52,0.00027216373951315304],["leaf",53,0.0027517099818101395],["leaf",54,0.015378562779077037],["leaf",55,0.008118703234176114],["leaf",56,-0.005128942207912257],["leaf",57,-0.008304724110076057],["leaf",58,-0.030397734887119485]],[["split",0,1.0,2,0],["split",1,11.0,25,1],["split",2,25.0,30,1],["split",3,9.0,10,2],["split",4,25.0,14,2],["split",5,21.0,30,2],["split",6,21.0,25,2],["split",7,5.0,21,3],["split",8,22.0,14,3],["leaf",9,-0.0023681776057100385],["leaf",10,-0.022257835913977626],["split",11,9.0,22,3],["split",12,3.0,11,3],["split",13,1.0,17,3],["leaf",14,0.026471059300914562],["split",15,2.0,19,4],["split",16,26.0,9,4],["split",17,4.0,25,4],["split",18,1.0,1,4],["split",23,21.0,15,4],["split",24,1.0,17,4],["split",25,1.0,19,4],["leaf",26,-0.019861193282161512],["leaf",27,-0.00615971920440599],["leaf",28,0.007459845655253764],["leaf",31,-0.020651399034082484],["leaf",32,0.001808276485346386],["leaf",33,0.024149249192374638],["leaf",34,0.0022194306584168814],["leaf",35,-0.007351711830119839],["leaf",36,-0.02207788895876875],["leaf",37,0.004651239276266922],["leaf",38,-0.014224993767567811],["leaf",47,-0.0021676750034431546],["leaf",48,0.0016018581370832034],["leaf",49,-0.0002381256899060807],["leaf",50,0.005225833847816677],["leaf",51,-7.092181496590018e-05],["leaf",52,-0.00815380247314975]],[["split",0,10.0,30,0],["split",1,2.0,1,1],["split",2,24.0,20,1],["split",3,9.0,10,2],["split",4,13.0,4,2],["split",5,25.0,19,2],["leaf",6,0.002715518695590335],["split",7,8.0,24,3],["split",8,7.0,22,3],["split",9,23.0,18,3],["split",10,10.0,11,3],["leaf",11,-0.019991110703813625],["leaf",12,-0.005928061430733437],["leaf",15,0.00299196651472001],["leaf",16,-0.010989979218234955],["split",17,13.0,17,4],["split",18,25.0,16,4],["split",19,25.0,20,4],["split",20,17.0,0,4],["split",21,26.0,2,4],["split",22,23.0,23,4],["leaf",35,0.012147269229167763],["leaf",36,-0.003045096304527345],["leaf",37,0.024787028812698376],["leaf",38,0.009498111168433687],["leaf",39,-0.009161416911765741],["leaf",40,0.007243801384808278],["leaf",41,0.007827450158361523],["leaf",42,-0.00988712453331399],["leaf",43,-0.014681818789977583],["leaf",44,-0.002874252930174395],["leaf",45,0.0010568804306866223],["leaf",46,-0.005875516088058672]],[["split",0,7.0,0,0],["split",1,23.0,27,1],["split",2,23.0,24,1],["split",3,3.0,8,2],["leaf",4,0.0005285845105443868],["split",5,1.0,30,2],["split",6,17.0,0,2],["leaf",7,-0.007649375720832445],["leaf",8,-0.02545635345148931],["split",11,7.0,27,3],["split",12,11.0,17,3],["split",13,20.0,0,3],["split",14,21.0,22,3],["split",23,20.0,0,4],["split",24,25.0,26,4],["leaf",25,-0.002862225941845868],["leaf",26,-0.01805559476458577],["split",27,7.0,30,4],["split",28,2.0,16,4],["split",29,9.0,11,4],["leaf",30,-0.02756560836607321],["leaf",47,0.0011441960678649288],["leaf",48,-0.003777112263154728],["leaf",49,0.004262393370418577],["leaf",50,0.015985380214280275],["leaf",55,-0.0038812952387252122],["leaf",56,0.01296913042460222],["leaf",57,-0.008922996540383954],["leaf",58,0.026614026491805422],["leaf",59,-0.018688432956893382],["leaf",60,-0.0032869775991567866]],[["split",0,7.0,18,0],["split",1,6.0,26,1],["split",2,6.0,20,1],["split",3,21.0,2,2],["leaf",4,-0.014396448733448322],["split",5,17.0,0,2],["split",6,3.0,11,2],["split",7,0.0,1,3],["split",8,10.0,2,3],["split",11,25.0,16,3],["split",12,12.0,16,3],["split",13,2.0,17,3],["split",14,19.0,0,3],["leaf",15,0.009741281096472057],["split",16,4.0,24,4],["split",17,12.0,18,4],["split",18,2.0,15,4],["split",23,0.0,12,4],["split",24,5.0,22,4],["leaf",25,0.009228628586397794],["leaf",26,-0.015890458771055143],["split",27,23.0,5,4],["split",28,5.0,19,4],["split",29,15.0,0,4],["split",30,26.0,14,4],["leaf",33,-0.004735631175083088],["leaf",34,-0.01786652684630572],["leaf",35,-0.0022406058366985057],["leaf",36,-0.021161913677290735],["leaf",37,0.0024302677160705113],["leaf",38,-0.001474914128056157],["leaf",47,0.008667559339774106],["leaf",48,3.4287496060554645e-05],["leaf",49,0.008450961637678445],["leaf",50,0.02122117495021233],["leaf",55,-0.013933520471601584],["leaf",56,0.0010765827135002836],["leaf",57,0.0004405469741415303],["leaf",58,-0.026192711116726834],["leaf",59,0.002158652166877353],["leaf",60,0.01063788421356924],["leaf",61,0.002740974420843797],["leaf",62,-0.01113300687079852]],[["split",0,22.0,12,0],["split",1,20.0,0,1],["split",2,17.0,0,1],["split",3,3.0,24,2],["split",4,2.0,16,2],["split",5,3.0,1,2],["split",6,10.0,13,2],["split",7,2.0,15,3],["split",8,1.0,6,3],["split",9,25.0,13,3],["split",10,21.0,14,3],["split",11,4.0,28,3],["split",12,11.0,7,3],["split",13,3.0,20,3],["split",14,0.0,5,3],["split",15,12.0,4,4],["split",16,17.0,0,4],["leaf",17,0.016742913874061013],["split",18,24.0,12,4],["leaf",19,-0.009831979923986463],["split",20,2.0,8,4],["leaf",21,0.0017100257129185085],["leaf",22,0.03016146823991],["split",23,2.0,12,4],["leaf",24,0.0034800556614215054],["split",25,20.0,0,4],["split",26,25.0,28,4],["split",27,11.0,20,4],["leaf",28,0.00891801565195051],["leaf",29,-0.005635252294050786],["split",30,10.0,26,4],["leaf",31,0.0169861548590052],["leaf",32,0.0009880056596280142],["leaf",33,-0.002294980132257126],["leaf",34,-0.013930428295015982],["leaf",37,-0.0012803309995596418],["leaf",38,0.006611268955996911],["leaf",41,-0.009082547235456756],["leaf",42,0.012518081138381704],["leaf",47,-0.0008123399998303987],["leaf",48,-0.02061052644721948],["leaf",51,-0.0026448350601101338],["leaf",52,-0.015319386491258682],["leaf",53,-0.0019747954288839003],["leaf",54,0.007740888992948996],["leaf",55,-0.01072423759955013],["leaf",56,0.003274890893379581],["leaf",61,0.015305219031905102],["leaf",62,-0.0008768482014643006]],[["split",0,1.0,3,0],["split",1,8.0,13,1],["split",2,23.0,29,1],["split",3,11.0,17,2],["split",4,10.0,21,2],["split",5,21.0,29,2],["split",6,1.0,10,2],["split",7,4.0,19,3],["split",8,7.0,9,3],["split",9,11.0,12,3],["leaf",10,-0.0227070874172086],["split",11,11.0,26,3],["split",12,11.0,6,3],["leaf",13,-0.017610968598969406],["split",14,0.0,21,3],["leaf",15,0.006650394902307626],["split",16,5.0,24,4],["split",17,21.0,7,4],["leaf",18,0.01324796854689017],["leaf",19,0.004113381853947254],["leaf",20,-0.014228095972291223],["split",23,11.0,11,4],["split",24,0.0,2,4],["leaf",25,-0.004592150174223956],["split",26,24.0,8,4],["split",29,4.0,20,4],["leaf",30,-0.011060901875284922],["leaf",33,-0.02250285950422295],["leaf",34,-0.0067428954262772345],["leaf",35,0.011286270380604639],["leaf",36,-0.004628549665898528],["leaf",47,-0.0006848819109079564],["leaf",48,0.002621690184999769],["leaf",49,0.00862517949403656],["leaf",50,-0.003919206625235713],["leaf",53,0.02407836370165634],["leaf",54,0.007597882525949962],["leaf",59,0.01048248452562623],["leaf",60,-0.0020512819416524707]],[["split",0,3.0,25,0],["split",1,11.0,23,1],["split",2,17.0,0,1],["split",3,3.0,3,2],["split",4,3.0,4,2],["split",5,20.0,0,2],["split",6,3.0,29,2],["split",7,11.0,4,3],["split",8,13.0,0,3],["split",9,10.0,18,3],["split",10,20.0,0,3],["split",11,24.0,3,3],["split",12,21.0,21,3],["split",13,1.0,14,3],["leaf",14,0.025870217983455106],["split",15,6.0,8,4],["split",16,24.0,15,4],["leaf",17,-0.01718773019278838],["split",18,7.0,22,4],["split",19,5.0,21,4],["split",20,1.0,19,4],["split",21,1.0,3,4],["split",22,21.0,22,4],["leaf",23,0.014306979892655756],["split",24,1.0,26,4],["split",25,2.0,11,4],["leaf",26,0.013259599955085983],["leaf",27,-0.0007000082437065864],["leaf",28,0.01562998928641478],["leaf",31,0.01494718659509212],["leaf",32,-0.001656673414893971],["leaf",33,-0.002617385662123712],["leaf",34,-0.01685472647638705],["leaf",37,-0.0023311181195269943],["leaf",38,0.0009883954676771784],["leaf",39,-0.007241115598643823],["leaf",40,0.009095394213262333],["leaf",41,0.027037858040159228],["leaf",42,0.009381857500511664],["leaf",43,-0.013091261275000874],["leaf",44,9.526269098519075e-05],["leaf",45,-0.0006099719256952388],["leaf",46,0.023697663271715828],["leaf",49,0.00036891590626401255],["leaf",50,0.01202822873259897],["leaf",51,-0.02544384584038568],["leaf",52,-0.01010995185762117]],[["split",0,13.0,13,0],["split",1,7.0,21,1],["split",2,22.0,28,1],["split",3,14.0,0,2],["split",4,11.0,15,2],["split",5,23.0,29,2],["split",6,6.0,28,2],["split",7,3.0,27,3],["split",8,4.0,28,3],["leaf",9,0.02138382679684777],["leaf",10,0.008895806927555833],["split",11,0.0,30,3],["split",12,1.0,11,3],["split",13,4.0,11,3],["leaf",14,0.003091026962213284],["split",15,22.0,16,4],["split",16,23.0,24,4],["split",17,3.0,18,4],["leaf",18,-0.01692106570372891],["split",23,21.0,0,4],["leaf",24,-0.011473620891763769],["leaf",25,-0.024355529338399455],["leaf",26,-0.0028868153386574307],["split",27,1.0,14,4],["split",28,10.0,13,4],["leaf",31,-0.0009227785184777172],["leaf",32,0.005023517805605953],["leaf",33,0.004194325883131873],["leaf",34,0.021171061203908798],["leaf",35,-0.004580368304275252],["leaf",36,0.005846782252743166],["leaf",47,-0.012481418600757773],["leaf",48,0.0007515204367465468],["leaf",55,-0.015619703423871239],["leaf",56,-0.00014803411078182855],["leaf",57,-0.02862263608727377],["leaf",58,-0.014790921929559976]],[["split",0,3.0,2,0],["split",1,18.0,0,1],["split",2,0.0,2,1],["split",3,24.0,22,2],["leaf",4,0.016217260086556246],["split",5,23.0,23,2],["split",6,21.0,3,2],["split",7,19.0,0,3],["split",8,6.0,10,3],["split",11,9.0,24,3],["split",12,11.0,15,3],["split",13,0.0,11,3],["split",14,21.0,30,3],["split",15,7.0,26,4],["leaf",16,0.02078366068792112],["split",17,7.0,4,4],["leaf",18,-0.014698647072853703],["split",23,7.0,22,4],["leaf",24,-0.012555750942668966],["leaf",25,-0.0034005062188586176],["leaf",26,-0.023854539914899015],["split",27,22.0,26,4],["split",28,1.0,11,4],["split",29,13.0,12,4],["split",30,4.0,21,4],["leaf",31,4.397358985090846e-05],["leaf",32,0.016077382349298435],["leaf",35,-0.006314638890601174],["leaf",36,0.01066792295377431],["leaf",47,-0.003428437552665864],["leaf",48,0.006841012275259673],["leaf",55,-0.0030350018737005506],["leaf",56,0.012523037109538044],["leaf",57,-0.01753817543561083],["leaf",58,-0.006587688435883303],["leaf",59,0.003022428161782264],["leaf",60,-0.00044004100861882946],["leaf",61,-0.0013165389769825928],["leaf",62,-0.02085349735270214]],[["split",0,7.0,28,0],["split",1,8.0,2,1],["split",2,19.0,0,1],["split",3,23.0,17,2],["split",4,21.0,9,2],["split",5,2.0,1,2],["leaf",6,-0.00916010307110108],["split",7,6.0,1,3],["split",8,26.0,6,3],["split",9,20.0,0,3],["split",10,23.0,29,3],["leaf",11,0.024728697220817643],["split",12,1.0,26,3],["leaf",15,-0.008728312372788882],["leaf",16,0.008383621311219246],["split",17,10.0,18,4],["split",18,9.0,9,4],["split",19,11.0,5,4],["split",20,2.0,14,4],["split",21,13.0,21,4],["split",22,10.0,8,4],["split",25,1.0,11,4],["split",26,10.0,20,4],["leaf",35,0.019481776439693933],["leaf",36,0.0018490974697351228],["leaf",37,0.012551587697920865],["leaf",38,-0.0023630353310642237],["leaf",39,0.003495450870893593],["leaf",40,-0.002933038794570498],["leaf",41,-0.021039763448465747],["leaf",42,0.0014053044919390145],["leaf",43,0.0020219094662507213],["leaf",44,-0.0018075753706432126],["leaf",45,0.0007398208170443933],["leaf",46,-0.0193485648494505],["leaf",51,0.000467402417902837],["leaf",52,0.015476374343669832],["leaf",53,0.006670891444691582],["leaf",54,-0.01087708655970481]],[["split",0,7.0,27,0],["split",1,13.0,2,1],["split",2,19.0,0,1],["split",3,26.0,22,2],["split",4,3.0,15,2],["split",5,18.0,0,2],["split",6,1.0,20,2],["split",7,10.0,25,3],["leaf",8,0.019436908874140044],["split",9,16.0,0,3],["split",10,7.0,3,3],["split",11,10.0,3,3],["split",12,10.0,13,3],["leaf",13,-0.0027377390227759126],["leaf",14,-0.017592657783835855],["split",15,7.0,6,4],["leaf",16,-0.008572862489958574],["split",19,5.0,21,4],["split",20,22.0,24,4],["split",21,21.0,19,4],["split",22,19.0,0,4],["leaf",23,-0.0057940997506070995],["split",24,23.0,28,4],["leaf",25,-0.009577151451093183],["leaf",26,0.004946764279639916],["leaf",31,0.007373572214057704],["leaf",32,-0.009291260669436268],["leaf",39,-0.0042078983287720025],["leaf",40,0.002398458800795637],["leaf",41,-0.015350875289654297],["leaf",42,-0.0008503592597974433],["leaf",43,-0.013194709882254735],["leaf",44,0.0018815289076499963],["leaf",45,0.002116843211885293],["leaf",46,-0.0038592705281150766],["leaf",49,0.009316019027397414],["leaf",50,0.022020159363433916]],[["split",0,4.0,29,0],["split",1,13.0,1,1],["split",2,24.0,3,1],["split",3,5.0,26,2],["split",4,22.0,0,2],["leaf",5,0.008497820592742051],["split",6,7.0,1,2],["split",7,24.0,9,3],["split",8,2.0,21,3],["split",9,7.0,19,3],["split",10,25.0,0,3],["split",13,21.0,15,3],["split",14,1.0,3,3],["leaf",15,-0.013824589346640171],["leaf",16,0.008690322186197223],["leaf",17,0.026362140204072237],["leaf",18,0.006841394211559555],["leaf",19,-0.018850144578628436],["leaf",20,-0.0007193956653038779],["split",21,24.0,14,4],["split",22,1.0,27,4],["leaf",27,-0.02793717428378975],["leaf",28,-0.010152280714685463],["leaf",29,-0.016980934520436707],["split",30,23.0,15,4],["leaf",43,-0.01760408220892832],["leaf",44,0.0020402737881371207],["leaf",45,0.0009654707671476883],["leaf",46,-0.0037634750680096804],["leaf",61,-0.008221787026878997],["leaf",62,0.007631603520632661]],[["split",0,24.0,13,0],["split",1,22.0,29,1],["split",2,9.0,2,1],["split",3,17.0,0,2],["split",4,3.0,8,2],["split",5,25.0,4,2],["split",6,5.0,30,2],["split",7,23.0,22,3],["split",8,23.0,23,3],["leaf",9,-0.0013946408293494968],["split",10,2.0,15,3],["leaf",11,0.012737824086751136],["split",12,0.0,26,3],["split",13,20.0,0,3],["split",14,3.0,5,3],["split",15,7.0,27,4],["split",16,9.0,11,4],["split",17,2.0,6,4],["leaf",18,-0.022923536791924232],["leaf",21,-0.008015623449282375],["leaf",22,-0.025536834185673277],["split",25,23.0,10,4],["leaf",26,-0.019493497042972544],["split",27,7.0,20,4],["split",28,23.0,22,4],["leaf",29,0.02092431826385639],["leaf",30,0.002148620723370825],["leaf",31,-0.0025858063495473615],["leaf",32,0.0047230178150547775],["leaf",33,0.010953241203173014],["leaf",34,0.0011300159404983683],["leaf",35,0.010557878714879954],["leaf",36,-0.007418873939951899],["leaf",51,0.006514569308810582],["leaf",52,-0.010046157076300044],["leaf",55,-0.0008135303816003579],["leaf",56,0.0037582836980173734],["leaf",57,0.00040995152497056125],["leaf",58,0.02347626377416187]],[["split",0,10.0,6,0],["split",1,21.0,25,1],["split",2,13.0,6,1],["split",3,26.0,9,2],["split",4,2.0,17,2],["split",5,25.0,22,2],["split",6,23.0,18,2],["split",7,1.0,10,3],["split",8,26.0,27,3],["split",9,2.0,9,3],["split",10,25.0,14,3],["split",11,22.0,20,3],["split",12,9.0,24,3],["split",13,11.0,2,3],["split",14,26.0,22,3],["split",15,0.0,11,4],["split",16,26.0,2,4],["split",17,8.0,21,4],["split",18,0.0,16,4],["leaf",19,-0.005906323394541739],["leaf",20,-0.027101006653191387],["leaf",21,0.0024319872885020474],["leaf",22,-0.011340714564071095],["split",23,4.0,16,4],["split",24,3.0,12,4],["split",25,10.0,20,4],["leaf",26,-0.014858967543841493],["split",27,12.0,12,4],["split",28,20.0,0,4],["split",29,24.0,22,4],["split",30,10.0,28,4],["leaf",31,0.007571196972160294],["leaf",32,-0.008286889454325246],["leaf",33,-0.022238935516573775],["leaf",34,-0.00632345653113464],["leaf",35,0.004454263731199764],["leaf",36,-0.003490374707194781],["leaf",37,-0.011842593937064638],["leaf",38,0.000696174725843977],["leaf",47,0.013093695403772776],["leaf",48,0.0007004428589418],["leaf",49,0.018690174413578422],["leaf",50,0.0009940416544610938],["leaf",51,-0.003012039567021929],["leaf",52,0.009436910214436081],["leaf",55,-0.013616544192758404],["leaf",56,-0.0034017143954936887],["leaf",57,0.0027161204126910843],["leaf",58,-0.0041481982782225375],["leaf",59,-0.008896305849254906],["leaf",60,0.0006184550788081697],["leaf",61,0.003201952389894204],["leaf",62,-0.013769412364179225]],[["split",0,21.0,1,0],["split",1,9.0,21,1],["split",2,22.0,23,1],["split",3,7.0,27,2],["split",4,6.0,16,2],["split",5,1.0,27,2],["split",6,25.0,1,2],["split",7,24.0,23,3],["leaf",8,0.023933065550210016],["leaf",9,-0.008384820822987669],["leaf",10,0.007560540151033768],["split",11,11.0,24,3],["split",12,7.0,18,3],["leaf",13,-0.01607627752378718],["split",14,4.0,28,3],["split",15,0.0,4,4],["leaf",16,0.0189686330752063],["split",23,7.0,3,4],["split",24,3.0,3,4],["split",25,9.0,21,4],["split",26,2.0,16,4],["split",29,10.0,20,4],["split",30,11.0,14,4],["leaf",31,0.010056891481202581],["leaf",32,-0.004488131061482024],["leaf",47,-0.004867243733022292],["leaf",48,0.0014118580065180693],["leaf",49,0.020364822417950106],["leaf",50,0.0036574668141551277],["leaf",51,-0.012983623906449509],["leaf",52,0.0015049273976585902],["leaf",53,0.00501208398847498],["leaf",54,-0.011629961924672475],["leaf",59,-0.005876781727744386],["leaf",60,0.00015950538863156562],["leaf",61,0.014490365204513684],["leaf",62,0.0012731629728083625]],[["split",0,26.0,21,0],["split",1,19.0,0,1],["split",2,13.0,22,1],["split",3,2.0,8,2],["split",4,26.0,14,2],["split",5,1.0,4,2],["split",6,24.0,23,2],["split",7,20.0,0,3],["split",8,20.0,0,3],["split",9,13.0,11,3],["split",10,23.0,18,3],["split",11,6.0,5,3],["split",12,9.0,9,3],["split",13,9.0,27,3],["split",14,12.0,22,3],["split",15,23.0,16,4],["split",16,1.0,18,4],["split",17,13.0,2,4],["split",18,21.0,8,4],["split",19,22.0,18,4],["split",20,22.0,15,4],["split",21,10.0,18,4],["leaf",22,0.005686789723715804],["leaf",23,0.008978371691237889],["split",24,24.0,16,4],["split",25,10.0,15,4],["split",26,23.0,1,4],["split",27,0.0,26,4],["split",28,26.0,26,4],["leaf",29,-0.017374560414223937],["split",30,2.0,8,4],["leaf",31,0.006464557717222703],["leaf",32,-0.001796441406043304],["leaf",33,-0.018594368813658627],["leaf",34,-0.002636187300597109],["leaf",35,0.0013893984031231508],["leaf",36,-0.005281653359504956],["leaf",37,-0.009262819569788423],["leaf",38,0.009187812594718073],["leaf",39,0.006131763151076922],["leaf",40,0.022024721755464143],["leaf",41,0.008304523476800064],["leaf",42,-0.004280123645305833],["leaf",43,-0.011493841910259229],["leaf",44,-0.0007929913957102164],["leaf",49,-0.011995474577820896],["leaf",50,-0.002540486160052589],["leaf",51,0.004174163444351233],["leaf",52,-0.005070286975503834],["leaf",53,-0.004051660617356606],["leaf",54,0.007859276467769464],["leaf",55,0.0005232723097698292],["leaf",56,-0.01301265587784537],["leaf",57,0.0006640464291892886],["leaf",58,0.013971057061219158],["leaf",61,-0.010892958453373217],["leaf",62,0.003963086861503275]],[["split",0,1.0,1,0],["split",1,6.0,7,1],["split",2,24.0,3,1],["split",3,4.0,17,2],["split",4,21.0,15,2],["split",5,16.0,0,2],["split",6,25.0,29,2],["leaf",7,0.009892906418809048],["split",8,22.0,21,3],["leaf",9,-0.004134787540873821],["leaf",10,-0.02488873624051517],["split",11,0.0,21,3],["split",12,11.0,14,3],["split",13,22.0,18,3],["split",14,21.0,26,3],["split",17,4.0,26,4],["leaf",18,-0.015503221384205226],["split",23,25.0,5,4],["split",24,5.0,20,4],["leaf",25,-0.004683823531540399],["leaf",26,-0.021233802988249992],["split",27,13.0,0,4],["split",28,3.0,2,4],["split",29,8.0,15,4],["leaf",30,0.017232834235014087],["leaf",35,-0.009825994546172272],["leaf",36,0.003426287341480472],["leaf",47,-0.005964435087490219],["leaf",48,0.003789990562310121],["leaf",49,-0.01485586254001967],["leaf",50,0.0027313084342613135],["leaf",55,0.012776846061022916],["leaf",56,0.00213944250060546],["leaf",57,-0.00976874907318765],["leaf",58,-0.00036275353456662404],["leaf",59,0.008887083233540902],["leaf",60,-0.0018597400729711822]],[["split",0,26.0,7,0],["split",1,1.0,4,1],["split",2,13.0,24,1],["split",3,23.0,20,2],["split",4,2.0,15,2],["split",5,21.0,2,2],["split",6,7.0,28,2],["split",7,2.0,24,3],["split",8,26.0,3,3],["split",9,12.0,11,3],["split",10,25.0,27,3],["split",11,12.0,4,3],["split",12,11.0,19,3],["split",13,23.0,23,3],["split",14,26.0,22,3],["split",15,4.0,19,4],["leaf",16,0.005918267639929167],["leaf",17,0.01779519383737504],["leaf",18,-0.0006103528511232156],["split",19,23.0,14,4],["split",20,7.0,18,4],["split",21,10.0,5,4],["leaf",22,0.003948018016589431],["leaf",23,0.00600343235877145],["split",24,11.0,16,4],["split",25,7.0,2,4],["split",26,23.0,6,4],["split",27,21.0,14,4],["split",28,0.0,13,4],["split",29,0.0,22,4],["leaf",30,-0.005950993956964382],["leaf",31,-0.0005758366275837925],["leaf",32,-0.01893570273997122],["leaf",39,0.008510870239823517],["leaf",40,-0.0038233327886505256],["leaf",41,-0.013718876341798115],["leaf",42,-0.0014452432526013714],["leaf",43,-0.024295279045781736],["leaf",44,-0.01014269136907769],["leaf",49,-0.014313528135096046],["leaf",50,-0.00038263470640966944],["leaf",51,-0.006624843740960105],["leaf",52,0.0005508891247558448],["leaf",53,0.008283574043729971],["leaf",54,0.002434349589865642],["leaf",55,0.00043364230963123616],["leaf",56,-0.006856202129620337],["leaf",57,-0.022954772642922724],["leaf",58,-0.007488999547061195],["leaf",59,0.003424649349103362],["leaf",60,0.017237161876834765]],[["split",0,3.0,19,0],["split",1,6.0,28,1],["split",2,22.0,8,1],["split",3,3.0,14,2],["split",4,12.0,28,2],["split",5,23.0,4,2],["split",6,19.0,0,2],["split",7,10.0,29,3],["split",8,13.0,21,3],["split",9,23.0,20,3],["leaf",10,-0.016127051367966154],["split",11,7.0,11,3],["split",12,0.0,27,3],["split",13,20.0,0,3],["split",14,3.0,23,3],["split",15,0.0,30,4],["split",16,2.0,18,4],["split",17,11.0,25,4],["split",18,25.0,23,4],["leaf",19,0.00041742793800378527],["leaf",20,-0.01182052258333444],["leaf",23,-0.008630242626511343],["split",24,2.0,13,4],["split",25,26.0,19,4],["leaf",26,0.019022943220126265],["split",27,2.0,3,4],["split",28,11.0,9,4],["leaf",29,0.003185055962416358],["split",30,13.0,15,4],["leaf",31,0.0002460300185785245],["leaf",32,0.00964881120666666],["leaf",33,-0.0025307351625370195],["leaf",34,-0.014358187636595545],["leaf",35,-5.731676521014585e-05],["leaf",36,-0.010432413383545155],["leaf",37,-0.012368139821903367],["leaf",38,0.003247570319136713],["leaf",49,0.009455924108636011],["leaf",50,-0.002379219149619899],["leaf",51,0.012759084272939354],["leaf",52,0.0027160215510625843],["leaf",55,0.010451072316720081],["leaf",56,0.00019869426488729628],["leaf",57,-0.020026933541582182],["leaf",58,0.0012069742928105967],["leaf",61,-0.0015503779314596964],["leaf",62,-0.013312911765765524]],[["split",0,0.0,12,0],["split",1,26.0,2,1],["split",2,21.0,3,1],["split",3,23.0,14,2],["split",4,10.0,26,2],["split",5,0.0,20,2],["split",6,6.0,30,2],["split",7,5.0,21,3],["split",8,11.0,9,3],["split",9,23.0,29,3],["split",10,2.0,23,3],["split",11,1.0,12,3],["leaf",12,-0.014489512763190878],["split",13,12.0,26,3],["split",14,1.0,21,3],["split",15,7.0,22,4],["leaf",16,-0.016168746077452008],["leaf",17,0.01901438270978521],["leaf",18,0.0057033534463948755],["split",19,3.0,22,4],["split",20,22.0,18,4],["split",21,23.0,22,4],["leaf",22,-0.020813443639327696],["leaf",23,0.009329822803206897],["leaf",24,-0.005232833119568056],["split",27,20.0,0,4],["split",28,3.0,15,4],["leaf",29,0.0017069974156650848],["leaf",30,0.018721537619578874],["leaf",31,0.0017263587477552099],["leaf",32,0.013420437109549762],["leaf",39,-0.00019499670323725599],["leaf",40,-0.004272462272228575],["leaf",41,-0.0003612949845077983],["leaf",42,0.01852348165059089],["leaf",43,-0.0002216521277367215],["leaf",44,-0.02017700478537949],["leaf",55,0.0017128431492652843],["leaf",56,0.006916098037072947],["leaf",57,-0.009090397803497196],["leaf",58,0.0009489776826918057]],[["split",0,8.0,11,0],["split",1,7.0,4,1],["split",2,6.0,23,1],["split",3,7.0,0,2],["split",4,7.0,11,2],["split",5,5.0,17,2],["split",6,4.0,9,2],["split",7,10.0,14,3],["split",8,4.0,14,3],["split",9,10.0,10,3],["split",10,10.0,4,3],["split",11,6.0,15,3],["split",12,13.0,17,3],["split",13,20.0,0,3],["split",14,4.0,20,3],["split",15,4.0,29,4],["leaf",16,-0.06848287925108772],["split",17,8.0,5,4],["split",18,20.0,0,4],["split",19,20.0,0,4],["split",20,4.0,13,4],["split",21,0.0,14,4],["split",22,25.0,11,4],["split",23,5.0,10,4],["split",24,20.0,0,4],["split",25,5.0,25,4],["split",26,21.0,3,4],["split",27,19.0,0,4],["split",28,21.0,19,4],["split",29,13.0,29,4],["split",30,21.0,17,4],["leaf",31,-0.08659580802092899],["leaf",32,-0.10357242368758941],["leaf",35,-0.03915453561955006],["leaf",36,-0.012360976904243166],["leaf",37,-0.0668115920715894],["leaf",38,-0.03151021269458653],["leaf",39,-0.04021446152208551],["leaf",40,-0.015763351445635298],["leaf",41,-0.0027809005768952727],["leaf",42,-0.02705053064108368],["leaf",43,-0.03165221043275806],["leaf",44,-0.017355535297424767],["leaf",45,-0.01624463491319362],["leaf",46,-0.0039591819718343396],["leaf",47,0.009626484296794652],["leaf",48,-0.0028143062724097717],["leaf",49,0.023253819661407733],["leaf",50,0.005348878899247081],["leaf",51,-0.00568793247082524],["leaf",52,-0.023676962634430927],["leaf",53,-0.009640052641207165],["leaf",54,0.008590536911495415],["leaf",55,0.058868458414489636],["leaf",56,0.03924619484530833],["leaf",57,0.014639777180170493],["leaf",58,0.04866207959636516],["leaf",59,0.03596365151315926],["leaf",60,0.049962807753814995],["leaf",61,0.01359609682861014],["leaf",62,0.03446972905731054]],[["split",0,8.0,12,0],["split",1,8.0,6,1],["split",2,13.0,23,1],["split",3,7.0,2,2],["split",4,4.0,17,2],["split",5,4.0,13,2],["split",6,13.0,29,2],["split",7,4.0,22,3],["split",8,7.0,8,3],["split",9,7.0,13,3],["split",10,7.0,6,3],["split",11,6.0,17,3],["split",12,6.0,18,3],["split",13,4.0,4,3],["split",14,2.0,13,3],["split",15,25.0,19,4],["split",16,7.0,0,4],["split",17,19.0,0,4],["split",18,7.0,10,4],["split",19,26.0,16,4],["split",20,25.0,18,4],["split",21,9.0,21,4],["split",22,23.0,25,4],["split",23,24.0,12,4],["split",24,20.0,0,4],["split",25,26.0,10,4],["split",26,26.0,3,4],["split",27,9.0,13,4],["split",28,12.0,22,4],["split",29,4.0,3,4],["split",30,23.0,13,4],["leaf",31,-0.05343565699228633],["leaf",32,-0.03528856059974038],["leaf",33,-0.09107139717221678],["leaf",34,-0.06706721937378995],["leaf",35,-0.04577657997455891],["leaf",36,-0.024058624585538802],["leaf",37,-0.02579616783955691],["leaf",38,-0.00848560309875119],["leaf",39,-0.018626446098719877],["leaf",40,-0.005468519060083691],["leaf",41,-0.002229431400530907],["leaf",42,0.011182657830579992],["leaf",43,-0.05328713844881115],["leaf",44,-0.02695480099561285],["leaf",45,-0.015480359348047684],["leaf",46,-0.03206659886478538],["leaf",47,0.003954699468092583],["leaf",48,0.017888100882260498],["leaf",49,0.029696555331378865],["leaf",50,0.010500170908848859],["leaf",51,-0.011628462865666878],["leaf",52,0.0018878641631783005],["leaf",53,-0.0039643379234193905],["leaf",54,0.012695058707771205],["leaf",55,0.03650904235164294],["leaf",56,0.052985230574391154],["leaf",57,0.024514127911300178],["leaf",58,0.0363484677087564],["leaf",59,0.06727260028527907],["leaf",60,0.0544357409730721],["leaf",61,0.048963857911283326],["leaf",62,0.03463924194319719]],[["split",0,6.0,11,0],["split",1,6.0,4,1],["split",2,6.0,24,1],["split",3,7.0,1,2],["split",4,7.0,3,2],["split",5,4.0,16,2],["split",6,4.0,10,2],["split",7,25.0,26,3],["split",8,7.0,6,3],["split",9,5.0,26,3],["split",10,7.0,6,3],["split",11,6.0,18,3],["split",12,6.0,20,3],["split",13,20.0,0,3],["split",14,12.0,26,3],["split",15,10.0,4,4],["leaf",16,-0.053008318585697246],["split",17,10.0,2,4],["split",18,21.0,5,4],["split",19,1.0,14,4],["leaf",20,-0.0703872857510563],["split",21,10.0,20,4],["split",22,12.0,7,4],["split",23,8.0,14,4],["split",24,20.0,0,4],["split",25,25.0,12,4],["split",26,21.0,7,4],["split",27,6.0,27,4],["split",28,25.0,17,4],["split",29,4.0,16,4],["split",30,1.0,16,4],["leaf",31,-0.09298918984038787],["leaf",32,-0.07507296438297432],["leaf",35,-0.06802786894600385],["leaf",36,-0.043717047885069804],["leaf",37,-0.032831393535421104],["leaf",38,-0.013716619104169813],["leaf",39,-0.054967989271323396],["leaf",40,-0.032650591422129865],["leaf",43,-0.036846217000394046],["leaf",44,-0.01648849509900392],["leaf",45,-0.022201352449122254],["leaf",46,-0.009715760140194776],["leaf",47,0.0009409581486059846],["leaf",48,0.013341427976544732],["leaf",49,0.02890491688879431],["leaf",50,0.004408078527792877],["leaf",51,-0.011913015348037778],["leaf",52,-0.0011186448551368429],["leaf",53,-0.004662856043066736],["leaf",54,0.014676144871546544],["leaf",55,0.04318949454708671],["leaf",56,0.05696715940284179],["leaf",57,0.006971747424803436],["leaf",58,0.03826305183099503],["leaf",59,0.032294548353139566],["leaf",60,0.01765252774789708],["leaf",61,0.02380270335867769],["leaf",62,0.0461484796005066]],[["split",0,13.0,13,0],["split",1,7.0,3,1],["split",2,6.0,24,1],["split",3,20.0,0,2],["split",4,4.0,15,2],["split",5,5.0,17,2],["split",6,8.0,29,2],["split",7,2.0,23,3],["split",8,25.0,15,3],["split",9,7.0,6,3],["split",10,7.0,7,3],["split",11,20.0,0,3],["split",12,26.0,6,3],["split",13,4.0,17,3],["split",14,2.0,1,3],["split",15,3.0,19,4],["split",16,3.0,23,4],["leaf",17,-0.05043708416703779],["leaf",18,-0.010252485013173015],["split",19,3.0,26,4],["split",20,21.0,5,4],["split",21,20.0,0,4],["split",22,26.0,7,4],["split",23,13.0,20,4],["split",24,25.0,16,4],["split",25,12.0,19,4],["split",26,0.0,4,4],["split",27,2.0,13,4],["split",28,23.0,21,4],["leaf",29,0.06836137114758162],["split",30,1.0,14,4],["leaf",31,-0.0547925084024967],["leaf",32,-0.027351871955352713],["leaf",33,-0.08131031442419862],["leaf",34,-0.04216702958225008],["leaf",39,-0.028470406650230463],["leaf",40,-0.006035175732716999],["leaf",41,-0.016464630143487615],["leaf",42,0.002048519008320899],["leaf",43,-0.03931121173442925],["leaf",44,-0.008422826645019345],["leaf",45,-0.02873966542155661],["leaf",46,-0.012577461196361467],["leaf",47,0.014957118756930673],["leaf",48,0.02885706344190095],["leaf",49,-0.010078578287522594],["leaf",50,0.01717531859256885],["leaf",51,-0.016064848194077642],["leaf",52,0.004357633136792191],["leaf",53,-0.01016244820818544],["leaf",54,0.008217073700522975],["leaf",55,0.04187422121837538],["leaf",56,0.024804106203228468],["leaf",57,0.02220334729512895],["leaf",58,0.005537142261954833],["leaf",61,0.03764163228718803],["leaf",62,0.05343526464700034]],[["split",0,6.0,11,0],["split",1,7.0,2,1],["split",2,13.0,22,1],["split",3,4.0,21,2],["split",4,7.0,5,2],["split",5,4.0,10,2],["split",6,4.0,11,2],["split",7,3.0,20,3],["split",8,10.0,9,3],["split",9,4.0,19,3],["split",10,10.0,4,3],["split",11,12.0,16,3],["split",12,21.0,5,3],["split",13,20.0,0,3],["split",14,4.0,21,3],["leaf",15,-0.04092083416373314],["leaf",16,-0.01953042773973624],["split",17,4.0,28,4],["split",18,4.0,30,4],["split",19,25.0,11,4],["split",20,20.0,0,4],["split",21,4.0,24,4],["split",22,7.0,14,4],["split",23,22.0,12,4],["split",24,25.0,23,4],["split",25,6.0,16,4],["split",26,4.0,25,4],["split",27,2.0,3,4],["split",28,21.0,14,4],["split",29,12.0,15,4],["split",30,3.0,8,4],["leaf",35,-0.06487492776284351],["leaf",36,-0.08471279981187593],["leaf",37,-0.048628720652468434],["leaf",38,-0.06808607203052904],["leaf",39,-0.030516086779331115],["leaf",40,-0.013335400235002539],["leaf",41,-0.04470960733636274],["leaf",42,-0.01468218717361833],["leaf",43,-0.021194177934412476],["leaf",44,-0.03794322580512517],["leaf",45,-0.015578358693316419],["leaf",46,-0.0024919855679459172],["leaf",47,0.016768068453028143],["leaf",48,0.002505785861507253],["leaf",49,0.018595629531714914],["leaf",50,0.03256729203427035],["leaf",51,-0.02421952765626573],["leaf",52,-0.002350685452999506],["leaf",53,0.005228639050866241],["leaf",54,-0.011089023197831043],["leaf",55,0.05341386209126267],["leaf",56,0.03751290139095272],["leaf",57,-0.002393668257511058],["leaf",58,0.027844871931778453],["leaf",59,0.011578328656055134],["leaf",60,0.029426027628369805],["leaf",61,-0.004148334168920613],["leaf",62,0.016494157770358713]],[["split",0,8.0,13,0],["split",1,7.0,3,1],["split",2,8.0,23,1],["split",3,7.0,0,2],["split",4,7.0,7,2],["split",5,4.0,10,2],["split",6,13.0,28,2],["split",7,11.0,21,3],["split",8,20.0,0,3],["split",9,4.0,18,3],["split",10,25.0,12,3],["split",11,12.0,13,3],["split",12,4.0,18,3],["split",13,12.0,21,3],["split",14,0.0,24,3],["split",15,11.0,7,4],["leaf",16,-0.0796514958862438],["split",17,19.0,0,4],["leaf",18,-0.019317309512777374],["split",19,2.0,9,4],["split",20,21.0,27,4],["split",21,6.0,4,4],["split",22,4.0,23,4],["split",23,7.0,14,4],["split",24,10.0,8,4],["split",25,13.0,18,4],["split",26,12.0,24,4],["split",27,2.0,4,4],["split",28,4.0,10,4],["split",29,25.0,8,4],["split",30,9.0,14,4],["leaf",31,-0.07243620939441217],["leaf",32,-0.05008306416776558],["leaf",35,-0.05001075527153297],["leaf",36,-0.027202677140177996],["leaf",39,0.002597286769211076],["leaf",40,-0.018364034489939812],["leaf",41,-0.033840805969745875],["leaf",42,-0.013719970124844296],["leaf",43,-0.03279051252981752],["leaf",44,-0.012462029128819772],["leaf",45,0.0003358104498436356],["leaf",46,-0.015384883815098782],["leaf",47,-0.004881577852998592],["leaf",48,0.01480674344740418],["leaf",49,0.01053923432984405],["leaf",50,0.026309442240939555],["leaf",51,0.0021805649751293604],["leaf",52,0.01316197500195657],["leaf",53,-0.006013508077888836],["leaf",54,0.011343973048616405],["leaf",55,0.03316398309413893],["leaf",56,0.016435040891777816],["leaf",57,0.03896681677601765],["leaf",58,0.02583287953901315],["leaf",59,0.02983480933763459],["leaf",60,0.043242198765755344],["leaf",61,0.04099228402054412],["leaf",62,0.05597025112584316]],[["split",0,6.0,11,0],["split",1,7.0,3,1],["split",2,13.0,21,1],["split",3,3.0,27,2],["split",4,7.0,7,2],["split",5,4.0,14,2],["split",6,5.0,13,2],["split",7,10.0,6,3],["leaf",8,-0.009082281471281638],["split",9,26.0,20,3],["split",10,4.0,13,3],["split",11,26.0,10,3],["split",12,7.0,20,3],["split",13,20.0,0,3],["split",14,23.0,22,3],["split",15,7.0,0,4],["split",16,2.0,26,4],["split",19,10.0,6,4],["split",20,5.0,13,4],["split",21,12.0,15,4],["split",22,7.0,13,4],["split",23,24.0,7,4],["split",24,12.0,23,4],["split",25,12.0,15,4],["split",26,10.0,5,4],["split",27,19.0,0,4],["split",28,24.0,18,4],["split",29,25.0,20,4],["split",30,2.0,13,4],["leaf",31,-0.07910378633351645],["leaf",32,-0.05907138118246465],["leaf",33,-0.03765019300828393],["leaf",34,-0.05440079562470979],["leaf",39,-0.04294317063315731],["leaf",40,-0.027123881802841126],["leaf",41,0.005048887004609709],["leaf",42,-0.024817991220296782],["leaf",43,-0.006303963810778021],["leaf",44,0.009679824618069332],["leaf",45,-0.01871948143045335],["leaf",46,-0.007439518441206459],["leaf",47,-0.007813274426682112],["leaf",48,0.01205013282794301],["leaf",49,0.014200887372512424],["leaf",50,0.02832739659492134],["leaf",51,-0.011829446668688877],["leaf",52,0.003056928965453291],["leaf",53,-0.005994717313442249],["leaf",54,0.0102081587605797],["leaf",55,0.03972502183463127],["leaf",56,0.021549643971530235],["leaf",57,0.00019964427854218667],["leaf",58,0.02451712750713858],["leaf",59,0.014672576394062524],["leaf",60,0.03144280015799197],["leaf",61,0.01712598090106048],["leaf",62,-0.006871358014768318]],[["split",0,13.0,10,0],["split",1,7.0,4,1],["split",2,8.0,20,1],["split",3,7.0,1,2],["split",4,7.0,8,2],["split",5,4.0,10,2],["split",6,4.0,14,2],["split",7,20.0,0,3],["split",8,2.0,25,3],["split",9,20.0,0,3],["split",10,17.0,0,3],["split",11,12.0,26,3],["split",12,2.0,8,3],["split",13,20.0,0,3],["split",14,26.0,9,3],["split",15,10.0,10,4],["leaf",16,-0.02593135424303891],["split",17,10.0,9,4],["split",18,21.0,12,4],["split",19,2.0,18,4],["leaf",20,-0.00601239585541079],["split",21,7.0,16,4],["leaf",22,-0.02624852044923631],["split",23,20.0,0,4],["split",24,4.0,4,4],["split",25,22.0,19,4],["split",26,13.0,14,4],["split",27,4.0,5,4],["split",28,25.0,20,4],["split",29,22.0,22,4],["split",30,6.0,25,4],["leaf",31,-0.06795953727798157],["leaf",32,-0.04308027492403454],["leaf",35,-0.03701444221273642],["leaf",36,-0.02161336506795954],["leaf",37,-0.05278727620204141],["leaf",38,-0.03613979824005797],["leaf",39,-0.016955574328106793],["leaf",40,-0.02985368317839198],["leaf",43,-0.008770490741035208],["leaf",44,0.007227196261698227],["leaf",47,0.0112305966250842],["leaf",48,-0.003451385766045445],["leaf",49,0.032487985285259105],["leaf",50,0.019004402358261986],["leaf",51,0.013401486424949323],["leaf",52,-0.0013809726661489882],["leaf",53,-0.01117332556710055],["leaf",54,0.0009632971456294673],["leaf",55,0.03838704165911952],["leaf",56,0.02590898871357854],["leaf",57,0.002687137749950916],["leaf",58,0.026359631727983585],["leaf",59,0.00587811336047146],["leaf",60,-0.008609526013308002],["leaf",61,0.010264964616318498],["leaf",62,0.0257275154892393]],[["split",0,13.0,10,0],["split",1,7.0,3,1],["split",2,6.0,24,1],["split",3,7.0,1,2],["split",4,4.0,17,2],["split",5,12.0,12,2],["split",6,5.0,7,2],["split",7,17.0,0,3],["split",8,20.0,0,3],["split",9,24.0,20,3],["split",10,20.0,0,3],["split",11,2.0,10,3],["split",12,5.0,10,3],["split",13,20.0,0,3],["split",14,12.0,25,3],["split",15,15.0,0,4],["leaf",16,-0.06804955393742668],["split",17,19.0,0,4],["leaf",18,-0.008789148169955257],["split",19,17.0,0,4],["split",20,10.0,9,4],["split",21,10.0,6,4],["split",22,25.0,19,4],["split",23,13.0,18,4],["split",24,21.0,7,4],["split",25,7.0,8,4],["split",26,25.0,13,4],["split",27,19.0,0,4],["split",28,25.0,15,4],["split",29,14.0,0,4],["split",30,22.0,10,4],["leaf",31,-0.044211650182900485],["leaf",32,-0.06314624263844026],["leaf",35,-0.04280403733544814],["leaf",36,-0.017354998051623653],["leaf",39,-0.0073339000302747245],["leaf",40,-0.020438997971677638],["leaf",41,-0.0036023215723360548],["leaf",42,0.012262455984898835],["leaf",43,-0.037994345538565434],["leaf",44,-0.021375302493946674],["leaf",45,-0.01358827066695129],["leaf",46,0.015386149430729624],["leaf",47,-0.0007366851089181731],["leaf",48,0.018585482037274018],["leaf",49,-0.017817047070025058],["leaf",50,-0.004655948356528719],["leaf",51,-0.0013178616142995947],["leaf",52,0.018795763198834026],["leaf",53,0.0009514217928416976],["leaf",54,0.010900320297276606],["leaf",55,0.04181278485706766],["leaf",56,0.02186750993801468],["leaf",57,0.0006475871572028118],["leaf",58,0.02266736931573518],["leaf",59,0.013896460100533255],["leaf",60,0.028174798688240663],["leaf",61,0.040356165249376144],["leaf",62,0.02337217585887741]],[["split",0,8.0,10,0],["split",1,7.0,2,1],["split",2,6.0,24,1],["split",3,20.0,0,2],["split",4,21.0,9,2],["split",5,13.0,16,2],["split",6,5.0,5,2],["split",7,2.0,27,3],["leaf",8,-0.0171227112393386],["split",9,7.0,8,3],["split",10,20.0,0,3],["split",11,4.0,11,3],["split",12,5.0,2,3],["split",13,20.0,0,3],["split",14,21.0,14,3],["split",15,10.0,11,4],["split",16,4.0,28,4],["split",19,10.0,7,4],["split",20,1.0,5,4],["split",21,4.0,21,4],["split",22,23.0,17,4],["split",23,10.0,7,4],["split",24,4.0,24,4],["split",25,12.0,20,4],["split",26,21.0,6,4],["split",27,19.0,0,4],["leaf",28,0.014413123172148013],["split",29,2.0,12,4],["split",30,7.0,21,4],["leaf",31,-0.04855232392634234],["leaf",32,-0.027271126758607347],["leaf",33,-0.04707260301328048],["leaf",34,-0.07305787935498638],["leaf",39,-0.043019651974173255],["leaf",40,-0.027788407701154854],["leaf",41,-0.028807731352625262],["leaf",42,-0.009661985813080148],["leaf",43,-0.007351004891165734],["leaf",44,-0.02402267951549837],["leaf",45,-0.002954970528116767],["leaf",46,0.020729190474946948],["leaf",47,-0.004327121161383944],["leaf",48,0.01039056912067019],["leaf",49,-0.003837092125083018],["leaf",50,-0.015825513431155583],["leaf",51,0.0191906106636875],["leaf",52,0.033929533931522135],["leaf",53,2.9301125996882743e-05],["leaf",54,0.011178910501555221],["leaf",55,0.04341587116371632],["leaf",56,0.01700286954240669],["leaf",59,0.019987691437950268],["leaf",60,0.006610179758503848],["leaf",61,0.011334056406161459],["leaf",62,0.028650020439468356]],[["split",0,8.0,9,0],["split",1,7.0,1,1],["split",2,6.0,20,1],["split",3,24.0,24,2],["split",4,7.0,6,2],["split",5,4.0,12,2],["split",6,4.0,5,2],["split",7,4.0,25,3],["split",8,4.0,28,3],["split",9,25.0,13,3],["split",10,4.0,15,3],["split",11,12.0,16,3],["split",12,25.0,5,3],["split",13,12.0,28,3],["split",14,13.0,27,3],["leaf",15,-0.03377588480193717],["split",16,9.0,14,4],["leaf",17,-0.020911255458337625],["leaf",18,-0.03641853913702105],["split",19,5.0,16,4],["split",20,17.0,0,4],["split",21,11.0,7,4],["split",22,26.0,3,4],["split",23,8.0,15,4],["split",24,3.0,22,4],["split",25,25.0,1,4],["split",26,4.0,28,4],["split",27,13.0,30,4],["split",28,9.0,17,4],["split",29,2.0,2,4],["split",30,3.0,18,4],["leaf",33,-0.071209650742583],["leaf",34,-0.04847094256754182],["leaf",39,-0.01638881178544186],["leaf",40,-0.03584431248519119],["leaf",41,-0.016333715396171122],["leaf",42,-0.03916683253003061],["leaf",43,0.014499409426822374],["leaf",44,-0.007028350430619139],["leaf",45,-0.027921966527934074],["leaf",46,-0.010738931751089599],["leaf",47,-0.001876109434195973],["leaf",48,0.0067798418482884724],["leaf",49,0.006014051143996496],["leaf",50,0.017984867171516847],["leaf",51,-0.024452246517512627],["leaf",52,-0.012605272743023139],["leaf",53,0.0004115734857688533],["leaf",54,-0.01673790425752766],["leaf",55,0.02378199483009007],["leaf",56,0.045155113806102036],["leaf",57,0.03637119530791107],["leaf",58,0.048496640682466834],["leaf",59,0.029804611273544418],["leaf",60,0.011110780548243854],["leaf",61,0.018677480487377756],["leaf",62,0.03081620371147416]],[["split",0,8.0,8,0],["split",1,7.0,4,1],["split",2,6.0,19,1],["split",3,10.0,9,2],["split",4,7.0,17,2],["split",5,4.0,20,2],["split",6,5.0,8,2],["split",7,3.0,22,3],["split",8,23.0,24,3],["split",9,17.0,0,3],["leaf",10,0.007916098417087066],["split",11,21.0,9,3],["split",12,8.0,13,3],["split",13,20.0,0,3],["split",14,9.0,27,3],["split",15,3.0,13,4],["leaf",16,-0.01583436061333023],["split",17,3.0,7,4],["split",18,21.0,16,4],["split",19,15.0,0,4],["split",20,23.0,19,4],["split",23,4.0,11,4],["split",24,26.0,22,4],["split",25,21.0,11,4],["split",26,12.0,7,4],["split",27,6.0,27,4],["split",28,21.0,21,4],["split",29,1.0,16,4],["split",30,12.0,19,4],["leaf",31,-0.04041290437737452],["leaf",32,-0.05851655894192457],["leaf",35,-0.012207504296110016],["leaf",36,-0.02874737375404438],["leaf",37,-0.04970987375908109],["leaf",38,-0.029812992521807167],["leaf",39,-0.014453730272774553],["leaf",40,-0.03156565121978096],["leaf",41,-0.02118469546846626],["leaf",42,-0.054026400030495136],["leaf",47,0.003188710405783507],["leaf",48,-0.009699388079300786],["leaf",49,0.004178609142974409],["leaf",50,0.014270471587553553],["leaf",51,-0.023815947084292223],["leaf",52,-0.007604144020079001],["leaf",53,-0.016060954884090956],["leaf",54,-0.0002507869338268445],["leaf",55,0.023803141489562023],["leaf",56,0.03665695185574814],["leaf",57,0.001205838104186812],["leaf",58,0.02760839105316345],["leaf",59,0.00591621387532051],["leaf",60,0.01470262839331046],["leaf",61,0.015561459452784893],["leaf",62,0.03181397046429842]],[["split",0,8.0,13,0],["split",1,8.0,6,1],["split",2,8.0,26,1],["split",3,7.0,1,2],["split",4,10.0,7,2],["split",5,4.0,16,2],["split",6,12.0,25,2],["split",7,4.0,25,3],["split",8,24.0,23,3],["split",9,20.0,0,3],["split",10,4.0,25,3],["split",11,20.0,0,3],["split",12,22.0,26,3],["split",13,2.0,5,3],["split",14,4.0,14,3],["split",15,2.0,23,4],["split",16,21.0,5,4],["split",17,19.0,0,4],["split",18,25.0,23,4],["split",19,23.0,18,4],["leaf",20,0.0014350136944238105],["split",21,7.0,8,4],["split",22,7.0,7,4],["split",23,8.0,18,4],["split",24,25.0,18,4],["split",25,26.0,12,4],["split",26,23.0,14,4],["split",27,7.0,26,4],["split",28,24.0,5,4],["split",29,2.0,11,4],["leaf",30,0.014188719696331084],["leaf",31,-0.036151528172955],["leaf",32,-0.016702836420775975],["leaf",33,-0.06875335954789297],["leaf",34,-0.045816487619648745],["leaf",35,-0.02774938168476863],["leaf",36,-0.00574950031033372],["leaf",37,-0.016722662836170552],["leaf",38,0.0037195847693436827],["leaf",39,-0.012284931917763678],["leaf",40,-0.028606082844524018],["leaf",43,-0.007419127494827448],["leaf",44,0.002816264295337405],["leaf",45,-0.0254802379554968],["leaf",46,-0.008896890348325207],["leaf",47,0.00640273320202034],["leaf",48,0.0187769340931487],["leaf",49,-0.014204733956631322],["leaf",50,0.017772143669388573],["leaf",51,-0.002149190819368991],["leaf",52,0.007507200553333838],["leaf",53,-0.001139275108691058],["leaf",54,-0.019846356697407707],["leaf",55,0.01735624408067331],["leaf",56,0.03847666026691635],["leaf",57,-0.0001983032310977745],["leaf",58,0.018519629747428983],["leaf",59,0.041241298274208575],["leaf",60,0.02667339883111892]],[["split",0,7.0,9,0],["split",1,13.0,8,1],["split",2,6.0,23,1],["split",3,7.0,0,2],["split",4,5.0,11,2],["split",5,5.0,15,2],["split",6,2.0,5,2],["split",7,9.0,15,3],["split",8,20.0,0,3],["split",9,1.0,12,3],["split",10,0.0,5,3],["split",11,12.0,17,3],["split",12,12.0,8,3],["split",13,20.0,0,3],["split",14,12.0,28,3],["split",15,3.0,11,4],["leaf",16,-0.03615873804174155],["split",17,10.0,5,4],["split",18,25.0,15,4],["leaf",19,-0.0036005999079583312],["split",20,3.0,26,4],["leaf",21,-0.0325023299207759],["split",22,23.0,19,4],["split",23,5.0,8,4],["split",24,10.0,9,4],["split",25,4.0,26,4],["split",26,23.0,22,4],["split",27,7.0,26,4],["leaf",28,0.0016987911883412576],["split",29,0.0,2,4],["split",30,9.0,17,4],["leaf",31,-0.06182981785705301],["leaf",32,-0.03725148445516125],["leaf",35,-0.03394208134587616],["leaf",36,-0.022309955344123996],["leaf",37,-0.020843123718680148],["leaf",38,0.00732178309979184],["leaf",41,-0.0005381430424821684],["leaf",42,0.01825061819707983],["leaf",45,-0.003751471503335978],["leaf",46,-0.018925648713995796],["leaf",47,0.012444204135513531],["leaf",48,-0.0008645799789006967],["leaf",49,0.0027577060126580477],["leaf",50,0.016679070277694143],["leaf",51,-0.007273482932477836],["leaf",52,-0.02155609559212335],["leaf",53,0.0036337921755903186],["leaf",54,-0.006708806482884181],["leaf",55,0.02529261272380953],["leaf",56,0.03994683590830607],["leaf",59,-0.0011909591760978088],["leaf",60,0.015328045106076878],["leaf",61,0.01949416939905983],["leaf",62,0.03145922184788295]],[["split",0,7.0,9,0],["split",1,7.0,5,1],["split",2,6.0,21,1],["split",3,4.0,22,2],["split",4,21.0,6,2],["split",5,12.0,10,2],["split",6,8.0,30,2],["split",7,15.0,0,3],["split",8,7.0,0,3],["split",9,23.0,15,3],["split",10,12.0,5,3],["split",11,26.0,9,3],["split",12,23.0,29,3],["split",13,4.0,17,3],["split",14,0.0,17,3],["split",15,10.0,17,4],["leaf",16,-0.03999370709390915],["split",17,3.0,9,4],["split",18,14.0,0,4],["leaf",19,-0.014384540109973526],["leaf",20,-0.028745355288280695],["split",21,9.0,19,4],["split",22,26.0,25,4],["split",23,6.0,13,4],["split",24,21.0,17,4],["split",25,4.0,3,4],["leaf",26,-0.018554655220191153],["split",27,19.0,0,4],["split",28,5.0,21,4],["split",29,10.0,20,4],["split",30,23.0,17,4],["leaf",31,-0.017564109314593335],["leaf",32,-0.0011457683437902089],["leaf",35,-0.03540837290043402],["leaf",36,-0.06640645694846964],["leaf",37,-0.025798794884911574],["leaf",38,-0.043930573212503675],["leaf",43,-0.007695689675316205],["leaf",44,-0.025943908232820875],["leaf",45,-0.007962050233590484],["leaf",46,0.004355178438513324],["leaf",47,-0.006706477903467777],["leaf",48,-0.024478027363853603],["leaf",49,-0.006490204045223161],["leaf",50,0.008192260572217173],["leaf",51,0.021619189217437218],["leaf",52,0.005453926409457702],["leaf",55,0.020352397771272655],["leaf",56,0.008457812066334796],["leaf",57,0.011465490733696663],["leaf",58,-0.0009973029238500952],["leaf",59,0.034001688703093065],["leaf",60,0.01822477224858881],["leaf",61,0.051220705037065654],["leaf",62,0.0326092895784739]],[["split",0,6.0,11,0],["split",1,7.0,1,1],["split",2,8.0,26,1],["split",3,21.0,25,2],["split",4,20.0,0,2],["split",5,4.0,12,2],["split",6,13.0,30,2],["split",7,3.0,19,3],["split",8,26.0,9,3],["split",9,4.0,10,3],["split",10,21.0,20,3],["split",11,20.0,0,3],["split",12,23.0,24,3],["split",13,12.0,18,3],["split",14,22.0,17,3],["split",15,12.0,5,4],["leaf",16,-0.02079413826841528],["leaf",17,-0.022654883344616638],["leaf",18,-0.012953324225184562],["split",19,1.0,9,4],["split",20,23.0,28,4],["split",21,2.0,15,4],["split",22,23.0,18,4],["split",23,2.0,11,4],["split",24,21.0,16,4],["split",25,12.0,22,4],["split",26,9.0,12,4],["split",27,26.0,19,4],["split",28,7.0,22,4],["split",29,5.0,3,4],["leaf",30,0.023586673730499608],["leaf",31,-0.040166883402618404],["leaf",32,-0.07003273265039045],["leaf",39,-0.009830830152040919],["leaf",40,0.003252258076425019],["leaf",41,-0.015379019899597595],["leaf",42,-0.03264467152019348],["leaf",43,-0.03138607035831037],["leaf",44,-0.00022878828645669596],["leaf",45,0.011547000251608089],["leaf",46,0.03385566396451452],["leaf",47,0.017391975334476624],["leaf",48,0.007045697387072273],["leaf",49,-0.016986915292260273],["leaf",50,0.010839008950168405],["leaf",51,-7.929910543451492e-05],["leaf",52,0.010905671321387776],["leaf",53,-0.017429261066368704],["leaf",54,8.65294863021866e-05],["leaf",55,-0.0009327092806844669],["leaf",56,0.013082459013037676],["leaf",57,0.011565930134381662],["leaf",58,0.02299024849349838],["leaf",59,0.04845527378868323],["leaf",60,0.028745669408857775]],[["split",0,13.0,10,0],["split",1,7.0,3,1],["split",2,13.0,23,1],["split",3,20.0,0,2],["split",4,4.0,20,2],["split",5,8.0,15,2],["split",6,2.0,11,2],["split",7,10.0,18,3],["split",8,21.0,13,3],["split",9,7.0,15,3],["split",10,7.0,15,3],["split",11,5.0,19,3],["split",12,2.0,14,3],["split",13,20.0,0,3],["split",14,22.0,25,3],["split",15,4.0,28,4],["split",16,2.0,17,4],["leaf",17,-0.02615610239815141],["leaf",18,0.009049443246362506],["split",19,20.0,0,4],["split",20,21.0,15,4],["split",21,10.0,10,4],["leaf",22,-0.00035111637454004986],["split",23,7.0,6,4],["split",24,26.0,17,4],["split",25,20.0,0,4],["split",26,26.0,4,4],["split",27,19.0,0,4],["split",28,10.0,19,4],["split",29,3.0,18,4],["split",30,25.0,15,4],["leaf",31,-0.025603503640498113],["leaf",32,-0.0444728290893958],["leaf",33,0.005258420190398671],["leaf",34,-0.025527512836084173],["leaf",39,-0.010291853053355877],["leaf",40,0.004917399125785608],["leaf",41,-0.0004563403148677607],["leaf",42,0.015459360354366493],["leaf",43,-0.024250617065036856],["leaf",44,-0.01280601379006424],["leaf",47,-0.015844544333710433],["leaf",48,0.003838713851961815],["leaf",49,-0.01402493162319492],["leaf",50,-0.0024547927455108457],["leaf",51,0.013445724605480746],["leaf",52,-0.008088398185031513],["leaf",53,-0.01326436215582094],["leaf",54,0.004428355581180432],["leaf",55,0.027412337196359212],["leaf",56,0.01391833016332803],["leaf",57,-0.005225892504165645],["leaf",58,0.014667347827186412],["leaf",59,0.005440857048083186],["leaf",60,0.019359731614909493],["leaf",61,-0.010434962857253519],["leaf",62,0.006931759728995974]],[["split",0,7.0,8,0],["split",1,7.0,1,1],["split",2,12.0,15,1],["split",3,20.0,0,2],["split",4,5.0,10,2],["split",5,3.0,21,2],["split",6,6.0,26,2],["split",7,19.0,0,3],["leaf",8,-0.010198841284489842],["split",9,1.0,11,3],["split",10,20.0,0,3],["split",11,2.0,15,3],["split",12,1.0,6,3],["split",13,10.0,11,3],["split",14,6.0,30,3],["split",15,17.0,0,4],["leaf",16,-0.014209473739651006],["leaf",19,-0.008108641746779027],["split",20,26.0,20,4],["split",21,7.0,5,4],["split",22,21.0,18,4],["split",23,7.0,20,4],["split",24,20.0,0,4],["leaf",25,-0.009652481780131372],["split",26,25.0,18,4],["split",27,25.0,4,4],["split",28,26.0,24,4],["split",29,22.0,18,4],["split",30,12.0,29,4],["leaf",31,-0.03540722141637011],["leaf",32,-0.05707129354547568],["leaf",41,-0.0029910241432926366],["leaf",42,0.009900762856533019],["leaf",43,-0.023248130969333435],["leaf",44,-0.011920408688662308],["leaf",45,-0.01283527379523517],["leaf",46,0.014889937428475627],["leaf",47,-0.003587799911669978],["leaf",48,0.010711352734329811],["leaf",49,-0.011462160749129284],["leaf",50,0.005569402403676286],["leaf",53,0.005300991980132884],["leaf",54,0.016914470332849926],["leaf",55,-0.011162138987667759],["leaf",56,0.005565863014334068],["leaf",57,0.008961953269569492],["leaf",58,0.018160186185958493],["leaf",59,0.02155081208510568],["leaf",60,0.010345052600715153],["leaf",61,0.023400363682135796],["leaf",62,0.039991860294219445]],[["split",0,7.0,8,0],["split",1,4.0,23,1],["split",2,8.0,22,1],["split",3,3.0,27,2],["split",4,7.0,4,2],["split",5,3.0,20,2],["split",6,8.0,28,2],["split",7,10.0,3,3],["split",8,2.0,24,3],["split",9,3.0,6,3],["split",10,24.0,12,3],["split",11,21.0,13,3],["split",12,12.0,18,3],["split",13,5.0,18,3],["split",14,20.0,0,3],["split",15,12.0,6,4],["split",16,20.0,0,4],["leaf",17,0.0071072740154759055],["leaf",18,-0.006405964783570314],["split",19,12.0,6,4],["split",20,20.0,0,4],["split",21,22.0,12,4],["split",22,1.0,16,4],["split",23,25.0,1,4],["split",24,10.0,28,4],["split",25,25.0,19,4],["split",26,10.0,15,4],["split",27,17.0,0,4],["split",28,7.0,27,4],["split",29,5.0,7,4],["leaf",30,0.007638118349398298],["leaf",31,-0.02219978126323558],["leaf",32,-0.0336761825502366],["leaf",33,-0.012684377021610065],["leaf",34,0.0018163108361108858],["leaf",39,-0.015884241927174577],["leaf",40,-0.0328181571024316],["leaf",41,-0.03703865161232277],["leaf",42,-0.0040834338319908434],["leaf",43,-0.02702338059233692],["leaf",44,-0.012593042464522123],["leaf",45,-0.010386266251678438],["leaf",46,0.004216980446845676],["leaf",47,-0.020311448231372865],["leaf",48,-0.0043601354961109355],["leaf",49,0.0015875240748675762],["leaf",50,0.013379886980864265],["leaf",51,-0.0006866973276076979],["leaf",52,0.01113494244595789],["leaf",53,0.0065061282578555936],["leaf",54,0.02151613315707546],["leaf",55,0.00995381852366224],["leaf",56,0.022495753368181423],["leaf",57,-0.00687031025315135],["leaf",58,0.009340109617788006],["leaf",59,0.026912759868777062],["leaf",60,0.01568156680824794]],[["split",0,7.0,6,0],["split",1,3.0,21,1],["split",2,4.0,16,1],["split",3,10.0,8,2],["split",4,2.0,27,2],["split",5,12.0,24,2],["split",6,7.0,11,2],["split",7,20.0,0,3],["split",8,1.0,2,3],["split",9,4.0,6,3],["split",10,10.0,12,3],["split",11,13.0,22,3],["split",12,7.0,26,3],["split",13,13.0,14,3],["split",14,1.0,5,3],["split",15,12.0,5,4],["leaf",16,-0.0009044077008608209],["split",17,21.0,15,4],["split",18,7.0,0,4],["leaf",19,0.010098807262326119],["split",20,21.0,14,4],["leaf",21,-0.010954916521039115],["leaf",22,-0.024012548830528703],["split",23,10.0,4,4],["split",24,20.0,0,4],["split",25,9.0,29,4],["split",26,2.0,4,4],["split",27,24.0,24,4],["leaf",28,0.0030075799764649543],["split",29,8.0,9,4],["split",30,25.0,20,4],["leaf",31,-0.03186129617970044],["leaf",32,-0.049230869084056286],["leaf",35,-0.040388263811343104],["leaf",36,-0.014778244221491299],["leaf",37,-0.02908334051591356],["leaf",38,-0.012567083375996753],["leaf",41,-0.014901748339513942],["leaf",42,0.0006345570761247681],["leaf",47,-0.004346460347104547],["leaf",48,0.004263560923011777],["leaf",49,0.01434443578920745],["leaf",50,-0.00641086908836333],["leaf",51,0.012033014363419388],["leaf",52,0.029877537866619508],["leaf",53,0.035602429599295514],["leaf",54,0.018822680699321838],["leaf",55,-0.01496985267762445],["leaf",56,-0.0034534503461704233],["leaf",59,0.0027096677077736325],["leaf",60,-0.015472467053505681],["leaf",61,-0.000955699618632269],["leaf",62,0.008928755058944183]],[["split",0,7.0,12,0],["split",1,3.0,26,1],["split",2,8.0,25,1],["split",3,7.0,3,2],["split",4,4.0,3,2],["split",5,25.0,0,2],["split",6,22.0,24,2],["split",7,20.0,0,3],["split",8,26.0,18,3],["split",9,11.0,13,3],["split",10,11.0,17,3],["leaf",11,-0.01583240107094967],["split",12,12.0,16,3],["split",13,25.0,26,3],["split",14,9.0,27,3],["split",15,10.0,7,4],["split",16,25.0,17,4],["split",17,5.0,26,4],["split",18,21.0,13,4],["leaf",19,0.018503977908937142],["leaf",20,0.006802322347188424],["split",21,4.0,16,4],["split",22,0.0,19,4],["split",25,4.0,16,4],["split",26,21.0,7,4],["split",27,20.0,0,4],["split",28,21.0,21,4],["split",29,1.0,17,4],["leaf",30,0.023976800548331997],["leaf",31,-0.039513582212066516],["leaf",32,-0.019926422192078873],["leaf",33,-0.020426044020050358],["leaf",34,0.018686728918389107],["leaf",35,-0.008926465782402197],["leaf",36,-0.021489775205986286],["leaf",37,-0.008175726463160253],["leaf",38,0.0006283051739782544],["leaf",43,-0.009520660078138255],["leaf",44,0.0032043142202938823],["leaf",45,0.0011986028598037523],["leaf",46,0.010811681179164923],["leaf",51,0.005434795222714187],["leaf",52,-0.003675332919752729],["leaf",53,0.0008191257960820461],["leaf",54,0.011381523890209093],["leaf",55,0.019573252908365875],["leaf",56,0.003721535322902102],["leaf",57,0.0263426338019223],["leaf",58,0.03967217244627882],["leaf",59,-0.0075284648962176464],["leaf",60,0.011587040536456542]],[["split",0,8.0,15,0],["split",1,7.0,3,1],["split",2,5.0,2,1],["split",3,22.0,18,2],["split",4,4.0,25,2],["split",5,20.0,0,2],["split",6,1.0,2,2],["split",7,21.0,5,3],["split",8,15.0,0,3],["split",9,1.0,1,3],["split",10,17.0,0,3],["split",11,26.0,19,3],["leaf",12,-0.0024110987663802865],["split",13,24.0,17,3],["split",14,21.0,8,3],["split",15,23.0,22,4],["split",16,20.0,0,4],["split",17,0.0,25,4],["leaf",18,-0.02983356222716354],["split",19,24.0,16,4],["split",20,21.0,6,4],["split",21,14.0,0,4],["leaf",22,-0.02377148170954485],["split",23,7.0,24,4],["split",24,13.0,30,4],["leaf",27,-0.01936385964850746],["leaf",28,-0.0007912500541353101],["split",29,20.0,0,4],["split",30,25.0,28,4],["leaf",31,-0.02752295123849933],["leaf",32,-0.04622482317404249],["leaf",33,-0.025666368090422605],["leaf",34,-0.005667225477966461],["leaf",35,-0.014076417598424679],["leaf",36,0.010688558819446855],["leaf",39,-0.024851510059935827],["leaf",40,-0.011328223365704618],["leaf",41,-0.009981078067595975],["leaf",42,0.00039271493775228404],["leaf",43,-0.007017946756500705],["leaf",44,-0.02289769551881223],["leaf",47,0.004579645278611926],["leaf",48,0.02426969250130101],["leaf",49,0.024995977379636446],["leaf",50,0.038035000518149745],["leaf",59,0.004934662496616437],["leaf",60,-0.01571198171495643],["leaf",61,0.008695674114930895],["leaf",62,0.020058842272629215]],[["split",0,13.0,9,0],["split",1,7.0,3,1],["split",2,8.0,23,1],["split",3,17.0,0,2],["split",4,10.0,12,2],["split",5,4.0,20,2],["split",6,5.0,19,2],["split",7,4.0,18,3],["split",8,2.0,23,3],["split",9,17.0,0,3],["split",10,4.0,29,3],["split",11,0.0,7,3],["split",12,21.0,7,3],["split",13,7.0,28,3],["split",14,26.0,16,3],["split",15,9.0,12,4],["split",16,3.0,11,4],["leaf",17,-0.018389964965905274],["leaf",18,-0.04677844496831213],["split",19,19.0,0,4],["leaf",20,-0.027685155884689783],["split",21,26.0,1,4],["leaf",22,-0.018995268077427307],["split",23,17.0,0,4],["split",24,12.0,28,4],["split",25,5.0,23,4],["split",26,15.0,0,4],["split",27,17.0,0,4],["split",28,2.0,6,4],["leaf",29,-0.007063453312783359],["leaf",30,0.006788590713854223],["leaf",31,0.009076903601146722],["leaf",32,-0.008779542205816743],["leaf",33,-0.01738655366812342],["leaf",34,-0.029464842069659453],["leaf",39,-0.0114417144838838],["leaf",40,0.0035307948106970024],["leaf",43,-0.01551316751153556],["leaf",44,0.0006194781986950752],["leaf",47,-0.005738682447548141],["leaf",48,0.013206760301752487],["leaf",49,0.004623499675551109],["leaf",50,0.015496003464892136],["leaf",51,-0.005647014620038343],["leaf",52,-0.024973563918630055],["leaf",53,0.0011213855162951944],["leaf",54,-0.015330458896024475],["leaf",55,0.009931597736410603],["leaf",56,0.024595118993550928],["leaf",57,0.0341829810813256],["leaf",58,0.017634076020606848]],[["split",0,7.0,13,0],["split",1,4.0,18,1],["split",2,6.0,28,1],["split",3,21.0,14,2],["split",4,7.0,4,2],["split",5,24.0,3,2],["split",6,24.0,9,2],["split",7,20.0,0,3],["split",8,9.0,18,3],["split",9,22.0,18,3],["split",10,23.0,19,3],["split",11,21.0,4,3],["split",12,4.0,24,3],["split",13,22.0,14,3],["split",14,3.0,21,3],["split",15,4.0,2,4],["leaf",16,-0.016914188639034964],["split",17,13.0,4,4],["split",18,24.0,20,4],["split",19,17.0,0,4],["split",20,10.0,4,4],["split",21,10.0,9,4],["split",22,14.0,0,4],["split",23,10.0,14,4],["split",24,26.0,12,4],["split",25,25.0,13,4],["split",26,22.0,11,4],["leaf",27,0.019898412572298053],["leaf",28,-0.0029436297290892665],["split",29,11.0,21,4],["split",30,7.0,25,4],["leaf",31,0.011313562997893144],["leaf",32,-0.006658735232731517],["leaf",35,0.008475979139463515],["leaf",36,-0.004485795435104795],["leaf",37,0.004053346227615653],["leaf",38,0.0161100995579057],["leaf",39,-0.02162026606278654],["leaf",40,-0.03622070676805705],["leaf",41,-0.026654222537865407],["leaf",42,-0.009845779260535854],["leaf",43,-0.013726839628917352],["leaf",44,0.00018197087115898734],["leaf",45,-0.014476991798834809],["leaf",46,-0.02801346913368649],["leaf",47,-0.026125263779769547],["leaf",48,-0.012521303377359823],["leaf",49,-0.005982956557288984],["leaf",50,0.008592951377689409],["leaf",51,0.003314602344201359],["leaf",52,0.010532573253213802],["leaf",53,0.002811202611090975],["leaf",54,-0.010306410113907679],["leaf",59,0.009912029956341425],["leaf",60,0.023283261149463514],["leaf",61,0.017253701939841742],["leaf",62,0.03626333298237416]],[["split",0,6.0,15,0],["split",1,7.0,0,1],["split",2,5.0,8,1],["split",3,21.0,9,2],["split",4,7.0,6,2],["split",5,7.0,30,2],["split",6,21.0,13,2],["leaf",7,-0.04031667787412117],["split",8,9.0,17,3],["split",9,21.0,15,3],["split",10,12.0,6,3],["split",11,19.0,0,3],["leaf",12,0.035111100449023276],["split",13,22.0,28,3],["split",14,10.0,5,3],["leaf",17,-0.01555750022259104],["leaf",18,-0.02940444883228506],["split",19,1.0,3,4],["split",20,20.0,0,4],["split",21,9.0,6,4],["split",22,21.0,2,4],["split",23,20.0,0,4],["split",24,12.0,25,4],["split",27,12.0,24,4],["leaf",28,-0.019025403981816708],["split",29,22.0,19,4],["split",30,7.0,28,4],["leaf",39,-0.025985010011653605],["leaf",40,-0.013471799097621603],["leaf",41,-0.009150044558619456],["leaf",42,0.009665457386244986],["leaf",43,-0.018395848530816637],["leaf",44,-0.006600049954811882],["leaf",45,-0.012713493378308873],["leaf",46,0.00010269469825776143],["leaf",47,0.01524843935428556],["leaf",48,7.342335944456208e-05],["leaf",49,-0.008236580098660328],["leaf",50,0.008601305397745567],["leaf",55,-0.0046648092718857915],["leaf",56,0.007974573653651552],["leaf",59,0.0032620872649319526],["leaf",60,-0.016867457779685856],["leaf",61,0.007569432918216639],["leaf",62,0.01881244287488679]],[["split",0,7.0,10,0],["split",1,7.0,1,1],["split",2,3.0,21,1],["split",3,9.0,17,2],["split",4,0.0,21,2],["split",5,2.0,8,2],["split",6,20.0,0,2],["split",7,23.0,29,3],["split",8,25.0,10,3],["split",9,15.0,0,3],["split",10,3.0,27,3],["split",11,7.0,19,3],["split",12,21.0,8,3],["split",13,12.0,26,3],["split",14,25.0,10,3],["split",15,3.0,9,4],["leaf",16,-0.010881602491543923],["leaf",17,-0.0001538400789390161],["split",18,24.0,16,4],["split",19,4.0,14,4],["split",20,23.0,10,4],["split",21,20.0,0,4],["leaf",22,0.012226585124962101],["split",23,23.0,16,4],["split",24,20.0,0,4],["split",25,10.0,2,4],["split",26,20.0,0,4],["split",27,10.0,14,4],["split",28,6.0,29,4],["leaf",29,-0.017666156359575337],["split",30,25.0,21,4],["leaf",31,-0.023753330124734205],["leaf",32,-0.042172136286136946],["leaf",37,-0.032132274795343495],["leaf",38,-0.010725626252001037],["leaf",39,-0.0011079309130409898],["leaf",40,-0.01083048466166605],["leaf",41,-0.009330373530180586],["leaf",42,-0.028727905129467192],["leaf",43,-0.0067834294578115305],["leaf",44,0.012103174194362399],["leaf",47,0.004089328355662057],["leaf",48,-0.012554218080400921],["leaf",49,0.012331571103659763],["leaf",50,-0.0017877882202110141],["leaf",51,-0.02326374192107578],["leaf",52,-0.00612033686773986],["leaf",53,-0.0011359438275923125],["leaf",54,0.017552591319049505],["leaf",55,0.004691008435849008],["leaf",56,0.014430378808892719],["leaf",57,0.017970887969522228],["leaf",58,0.0318265981186823],["leaf",61,-0.0035370298744028758],["leaf",62,0.0072498262784845965]],[["split",0,6.0,6,0],["split",1,4.0,8,1],["split",2,12.0,18,1],["leaf",3,0.010513792098175332],["split",4,7.0,0,2],["split",5,25.0,8,2],["split",6,4.0,24,2],["split",9,10.0,8,3],["split",10,10.0,4,3],["split",11,20.0,0,3],["split",12,4.0,4,3],["split",13,1.0,4,3],["split",14,25.0,10,3],["leaf",19,-0.03850371907102069],["split",20,22.0,14,4],["split",21,24.0,24,4],["split",22,11.0,19,4],["split",23,18.0,0,4],["split",24,21.0,6,4],["split",25,7.0,17,4],["split",26,1.0,3,4],["split",27,3.0,10,4],["split",28,10.0,8,4],["leaf",29,-0.02131616295377709],["leaf",30,-0.007001580605479524],["leaf",41,-0.030718493684503287],["leaf",42,-0.00811447734179009],["leaf",43,-0.025911135742413962],["leaf",44,-0.011143105809403342],["leaf",45,-0.011557046126238873],["leaf",46,-0.0040094736903414875],["leaf",47,-0.0010565293389609887],["leaf",48,-0.01725852989131889],["leaf",49,-0.03736991084623231],["leaf",50,-0.008323978864081791],["leaf",51,0.006339371401131216],["leaf",52,0.017747880872530042],["leaf",53,-0.011983429159800376],["leaf",54,0.0007167188123285466],["leaf",55,0.003589781351016843],["leaf",56,-0.0107728065335756],["leaf",57,0.0031936917628989083],["leaf",58,0.013088661066096479]],[["split",0,8.0,16,0],["split",1,4.0,20,1],["split",2,8.0,27,1],["split",3,13.0,6,2],["split",4,20.0,0,2],["split",5,2.0,10,2],["split",6,23.0,23,2],["split",7,25.0,21,3],["split",8,9.0,3,3],["split",9,10.0,13,3],["split",10,21.0,16,3],["split",11,20.0,0,3],["split",12,9.0,18,3],["split",13,20.0,0,3],["split",14,0.0,21,3],["split",15,1.0,17,4],["split",16,11.0,19,4],["split",17,8.0,13,4],["split",18,0.0,14,4],["split",19,7.0,1,4],["split",20,19.0,0,4],["split",21,25.0,7,4],["split",22,25.0,23,4],["split",23,7.0,26,4],["split",24,21.0,11,4],["split",25,3.0,15,4],["split",26,26.0,29,4],["split",27,6.0,30,4],["leaf",28,0.002432463708087549],["split",29,9.0,13,4],["leaf",30,0.016422901490190458],["leaf",31,-0.008365174492846158],["leaf",32,-0.021542119955092913],["leaf",33,0.0054448033416370605],["leaf",34,-0.006270799233443955],["leaf",35,-0.014256606362853233],["leaf",36,0.0018958327983129473],["leaf",37,-0.0020035838639058403],["leaf",38,0.006019146512890615],["leaf",39,-0.033471375750299404],["leaf",40,-0.014824554730077362],["leaf",41,-0.009511231385780456],["leaf",42,0.005771900882645273],["leaf",43,-0.02746004205165621],["leaf",44,-0.002856228563833415],["leaf",45,0.006993644452917373],["leaf",46,0.035499699761542367],["leaf",47,0.008547675305081897],["leaf",48,0.019610630565984798],["leaf",49,-0.024828920316139043],["leaf",50,0.003932172811818512],["leaf",51,-0.006612832631139968],["leaf",52,0.001559011485614743],["leaf",53,0.007754407224019898],["leaf",54,0.0232693627178351],["leaf",55,0.016465220888610167],["leaf",56,0.02881234020131176],["leaf",59,0.005374436279658694],["leaf",60,-0.008525207754598245]],[["split",0,7.0,11,0],["split",1,3.0,27,1],["split",2,7.0,27,1],["split",3,20.0,0,2],["split",4,5.0,2,2],["split",5,3.0,20,2],["split",6,23.0,29,2],["split",7,10.0,11,3],["split",8,21.0,4,3],["leaf",9,0.01064271326230224],["split",10,25.0,19,3],["split",11,10.0,0,3],["split",12,20.0,0,3],["split",13,7.0,30,3],["leaf",14,-0.005263333974943937],["split",15,7.0,0,4],["split",16,2.0,9,4],["leaf",17,-0.025340258682872412],["split",18,2.0,18,4],["split",21,4.0,13,4],["leaf",22,-0.006898230486378678],["leaf",23,-0.018082198523054684],["split",24,21.0,11,4],["split",25,26.0,16,4],["split",26,25.0,12,4],["split",27,25.0,4,4],["split",28,25.0,20,4],["leaf",31,-0.035654779851569854],["leaf",32,-0.014641214146622237],["leaf",33,0.003479380803090311],["leaf",34,-0.008713314587331678],["leaf",37,-0.0066259682671075465],["leaf",38,0.017712709925194175],["leaf",43,-0.0014248610505201314],["leaf",44,0.012499093731745507],["leaf",49,-0.0040917835013377455],["leaf",50,0.0028184508010540535],["leaf",51,0.0032177669401010443],["leaf",52,0.013234552771538774],["leaf",53,-0.01964605692680861],["leaf",54,0.004108079625315007],["leaf",55,0.0005102487668965333],["leaf",56,0.014286434954482811],["leaf",57,0.013964372186351557],["leaf",58,0.03954319890780172]],[["split",0,7.0,15,0],["split",1,10.0,6,1],["split",2,3.0,20,1],["split",3,20.0,0,2],["split",4,26.0,3,2],["split",5,1.0,17,2],["split",6,20.0,0,2],["split",7,4.0,7,3],["split",8,21.0,15,3],["split",9,5.0,28,3],["split",10,1.0,0,3],["split",11,17.0,0,3],["split",12,26.0,7,3],["split",13,19.0,0,3],["split",14,25.0,20,3],["leaf",15,0.008171412597897129],["split",16,4.0,28,4],["leaf",17,-0.007805182009447077],["leaf",18,0.013000045646882872],["split",19,4.0,20,4],["leaf",20,-0.0028669391606127082],["leaf",21,-0.024097674996194134],["split",22,3.0,24,4],["split",23,7.0,26,4],["split",24,12.0,18,4],["split",25,9.0,9,4],["split",26,7.0,29,4],["split",27,2.0,22,4],["split",28,9.0,14,4],["split",29,3.0,25,4],["leaf",30,0.011115755294903886],["leaf",33,-0.017932063491054515],["leaf",34,-0.03164228931138875],["leaf",39,-0.011397026474368925],["leaf",40,-0.024511424799646082],["leaf",45,-0.004412347412060846],["leaf",46,0.0022662753325735966],["leaf",47,-0.0021622293168849084],["leaf",48,0.006573136380065014],["leaf",49,0.0057863231308022305],["leaf",50,-0.023529753283907866],["leaf",51,-0.007302127690007764],["leaf",52,0.003721589607997348],["leaf",53,0.006159672781570569],["leaf",54,0.022241395910610726],["leaf",55,0.017418954852232868],["leaf",56,0.0042865187364756205],["leaf",57,-0.011531668864670042],["leaf",58,0.0043849737916227954],["leaf",59,-0.0010311786689452867],["leaf",60,-0.019243400794682224]],[["split",0,7.0,13,0],["split",1,4.0,18,1],["split",2,7.0,29,1],["split",3,21.0,1,2],["split",4,7.0,0,2],["split",5,21.0,15,2],["split",6,2.0,4,2],["leaf",7,-0.024316455036190083],["split",8,10.0,12,3],["split",9,5.0,27,3],["split",10,21.0,16,3],["split",11,20.0,0,3],["split",12,9.0,29,3],["split",13,7.0,30,3],["split",14,23.0,23,3],["split",17,15.0,0,4],["split",18,23.0,16,4],["leaf",19,-0.03658593221386611],["split",20,24.0,12,4],["split",21,24.0,9,4],["split",22,14.0,0,4],["split",23,23.0,21,4],["split",24,23.0,13,4],["split",25,8.0,26,4],["split",26,10.0,19,4],["leaf",27,0.019294387083296543],["leaf",28,0.04012737201611554],["split",29,25.0,20,4],["split",30,21.0,19,4],["leaf",35,-0.005372984480088991],["leaf",36,-0.021342915630166544],["leaf",37,0.0051456029397149005],["leaf",38,-0.0036523740623742402],["leaf",41,-0.0358402852545901],["leaf",42,-0.008971244041455936],["leaf",43,-0.02020107681679538],["leaf",44,-0.010548156754061405],["leaf",45,-0.0018183277411196043],["leaf",46,-0.01771189012472549],["leaf",47,0.00597213629687722],["leaf",48,-0.005329077985459145],["leaf",49,-0.02089860057094219],["leaf",50,0.0016877845802277488],["leaf",51,0.004300822066084513],["leaf",52,0.011891935236404779],["leaf",53,0.011620202063753087],["leaf",54,0.02870076984303766],["leaf",59,0.008575257822026028],["leaf",60,0.033539685820121894],["leaf",61,-0.006690075103590339],["leaf",62,0.006889389892275278]],[["split",0,6.0,17,0],["split",1,10.0,2,1],["split",2,7.0,30,1],["split",3,3.0,23,2],["split",4,7.0,6,2],["split",5,1.0,4,2],["split",6,2.0,6,2],["split",7,25.0,7,3],["leaf",8,0.002098579701883868],["split",9,15.0,0,3],["split",10,26.0,1,3],["split",11,2.0,18,3],["split",12,22.0,21,3],["leaf",13,0.033513585395368035],["split",14,24.0,12,3],["leaf",15,-0.02700741430820077],["split",16,21.0,9,4],["split",19,24.0,17,4],["split",20,13.0,3,4],["split",21,1.0,16,4],["split",22,23.0,14,4],["split",23,11.0,14,4],["leaf",24,0.003899629432231359],["split",25,3.0,18,4],["split",26,3.0,20,4],["leaf",29,-0.0038920156592551105],["leaf",30,0.017315437140040038],["leaf",33,-0.02368395303302054],["leaf",34,-0.006147803828546291],["leaf",39,-0.008323247327922521],["leaf",40,-0.0003042742210110897],["leaf",41,-0.00780895420320734],["leaf",42,-0.02340751294679786],["leaf",43,-0.0034131725684937745],["leaf",44,-0.022261703749664845],["leaf",45,0.0043948021645990945],["leaf",46,-0.002830045712843115],["leaf",47,-0.022830582865693696],["leaf",48,-0.005620681406553471],["leaf",51,0.004516082113506168],["leaf",52,0.010710699731280086],["leaf",53,-0.00412573409185052],["leaf",54,0.004782162189237275]],[["split",0,7.0,12,0],["split",1,4.0,26,1],["split",2,2.0,7,1],["split",3,4.0,2,2],["split",4,2.0,24,2],["split",5,7.0,27,2],["split",6,3.0,14,2],["split",7,8.0,13,3],["split",8,22.0,24,3],["split",9,1.0,25,3],["split",10,20.0,0,3],["split",11,20.0,0,3],["split",12,0.0,6,3],["split",13,21.0,0,3],["split",14,3.0,28,3],["leaf",15,0.01038452021571608],["leaf",16,-0.0012525254177252134],["split",17,22.0,17,4],["split",18,12.0,15,4],["split",19,12.0,10,4],["leaf",20,-0.017736871640049764],["split",21,1.0,1,4],["leaf",22,0.0006673269431860044],["split",23,3.0,29,4],["split",24,24.0,13,4],["leaf",25,0.00758393897616636],["split",26,25.0,13,4],["leaf",27,-0.02375904939306507],["split",28,20.0,0,4],["split",29,24.0,3,4],["split",30,9.0,16,4],["leaf",35,-0.006045823162279472],["leaf",36,0.0022128581120438745],["leaf",37,-0.014388493221994023],["leaf",38,0.0006075795983557146],["leaf",39,-0.00017252371087722772],["leaf",40,-0.016439936803881623],["leaf",43,-0.03164840679978735],["leaf",44,-0.017232040246858434],["leaf",47,0.0063993670685119535],["leaf",48,0.022949249903698465],["leaf",49,-0.025322070149070283],["leaf",50,-0.0025095208619222904],["leaf",53,0.015266007090297948],["leaf",54,0.026042542429845838],["leaf",57,-0.0031240994296015696],["leaf",58,0.006981863731777367],["leaf",59,-0.008191981586808315],["leaf",60,0.00463354195791118],["leaf",61,0.002880266401752632],["leaf",62,0.015501374564185565]],[["split",0,7.0,4,0],["split",1,10.0,2,1],["split",2,8.0,19,1],["split",3,26.0,14,2],["split",4,3.0,5,2],["split",5,9.0,4,2],["split",6,19.0,0,2],["leaf",7,-0.012505505859663849],["leaf",8,-0.031179053588403675],["split",9,12.0,3,3],["split",10,20.0,0,3],["split",11,1.0,5,3],["split",12,13.0,3,3],["split",13,3.0,25,3],["split",14,26.0,9,3],["split",19,4.0,27,4],["split",20,2.0,21,4],["split",21,2.0,11,4],["leaf",22,0.00728348868771287],["split",23,26.0,14,4],["split",24,21.0,10,4],["split",25,1.0,12,4],["split",26,24.0,16,4],["split",27,2.0,4,4],["split",28,20.0,0,4],["leaf",29,0.005796235749794768],["split",30,9.0,15,4],["leaf",39,0.014948064262599833],["leaf",40,-0.00196442293514081],["leaf",41,-0.0019072965140317058],["leaf",42,-0.02160908909826025],["leaf",43,0.0037079754574973036],["leaf",44,-0.018184202328363906],["leaf",47,-0.02031040776485524],["leaf",48,-0.006994925169908534],["leaf",49,-0.010958817578469029],["leaf",50,0.001713117178716689],["leaf",51,0.019294888330495297],["leaf",52,0.0007946769519240009],["leaf",53,-0.0023155112698965015],["leaf",54,0.004100910242052575],["leaf",55,0.012634259948490129],["leaf",56,0.0035278512432958195],["leaf",57,0.01740974098609573],["leaf",58,-0.0025359900048240237],["leaf",61,-0.013196374234563551],["leaf",62,0.0022731306811986343]],[["split",0,8.0,18,0],["split",1,4.0,25,1],["split",2,12.0,20,1],["split",3,10.0,4,2],["split",4,2.0,30,2],["split",5,26.0,1,2],["split",6,19.0,0,2],["split",7,7.0,14,3],["split",8,14.0,0,3],["split",9,19.0,0,3],["split",10,9.0,9,3],["leaf",11,-0.016256321672491962],["split",12,0.0,28,3],["split",13,4.0,6,3],["split",14,24.0,19,3],["split",15,3.0,24,4],["split",16,26.0,15,4],["split",17,24.0,19,4],["split",18,7.0,5,4],["split",19,9.0,28,4],["split",20,10.0,15,4],["leaf",21,-0.03624751318219533],["leaf",22,-0.020632214774361807],["split",25,7.0,13,4],["leaf",26,0.015758301069133094],["split",27,24.0,22,4],["split",28,22.0,29,4],["split",29,26.0,13,4],["leaf",30,0.008256317322174364],["leaf",31,-0.016857470208915384],["leaf",32,-0.00034205852871416556],["leaf",33,0.008159605376379202],["leaf",34,-0.011100484177980841],["leaf",35,-0.002094513697408514],["leaf",36,0.005655099537064402],["leaf",37,-0.020152671604712166],["leaf",38,-0.004384137543708122],["leaf",39,-0.011069895965281774],["leaf",40,0.007461301881118779],["leaf",41,-0.005176986937194963],["leaf",42,0.013881024969511931],["leaf",51,-0.014408395702484124],["leaf",52,0.0019576944462894904],["leaf",55,0.010468053826162727],["leaf",56,0.021254157604179043],["leaf",57,0.0077137401480406815],["leaf",58,-0.010810795433259598],["leaf",59,0.004392890742751891],["leaf",60,-0.01213042667751527]],[["split",0,7.0,14,0],["split",1,2.0,13,1],["split",2,2.0,6,1],["split",3,20.0,0,2],["split",4,20.0,0,2],["split",5,15.0,0,2],["split",6,10.0,8,2],["split",7,4.0,25,3],["split",8,21.0,17,3],["split",9,3.0,20,3],["split",10,23.0,25,3],["split",11,7.0,29,3],["split",12,7.0,25,3],["split",13,4.0,25,3],["split",14,21.0,3,3],["split",15,26.0,8,4],["split",16,3.0,5,4],["leaf",17,-0.029791005583101455],["leaf",18,-0.002571824679654515],["split",19,5.0,28,4],["split",20,22.0,14,4],["split",21,21.0,6,4],["leaf",22,0.02141305942184009],["split",23,25.0,5,4],["split",24,11.0,17,4],["leaf",25,0.012433223815476792],["leaf",26,0.03004217308243965],["split",27,2.0,25,4],["leaf",28,-0.019461608639969437],["split",29,8.0,14,4],["split",30,23.0,26,4],["leaf",31,-0.012113808931513514],["leaf",32,0.004774133361067612],["leaf",33,0.026552328788163593],["leaf",34,0.006533728848352151],["leaf",39,-0.015195073437968694],["leaf",40,-0.005353928389919034],["leaf",41,0.003156712733154792],["leaf",42,-0.006180056070259904],["leaf",43,-0.02246091806983356],["leaf",44,0.006964297178582178],["leaf",47,-0.0074563963438482785],["leaf",48,0.007473688433597088],["leaf",49,0.02866343628080817],["leaf",50,0.006458975509149478],["leaf",55,-0.0042305429656348155],["leaf",56,0.011591900280622246],["leaf",59,0.007157995959482702],["leaf",60,-0.01030037917714814],["leaf",61,0.005826755240836067],["leaf",62,-0.005567308988560466]],[["split",0,7.0,17,0],["split",1,10.0,16,1],["split",2,22.0,23,1],["split",3,20.0,0,2],["split",4,17.0,0,2],["split",5,21.0,8,2],["split",6,6.0,29,2],["split",7,23.0,22,3],["split",8,21.0,13,3],["split",9,21.0,1,3],["split",10,1.0,16,3],["split",11,20.0,0,3],["split",12,26.0,5,3],["split",13,20.0,0,3],["split",14,4.0,4,3],["split",15,26.0,1,4],["split",16,17.0,0,4],["split",17,23.0,15,4],["split",18,11.0,21,4],["leaf",19,-0.01564205984135573],["split",20,11.0,21,4],["split",21,5.0,15,4],["split",22,26.0,18,4],["split",23,0.0,20,4],["leaf",24,-0.017056325179555416],["split",25,9.0,5,4],["split",26,1.0,2,4],["split",27,0.0,28,4],["leaf",28,-0.017663889569877807],["leaf",29,0.01924953238738681],["leaf",30,0.001151417058166238],["leaf",31,-0.02151445231445108],["leaf",32,-0.0028958702805510924],["leaf",33,-0.008402408918376725],["leaf",34,-0.0340503790861662],["leaf",35,-0.019481067014630343],["leaf",36,-0.00271895849791598],["leaf",37,0.007707232312497093],["leaf",38,0.029285886440445188],["leaf",41,-0.0023753430749627455],["leaf",42,0.00544007453265457],["leaf",43,0.01363889154346783],["leaf",44,-0.0022670739084094563],["leaf",45,0.011491530917236448],["leaf",46,0.029590792710172734],["leaf",47,0.006356353722841387],["leaf",48,-0.008555647542152848],["leaf",51,0.015836277309942956],["leaf",52,-0.0020403471606567566],["leaf",53,-0.005117045430948943],["leaf",54,0.01136163897029121],["leaf",55,-6.300347935333542e-05],["leaf",56,-0.013229369236382585]],[["split",0,1.0,11,0],["split",1,17.0,0,1],["split",2,26.0,22,1],["split",3,1.0,0,2],["split",4,23.0,27,2],["split",5,25.0,4,2],["split",6,19.0,0,2],["split",7,2.0,28,3],["split",8,3.0,2,3],["split",9,10.0,10,3],["leaf",10,-0.030538317386194357],["split",11,21.0,1,3],["split",12,7.0,28,3],["split",13,12.0,21,3],["split",14,5.0,14,3],["split",15,3.0,8,4],["leaf",16,-0.02089977628280573],["split",17,21.0,13,4],["split",18,7.0,5,4],["split",19,22.0,13,4],["split",20,2.0,19,4],["leaf",23,-0.021859743948159985],["split",24,26.0,1,4],["split",25,26.0,0,4],["split",26,21.0,10,4],["split",27,10.0,19,4],["split",28,22.0,20,4],["split",29,10.0,20,4],["leaf",30,0.0025588285580269514],["leaf",31,0.0015873996696772173],["leaf",32,-0.018627276388947622],["leaf",35,-0.00483681808814477],["leaf",36,0.016095854193294474],["leaf",37,-0.009408704024955453],["leaf",38,-0.0010839826366728128],["leaf",39,-0.027229932099823356],["leaf",40,-0.007996743011896033],["leaf",41,0.006979215427928494],["leaf",42,-0.013080420975420361],["leaf",49,-0.01682822614429719],["leaf",50,-0.003977775663537961],["leaf",51,-0.015840357215411036],["leaf",52,-0.0004609434079467386],["leaf",53,0.0008837710049854517],["leaf",54,0.015456289165691391],["leaf",55,-7.667772470524869e-05],["leaf",56,0.01106789290115545],["leaf",57,0.016342178732681286],["leaf",58,0.008788063999695407],["leaf",59,-0.003964067847021207],["leaf",60,-0.012826842293124035]],[["split",0,7.0,15,0],["split",1,23.0,23,1],["split",2,7.0,27,1],["split",3,1.0,2,2],["split",4,17.0,0,2],["split",5,25.0,4,2],["split",6,22.0,18,2],["split",7,0.0,16,3],["split",8,17.0,0,3],["split",9,20.0,0,3],["split",10,2.0,26,3],["split",11,20.0,0,3],["split",12,22.0,25,3],["split",13,11.0,12,3],["split",14,12.0,29,3],["leaf",15,-0.01726714657491243],["leaf",16,-0.005150708752151284],["split",17,24.0,7,4],["split",18,11.0,10,4],["split",19,3.0,22,4],["split",20,25.0,14,4],["leaf",21,-0.013368306492934256],["leaf",22,-0.0369815654793784],["split",23,5.0,23,4],["leaf",24,-0.021252288260538876],["split",25,9.0,8,4],["split",26,9.0,24,4],["split",27,2.0,14,4],["split",28,25.0,21,4],["split",29,26.0,7,4],["leaf",30,0.013264990275983863],["leaf",35,-0.007166453325679269],["leaf",36,-0.0005453425047240159],["leaf",37,0.016781827167447454],["leaf",38,0.0015608751981102257],["leaf",39,-0.013844251062189486],["leaf",40,0.002174000156556039],["leaf",41,0.00014723049758072137],["leaf",42,0.02551473469309165],["leaf",47,-0.0009199225710922377],["leaf",48,-0.018900128164993736],["leaf",51,-0.000912039616561874],["leaf",52,0.006725434829205734],["leaf",53,-0.00044809948207194853],["leaf",54,-0.011102781938742833],["leaf",55,0.011423861926771833],["leaf",56,-0.0019930557093663214],["leaf",57,0.023254615204897435],["leaf",58,0.01118154377314546],["leaf",59,-0.010612847871586195],["leaf",60,0.006700226840127818]],[["split",0,7.0,1,0],["split",1,2.0,27,1],["split",2,7.0,16,1],["split",3,3.0,1,2],["split",4,24.0,18,2],["split",5,26.0,1,2],["split",6,0.0,9,2],["leaf",7,0.003556062671292372],["split",8,22.0,7,3],["leaf",9,-0.03179841787593476],["leaf",10,-0.00959513323354507],["split",11,6.0,2,3],["split",12,23.0,29,3],["split",13,17.0,0,3],["split",14,22.0,30,3],["leaf",17,-0.0029566510058793337],["split",18,3.0,11,4],["leaf",23,0.0017550446635851147],["split",24,23.0,10,4],["split",25,21.0,7,4],["split",26,1.0,15,4],["split",27,21.0,2,4],["split",28,6.0,24,4],["split",29,13.0,18,4],["leaf",30,-0.011508564153327776],["leaf",37,-0.02540381989289061],["leaf",38,-0.008262504741883265],["leaf",49,-0.012443013598559476],["leaf",50,-0.031132440447446535],["leaf",51,-0.006572111735873661],["leaf",52,0.0010228487774305566],["leaf",53,-0.008238066079799627],["leaf",54,-0.024938722488433835],["leaf",55,-0.009881158953033129],["leaf",56,-0.0009184355169676034],["leaf",57,0.00048689181285452326],["leaf",58,0.01968610478126632],["leaf",59,0.0007722457054046296],["leaf",60,0.008456859949549261]],[["split",0,4.0,19,0],["split",1,8.0,19,1],["split",2,7.0,4,1],["split",3,26.0,1,2],["split",4,18.0,0,2],["split",5,11.0,29,2],["split",6,24.0,11,2],["split",7,7.0,16,3],["split",8,24.0,19,3],["split",9,22.0,9,3],["split",10,1.0,24,3],["split",11,24.0,10,3],["leaf",12,-0.028440833676194155],["split",13,26.0,21,3],["split",14,1.0,28,3],["leaf",15,-0.022320506926328386],["leaf",16,-0.003557213087701335],["split",17,1.0,8,4],["split",18,22.0,19,4],["split",19,2.0,10,4],["split",20,7.0,30,4],["split",21,24.0,15,4],["leaf",22,-0.010760260972793116],["split",23,1.0,21,4],["split",24,19.0,0,4],["split",27,19.0,0,4],["split",28,21.0,15,4],["split",29,13.0,3,4],["leaf",30,-0.01208708142451906],["leaf",35,-0.0064417598366770905],["leaf",36,0.0012588892359267755],["leaf",37,0.008574210650142485],["leaf",38,-2.7937182126617034e-05],["leaf",39,0.017218633642113144],["leaf",40,0.008183234793788439],["leaf",41,0.004569100310868294],["leaf",42,0.01793661674581788],["leaf",43,-0.0046558151942432706],["leaf",44,0.00898221187342997],["leaf",47,-0.013629433673259818],["leaf",48,-0.025228557994164593],["leaf",49,-0.008691040345857557],["leaf",50,0.006702133473872139],["leaf",55,-0.011156744380901844],["leaf",56,0.0008407361843812845],["leaf",57,-0.003949304830912757],["leaf",58,0.010652585781547702],["leaf",59,0.010881326652829313],["leaf",60,-0.00016445499391226965]],[["split",0,7.0,12,0],["split",1,26.0,26,1],["split",2,21.0,3,1],["split",3,0.0,26,2],["split",4,16.0,0,2],["split",5,20.0,0,2],["split",6,8.0,30,2],["split",7,4.0,28,3],["split",8,11.0,21,3],["split",9,1.0,4,3],["leaf",10,0.012676037939157812],["split",11,2.0,21,3],["leaf",12,-0.021817743447190598],["split",13,23.0,30,3],["split",14,9.0,22,3],["split",15,21.0,1,4],["split",16,26.0,8,4],["split",17,2.0,22,4],["split",18,6.0,4,4],["leaf",19,-0.013183779448462482],["split",20,1.0,15,4],["split",23,21.0,0,4],["leaf",24,-0.011033339639483213],["split",27,22.0,8,4],["leaf",28,-0.017157373405452828],["split",29,10.0,20,4],["leaf",30,0.025938513657159767],["leaf",31,-0.016647959251132348],["leaf",32,-0.005259728319887801],["leaf",33,-0.0063745026462961015],["leaf",34,-0.0218820813139961],["leaf",35,-0.012804231670660162],["leaf",36,0.005542458506691841],["leaf",37,0.020203249258919146],["leaf",38,0.0016366752981454514],["leaf",41,0.005906668084908662],["leaf",42,-0.000929016575943939],["leaf",47,-0.0063334718139931015],["leaf",48,0.005531616956892526],["leaf",55,0.008330710511772815],["leaf",56,0.0031366293969357518],["leaf",59,0.012233565761782887],["leaf",60,0.0029894367505459624]],[["split",0,7.0,18,0],["split",1,3.0,28,1],["split",2,10.0,11,1],["split",3,21.0,0,2],["split",4,20.0,0,2],["split",5,26.0,7,2],["split",6,3.0,6,2],["split",7,11.0,14,3],["split",8,10.0,15,3],["split",9,11.0,14,3],["leaf",10,-0.013089916017932279],["split",11,23.0,12,3],["split",12,23.0,28,3],["split",13,7.0,30,3],["split",14,21.0,26,3],["leaf",15,-0.006269557894469785],["leaf",16,-0.026011021191711312],["split",17,24.0,23,4],["split",18,26.0,29,4],["split",19,23.0,10,4],["split",20,10.0,17,4],["split",23,11.0,15,4],["split",24,25.0,13,4],["split",25,21.0,16,4],["leaf",26,-0.009297778404118478],["split",27,23.0,19,4],["leaf",28,0.016103244337089397],["split",29,10.0,29,4],["split",30,1.0,22,4],["leaf",35,-0.00730316146661037],["leaf",36,0.0008971192910311864],["leaf",37,-0.0012612648386763254],["leaf",38,0.014427039745099332],["leaf",39,0.008651629590157923],["leaf",40,0.0227806607933097],["leaf",41,0.014379911423056328],["leaf",42,0.0007579676325070041],["leaf",47,-0.008575720681833397],["leaf",48,-0.022807742630858868],["leaf",49,-0.012241107576323913],["leaf",50,0.009710424530180993],["leaf",51,7.973474902200437e-05],["leaf",52,0.007845001022917603],["leaf",55,0.0016490356659176766],["leaf",56,-0.012864314120572175],["leaf",59,0.005991270173382402],["leaf",60,-0.004985846214690197],["leaf",61,0.00961488297767597],["leaf",62,0.02368878273442864]],[["split",0,7.0,15,0],["split",1,25.0,4,1],["split",2,9.0,28,1],["split",3,7.0,5,2],["split",4,16.0,0,2],["split",5,22.0,29,2],["split",6,12.0,29,2],["split",7,13.0,2,3],["split",8,12.0,15,3],["split",9,2.0,29,3],["split",10,9.0,26,3],["split",11,7.0,30,3],["split",12,24.0,8,3],["split",13,23.0,19,3],["leaf",14,0.025975611612933727],["split",15,1.0,5,4],["split",16,26.0,12,4],["split",17,12.0,8,4],["leaf",18,-0.01265360164392444],["split",19,21.0,3,4],["split",20,4.0,27,4],["split",21,6.0,12,4],["leaf",22,0.005408562633756035],["split",23,23.0,18,4],["split",24,12.0,22,4],["leaf",25,-0.022432913509545713],["split",26,11.0,15,4],["split",27,1.0,11,4],["leaf",28,0.021261646920644528],["leaf",31,0.002996702669252195],["leaf",32,-0.01446331645012532],["leaf",33,-0.031437630524300104],["leaf",34,-0.013049785745596607],["leaf",35,-0.007305042524972012],["leaf",36,0.008664765929721013],["leaf",39,-0.005770767106403073],["leaf",40,0.0018197212429692782],["leaf",41,-0.0024147647975570163],["leaf",42,-0.017363527333960363],["leaf",43,-0.014543550594933289],["leaf",44,0.0010204267053677389],["leaf",47,0.0049311298970715145],["leaf",48,-0.00152574374746319],["leaf",49,-0.0016954275475852647],["leaf",50,0.019002762253082842],["leaf",53,-0.012166830445080868],["leaf",54,0.010034278972196697],["leaf",55,0.014979050636567794],["leaf",56,-0.00028587109305095327]],[["split",0,7.0,20,0],["split",1,10.0,14,1],["split",2,2.0,8,1],["split",3,17.0,0,2],["split",4,12.0,0,2],["split",5,20.0,0,2],["split",6,20.0,0,2],["split",7,15.0,0,3],["split",8,23.0,21,3],["leaf",9,0.014450420773653733],["split",10,4.0,22,3],["split",11,25.0,28,3],["split",12,24.0,14,3],["split",13,23.0,19,3],["split",14,25.0,18,3],["split",15,20.0,0,4],["split",16,2.0,27,4],["split",17,2.0,22,4],["split",18,12.0,9,4],["split",21,17.0,0,4],["split",22,22.0,16,4],["split",23,22.0,19,4],["leaf",24,0.021363482646260572],["leaf",25,-0.013939905705846375],["leaf",26,0.006895423986589783],["split",27,10.0,15,4],["split",28,9.0,23,4],["split",29,11.0,16,4],["leaf",30,0.03196219639067949],["leaf",31,-0.004164078422409418],["leaf",32,0.007509128677514991],["leaf",33,-0.007149156436703075],["leaf",34,-0.026779680865082128],["leaf",35,0.004216148918676209],["leaf",36,-0.014059340329031042],["leaf",37,-0.020567574590267507],["leaf",38,-0.03880796051067724],["leaf",43,0.00189984044267748],["leaf",44,0.012810718970838506],["leaf",45,-0.008027623775052006],["leaf",46,0.002433838128676729],["leaf",47,0.011649694365949889],["leaf",48,0.0034601561713963405],["leaf",55,-0.0006052037538218881],["leaf",56,0.00988335572780034],["leaf",57,-0.01133274961608211],["leaf",58,0.004516382866647697],["leaf",59,-0.013009090913989746],["leaf",60,0.009450755961931637]],[["split",0,4.0,14,0],["split",1,7.0,30,1],["split",2,10.0,4,1],["split",3,1.0,7,2],["split",4,1.0,24,2],["split",5,20.0,0,2],["split",6,19.0,0,2],["split",7,5.0,18,3],["split",8,20.0,0,3],["leaf",9,0.00764092415887796],["leaf",10,0.02825611159220239],["split",11,13.0,21,3],["leaf",12,0.007204639749241859],["split",13,11.0,23,3],["split",14,8.0,2,3],["split",15,2.0,17,4],["leaf",16,0.010036521019335496],["split",17,9.0,30,4],["split",18,21.0,10,4],["split",23,3.0,20,4],["leaf",24,0.0029700281503025707],["split",27,24.0,0,4],["split",28,20.0,0,4],["leaf",29,0.01828103036923449],["split",30,21.0,19,4],["leaf",31,-0.010881353652206494],["leaf",32,-0.0024440014180704068],["leaf",35,0.004215904642249862],["leaf",36,0.02037497489744853],["leaf",37,-0.02094313331850167],["leaf",38,0.0011262593182316894],["leaf",47,-0.01702217048045912],["leaf",48,-0.002305948402726291],["leaf",55,-0.020486059325664517],["leaf",56,-0.003228217276195638],["leaf",57,1.4703785787487516e-05],["leaf",58,0.012245737220678342],["leaf",61,-0.0012841460098068254],["leaf",62,0.009103467015293491]],[["split",0,4.0,23,0],["split",1,21.0,10,1],["split",2,25.0,30,1],["split",3,20.0,0,2],["split",4,20.0,0,2],["split",5,20.0,0,2],["leaf",6,0.016285565135284806],["split",7,23.0,24,3],["split",8,3.0,12,3],["split",9,4.0,2,3],["split",10,2.0,20,3],["split",11,2.0,26,3],["split",12,25.0,19,3],["split",15,2.0,9,4],["split",16,3.0,23,4],["leaf",17,0.0007735301431877474],["split",18,25.0,9,4],["split",19,17.0,0,4],["split",20,23.0,21,4],["split",21,23.0,20,4],["split",22,23.0,15,4],["split",23,13.0,3,4],["split",24,21.0,8,4],["split",25,23.0,20,4],["leaf",26,0.020854284629624756],["leaf",31,0.008320699985666744],["leaf",32,0.0008202926941055483],["leaf",33,-0.010837167176131831],["leaf",34,0.006893129327690173],["leaf",37,-0.027295906974583832],["leaf",38,-0.008450700237893457],["leaf",39,0.0069640444147106755],["leaf",40,0.02538697859362013],["leaf",41,0.003969117636226758],["leaf",42,-0.0037710401655620053],["leaf",43,-0.002099967950957529],["leaf",44,0.014431131617524504],["leaf",45,0.016344917676409894],["leaf",46,0.03977392029816642],["leaf",47,0.0022243296824104885],["leaf",48,-0.006132271850718315],["leaf",49,-5.36669942980713e-05],["leaf",50,-0.0159542998864597],["leaf",51,-0.01766970252654723],["leaf",52,0.011888442642094859]],[["split",0,7.0,10,0],["split",1,24.0,17,1],["split",2,26.0,20,1],["split",3,4.0,28,2],["split",4,15.0,0,2],["split",5,7.0,22,2],["split",6,21.0,3,2],["split",7,21.0,1,3],["split",8,7.0,1,3],["split",9,4.0,27,3],["split",10,2.0,21,3],["split",11,13.0,8,3],["split",12,25.0,8,3],["split",13,6.0,22,3],["split",14,19.0,0,3],["leaf",15,-0.01374249984561958],["split",16,15.0,0,4],["split",17,26.0,11,4],["split",18,6.0,4,4],["split",19,26.0,26,4],["split",20,22.0,24,4],["leaf",21,0.0018118626612481233],["leaf",22,-0.018274519313791517],["split",23,0.0,21,4],["split",24,26.0,9,4],["split",25,14.0,0,4],["split",26,22.0,17,4],["leaf",27,-0.012791438651881247],["leaf",28,0.0015615328628195329],["split",29,12.0,29,4],["split",30,0.0,22,4],["leaf",33,-0.0016801116508393455],["leaf",34,-0.010809448672635872],["leaf",35,-0.011844516276223033],["leaf",36,-0.03680633645359656],["leaf",37,-0.015848348315275495],["leaf",38,-0.002850261707602219],["leaf",39,0.0026937870049088273],["leaf",40,0.014165492124747636],["leaf",41,-0.006390099140952318],["leaf",42,0.0067575377054220395],["leaf",47,0.007881513124860505],["leaf",48,-0.004726375716910095],["leaf",49,-0.009193409690919377],["leaf",50,-0.0010829148264869582],["leaf",51,-0.00513767936564755],["leaf",52,0.015408765202422106],["leaf",53,0.009786351229801956],["leaf",54,-0.0003400250162236931],["leaf",59,0.00612922779471512],["leaf",60,0.014393243832991962],["leaf",61,0.0035468280812209966],["leaf",62,-0.010663455688692381]],[["split",0,3.0,21,0],["split",1,1.0,30,1],["split",2,17.0,0,1],["split",3,7.0,12,2],["split",4,24.0,12,2],["split",5,1.0,28,2],["split",6,5.0,14,2],["split",7,25.0,16,3],["split",8,22.0,24,3],["leaf",9,-0.023824776527322195],["split",10,9.0,11,3],["split",11,7.0,8,3],["leaf",12,0.01270573706297096],["split",13,25.0,13,3],["leaf",14,-0.0027886739374105318],["split",15,26.0,1,4],["split",16,20.0,0,4],["split",17,13.0,24,4],["split",18,26.0,15,4],["leaf",21,-0.013177454263780117],["leaf",22,0.003991801060269735],["split",23,3.0,29,4],["split",24,14.0,0,4],["leaf",27,0.02797156311971497],["split",28,26.0,22,4],["leaf",31,-0.02052944004638206],["leaf",32,-0.005122175844830667],["leaf",33,-0.0033626530766446084],["leaf",34,0.014563891466306952],["leaf",35,0.002933976314382063],["leaf",36,-0.0031258658424548336],["leaf",37,-0.009300944007143588],["leaf",38,0.0009533118092131866],["leaf",47,-0.0070169271945980044],["leaf",48,0.006783924554394725],["leaf",49,0.0008578240200428364],["leaf",50,0.010210840894628724],["leaf",57,0.00397410099262644],["leaf",58,0.014350432725699724]],[["split",0,7.0,8,0],["split",1,2.0,30,1],["split",2,12.0,11,1],["split",3,9.0,7,2],["split",4,21.0,14,2],["split",5,6.0,18,2],["split",6,19.0,0,2],["split",7,8.0,1,3],["split",8,1.0,28,3],["leaf",9,-0.006518708285293152],["leaf",10,-0.02124483797811946],["split",11,16.0,0,3],["split",12,13.0,21,3],["split",13,9.0,27,3],["split",14,7.0,22,3],["leaf",15,0.015892366376005],["split",16,26.0,17,4],["split",17,10.0,23,4],["split",18,6.0,3,4],["split",23,0.0,12,4],["split",24,7.0,12,4],["split",25,9.0,15,4],["split",26,24.0,14,4],["split",27,7.0,30,4],["split",28,20.0,0,4],["split",29,8.0,12,4],["split",30,26.0,6,4],["leaf",33,-0.007971061089136693],["leaf",34,0.0021468736942420233],["leaf",35,-0.006812670856743382],["leaf",36,0.0016069281217752055],["leaf",37,-0.02841475113824966],["leaf",38,-0.003654709526262002],["leaf",47,0.004950110540114716],["leaf",48,-0.0027300046383236295],["leaf",49,-0.021244313622001708],["leaf",50,-0.0030373668015536708],["leaf",51,-0.030287805590436777],["leaf",52,-0.006187648080085116],["leaf",53,0.003205502016448666],["leaf",54,-0.011034294067056203],["leaf",55,0.0028334800880476774],["leaf",56,0.013840613786065693],["leaf",57,0.015106591661781228],["leaf",58,-0.003907834125132358],["leaf",59,0.003338172216988376],["leaf",60,-0.008313746440691109],["leaf",61,-0.008452181177223012],["leaf",62,0.006368861632157402]],[["split",0,4.0,19,0],["split",1,25.0,29,1],["split",2,7.0,0,1],["split",3,26.0,0,2],["split",4,24.0,22,2],["split",5,25.0,23,2],["split",6,24.0,29,2],["leaf",7,-0.012481476606412719],["split",8,20.0,0,3],["split",9,4.0,10,3],["leaf",10,0.020683581135351892],["split",11,23.0,25,3],["leaf",12,-0.02650263104435404],["split",13,13.0,1,3],["split",14,5.0,27,3],["split",17,3.0,27,4],["split",18,3.0,27,4],["leaf",19,0.01175333143524356],["leaf",20,-0.001888149836429469],["leaf",23,-0.019703781838711968],["leaf",24,-0.004261396205352894],["split",27,5.0,26,4],["split",28,23.0,24,4],["split",29,10.0,14,4],["leaf",30,0.017053212765390065],["leaf",35,0.0018205693676200453],["leaf",36,0.007100633618957498],["leaf",37,0.0007763976381749046],["leaf",38,-0.020173638775858784],["leaf",55,-0.007319217380489251],["leaf",56,0.0104109494000169],["leaf",57,-0.0023359539526025715],["leaf",58,-0.010043858785082498],["leaf",59,-0.005656338158143875],["leaf",60,0.010880690683784887]],[["split",0,4.0,14,0],["split",1,15.0,0,1],["split",2,10.0,2,1],["split",3,21.0,30,2],["split",4,7.0,11,2],["split",5,25.0,25,2],["split",6,8.0,7,2],["split",7,8.0,10,3],["leaf",8,-0.018765760887249603],["leaf",9,-0.005333447573757074],["split",10,9.0,18,3],["split",11,0.0,16,3],["leaf",12,0.002046152723891086],["split",13,12.0,7,3],["split",14,21.0,16,3],["split",15,18.0,0,4],["split",16,2.0,2,4],["split",21,6.0,22,4],["split",22,25.0,19,4],["split",23,0.0,4,4],["split",24,3.0,11,4],["split",27,23.0,13,4],["split",28,17.0,0,4],["split",29,22.0,29,4],["split",30,1.0,6,4],["leaf",31,0.006209401337175899],["leaf",32,0.019542644088734615],["leaf",33,0.006991504889146302],["leaf",34,-0.000873913451282753],["leaf",43,0.014010705285426518],["leaf",44,0.0015078690573002134],["leaf",45,0.022977615405950315],["leaf",46,0.00733870971529896],["leaf",47,-0.018733407414441135],["leaf",48,-0.005184842547816382],["leaf",49,-0.030773721972347883],["leaf",50,-0.01410882198302974],["leaf",55,-0.00858619365511286],["leaf",56,0.001341633087802892],["leaf",57,-0.007112122994195309],["leaf",58,-0.030520423018817026],["leaf",59,-0.0007048133045770231],["leaf",60,-0.017412990268355386],["leaf",61,-0.004016265763949354],["leaf",62,0.005422124274911686]],[["split",0,4.0,3,0],["split",1,2.0,1,1],["split",2,24.0,3,1],["split",3,11.0,13,2],["split",4,1.0,9,2],["split",5,11.0,20,2],["split",6,25.0,30,2],["leaf",7,0.006246780112831266],["leaf",8,0.02072162515188588],["split",9,25.0,20,3],["split",10,20.0,0,3],["split",11,10.0,27,3],["split",12,23.0,24,3],["split",13,23.0,29,3],["split",14,0.0,25,3],["split",19,10.0,15,4],["leaf",20,-0.01559133399160574],["split",21,19.0,0,4],["leaf",22,-0.007814018999712746],["split",23,2.0,6,4],["leaf",24,0.0053716187078733085],["split",25,21.0,6,4],["leaf",26,-0.016997587370228363],["split",27,4.0,30,4],["split",28,26.0,14,4],["split",29,23.0,15,4],["leaf",30,0.025200991227329612],["leaf",39,0.006441121631890259],["leaf",40,-0.0038462896255636566],["leaf",43,0.00932262776291633],["leaf",44,-0.0061059354149950685],["leaf",47,-0.0012204765404682531],["leaf",48,-0.014808804682503191],["leaf",51,-0.006127581960120222],["leaf",52,0.008340659075089027],["leaf",55,2.818602272288523e-05],["leaf",56,-0.01220221056235548],["leaf",57,0.0048551376841066566],["leaf",58,-0.0161606295701937],["leaf",59,0.009309406031397394],["leaf",60,-0.0031114748860712944]],[["split",0,6.0,30,0],["split",1,13.0,1,1],["split",2,25.0,11,1],["split",3,0.0,27,2],["split",4,7.0,2,2],["leaf",5,0.003890424927424945],["split",6,1.0,18,2],["split",7,7.0,0,3],["leaf",8,-0.009657116925275882],["split",9,26.0,7,3],["split",10,26.0,5,3],["leaf",13,0.009693824350628714],["leaf",14,0.02395634741643733],["split",15,22.0,10,4],["split",16,1.0,9,4],["leaf",19,-0.021102169738022748],["split",20,0.0,17,4],["split",21,13.0,7,4],["split",22,1.0,3,4],["leaf",31,-0.011632193685179327],["leaf",32,0.0066184423533382],["leaf",33,0.008216502128100659],["leaf",34,0.023746959909635632],["leaf",41,-0.013672841149721605],["leaf",42,0.002283850699857328],["leaf",43,0.0029911853113425207],["leaf",44,-0.006755653475203915],["leaf",45,-0.006001160265923849],["leaf",46,0.0011435871189858717]],[["split",0,10.0,7,0],["split",1,7.0,30,1],["split",2,12.0,6,1],["split",3,4.0,4,2],["leaf",4,0.016046969657534122],["split",5,22.0,2,2],["split",6,21.0,2,2],["split",7,6.0,23,3],["split",8,20.0,0,3],["leaf",11,-0.01564693633461633],["split",12,20.0,0,3],["split",13,11.0,22,3],["split",14,24.0,9,3],["split",15,7.0,14,4],["leaf",16,-0.00193807293617261],["split",17,7.0,1,4],["split",18,2.0,18,4],["split",25,7.0,11,4],["split",26,25.0,11,4],["split",27,13.0,11,4],["leaf",28,0.003228028806255165],["split",29,5.0,13,4],["split",30,25.0,4,4],["leaf",31,0.0011510210269690293],["leaf",32,0.019152339291071428],["leaf",35,-0.019729320247171625],["leaf",36,-0.006134114241477993],["leaf",37,-0.008971738261895821],["leaf",38,0.015226880987407076],["leaf",51,0.0019347000962095233],["leaf",52,-0.012358510322461906],["leaf",53,-0.02127921409845943],["leaf",54,-0.003363979971923251],["leaf",55,-0.023780574261483847],["leaf",56,-0.004713945008292723],["leaf",59,-0.002625754071589157],["leaf",60,0.0028813259708334485],["leaf",61,-0.001382350340866713],["leaf",62,0.004970495457678974]],[["split",0,10.0,8,0],["split",1,17.0,0,1],["split",2,21.0,16,1],["split",3,5.0,27,2],["split",4,8.0,23,2],["split",5,20.0,0,2],["split",6,3.0,20,2],["split",7,9.0,26,3],["split",8,4.0,27,3],["split",9,25.0,12,3],["leaf",10,0.006336917705122761],["split",11,1.0,4,3],["split",12,23.0,24,3],["split",13,14.0,0,3],["split",14,19.0,0,3],["split",15,4.0,22,4],["split",16,5.0,11,4],["leaf",17,0.018224207465403833],["split",18,22.0,9,4],["leaf",19,-0.025215375097577826],["split",20,23.0,17,4],["split",23,8.0,2,4],["split",24,7.0,19,4],["split",25,2.0,6,4],["leaf",26,0.010743068168221159],["split",27,13.0,7,4],["split",28,24.0,25,4],["split",29,24.0,12,4],["split",30,1.0,14,4],["leaf",31,-0.0001169929613053295],["leaf",32,-0.007804777645066883],["leaf",33,-0.0047846638057203105],["leaf",34,-0.02161446256975929],["leaf",37,0.014769659307345858],["leaf",38,-0.003944050516987394],["leaf",41,0.005004383500712505],["leaf",42,-0.017472208962274898],["leaf",47,0.007605347182523394],["leaf",48,-0.013899104486659911],["leaf",49,-4.755721475640373e-06],["leaf",50,0.006445655672304767],["leaf",51,-0.02764295309340871],["leaf",52,-0.0072986415310116026],["leaf",55,0.008842865377715994],["leaf",56,4.4750066784507805e-05],["leaf",57,-0.0025647760746029375],["leaf",58,-0.019066679677671807],["leaf",59,0.0028039714083169806],["leaf",60,0.010390830250736546],["leaf",61,-0.01087341873571785],["leaf",62,0.004332285983014]],[["split",0,21.0,0,0],["split",1,23.0,11,1],["split",2,4.0,26,1],["leaf",3,0.0008725093198891654],["split",4,2.0,19,2],["split",5,13.0,0,2],["split",6,20.0,0,2],["leaf",9,-0.018581208442415825],["leaf",10,-0.004937347515182334],["leaf",11,0.018736978888925827],["split",12,1.0,6,3],["split",13,19.0,0,3],["split",14,21.0,20,3],["split",25,0.0,3,4],["split",26,8.0,27,4],["split",27,25.0,13,4],["split",28,9.0,10,4],["leaf",29,-0.0025742079166087358],["leaf",30,0.01905435846804251],["leaf",51,-0.012762029682685033],["leaf",52,-0.0013781292819622702],["leaf",53,0.0013662021486749276],["leaf",54,0.00589613549481348],["leaf",55,-0.001835726220215462],["leaf",56,-0.011122402541675909],["leaf",57,-0.003160750596812512],["leaf",58,0.007303573482318764]],[["split",0,26.0,2,0],["split",1,25.0,3,1],["split",2,7.0,3,1],["split",3,9.0,20,2],["split",4,24.0,22,2],["split",5,4.0,13,2],["split",6,22.0,21,2],["split",7,7.0,19,3],["leaf",8,-0.024001548786068572],["split",9,1.0,17,3],["split",10,22.0,11,3],["leaf",11,0.017609469419612903],["split",12,3.0,5,3],["split",13,23.0,29,3],["split",14,20.0,0,3],["leaf",15,-0.018454537857760796],["leaf",16,-0.00616754940634912],["split",19,3.0,4,4],["split",20,5.0,18,4],["leaf",21,-0.0022187824535473315],["leaf",22,0.010100450664013907],["split",25,22.0,13,4],["split",26,17.0,0,4],["split",27,25.0,27,4],["split",28,12.0,18,4],["split",29,23.0,16,4],["split",30,24.0,18,4],["leaf",39,0.006785109704806704],["leaf",40,-0.00890422508867935],["leaf",41,-0.003121656276993276],["leaf",42,-0.020717371972198233],["leaf",51,-0.008817264431955607],["leaf",52,0.014148352176600537],["leaf",53,-0.010312380919666336],["leaf",54,-0.028245049536880642],["leaf",55,0.002957374787915196],["leaf",56,0.009427411747895853],["leaf",57,0.003571723582042441],["leaf",58,-0.018616356993603568],["leaf",59,0.004313170108021861],["leaf",60,-0.0030967431300939798],["leaf",61,-0.01823036458676667],["leaf",62,0.005063891051651158]],[["split",0,5.0,18,0],["split",1,25.0,22,1],["split",2,20.0,0,1],["split",3,17.0,0,2],["split",4,20.0,0,2],["split",5,19.0,0,2],["split",6,2.0,18,2],["split",7,20.0,0,3],["split",8,23.0,25,3],["split",9,23.0,5,3],["split",10,13.0,20,3],["split",11,2.0,16,3],["split",12,13.0,2,3],["split",13,25.0,21,3],["split",14,23.0,17,3],["split",15,2.0,7,4],["split",16,3.0,17,4],["split",17,2.0,5,4],["leaf",18,-0.013443888739940707],["split",19,2.0,17,4],["split",20,4.0,1,4],["leaf",21,0.025830194674776805],["leaf",22,0.002628409317897423],["split",23,8.0,7,4],["split",24,7.0,0,4],["leaf",25,0.014916113480878382],["split",26,9.0,4,4],["split",27,0.0,11,4],["leaf",28,0.0006199303902894878],["leaf",29,0.0002750349705200789],["split",30,21.0,19,4],["leaf",31,0.004168491398474922],["leaf",32,-0.000899402103720734],["leaf",33,0.0052317859269108416],["leaf",34,-0.012410969688944914],["leaf",35,0.01986034243083996],["leaf",36,0.00460496510375154],["leaf",39,0.013055387246999763],["leaf",40,-0.002701265967064209],["leaf",41,0.013532575829778454],["leaf",42,0.0007168045528223535],["leaf",47,0.007502452358263234],["leaf",48,-0.004014859947634703],["leaf",49,-0.017995267491358727],["leaf",50,-0.006164460470228438],["leaf",53,-0.012157150588774613],["leaf",54,0.0023024862126297635],["leaf",55,-0.009915532390555785],["leaf",56,-0.02629634507368322],["leaf",61,0.017497633174544683],["leaf",62,0.038518726174742626]],[["split",0,4.0,26,0],["split",1,5.0,28,1],["split",2,1.0,15,1],["split",3,7.0,27,2],["leaf",4,0.014283832689079769],["split",5,19.0,0,2],["split",6,5.0,29,2],["split",7,26.0,16,3],["split",8,19.0,0,3],["split",11,9.0,2,3],["leaf",12,0.006965389793614554],["split",13,13.0,5,3],["leaf",14,0.010363804439610198],["split",15,15.0,0,4],["split",16,0.0,28,4],["split",17,24.0,11,4],["split",18,1.0,18,4],["leaf",23,0.004021141185713445],["split",24,21.0,27,4],["split",27,9.0,15,4],["split",28,3.0,12,4],["leaf",31,1.852403663357376e-05],["leaf",32,-0.007883017397688027],["leaf",33,0.0018033617896603597],["leaf",34,0.009748128105356456],["leaf",35,0.0021480179062543657],["leaf",36,0.011670939258249196],["leaf",37,0.001366595825951909],["leaf",38,-0.012460342948852393],["leaf",49,-0.014810693175066551],["leaf",50,-0.0022404326568652277],["leaf",55,-0.012717841247758002],["leaf",56,-0.009483172239654496],["leaf",57,-0.004434256831946217],["leaf",58,0.007738215367780828]],[["split",0,4.0,30,0],["split",1,5.0,27,1],["split",2,23.0,20,1],["split",3,7.0,23,2],["split",4,19.0,0,2],["leaf",5,-0.014237684442624696],["split",6,22.0,16,2],["split",7,13.0,0,3],["split",8,18.0,0,3],["split",9,8.0,15,3],["leaf",10,0.01873804951021659],["leaf",13,-0.013064579932052945],["leaf",14,0.004724135441575796],["leaf",15,0.011126680217048518],["split",16,26.0,2,4],["split",17,13.0,15,4],["split",18,25.0,21,4],["split",19,18.0,0,4],["leaf",20,-0.008971933997805468],["leaf",33,-0.012640583130735406],["leaf",34,-0.0014277101035397384],["leaf",35,0.01788311819869982],["leaf",36,0.0038813432143326274],["leaf",37,-0.012651860531983453],["leaf",38,0.00514709822719831],["leaf",39,0.002279248249322809],["leaf",40,0.015068353781929875]],[["split",0,4.0,9,0],["split",1,7.0,28,1],["split",2,23.0,23,1],["split",3,9.0,4,2],["split",4,3.0,9,2],["split",5,7.0,0,2],["split",6,17.0,0,2],["split",7,23.0,13,3],["split",8,10.0,4,3],["leaf",9,-0.0030470748273737965],["split",10,2.0,5,3],["leaf",11,-0.015102864379368173],["split",12,26.0,10,3],["split",13,20.0,0,3],["split",14,12.0,13,3],["split",15,23.0,5,4],["split",16,13.0,15,4],["split",17,1.0,18,4],["split",18,25.0,24,4],["leaf",21,0.023907528391611352],["leaf",22,0.011638690813438953],["split",25,2.0,9,4],["split",26,17.0,0,4],["split",27,9.0,11,4],["split",28,2.0,19,4],["split",29,24.0,13,4],["leaf",30,-0.03639817239271745],["leaf",31,-0.007361309672611419],["leaf",32,0.006465682839606938],["leaf",33,-0.0027521136223426852],["leaf",34,-0.01576286197274795],["leaf",35,0.00012122426776222448],["leaf",36,-0.009210791180212372],["leaf",37,0.0022010351192726656],["leaf",38,0.01044384200414867],["leaf",51,0.004411831130858596],["leaf",52,-0.004729410574704718],["leaf",53,0.0013918129063475644],["leaf",54,0.008478183652261929],["leaf",55,0.000298053229061088],["leaf",56,-0.008307127336896244],["leaf",57,-0.006480329663760166],["leaf",58,0.03006525924086923],["leaf",59,-0.029886257840176077],["leaf",60,-0.013371225967803529]],[["split",0,13.0,2,0],["split",1,17.0,0,1],["split",2,12.0,5,1],["split",3,0.0,9,2],["leaf",4,-0.005471608515649829],["split",5,2.0,11,2],["split",6,1.0,0,2],["split",7,22.0,21,3],["split",8,5.0,23,3],["split",11,24.0,18,3],["split",12,20.0,0,3],["leaf",13,-0.01414401157766957],["split",14,7.0,29,3],["split",15,23.0,20,4],["leaf",16,0.015033694170961159],["leaf",17,-0.0011389100842228235],["split",18,26.0,12,4],["leaf",23,-0.004779011164524293],["leaf",24,0.018308362907608712],["split",25,23.0,26,4],["leaf",26,0.005757766076941858],["split",29,10.0,5,4],["split",30,22.0,12,4],["leaf",31,-0.01705278522324337],["leaf",32,0.004440377811215451],["leaf",37,0.010823479445640917],["leaf",38,0.025310513834272078],["leaf",51,-0.006432752457420392],["leaf",52,-0.01964750421581749],["leaf",59,-0.003411782891421474],["leaf",60,0.0012349361438250158],["leaf",61,0.014935126540787708],["leaf",62,0.001429805276919982]],[["split",0,21.0,2,0],["split",1,20.0,0,1],["split",2,6.0,30,1],["split",3,2.0,18,2],["leaf",4,-0.015568433718380641],["split",5,21.0,28,2],["split",6,3.0,21,2],["split",7,5.0,15,3],["split",8,11.0,11,3],["split",11,21.0,20,3],["split",12,5.0,3,3],["leaf",13,0.017903937512605626],["split",14,26.0,26,3],["split",15,22.0,20,4],["split",16,3.0,4,4],["leaf",17,0.0008964808951442948],["split",18,24.0,11,4],["split",23,20.0,0,4],["split",24,2.0,5,4],["leaf",25,-0.02380666219021772],["split",26,20.0,0,4],["leaf",29,0.002503721429975191],["leaf",30,0.012769387620521142],["leaf",31,0.0018123153538245414],["leaf",32,-0.018116200582393337],["leaf",33,0.014746401079065725],["leaf",34,-0.0013967414019838199],["leaf",37,-0.007847596020611377],["leaf",38,-0.021543706702009936],["leaf",47,0.0003109721049907526],["leaf",48,-0.00659783899333761],["leaf",49,0.010689249153537686],["leaf",50,0.002252273485724841],["leaf",53,-0.005485458171887783],["leaf",54,0.015504999738926406]],[["split",0,7.0,28,0],["split",1,26.0,28,1],["split",2,17.0,0,1],["split",3,10.0,1,2],["split",4,21.0,11,2],["split",5,5.0,3,2],["leaf",6,-0.005065692911721369],["split",7,13.0,2,3],["split",8,23.0,29,3],["split",9,6.0,23,3],["split",10,9.0,14,3],["leaf",11,0.015676392238693722],["split",12,8.0,25,3],["leaf",15,0.003456525127578311],["split",16,21.0,25,4],["split",17,9.0,20,4],["split",18,2.0,21,4],["split",19,0.0,14,4],["leaf",20,-0.014024526709770443],["split",21,23.0,16,4],["split",22,3.0,22,4],["split",25,11.0,23,4],["split",26,6.0,29,4],["leaf",33,-0.007728571086416336],["leaf",34,-0.025976768897066713],["leaf",35,-0.00041787873262389715],["leaf",36,-0.0035747217867783294],["leaf",37,-0.017556215471360017],["leaf",38,-0.003340407175714436],["leaf",39,0.012449824684609725],["leaf",40,-0.004609744361868474],["leaf",43,-0.0036876317605541646],["leaf",44,0.005424659580824929],["leaf",45,0.021978350076712335],["leaf",46,0.008611859447591796],["leaf",51,0.004864435887112065],["leaf",52,0.020982504547234136],["leaf",53,-0.00626086955177995],["leaf",54,0.007830172003709094]],[["split",0,23.0,16,0],["split",1,26.0,0,1],["split",2,20.0,0,1],["split",3,25.0,9,2],["split",4,17.0,0,2],["split",5,3.0,29,2],["split",6,2.0,22,2],["leaf",7,-0.021925737714367382],["leaf",8,-0.0037221206921018066],["split",9,22.0,19,3],["split",10,22.0,21,3],["split",11,23.0,26,3],["split",12,13.0,18,3],["split",13,12.0,11,3],["split",14,23.0,25,3],["split",19,22.0,13,4],["split",20,20.0,0,4],["split",21,0.0,16,4],["split",22,23.0,8,4],["split",23,2.0,11,4],["split",24,17.0,0,4],["leaf",25,0.02237132575420855],["leaf",26,-0.0054174247664974],["leaf",27,-0.01271839369783535],["split",28,25.0,18,4],["leaf",29,0.010977246221130312],["leaf",30,0.030162096468792123],["leaf",39,0.0018173181340829106],["leaf",40,0.007108962706288603],["leaf",41,0.0005681672644475615],["leaf",42,-0.010140668188294148],["leaf",43,0.009537411048339377],["leaf",44,-0.0003724021353309146],["leaf",45,0.021823941922340433],["leaf",46,0.010826803294334199],["leaf",47,0.0020509344810561163],["leaf",48,-0.0037675043399498687],["leaf",49,-0.005131144166792373],["leaf",50,-0.02558421489532928],["leaf",57,-0.006426169126963169],["leaf",58,0.014900429920982509]],[["split",0,17.0,0,0],["split",1,24.0,7,1],["split",2,23.0,28,1],["split",3,10.0,28,2],["split",4,25.0,25,2],["split",5,3.0,29,2],["leaf",6,-0.021822414304956492],["split",7,20.0,0,3],["split",8,24.0,3,3],["split",9,12.0,2,3],["split",10,5.0,8,3],["split",11,1.0,19,3],["leaf",12,0.027952400327499655],["split",15,9.0,1,4],["split",16,11.0,17,4],["leaf",17,-0.019666942220319682],["leaf",18,-0.00541057851907506],["split",19,14.0,0,4],["split",20,5.0,29,4],["split",21,1.0,10,4],["split",22,20.0,0,4],["split",23,2.0,12,4],["split",24,24.0,15,4],["leaf",31,-0.016202780046676784],["leaf",32,-0.0005523229538979458],["leaf",33,-0.018131730292824166],["leaf",34,0.002140085317783556],["leaf",39,-0.003218024603949289],["leaf",40,-0.025539405891752777],["leaf",41,-0.0009898713974909067],["leaf",42,0.011039138925765987],["leaf",43,-0.01402983850636887],["leaf",44,0.0003428171633462029],["leaf",45,0.00362551554435738],["leaf",46,0.022476315141004816],["leaf",47,0.007751755212581851],["leaf",48,-0.0034113305324444732],["leaf",49,0.020238232209568312],["leaf",50,0.005182351111579297]],[["split",0,0.0,6,0],["split",1,11.0,1,1],["split",2,8.0,29,1],["leaf",3,-0.015631214262058556],["split",4,16.0,0,2],["split",5,10.0,4,2],["split",6,7.0,23,2],["split",9,8.0,24,3],["split",10,10.0,19,3],["split",11,18.0,0,3],["split",12,3.0,27,3],["leaf",13,-0.0008641568825648077],["split",14,3.0,19,3],["split",19,7.0,17,4],["split",20,1.0,18,4],["split",21,26.0,8,4],["leaf",22,-0.018868387567121402],["split",23,22.0,11,4],["split",24,1.0,13,4],["split",25,1.0,28,4],["split",26,19.0,0,4],["leaf",29,0.0023222031589534516],["split",30,1.0,21,4],["leaf",39,-0.002711969156308229],["leaf",40,0.005855243954426427],["leaf",41,-0.016099368275428492],["leaf",42,0.00042130577480093014],["leaf",43,0.005191587020286556],["leaf",44,-0.01060023068330975],["leaf",47,0.0026437651481751936],["leaf",48,-0.005726711314299365],["leaf",49,-0.011563691366158303],["leaf",50,-0.0245514208896621],["leaf",51,1.4200959414416544e-05],["leaf",52,0.0060426147548820275],["leaf",53,0.007412352731049569],["leaf",54,-0.0036926912529510365],["leaf",61,0.005095163364912026],["leaf",62,0.023118295412793366]],[["split",0,12.0,11,0],["split",1,10.0,29,1],["split",2,9.0,9,1],["split",3,7.0,27,2],["split",4,22.0,17,2],["split",5,13.0,29,2],["split",6,7.0,24,2],["split",7,19.0,0,3],["leaf",8,0.015472002899683394],["leaf",9,-0.018883983808029175],["leaf",10,-0.003044354835275747],["split",11,24.0,10,3],["split",12,9.0,4,3],["split",13,9.0,30,3],["split",14,8.0,19,3],["split",15,25.0,30,4],["split",16,3.0,5,4],["split",23,18.0,0,4],["split",24,7.0,29,4],["leaf",25,-0.0032681532778226926],["leaf",26,0.01145571219637407],["split",27,3.0,13,4],["split",28,22.0,17,4],["split",29,1.0,13,4],["split",30,24.0,3,4],["leaf",31,-0.004053552663107981],["leaf",32,0.013700567763891152],["leaf",33,0.012788592750029056],["leaf",34,5.9031133695408976e-05],["leaf",47,-0.010369516120080975],["leaf",48,0.006573409510830632],["leaf",49,-0.0012943810348860843],["leaf",50,0.009606526414323],["leaf",55,-0.0052874572055915245],["leaf",56,0.0018915717967973415],["leaf",57,0.020228512817959763],["leaf",58,0.004006316644922002],["leaf",59,0.03168535300816487],["leaf",60,0.007957727721974254],["leaf",61,-0.00573570592392672],["leaf",62,0.0075572046448436495]],[["split",0,7.0,23,0],["split",1,3.0,24,1],["split",2,13.0,15,1],["split",3,10.0,2,2],["split",4,20.0,0,2],["leaf",5,0.012380700875250044],["split",6,1.0,2,2],["split",7,17.0,0,3],["split",8,8.0,27,3],["split",9,19.0,0,3],["split",10,21.0,18,3],["leaf",13,-0.01141156428234299],["split",14,2.0,10,3],["split",15,14.0,0,4],["leaf",16,-0.023852746614304872],["split",17,26.0,21,4],["leaf",18,-0.01825237564231168],["split",19,25.0,4,4],["split",20,21.0,15,4],["split",21,13.0,18,4],["leaf",22,0.003978177426364756],["split",29,4.0,19,4],["split",30,0.0,15,4],["leaf",31,-0.004394370578359059],["leaf",32,-0.017916760439998684],["leaf",35,-0.0021952213932680638],["leaf",36,0.0034009671329385295],["leaf",39,0.016049452245682168],["leaf",40,0.004734574710388548],["leaf",41,0.0028375659394602607],["leaf",42,-0.008274868295706448],["leaf",43,-0.009326742956482148],["leaf",44,-0.019847129558760123],["leaf",59,0.004744508805533983],["leaf",60,0.01532037571387124],["leaf",61,-0.004056973363336156],["leaf",62,0.004455798983724024]],[["split",0,7.0,0,0],["split",1,11.0,13,1],["split",2,11.0,23,1],["leaf",3,-0.0007315718648290349],["split",4,3.0,7,2],["split",5,2.0,26,2],["split",6,17.0,0,2],["leaf",9,-0.02897674357864636],["leaf",10,-0.005350397205105976],["split",11,20.0,0,3],["split",12,7.0,4,3],["split",13,5.0,21,3],["split",14,23.0,17,3],["split",23,23.0,27,4],["split",24,25.0,23,4],["split",25,21.0,21,4],["split",26,23.0,28,4],["split",27,4.0,22,4],["split",28,21.0,14,4],["split",29,26.0,16,4],["split",30,12.0,14,4],["leaf",47,0.001422653927763112],["leaf",48,-0.007094610912902332],["leaf",49,-0.010646463411352607],["leaf",50,0.007324728375823353],["leaf",51,-0.016725468599839314],["leaf",52,0.002575354492673999],["leaf",53,-0.005180271394698807],["leaf",54,0.010405974388399542],["leaf",55,0.0025387451065277664],["leaf",56,-0.012788120252985456],["leaf",57,0.004481926802483849],["leaf",58,0.014418147708362224],["leaf",59,-0.005794839813687411],["leaf",60,0.014152201137045328],["leaf",61,-0.010226700811114571],["leaf",62,-0.01920930906790812]],[["split",0,26.0,25,0],["split",1,26.0,0,1],["split",2,1.0,3,1],["split",3,7.0,18,2],["split",4,7.0,24,2],["split",5,5.0,20,2],["split",6,9.0,11,2],["split",7,9.0,14,3],["split",8,1.0,19,3],["split",9,13.0,6,3],["split",10,22.0,19,3],["leaf",11,-0.021289588950373486],["leaf",12,0.0025775957809323286],["split",13,2.0,2,3],["split",14,17.0,0,3],["leaf",15,-0.003792833581889775],["leaf",16,-0.03174103579296904],["leaf",17,0.0034485192090180383],["leaf",18,0.0009596271842779589],["split",19,10.0,25,4],["split",20,7.0,14,4],["split",21,25.0,18,4],["split",22,26.0,20,4],["leaf",27,-0.012565579902538758],["split",28,25.0,26,4],["split",29,21.0,20,4],["split",30,2.0,14,4],["leaf",39,0.002681208158993648],["leaf",40,-0.009031433121301079],["leaf",41,-0.0057701382172365605],["leaf",42,-0.0020237664258485766],["leaf",43,0.002868017211036496],["leaf",44,0.009546787377200397],["leaf",45,-0.003968591348596608],["leaf",46,0.010204330689374028],["leaf",57,-0.0015073784016023824],["leaf",58,0.013303958857363591],["leaf",59,0.0034147773889067583],["leaf",60,0.009871208788130673],["leaf",61,0.02513778295642038],["leaf",62,0.0050627748460714975]],[["split",0,25.0,27,0],["split",1,2.0,11,1],["split",2,9.0,22,1],["split",3,17.0,0,2],["split",4,9.0,24,2],["split",5,20.0,0,2],["split",6,0.0,10,2],["split",7,20.0,0,3],["split",8,12.0,14,3],["split",9,3.0,6,3],["split",10,7.0,26,3],["split",11,7.0,7,3],["leaf",12,0.022496095530799773],["leaf",13,0.02098624007200329],["split",14,7.0,10,3],["split",15,21.0,29,4],["split",16,21.0,9,4],["leaf",17,0.016126795813678246],["split",18,8.0,22,4],["split",19,0.0,18,4],["split",20,7.0,10,4],["split",21,11.0,12,4],["leaf",22,-0.010132779362511016],["split",23,24.0,11,4],["split",24,1.0,9,4],["leaf",29,-0.0026364954869750887],["split",30,2.0,10,4],["leaf",31,0.003279150894162409],["leaf",32,-0.009698586896502909],["leaf",33,-0.024003864369212695],["leaf",34,-0.004594443670942919],["leaf",37,-0.003141336620637154],["leaf",38,0.011786509287624052],["leaf",39,-0.002226670961876036],["leaf",40,-0.012699034924925372],["leaf",41,-0.0043491634284899714],["leaf",42,0.00044300293100009723],["leaf",43,0.006542532747643008],["leaf",44,-0.0005507331043898601],["leaf",47,-0.003513010932108925],["leaf",48,0.012205064435087084],["leaf",49,-0.01873378062432629],["leaf",50,-0.0005079587466804307],["leaf",61,0.00795810497907931],["leaf",62,0.016065109632498815]],[["split",0,7.0,8,0],["split",1,6.0,11,1],["split",2,10.0,5,1],["split",3,26.0,29,2],["split",4,3.0,22,2],["split",5,1.0,26,2],["split",6,21.0,9,2],["split",7,3.0,29,3],["leaf",8,0.013530942298921454],["leaf",9,-0.02025027977182975],["split",10,10.0,21,3],["split",11,24.0,4,3],["split",12,3.0,14,3],["split",13,7.0,16,3],["split",14,25.0,30,3],["split",15,3.0,14,4],["leaf",16,0.013040538723620104],["leaf",21,-0.011383455859690049],["leaf",22,0.001345056099909914],["split",23,13.0,17,4],["split",24,2.0,25,4],["leaf",25,-0.02031059415964061],["leaf",26,-0.0020631984432237494],["split",27,20.0,0,4],["split",28,13.0,21,4],["split",29,21.0,25,4],["leaf",30,0.01630996992123375],["leaf",31,-0.0004138458875373779],["leaf",32,-0.006369129702312414],["leaf",47,-0.00177792612154192],["leaf",48,-0.01493078291583102],["leaf",49,-0.0014817421894394432],["leaf",50,0.008882895734362403],["leaf",55,-0.0033687170529029336],["leaf",56,-0.016600657750183143],["leaf",57,0.0075928605223437445],["leaf",58,-0.0011798506637352893],["leaf",59,0.0052325474150708715],["leaf",60,0.0011242004441316096]],[["split",0,23.0,28,0],["split",1,10.0,11,1],["split",2,17.0,0,1],["split",3,9.0,11,2],["split",4,0.0,17,2],["split",5,20.0,0,2],["leaf",6,-0.024750932044336083],["split",7,5.0,26,3],["split",8,22.0,27,3],["split",9,3.0,16,3],["split",10,8.0,3,3],["split",11,13.0,4,3],["leaf",12,0.006212974224665609],["split",15,21.0,4,4],["split",16,2.0,25,4],["split",17,4.0,29,4],["split",18,25.0,17,4],["split",19,16.0,0,4],["split",20,9.0,2,4],["split",21,10.0,20,4],["split",22,1.0,2,4],["split",23,10.0,9,4],["split",24,9.0,12,4],["leaf",31,-0.008550473844009693],["leaf",32,0.00214519482785932],["leaf",33,0.02260473895457296],["leaf",34,0.0025852043561965894],["leaf",35,-0.0021046977733778223],["leaf",36,-0.012905717505627738],["leaf",37,-0.017634793886161647],["leaf",38,-0.0058238591996267715],["leaf",39,-0.0006592534475499928],["leaf",40,-0.015436780406911975],["leaf",41,-0.009644608435953177],["leaf",42,0.004220837265712012],["leaf",43,0.02145899462306951],["leaf",44,0.005716857466686868],["leaf",45,-0.008098888260110126],["leaf",46,0.0036877879007615457],["leaf",47,0.00800642834676057],["leaf",48,-0.011253458868459507],["leaf",49,-0.017575867518204284],["leaf",50,-0.0009656194979634781]],[["split",0,1.0,1,0],["split",1,25.0,10,1],["split",2,2.0,0,1],["split",3,6.0,4,2],["split",4,25.0,25,2],["split",5,10.0,23,2],["split",6,21.0,14,2],["leaf",7,0.008456099685544493],["leaf",8,-0.00891972655300854],["split",9,12.0,7,3],["leaf",10,0.0022726463424674777],["split",11,22.0,16,3],["leaf",12,0.0035337198458271226],["split",13,24.0,30,3],["split",14,20.0,0,3],["leaf",19,-0.02326695898685864],["leaf",20,-0.010233416320651133],["leaf",23,0.01929692943383048],["leaf",24,0.0071883580428714275],["split",27,17.0,0,4],["leaf",28,-0.013759314012248218],["split",29,7.0,8,4],["split",30,23.0,24,4],["leaf",55,-0.0020097877204732044],["leaf",56,0.003458116874283292],["leaf",59,-0.003375780639122776],["leaf",60,0.002709953914936398],["leaf",61,0.002805961253416337],["leaf",62,0.02724237354772271]],[["split",0,9.0,30,0],["split",1,25.0,29,1],["split",2,23.0,19,1],["split",3,11.0,28,2],["split",4,20.0,0,2],["split",5,10.0,19,2],["leaf",6,0.023097050349113368],["split",7,24.0,6,3],["split",8,3.0,4,3],["split",9,2.0,6,3],["leaf",10,0.027278232229542666],["leaf",11,-0.0011677517055991703],["split",12,26.0,24,3],["split",15,25.0,27,4],["split",16,7.0,9,4],["leaf",17,0.006964111432435055],["split",18,9.0,26,4],["leaf",19,0.014685864049923216],["split",20,13.0,11,4],["leaf",25,0.018519314887589827],["leaf",26,0.004383385756145691],["leaf",31,-0.001149092724621188],["leaf",32,-0.011214636467500318],["leaf",33,-0.00013066435067837643],["leaf",34,0.0030626746375188034],["leaf",37,-0.009778146212433213],["leaf",38,0.003844271220384183],["leaf",41,0.008079635781878399],["leaf",42,-0.008918162397127722]],[["split",0,26.0,26,0],["split",1,22.0,22,1],["split",2,8.0,10,1],["split",3,7.0,21,2],["split",4,1.0,27,2],["split",5,1.0,4,2],["split",6,23.0,1,2],["split",7,9.0,30,3],["split",8,6.0,12,3],["split",9,13.0,3,3],["split",10,10.0,9,3],["leaf",11,0.0015782396965312025],["split",12,5.0,21,3],["leaf",13,-0.014130618218429578],["split",14,21.0,27,3],["split",15,6.0,22,4],["leaf",16,0.012284256264187651],["leaf",17,0.02449824248136549],["split",18,0.0,0,4],["split",19,10.0,11,4],["split",20,3.0,1,4],["leaf",21,-0.008190732428513977],["split",22,7.0,18,4],["leaf",25,0.0051200616050300456],["leaf",26,0.023197358533612256],["split",29,20.0,0,4],["leaf",30,0.016048451420874243],["leaf",31,-0.0014209855283082195],["leaf",32,-0.009970529198708286],["leaf",37,-0.011320360435980031],["leaf",38,0.0027075238464045297],["leaf",39,-0.00613170605346575],["leaf",40,0.010886591181199142],["leaf",41,-0.019906859024319233],["leaf",42,-0.005998354119097663],["leaf",45,0.01727209568265414],["leaf",46,0.0011676666019708136],["leaf",59,0.0036853605184976293],["leaf",60,-0.009106880387491518]],[["split",0,1.0,28,0],["split",1,3.0,1,1],["split",2,22.0,28,1],["split",3,21.0,25,2],["split",4,6.0,3,2],["split",5,12.0,6,2],["leaf",6,-0.0213478007368444],["split",7,22.0,9,3],["leaf",8,0.019891144994663493],["split",9,22.0,11,3],["split",10,8.0,4,3],["leaf",11,-0.016107673626718066],["split",12,22.0,5,3],["leaf",15,0.010723942219547785],["split",16,2.0,19,4],["split",19,24.0,17,4],["split",20,11.0,4,4],["split",21,1.0,3,4],["split",22,23.0,26,4],["split",25,3.0,9,4],["split",26,21.0,17,4],["leaf",33,-0.00016958566803825312],["leaf",34,-0.014363990050572416],["leaf",39,-0.010602434509887662],["leaf",40,0.007639084325701468],["leaf",41,0.02100837207986643],["leaf",42,0.0045509405256431755],["leaf",43,-0.019189676852343563],["leaf",44,-0.006744438986282458],["leaf",45,-4.2837843745404724e-05],["leaf",46,-0.005526311811029398],["leaf",51,-0.01916791807695589],["leaf",52,-0.003035734984282309],["leaf",53,-0.004584913022542159],["leaf",54,0.007125495237492865]],[["split",0,1.0,5,0],["split",1,0.0,20,1],["split",2,25.0,1,1],["split",3,6.0,12,2],["split",4,15.0,0,2],["split",5,1.0,17,2],["split",6,6.0,10,2],["split",7,19.0,0,3],["split",8,9.0,12,3],["split",9,26.0,4,3],["leaf",10,-0.016691661381389087],["split",11,4.0,21,3],["split",12,21.0,7,3],["split",13,0.0,18,3],["split",14,17.0,0,3],["split",15,0.0,10,4],["leaf",16,0.013774559273937084],["split",17,22.0,16,4],["split",18,23.0,19,4],["leaf",19,0.004768559085410886],["split",20,20.0,0,4],["leaf",23,-0.017783422665066224],["leaf",24,-0.0009722987626980405],["leaf",25,-0.015002892853696628],["split",26,4.0,14,4],["split",27,7.0,1,4],["split",28,1.0,21,4],["split",29,11.0,9,4],["split",30,24.0,6,4],["leaf",31,-0.004506946854213904],["leaf",32,0.005364939648479472],["leaf",35,0.004914120451906279],["leaf",36,-0.009865491604851229],["leaf",37,-0.019341188286823834],["leaf",38,-0.005852576775781462],["leaf",41,-0.010576259873412945],["leaf",42,0.002720291442062148],["leaf",53,-0.0005023403682307411],["leaf",54,0.014647660153974482],["leaf",55,-0.008695626498311964],["leaf",56,0.0025599390687740405],["leaf",57,0.010591305048269651],["leaf",58,0.0017135625106309827],["leaf",59,-0.003790845180982572],["leaf",60,0.0009919765537068817],["leaf",61,0.014465892170051457],["leaf",62,0.0018934578589415022]],[["split",0,0.0,0,0],["split",1,3.0,16,1],["split",2,10.0,1,1],["split",3,26.0,4,2],["leaf",4,-0.01856976036762066],["split",5,5.0,25,2],["split",6,24.0,1,2],["leaf",7,-0.010838825025106512],["leaf",8,0.004665132467657714],["split",11,1.0,8,3],["leaf",12,0.00630393877664881],["split",13,22.0,5,3],["split",14,6.0,30,3],["split",23,4.0,14,4],["split",24,4.0,10,4],["leaf",27,-0.015841191122923315],["split",28,3.0,4,4],["split",29,6.0,1,4],["split",30,26.0,22,4],["leaf",47,-0.00859240778793323],["leaf",48,0.009153461265102985],["leaf",49,-0.004117953445243244],["leaf",50,-0.0183558789707639],["leaf",57,0.007630263414568247],["leaf",58,-0.004582876126021047],["leaf",59,0.0066535647622836995],["leaf",60,0.0007694531636739384],["leaf",61,0.0021357345306849507],["leaf",62,0.01396155116215835]],[["split",0,25.0,29,0],["split",1,7.0,4,1],["split",2,9.0,3,1],["split",3,3.0,26,2],["split",4,0.0,26,2],["leaf",5,-0.012725290159320413],["split",6,20.0,0,2],["split",7,7.0,0,3],["leaf",8,0.009654672343991581],["split",9,13.0,25,3],["split",10,9.0,24,3],["split",13,2.0,11,3],["leaf",14,0.026698749549439182],["split",15,1.0,6,4],["split",16,3.0,9,4],["split",19,7.0,26,4],["split",20,4.0,8,4],["split",21,4.0,24,4],["split",22,12.0,26,4],["split",27,5.0,11,4],["split",28,1.0,10,4],["leaf",31,-0.004811749221641043],["leaf",32,-0.02622471356511756],["leaf",33,8.75034415196081e-05],["leaf",34,-0.009742521653807297],["leaf",39,0.00035235084035173115],["leaf",40,0.006653816406175062],["leaf",41,0.00026567970558215146],["leaf",42,-0.008267339847424143],["leaf",43,-0.004646823635568124],["leaf",44,-0.01606491268092794],["leaf",45,-0.002388865833939553],["leaf",46,0.011521092153425517],["leaf",55,0.009165406355012418],["leaf",56,0.023306394239796246],["leaf",57,-0.010878474474847175],["leaf",58,0.006377080940646626]],[["split",0,7.0,29,0],["split",1,1.0,26,1],["split",2,2.0,7,1],["split",3,9.0,0,2],["split",4,9.0,16,2],["split",5,10.0,10,2],["split",6,25.0,13,2],["split",7,8.0,6,3],["split",8,7.0,1,3],["split",9,26.0,9,3],["split",10,20.0,0,3],["leaf",11,-0.0004913410703094803],["split",12,6.0,28,3],["split",13,2.0,15,3],["split",14,3.0,8,3],["leaf",15,0.01988949601959875],["split",16,6.0,16,4],["split",17,11.0,6,4],["split",18,12.0,9,4],["split",19,23.0,18,4],["split",20,11.0,21,4],["split",21,24.0,16,4],["leaf",22,-0.02119211238102047],["leaf",25,0.010108366930859349],["leaf",26,0.025504951915431527],["leaf",27,-0.01713876275710136],["leaf",28,0.004830882338298982],["leaf",29,-0.0006627080984355271],["leaf",30,0.011701079013158953],["leaf",33,0.006558326201398029],["leaf",34,-0.005240087705422514],["leaf",35,-0.019206573065068115],["leaf",36,-0.0036650657018148973],["leaf",37,0.002258567150972722],["leaf",38,-0.0012928967872399525],["leaf",39,-0.009439895293511727],["leaf",40,-0.02378076221371823],["leaf",41,-0.009612657956920956],["leaf",42,0.004425868827499804],["leaf",43,-0.004424191865681378],["leaf",44,0.008415474570917859]],[["split",0,26.0,21,0],["split",1,22.0,30,1],["split",2,9.0,27,1],["split",3,7.0,28,2],["split",4,24.0,11,2],["split",5,2.0,8,2],["split",6,0.0,9,2],["split",7,6.0,14,3],["split",8,11.0,22,3],["leaf",9,-0.025832486989276045],["leaf",10,-0.005018888897783401],["split",11,20.0,0,3],["split",12,7.0,9,3],["leaf",13,-0.0013815892511116687],["split",14,7.0,15,3],["split",15,19.0,0,4],["split",16,26.0,2,4],["split",17,20.0,0,4],["split",18,2.0,12,4],["split",23,19.0,0,4],["leaf",24,-0.018266556051922764],["split",25,22.0,8,4],["split",26,8.0,17,4],["split",29,2.0,10,4],["split",30,2.0,8,4],["leaf",31,-0.002064876244882365],["leaf",32,0.006309899805258495],["leaf",33,-0.01336284833333479],["leaf",34,-0.00388543446385488],["leaf",35,0.006979545050770316],["leaf",36,0.023949067688033593],["leaf",37,0.004774774260125672],["leaf",38,-0.020574301734466532],["leaf",47,0.0014721651051222884],["leaf",48,-0.014400503068661467],["leaf",51,-0.015608068643203863],["leaf",52,0.0014041544502278762],["leaf",53,0.01067402797520185],["leaf",54,0.0015066867339790756],["leaf",59,0.015459293296892454],["leaf",60,0.00031601634614187863],["leaf",61,0.008646876827298472],["leaf",62,0.02220321623831708]],[["split",0,2.0,29,0],["split",1,13.0,2,1],["split",2,5.0,26,1],["split",3,21.0,8,2],["split",4,24.0,3,2],["split",5,23.0,26,2],["split",6,26.0,12,2],["split",7,0.0,3,3],["split",8,7.0,5,3],["split",9,8.0,4,3],["split",10,1.0,0,3],["split",11,9.0,12,3],["leaf",12,0.012230199959606608],["leaf",13,-0.01960164796282765],["split",14,3.0,13,3],["leaf",15,0.011433011060286595],["split",16,24.0,16,4],["split",17,11.0,11,4],["leaf",18,0.02941593156112943],["leaf",19,0.010667717099699086],["split",20,2.0,11,4],["leaf",21,-0.014430184082171445],["split",22,9.0,20,4],["leaf",23,-0.010458016059098936],["leaf",24,0.006282542120827823],["leaf",29,0.0008799436409811007],["leaf",30,-0.011709447733165483],["leaf",33,-0.01555603987152062],["leaf",34,0.0031392102506134115],["leaf",35,0.017826260288285645],["leaf",36,0.0010030183097514305],["leaf",41,0.0003293612707511067],["leaf",42,-0.008797447019958934],["leaf",45,0.0020046350483397452],["leaf",46,-0.001051292205444098]],[["split",0,3.0,25,0],["split",1,11.0,0,1],["split",2,17.0,0,1],["split",3,23.0,19,2],["split",4,12.0,29,2],["split",5,1.0,2,2],["split",6,4.0,3,2],["leaf",7,-0.0022411162459319363],["leaf",8,-0.018357201759318112],["split",9,7.0,24,3],["split",10,5.0,11,3],["leaf",11,-0.012059127900588093],["split",12,19.0,0,3],["leaf",13,0.02041204258119682],["split",14,4.0,10,3],["split",19,25.0,1,4],["split",20,9.0,5,4],["split",21,11.0,16,4],["leaf",22,0.001928815367258437],["split",25,22.0,24,4],["split",26,13.0,20,4],["leaf",29,-0.0020561915838077168],["leaf",30,0.01539307927278564],["leaf",39,-0.008861148851115178],["leaf",40,-0.0004404778859110379],["leaf",41,-0.004960111619648839],["leaf",42,0.004964700667441716],["leaf",43,-0.004389823244460427],["leaf",44,-0.018562537134258185],["leaf",51,0.0047369837037324735],["leaf",52,-0.0018670176927581042],["leaf",53,0.0021321793824725984],["leaf",54,-0.012919228043205666]],[["split",0,12.0,0,0],["split",1,9.0,19,1],["split",2,26.0,0,1],["split",3,10.0,14,2],["leaf",4,-0.016176236897944833],["split",5,22.0,3,2],["split",6,13.0,3,2],["leaf",7,0.0009624792778700406],["leaf",8,-0.012082490324068903],["leaf",11,0.003041043563339274],["split",12,8.0,10,3],["split",13,1.0,23,3],["split",14,7.0,5,3],["leaf",25,-0.007664974943112915],["leaf",26,-0.01877123332757636],["split",27,0.0,12,4],["split",28,21.0,12,4],["split",29,0.0,9,4],["split",30,2.0,23,4],["leaf",55,0.0149353127321552],["leaf",56,0.004121005716198814],["leaf",57,-0.02030619001707937],["leaf",58,0.0048521695676659385],["leaf",59,0.0008146829705423773],["leaf",60,-0.00831960313186227],["leaf",61,0.0016519195373645567],["leaf",62,-0.0027385909451600695]],[["split",0,7.0,8,0],["split",1,15.0,0,1],["split",2,7.0,30,1],["split",3,8.0,10,2],["split",4,24.0,13,2],["split",5,9.0,23,2],["split",6,2.0,12,2],["split",7,2.0,28,3],["split",8,5.0,18,3],["split",9,12.0,5,3],["split",10,26.0,16,3],["split",11,0.0,29,3],["split",12,20.0,0,3],["split",13,0.0,15,3],["leaf",14,-0.0016214331255115953],["split",15,12.0,1,4],["split",16,17.0,0,4],["split",17,7.0,6,4],["leaf",18,-0.0009245992490768657],["leaf",19,0.004230596576302423],["leaf",20,-0.012326846496044581],["leaf",21,-0.024546959878742294],["leaf",22,-0.007030677193874383],["split",23,13.0,22,4],["split",24,24.0,21,4],["split",25,24.0,3,4],["split",26,5.0,6,4],["leaf",27,0.02557988617672432],["leaf",28,0.002491267496119061],["leaf",31,-0.005629136725426317],["leaf",32,0.0023049369383173843],["leaf",33,-0.004482933932453077],["leaf",34,-0.025867786130160313],["leaf",35,-0.02198234822346967],["leaf",36,-0.009292742492027434],["leaf",47,0.0013815800141529049],["leaf",48,-0.0026643295965915422],["leaf",49,-0.0009495080977576484],["leaf",50,-0.017226421504752672],["leaf",51,-0.005486548303953032],["leaf",52,0.005842604773905432],["leaf",53,-0.016187026033776922],["leaf",54,0.0012982645915215764]],[["split",0,7.0,30,0],["split",1,8.0,12,1],["split",2,10.0,16,1],["split",3,18.0,0,2],["split",4,23.0,13,2],["split",5,9.0,14,2],["split",6,1.0,26,2],["split",7,26.0,29,3],["split",8,9.0,15,3],["split",9,20.0,0,3],["split",10,26.0,11,3],["leaf",11,0.004385705009230603],["leaf",12,0.026839915371646202],["leaf",13,-0.008271196212509695],["leaf",14,0.011013438245407308],["split",15,20.0,0,4],["leaf",16,0.015162303311910078],["split",17,0.0,10,4],["split",18,1.0,11,4],["split",19,17.0,0,4],["split",20,25.0,20,4],["split",21,20.0,0,4],["split",22,19.0,0,4],["leaf",31,-0.0011279186054796405],["leaf",32,0.006001731374894775],["leaf",35,0.007459499771886659],["leaf",36,-0.0073067046265339036],["leaf",37,0.008536678727331699],["leaf",38,0.026380728062765625],["leaf",39,0.0010707271439443294],["leaf",40,0.008338108747447162],["leaf",41,-0.014616812993549801],["leaf",42,0.0010583777886054903],["leaf",43,-0.012432847218480498],["leaf",44,0.0035441039364366577],["leaf",45,-0.0006738136321225542],["leaf",46,-0.009033098802205465]],[["split",0,7.0,20,0],["split",1,12.0,17,1],["split",2,13.0,10,1],["split",3,23.0,28,2],["split",4,7.0,9,2],["leaf",5,0.019835372480151554],["split",6,23.0,18,2],["split",7,6.0,6,3],["split",8,3.0,22,3],["split",9,24.0,13,3],["split",10,10.0,6,3],["split",13,0.0,26,3],["split",14,4.0,22,3],["split",15,25.0,23,4],["split",16,17.0,0,4],["split",17,26.0,19,4],["leaf",18,0.02263088393090212],["leaf",19,-0.01815912321364737],["split",20,21.0,14,4],["split",21,7.0,16,4],["split",22,21.0,3,4],["split",27,10.0,14,4],["split",28,25.0,22,4],["split",29,1.0,2,4],["split",30,22.0,13,4],["leaf",31,-0.005704577056053237],["leaf",32,0.0007415034105713233],["leaf",33,-0.00047637308135136167],["leaf",34,0.007257864137462141],["leaf",35,0.005751679130437556],["leaf",36,-0.006315111781699142],["leaf",41,0.00955303153869041],["leaf",42,-0.011870581193705224],["leaf",43,-0.013885715331178395],["leaf",44,-0.0036444836794626953],["leaf",45,-0.01043590692266216],["leaf",46,-0.00022309168441654421],["leaf",55,0.0007202552540344218],["leaf",56,0.008434213061685814],["leaf",57,0.0025226353439290394],["leaf",58,-0.012026428684781092],["leaf",59,-0.014112313932340538],["leaf",60,0.0021598882595033615],["leaf",61,-0.0019868254767566994],["leaf",62,-0.0166861983199552]],[["split",0,25.0,30,0],["split",1,1.0,30,1],["split",2,24.0,23,1],["split",3,2.0,1,2],["split",4,12.0,11,2],["split",5,26.0,10,2],["leaf",6,0.023455326835513303],["split",7,25.0,25,3],["split",8,7.0,27,3],["leaf",9,0.0007377619848378811],["split",10,2.0,10,3],["leaf",11,0.011344935889290952],["leaf",12,-0.0011146889785110113],["split",15,13.0,26,4],["leaf",16,0.016520226066324],["split",17,8.0,0,4],["split",18,21.0,28,4],["leaf",21,-0.006497443047692882],["leaf",22,-0.022706722609937564],["leaf",31,0.009278683349620252],["leaf",32,-0.00032675784227212667],["leaf",35,0.006855825356743448],["leaf",36,-0.0009693672136165294],["leaf",37,0.006206099866193796],["leaf",38,-0.00795028658298477]],[["split",0,23.0,30,0],["split",1,24.0,14,1],["split",2,21.0,8,1],["split",3,26.0,6,2],["split",4,25.0,25,2],["leaf",5,0.006569770627133971],["split",6,24.0,19,2],["split",7,25.0,2,3],["split",8,22.0,3,3],["split",9,4.0,20,3],["split",10,4.0,22,3],["leaf",13,-0.007981137872412301],["leaf",14,-0.022512466871229247],["split",15,8.0,16,4],["split",16,11.0,15,4],["split",17,8.0,6,4],["split",18,20.0,0,4],["split",19,1.0,26,4],["split",20,19.0,0,4],["split",21,10.0,21,4],["split",22,11.0,16,4],["leaf",31,-0.023480434884309016],["leaf",32,-0.005486876518177273],["leaf",33,0.00043446153905795477],["leaf",34,-0.0068216895271355086],["leaf",35,0.018902009474100815],["leaf",36,0.004686968395777717],["leaf",37,-0.00012324515526635087],["leaf",38,-0.007874868673626596],["leaf",39,0.0025241842379895053],["leaf",40,-0.0038225749006610622],["leaf",41,-0.003973851678217368],["leaf",42,0.006637008112574955],["leaf",43,-0.00027244672182693966],["leaf",44,0.010418657707950628],["leaf",45,0.005957137331576311],["leaf",46,0.01841146649232015]],[["split",0,24.0,22,0],["split",1,1.0,30,1],["split",2,8.0,0,1],["split",3,4.0,23,2],["split",4,2.0,7,2],["leaf",5,0.019239198490372873],["split",6,21.0,2,2],["split",7,22.0,30,3],["split",8,24.0,18,3],["leaf",9,-0.0010628969582664747],["leaf",10,-0.01692026369219283],["split",13,5.0,16,3],["split",14,22.0,30,3],["split",15,11.0,28,4],["leaf",16,-0.013999422351456157],["split",17,22.0,15,4],["split",18,22.0,11,4],["leaf",27,-0.01560776479172257],["leaf",28,-0.0004626160141548032],["split",29,10.0,3,4],["leaf",30,0.014115498144056002],["leaf",31,0.0011023821157445232],["leaf",32,-0.005923281721814675],["leaf",35,-0.008743506780816084],["leaf",36,-0.0011417833586954257],["leaf",37,0.012674459123147333],["leaf",38,-0.004647208725805572],["leaf",59,-0.0051062978184479],["leaf",60,0.0026058739354426794]],[["split",0,24.0,11,0],["split",1,20.0,0,1],["split",2,4.0,30,1],["split",3,23.0,9,2],["split",4,2.0,8,2],["split",5,11.0,23,2],["split",6,12.0,3,2],["split",7,1.0,13,3],["split",8,12.0,3,3],["leaf",9,-0.02802504007382643],["split",10,23.0,11,3],["split",11,6.0,3,3],["split",12,22.0,12,3],["leaf",13,0.019901071547882482],["leaf",14,0.0008436296554546272],["split",15,22.0,23,4],["split",16,23.0,3,4],["split",17,26.0,5,4],["split",18,7.0,2,4],["leaf",21,-0.015853645659607573],["split",22,4.0,24,4],["split",23,21.0,17,4],["split",24,21.0,29,4],["split",25,26.0,18,4],["split",26,13.0,3,4],["leaf",31,-0.0002937600249870603],["leaf",32,-0.013102185363213226],["leaf",33,0.0011273825294756056],["leaf",34,0.010668987172068602],["leaf",35,-0.004353173809764078],["leaf",36,0.01125339009421055],["leaf",37,-0.017257927904900838],["leaf",38,-0.0038319805158688164],["leaf",45,0.01225234417706545],["leaf",46,-0.0035240788654009795],["leaf",47,-0.01485459615910131],["leaf",48,0.002323730275389367],["leaf",49,0.0011475586265551631],["leaf",50,-0.008172475256116316],["leaf",51,0.011963482809312623],["leaf",52,0.0014275737344977785],["leaf",53,0.01709940277786508],["leaf",54,-0.0005698192711078999]],[["split",0,25.0,2,0],["split",1,6.0,2,1],["split",2,7.0,26,1],["leaf",3,0.01825004548451784],["split",4,7.0,23,2],["split",5,13.0,12,2],["split",6,19.0,0,2],["split",9,22.0,6,3],["split",10,13.0,25,3],["split",11,20.0,0,3],["split",12,3.0,10,3],["split",13,22.0,26,3],["split",14,24.0,13,3],["leaf",19,0.00761712050438109],["split",20,10.0,17,4],["split",21,23.0,16,4],["leaf",22,0.00026177119849831934],["split",23,2.0,5,4],["split",24,2.0,18,4],["split",25,8.0,18,4],["split",26,23.0,9,4],["split",27,26.0,2,4],["split",28,24.0,13,4],["leaf",29,-0.015506328082814307],["leaf",30,0.0030651057275102975],["leaf",41,0.002573937001094918],["leaf",42,-0.008124059690294465],["leaf",43,0.01146774717647772],["leaf",44,0.02395185643797759],["leaf",47,0.013131074727054836],["leaf",48,-0.0014770660594634049],["leaf",49,-0.00557855544462045],["leaf",50,0.016635929221859407],["leaf",51,-0.004408489046022814],["leaf",52,-0.014194626237991418],["leaf",53,0.001387676915666422],["leaf",54,-0.0027504568630816744],["leaf",55,0.013778465796625426],["leaf",56,0.00423196401151115],["leaf",57,-0.010017650138990754],["leaf",58,0.003174272386119762]],[["split",0,11.0,0,0],["split",1,1.0,9,1],["split",2,23.0,26,1],["leaf",3,0.0222795657634487],["split",4,13.0,14,2],["split",5,7.0,0,2],["split",6,8.0,13,2],["leaf",9,-0.008138607594923894],["leaf",10,0.007563585102914947],["leaf",11,-0.011856739366339313],["split",12,21.0,16,3],["split",13,17.0,0,3],["split",14,5.0,17,3],["split",25,20.0,0,4],["split",26,7.0,26,4],["split",27,25.0,8,4],["leaf",28,-0.015736482299106586],["split",29,11.0,11,4],["split",30,2.0,17,4],["leaf",51,-7.553345719428457e-05],["leaf",52,-0.008868976860272717],["leaf",53,0.0009289542450100854],["leaf",54,0.007640432110329354],["leaf",55,-0.007795491272798801],["leaf",56,0.005594647868534059],["leaf",59,-0.013165740292723882],["leaf",60,-0.0012483454808800683],["leaf",61,-0.03365589732806956],["leaf",62,-0.012026966314930788]],[["split",0,13.0,5,0],["split",1,7.0,11,1],["split",2,3.0,7,1],["split",3,12.0,17,2],["leaf",4,0.013478941492056943],["split",5,20.0,0,2],["split",6,1.0,0,2],["split",7,19.0,0,3],["leaf",8,-0.016032029275256827],["split",11,22.0,1,3],["split",12,21.0,9,3],["leaf",13,-0.015584538180492354],["split",14,7.0,29,3],["split",15,3.0,8,4],["split",16,3.0,7,4],["leaf",23,-0.022381778145972284],["split",24,9.0,27,4],["leaf",25,-0.011375957173131645],["leaf",26,0.016256558571400778],["split",29,25.0,30,4],["split",30,22.0,23,4],["leaf",31,-0.002115602135550538],["leaf",32,0.004495073089680839],["leaf",33,0.019354587844093597],["leaf",34,0.002066257539263463],["leaf",49,-0.003916609448707242],["leaf",50,-0.019114311803829668],["leaf",59,-0.0007351800143151661],["leaf",60,0.009359972161434298],["leaf",61,0.011675805808128948],["leaf",62,-0.0017070695748241922]],[["split",0,11.0,21,0],["split",1,11.0,16,1],["split",2,22.0,4,1],["split",3,0.0,29,2],["split",4,1.0,25,2],["split",5,26.0,21,2],["split",6,25.0,26,2],["split",7,4.0,3,3],["split",8,11.0,9,3],["split",9,23.0,28,3],["split",10,0.0,8,3],["split",11,23.0,21,3],["leaf",12,0.01832666310063609],["split",13,8.0,4,3],["split",14,13.0,3,3],["split",15,22.0,8,4],["split",16,7.0,19,4],["split",17,5.0,20,4],["leaf",18,0.002810877997721514],["split",19,23.0,4,4],["leaf",20,-0.009499352089958857],["leaf",21,-0.000387463861096944],["split",22,26.0,17,4],["split",23,5.0,17,4],["leaf",24,-0.00260941468432014],["split",27,5.0,27,4],["split",28,12.0,16,4],["leaf",29,-0.005126149970317097],["split",30,2.0,20,4],["leaf",31,0.012864785420588198],["leaf",32,0.003075803182069676],["leaf",33,-0.0001286054888757718],["leaf",34,0.00410051827382742],["leaf",35,-0.01726545848783362],["leaf",36,-0.000396163875794911],["leaf",39,-0.006930444339262251],["leaf",40,0.0021650222840342015],["leaf",45,-0.022640154145189894],["leaf",46,-0.00838243493633831],["leaf",47,0.0011339879381220255],["leaf",48,0.018432894033498282],["leaf",55,0.014320817879683664],["leaf",56,0.00012587321120647918],["leaf",57,-0.0033299197808356515],["leaf",58,0.002542736367493443],["leaf",61,0.004379501830469199],["leaf",62,0.01870589143653758]],[["split",0,21.0,2,0],["split",1,12.0,16,1],["split",2,10.0,3,1],["split",3,7.0,10,2],["split",4,4.0,5,2],["split",5,12.0,28,2],["split",6,22.0,7,2],["split",7,2.0,13,3],["split",8,25.0,13,3],["leaf",9,0.0037617098865060407],["split",10,5.0,10,3],["split",11,3.0,12,3],["leaf",12,-0.016259038097492604],["split",13,20.0,0,3],["split",14,1.0,0,3],["leaf",15,-0.017476995823883058],["split",16,3.0,4,4],["leaf",17,-0.0031764244459928713],["leaf",18,0.009664322938744613],["leaf",21,-0.024391071475752545],["split",22,26.0,6,4],["split",23,22.0,4,4],["split",24,5.0,21,4],["split",27,9.0,5,4],["split",28,2.0,19,4],["leaf",29,-0.011661703511533126],["split",30,4.0,0,4],["leaf",33,-0.011256801314275991],["leaf",34,0.00012381806148028688],["leaf",45,-0.015697727136902755],["leaf",46,-0.004780759078410208],["leaf",47,-0.016865749972592894],["leaf",48,-0.0052862380986088454],["leaf",49,-0.00031220130753021463],["leaf",50,0.009201657972114374],["leaf",55,0.00945705852016225],["leaf",56,0.0006668746369829435],["leaf",57,0.004254897792931762],["leaf",58,0.029198873460692002],["leaf",61,0.009087086490312202],["leaf",62,-0.0001613369896948414]],[["split",0,21.0,30,0],["split",1,7.0,23,1],["split",2,5.0,21,1],["split",3,6.0,0,2],["split",4,10.0,27,2],["split",5,13.0,18,2],["leaf",6,-0.0022118015700354834],["split",7,3.0,5,3],["split",8,26.0,2,3],["split",9,19.0,0,3],["split",10,11.0,15,3],["leaf",11,-0.023506866549171834],["leaf",12,-0.005012080041703712],["split",15,0.0,16,4],["leaf",16,0.02407480109770229],["split",17,22.0,25,4],["split",18,10.0,0,4],["split",19,11.0,8,4],["split",20,9.0,16,4],["split",21,7.0,27,4],["split",22,0.0,13,4],["leaf",31,0.007318247935617712],["leaf",32,-0.007099720557490551],["leaf",35,-0.004857100155417623],["leaf",36,-0.025291892269494354],["leaf",37,-0.008311119944346137],["leaf",38,-0.00012788940424478512],["leaf",39,0.000385242298656061],["leaf",40,0.008008850455888063],["leaf",41,-0.009486917538389194],["leaf",42,0.0036174137856459375],["leaf",43,-0.0014346285549257957],["leaf",44,0.01020204213545846],["leaf",45,-0.0049125925335250726],["leaf",46,-0.02115295239734458]]]}""")


def _main_a(datasources, start_date, end_date):
    """【新根因子 第三轮·多根合成 1399】合成头部加九根含六十九十分钟资金流精简按月头部按季

    B = 490 头部原始值 + Σ w·rank(根)，09:30 置 0，股票池骨架、缺失填 0。根来自多统计量框架 1340_资金流窗口框架.py（原样复用），
    以 L_ 开头的季节性根先在同一股票连续截面上做 EWMA（α=0.75）再排名。各根含义：
      L_lab10  同时段季节性（下一个半小时段过去 10 日"收盘 ÷ VWAP − 1"均值）
      M_q_lin  聪明钱（|r|/v 加权的当日 VWAP ÷ 当日 VWAP − 1）
      N_dir    大单方向（Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|）
      W_vdir   量价同向（Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|）
      W_zero   当日零收益分钟占比
      Y_sd5    近 5 日单笔成交额离散度
      X_n5d    近 5 日大单方向
      Z_mfr60  近 60 分钟上涨分钟成交量占比（资金流）
      Z_mfr90  近 90 分钟上涨分钟成交量占比（资金流）
    权重：{"L_lab10": +0.20, "M_q_lin": +0.10, "N_dir": -0.10, "W_vdir": -0.10, "W_zero": +0.10, "Y_sd5": +0.10, "X_n5d": -0.10, "Z_mfr60": -0.15, "Z_mfr90": -0.10}。口径与评测机 values 合成（build_values_composite.py raw 模式）一致。
    """
    import numpy as np
    import pandas as pd
    import dai

    W = {"L_lab10": +0.20, "M_q_lin": +0.10, "N_dir": -0.10, "W_vdir": -0.10, "W_zero": +0.10, "Y_sd5": +0.10, "X_n5d": -0.10, "Z_mfr60": -0.15, "Z_mfr90": -0.10}
    ALPHA = 0.75
    MONTHLY = True                                                       # True：分钟类根按月分批（控制平台整段调用的内存）
    HEADQ = True                                                           # True：头部按自然季度分批
    _cache = {}
    _roots = {}
    _mparts = {}
    if MONTHLY:                                                               # 分钟类根按月分批：每月一次分钟 SQL，算完该月所有根就释放
        _mins = [st for st in W if not st.startswith("L_")]
        for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
            _s = max(pd.Timestamp(start_date), _ms.start_time)
            _e = min(pd.Timestamp(end_date), _ms.end_time)
            if _s > _e:
                continue
            _c = {}
            for st in _mins:
                _mparts.setdefault(st, []).append(
                    _root_impl(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), st, _c))
            del _c
    for st in W:
        if st.startswith("L_"):
            warm = (pd.to_datetime(start_date) - pd.Timedelta(days=21)).strftime("%Y-%m-%d %H:%M:%S")
            x = _root_impl(datasources, warm, end_date, st, {})
            x["date"] = pd.to_datetime(x["date"])
            x = x.sort_values(["instrument", "date"]).reset_index(drop=True)
            x["factor"] = x.groupby("instrument", sort=False)["factor"].transform(
                lambda s: s.ewm(alpha=1 - ALPHA, adjust=False).mean())
            x = x[x["date"] >= pd.Timestamp(start_date)]
        elif MONTHLY:
            x = pd.concat(_mparts.pop(st), ignore_index=True)
            x["date"] = pd.to_datetime(x["date"])
        else:
            x = _root_impl(datasources, start_date, end_date, st, _cache)
            x["date"] = pd.to_datetime(x["date"])
        x["instrument"] = x["instrument"].astype(str)
        x["r_" + st] = x.groupby("date")["factor"].rank(pct=True) - 0.5
        _roots[st] = x[["date", "instrument", "r_" + st]]
    import gc
    gc.collect()                                                              # 先算完各根（只留很小的结果），最后才算头部：峰值≈头部本身
    if HEADQ:                                                                 # 头部按自然季度循环（与评测机按季度评分逐位一致）
        _hp = []
        for _q in pd.period_range(pd.Timestamp(start_date).to_period("Q"), pd.Timestamp(end_date).to_period("Q"), freq="Q"):
            _s = max(pd.Timestamp(start_date), _q.start_time)
            _e = min(pd.Timestamp(end_date), _q.end_time)
            if _s > _e:
                continue
            _hp.append(_head_impl(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S")))
            gc.collect()
        _head = pd.concat(_hp, ignore_index=True).drop_duplicates(["date", "instrument"])
        del _hp
    else:
        _head = _head_impl(datasources, start_date, end_date)
    _head = _head.rename(columns={"factor": "fh"})
    _head["date"] = pd.to_datetime(_head["date"])
    _head["instrument"] = _head["instrument"].astype(str)
    _m = _head
    for st, x in _roots.items():
        _m = _m.merge(x, on=["date", "instrument"], how="left")
        _m["r_" + st] = _m["r_" + st].fillna(0.0)
    _m["factor"] = _m["fh"].astype("float64")
    for st, w in W.items():
        _m["factor"] = _m["factor"] + w * _m["r_" + st]

    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    _out = _pool.merge(_m[["date", "instrument", "factor"]].drop_duplicates(["date", "instrument"]), on=["date", "instrument"], how="left")
    _out["factor"] = _out["factor"].astype("float64").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    _out.loc[_out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return _out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _head_impl(datasources, start_date, end_date):
    """长窗口软同伴（240 日）+ 六特征 1.5 倍中性化（新增换手率与市值二次项）+（09:30 置零版）。

    148（换手率特征，0.834）与 149（市值二次项，0.834）各自与底座持平，这里同时放进中性化特征集，
      检验两个「官方风格未覆盖的暴露」一起剔除后能否超过 144。
    09:30 修复：同 104 号对照版本。"""
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 400
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS, PEER_POWER = 200, 240, 120, 2.0
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(open, 0) AS DOUBLE) AS px_open,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt,
            CAST(COALESCE(deal_number, 0) AS DOUBLE) AS deals
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            deals,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN px_high - px_low ELSE 0 END AS rng_c,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN px_high - GREATEST(px_open, px_close) ELSE 0 END AS upsh_c,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN ABS(px_close - px_open) ELSE 0 END AS body_c,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            SUM(body_c) AS sum_body,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment
                      OR (time_segment = 93000 AND hms = '092500') THEN px ELSE 0 END) AS last_px,
            MAX(pre_close) AS pre_close,
            MAX(CASE WHEN px_high > 0 THEN px_high END) AS win_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS win_low,
            SUM(rng_c) AS sum_rng,
            SUM(upsh_c) AS sum_upsh,
            SUM(deals) AS sum_deals
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.sum_body, f.last_px, f.pre_close, f.win_high, f.win_low, f.sum_rng, f.sum_upsh, f.sum_deals
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "sum_body", "last_px", "pre_close", "win_high", "win_low", "sum_rng", "sum_upsh", "sum_deals"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    exposure_table = datasources.get("exposure", "cpt_jyc_2026_exposure")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    # 官方暴露表（日频）里的流通市值：用于构造「官方风格没有覆盖」的中性化特征。
    _exp = dai.query(
        "SELECT date, instrument, float_market_cap FROM %s" % exposure_table,
        filters={"date": [query_start, end_date]}, compression=True,
    ).df()
    _exp["day"] = pd.to_datetime(_exp["date"]).dt.normalize()
    _exp["instrument"] = _exp["instrument"].astype(str)
    _mcap = _exp.groupby(["instrument", "day"], sort=False)["float_market_cap"].last()
    # 暴露表第 t 天的流通市值按当天 15:00 收盘价计算，盘中截面不可得：一律取该股票前一交易日的值。
    _mcap = _mcap.sort_index().groupby(level=0).shift(1)
    _midx = pd.MultiIndex.from_arrays([df["instrument"].values, df["day"].values])
    df["mcap"] = pd.Series(_mcap.reindex(_midx).to_numpy(), index=df.index)
    df["mcap"] = df["mcap"].where(df["mcap"] > 0)
    # 官方 10 风格暴露 + 行业码（与平台标签残差化口径一致），用于因子自身的中性化测试。
    _st = dai.query(
        "SELECT date, instrument, SIZE, BETA, MOMENTUM, RESVOL, SIZENL, BTOP, LIQUIDTY, EARNYILD, GROWTH, LEVERAGE, industry_level1_code FROM %s" % exposure_table,
        filters={"date": [query_start, end_date]}, compression=True,
    ).df()
    _st["day"] = pd.to_datetime(_st["date"]).dt.normalize()
    _st["instrument"] = _st["instrument"].astype(str)
    _st = _st.drop_duplicates(["instrument", "day"])
    # 风格暴露与行业同样取前一交易日的值（第 t 天的暴露在当天收盘后才可得）。
    _st = _st.sort_values(["instrument", "day"]).reset_index(drop=True)
    for _c in ["SIZE", "BETA", "MOMENTUM", "RESVOL", "SIZENL", "BTOP", "LIQUIDTY", "EARNYILD", "GROWTH", "LEVERAGE", "industry_level1_code"]:
        _st[_c] = _st.groupby("instrument")[_c].shift(1)
    _st = _st.dropna(subset=["industry_level1_code"])
    _sg = _st.set_index(["instrument", "day"])
    _sid = pd.MultiIndex.from_arrays([df["instrument"].values, df["day"].values])
    for _c in ['SIZE', 'BETA', 'MOMENTUM', 'RESVOL', 'SIZENL', 'BTOP', 'LIQUIDTY', 'EARNYILD', 'GROWTH', 'LEVERAGE']:
        df["s_" + _c] = pd.Series(_sg[_c].reindex(_sid).to_numpy(), index=df.index)
    _ind = _sg["industry_level1_code"].astype(str)
    for _c in sorted(_ind.dropna().unique()):
        df["i_" + _c] = (_ind.reindex(_sid).to_numpy() == _c).astype(float)

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))

    # 换手率代理：最近 40 个窗口成交额 ÷ 流通市值（官方风格里没有直接的换手率暴露）。
    df["c_turn"] = roll(df["sum_amt"], 40, "sum", 10) / df["mcap"]

    # 流通市值的非线性项：ln(流通市值) 的平方（官方 SIZE 风格只覆盖线性部分）。
    df["c_mcap2"] = np.log(df["mcap"]) ** 2
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 分钟上影线占比：窗口内 Σ(最高-实体上沿)/Σ(最高-最低)，40 窗口均值取负（冲高回落越多，反转越强）。
    _upsh = df["sum_upsh"] / df["sum_rng"].where(df["sum_rng"] > 0)
    df["c_body"] = -roll(df["sum_body"] / df["sum_rng"].where(df["sum_rng"] > 0), 40, "mean", 10)
    df["c_upsh"] = -_upsh.groupby(df["instrument"], sort=False).transform(lambda x: x.ewm(span=10, min_periods=8).mean())
    # 平均单笔成交额：Σ成交额/Σ委托笔数 的 40 窗口均值（大单/小单主导窗口的代理）。
    _avgtr = df["sum_amt"] / df["sum_deals"].where(df["sum_deals"] > 0)
    df["c_avgtr"] = roll(_avgtr, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的日收益（收盘段价格 / 昨收）计算相关系数。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    daily_amt = df.groupby(["day", "instrument"], sort=False)["sum_amt"].sum().unstack("instrument").reindex(columns=peer_insts).sort_index()
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    peers_by_month = {}
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    for m in out_months:
        in_month = all_days[day_month == m]
        if len(in_month) == 0:
            continue
        hist = daily_ret.loc[daily_ret.index < in_month[0]].tail(PEER_LOOKBACK)
        if len(hist) < PEER_MIN_OBS:
            continue
        ok = (hist.notna().sum() >= PEER_MIN_OBS).values
        corr = hist.loc[:, ok].corr(min_periods=PEER_MIN_OBS).values
        np.fill_diagonal(corr, np.nan)
        corr = np.nan_to_num(corr, nan=-1.0)          # 缺失相关性记 -1（等价于不选）
        sub = np.where(ok)[0]
        top = np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]
        idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
        idx[sub] = sub[top]
        cvals = corr[np.arange(len(corr))[:, None], top]
        # 规模加权：相关度^p × 该同伴前 PEER_LOOKBACK 日的平均成交额（大票的共同运动信息权重更高）。
        _aw = daily_amt.loc[daily_amt.index < in_month[0]].tail(PEER_LOOKBACK).mean().fillna(0.0).values
        _aw = np.where(_aw > 0, _aw, 0.0)
        wts = (np.clip(cvals, 0.0, None) ** PEER_POWER) * _aw[sub][:, None]
        wsum = wts.sum(axis=1, keepdims=True)
        wts = np.where(wsum > 0, wts / np.where(wsum > 0, wsum, 1.0), 0.0)
        full_w = np.zeros((len(peer_insts), PEER_K))
        full_w[sub] = wts
        peers_by_month[m] = (idx, full_w)

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_30", "r_rv4", "r_rv8", "r_day", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_turn", "c_mcap2", "c_upsh", "c_avgtr", "r_rv16", "c_body"]
    keep = keep + sorted([c for c in df.columns if c.startswith("i_")])
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col):
        """每个截面上同伴的 col 均值（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            rec = peers_by_month.get(m)
            if rec is None:
                continue
            idx, wts = rec
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            ww = np.where(valid[None, :, :], wts[None, :, :], 0.0)
            okm = ~np.isnan(g)
            cnt = okm.sum(axis=2)
            wsum = (ww * okm).sum(axis=2)
            pm = np.where((cnt >= 3) & (wsum > 0), np.nansum(np.where(okm, g, 0.0) * ww, axis=2) / np.where(wsum > 0, wsum, 1.0), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    def peer_ladder():
        cols = []
        for c in ["r_30", "r_rv4", "r_rv8", "r_day"]:
            result[c + "_p"] = result[c] - peer_mean(c).fillna(result.groupby("date")[c].transform("mean"))
            cols.append(c + "_p")
        return rank_blend(cols)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx", "c_turn", "c_mcap2"] + sorted([c for c in df.columns if c.startswith("i_")])
    result["ladder"] = peer_ladder()
    result["signal"] = rank_blend(["ladder", "c_upsh", "c_avgtr", "r_rv16", "c_body"], [1.0, 0.30, 0.15, 0.20, 0.15])
    result["factor"] = neutralize("signal", CHARS4, 2.5)
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    # 09:30 输出 0（与 104/103/107 同口径）：平台跳过全 0 截面。
    result.loc[result["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def _root_impl(datasources, start_date, end_date, STAT, _cache):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *, ABS(r) / GREATEST(v, 1) AS s2,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(r * (lats - lats_prev)) OVER w AS cnd,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * (lv - lv_prev)) OVER w AS cvd, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr, SUM(ABS(r)) OVER w5 AS a5d, SUM(r * (lats - lats_prev)) OVER w5 AS nd5, STDDEV_SAMP(lats) OVER w5 AS sd5,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        if "min" not in _cache:                                                    # 同一调用内只查一次
            _cache["min"] = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = _cache["min"].copy()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q_lin": lambda: (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
                "N_dir": lambda: df["cnd"] / cabs,
                "W_vdir": lambda: df["cvd"] / cabs,
                "X_n5d": lambda: df["nd5"] / df["a5d"].where(df["a5d"] > 0),
                "Y_sd5": lambda: df["sd5"],
                "Z_mfr60": lambda: df["vup60"] / df["v60"].where(df["v60"] > 0),
                "Z_mfr90": lambda: df["vup90"] / df["v90"].where(df["v90"] > 0),
                "W_zero": lambda: df["czero"] / df["cnr"].where(df["cnr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b1(datasources, start_date, end_date):
    """【1398：临时L_lab10】新根单独版（公榜探针），STAT = L_lab10，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b1(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "L_lab10"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b1(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                ABS(r) / SQRT(GREATEST(v, 1)) AS s1, ABS(r) / GREATEST(v, 1) AS s2, ABS(r) / POW(GREATEST(v, 1), 0.75) AS s3, ABS(r) / GREATEST(a, 1) AS s4,
                CASE WHEN bo > 0 AND ao > 0 AND bv > 0 AND av > 0 THEN LN((bv / bo) / (av / ao)) END AS osz,
                LN((bv1 + 1) / (av1 + 1)) AS q1,
                CASE WHEN ap1 > 0 AND bp1 > 0 AND ap1 >= bp1 THEN (ap1 - bp1) / ((ap1 + bp1) / 2) END AS spr,
                r * LAG(r) OVER (PARTITION BY instrument_id, d ORDER BY date) AS rr1,
                AVG(r * r) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS rv_prev,
                COUNT(r) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS n_prev,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev,
                CASE WHEN bo + ao > 0 THEN (bo - ao) / (bo + ao) END AS cimb,
                CASE WHEN h >= l AND c > 0 THEN (h - l) / c END AS hlr
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(s1 * a) OVER w AS cs1a, SUM(s1 * v) OVER w AS cs1v, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(r * (lats - lats_prev)) OVER w AS cnd,
                SUM(CASE WHEN lats > lats_prev THEN r ELSE 0 END) OVER w AS cbig,
                SUM(CASE WHEN lats <= lats_prev THEN r ELSE 0 END) OVER w AS csmall,
                SUM(ABS(r)) OVER w AS cabs, SUM(r) OVER w AS csum, SUM(r * r) OVER w AS crr, SUM(rr1) OVER w AS crr1,
                SUM(CASE WHEN n_prev >= 10 AND ABS(r) > 4 * SQRT(rv_prev) THEN 1 ELSE 0 END) OVER w AS cnj,
                SUM(CASE WHEN n_prev >= 10 AND ABS(r) > 4 * SQRT(rv_prev) THEN r ELSE 0 END) OVER w AS csj,
                AVG(osz) OVER w AS mosz,
                AVG(osz) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS mosz30,
                AVG(q1) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS mq5,
                AVG(spr) OVER w AS mspr,
                SUM(r * (lv - lv_prev)) OVER w AS cvd, AVG(cimb) OVER w AS mcimb,
                SUM(v * v) OVER w AS cvv, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr,
                STDDEV_SAMP(lats) OVER w AS sdlats, SUM(hlr) OVER w AS chl,
                SUM(s3 * a) OVER w AS cs3a, SUM(s3 * v) OVER w AS cs3v, SUM(s4 * a) OVER w AS cs4a, SUM(s4 * v) OVER w AS cs4v,
                SUM(r) OVER w60 AS r60, SUM(ABS(r)) OVER w60 AS a60, SUM(r) OVER w30 AS r30, SUM(ABS(r)) OVER w30 AS a30,
                SUM(r) OVER w5 AS r5d, SUM(ABS(r)) OVER w5 AS a5d, SUM(r * (lats - lats_prev)) OVER w5 AS nd5,
                SUM(a) OVER w5 AS a_5, SUM(v) OVER w5 AS v_5, SUM(s2 * a) OVER w5 AS s2a_5, SUM(s2 * v) OVER w5 AS s2v_5,
                SUM(r * (lv - lv_prev)) OVER w5 AS vd5, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w5 AS z5,
                COUNT(r) OVER w5 AS n5, STDDEV_SAMP(lats) OVER w5 AS sd5,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w AS vup, SUM(v) OVER w AS vall,
                SUM(CASE WHEN c >= ap1 THEN a WHEN c <= bp1 THEN -a ELSE 0 END) OVER w AS abid, SUM(a) OVER w AS aall,
                SUM(CASE WHEN r > 0 THEN a ELSE 0 END) OVER w AS aup, SUM(CASE WHEN r > 0 THEN n ELSE 0 END) OVER w AS nup,
                SUM(CASE WHEN r < 0 THEN a ELSE 0 END) OVER w AS adn, SUM(CASE WHEN r < 0 THEN n ELSE 0 END) OVER w AS ndn,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w30 AS vup30, SUM(v) OVER w30 AS v30,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w120 AS vup120, SUM(v) OVER w120 AS v120,
                SUM(CASE WHEN r < 0 THEN v ELSE 0 END) OVER w60 AS vdn60,
                SUM(CASE WHEN r > 0 THEN a ELSE 0 END) OVER w60 AS aup60, SUM(a) OVER w60 AS am60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60x AS vup60x, SUM(v) OVER w60x AS v60x
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q": (df["cs1a"] / df["cs1v"].where(df["cs1v"] > 0)) / vw - 1,
                "M_q_lin": (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
                "N_dir": df["cnd"] / cabs,
                "N_big": df["cbig"] - df["csmall"],
                "Q_eff": df["csum"].abs() / cabs,
                "Q_seff": df["csum"] / cabs,
                "Q_ac1": df["crr1"] / df["crr"].where(df["crr"] > 0),
                "U_nj": df["cnj"].astype(float),
                "U_sj": df["csj"],
                "S_osz": df["mosz"],
                "S_osz30": df["mosz30"],
                "O_q5": df["mq5"],
                "R_round": (df["c"] - np.round(df["c"])) / df["c"],
                "R_half": (df["c"] - np.round(df["c"] * 2) / 2) / df["c"],
                "V_spr": df["mspr"],
                "W_vdir": df["cvd"] / cabs,
                "X_q60": df["r60"] / df["a60"].where(df["a60"] > 0),
                "X_q30": df["r30"] / df["a30"].where(df["a30"] > 0),
                "X_q5d": df["r5d"] / df["a5d"].where(df["a5d"] > 0),
                "X_n5d": df["nd5"] / df["a5d"].where(df["a5d"] > 0),
                "X_m5d": (df["s2a_5"] / df["s2v_5"].where(df["s2v_5"] > 0)) / (df["a_5"] / df["v_5"].where(df["v_5"] > 0)) - 1,
                "X_m75": (df["cs3a"] / df["cs3v"].where(df["cs3v"] > 0)) / vw - 1,
                "X_ma": (df["cs4a"] / df["cs4v"].where(df["cs4v"] > 0)) / vw - 1,
                "Y_vd5": df["vd5"] / df["a5d"].where(df["a5d"] > 0),
                "Y_z5": df["z5"] / df["n5"].where(df["n5"] > 0),
                "Y_sd5": df["sd5"],
                "Y_mfr": df["vup"] / df["vall"].where(df["vall"] > 0),
                "Y_bid": df["abid"] / df["aall"].where(df["aall"] > 0),
                "Z_aup": df["aup"] / df["aall"].where(df["aall"] > 0),
                "Z_mfr60": df["vup60"] / df["v60"].where(df["v60"] > 0),
                "Z_ats": np.log((df["aup"] / df["nup"].where(df["nup"] > 0)) / (df["adn"] / df["ndn"].where(df["ndn"] > 0))),
                "Z_mfr30": df["vup30"] / df["v30"].where(df["v30"] > 0),
                "Z_mfr90": df["vup90"] / df["v90"].where(df["v90"] > 0),
                "Z_mfr120": df["vup120"] / df["v120"].where(df["v120"] > 0),
                "Z_net60": (df["vup60"] - df["vdn60"]) / df["v60"].where(df["v60"] > 0),
                "Z_amfr60": df["aup60"] / df["am60"].where(df["am60"] > 0),
                "Z_mfr60x": df["vup60x"] / df["v60x"].where(df["v60x"] > 0),
                "W_cnt": df["mcimb"],
                "W_hhi": df["cvv"] / (df["cv"] ** 2).where(df["cv"] > 0),
                "W_zero": df["czero"] / df["cnr"].where(df["cnr"] > 0),
                "W_atsd": df["sdlats"],
                "W_hl": df["chl"] / df["csum"].abs().where(df["csum"].abs() > 0),
            }[STAT.split("_res")[0]]
        df["signal"] = np.asarray(sig, dtype=float)
        if "_res" in STAT:                                                   # 逐截面对当日收益（_res2 再加量价同向）回归取残差
            Xr = [df["csum"].to_numpy(float)]
            if STAT.endswith("_res2"):
                Xr.append((df["cvd"] / cabs).to_numpy(float))
            Xr = np.column_stack(Xr)
            y_ = df["signal"].to_numpy(float)
            out_ = np.full(len(df), np.nan)
            for _, ii in df.groupby("date").indices.items():
                ok_ = np.isfinite(y_[ii]) & np.isfinite(Xr[ii]).all(1)
                if ok_.sum() < 50:
                    continue
                A_ = np.column_stack([np.ones(ok_.sum()), Xr[ii][ok_]])
                coef_, *_ = np.linalg.lstsq(A_, y_[ii][ok_], rcond=None)
                out_[ii[ok_]] = y_[ii][ok_] - A_ @ coef_
            df["signal"] = out_
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b2(datasources, start_date, end_date):
    """【1398：临时M_q_lin】新根单独版（公榜探针），STAT = M_q_lin，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b2(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "M_q_lin"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b2(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *, ABS(r) / GREATEST(v, 1) AS s2
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(ABS(r)) OVER w AS cabs
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q_lin": lambda: (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b3(datasources, start_date, end_date):
    """【1398：临时W_vdir】新根单独版（公榜探针），STAT = W_vdir，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b3(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "W_vdir"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b3(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * (lv - lv_prev)) OVER w AS cvd
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "W_vdir": lambda: df["cvd"] / cabs,
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b4(datasources, start_date, end_date):
    """【1398：临时W_zero】新根单独版（公榜探针），STAT = W_zero，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b4(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "W_zero"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b4(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "W_zero": lambda: df["czero"] / df["cnr"].where(df["cnr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b5(datasources, start_date, end_date):
    """【1398：临时Z_mfr60】新根单独版（公榜探针），STAT = Z_mfr60，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b5(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "Z_mfr60"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b5(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "Z_mfr60": lambda: df["vup60"] / df["v60"].where(df["v60"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b6(datasources, start_date, end_date):
    """【1398：临时Z_mfr90】新根单独版（公榜探针），STAT = Z_mfr90，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b6(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "Z_mfr90"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b6(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "Z_mfr90": lambda: df["vup90"] / df["v90"].where(df["v90"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b7(datasources, start_date, end_date):
    """
    152（倍增阶梯 + 四类价格锚点四网络 IQR + 低波 / 小盘 / 低价）的 IQR 内加四个盘口 / 成交价锚点分量。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    155 在 152 号基础上只在 IQR 内部加四个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_qmid  : ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)，涨跌停单边盘口的窗口不参与锚点、当期缺失；
      r_cmid  : 最近 40 个窗口 ln(窗口末成交价 / 窗口末当前价) 的均值，成交持续落在买一侧（卖压）为正；
      r_deep  : ln(最近 8 个窗口末价按三档挂单总量加权的均价 / 当前价)，挂单厚的窗口价格更可信；
      r_micro : ln(最近 8 个窗口成交 VWAP / 当前一档微价格)，微价格按对侧挂单量加权买一 / 卖一价。
    四个分量来自第二十六批“锚点”信号的分量级筛选（逐个及组合作为新分量放进 152 的 IQR），单独加入均为正，
    四个一起加入估计分差最高；窗口长度沿用已有的 8 与 40，无新增参数，所有候选统一用同一套网络与聚合。
    没有当前价（09:30 截面）时四个分量都缺失，该截面与 152 一样整体补 0。14 分量 x 四网络 = 56 列做 IQR Winsor 均值。
      验证段 IC 0.08401（152 0.08417）；ICIR 1.8031（1.7965）；Sharpe 53.38（53.06）；压力 1.7983（1.7921）；换手 0.3813（0.3892）。
    以下为 152 的说明。
    更长倍增阶梯 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    152 在 151 号基础上去掉两个分量 r_lo（当日低点距离）与 r_slope（VWAP 斜率），其余（同伴网络、中性化、聚合、慢特征融合）全部不变，
    即 146 的 6 个阶梯分量 + r_med / r_mid / r_lomed / r_calm 四个价格锚点，10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    原因：窗口末价格取截面那一分钟，09:30 截面没有这一分钟（1 分钟数据只有 09:25 竞价和 09:31 之后），当前价缺失。
    146 在该截面所有分量缺失、整体补 0 而不参与评估；r_slope 只用成交额与成交量、不含当前价，147–151 在该截面只剩它一个分量，
    该截面 IC 仅约 0.02，拖低整体（本地按同一口径复现：147 相对 146 估计分差 -0.13，与平台 -0.11 一致）。
    152 的所有分量都以当前价为分母，09:30 截面与 146 相同地整体补 0。四个锚点作为一组加入 146 的 IQR，
    在 09:30 截面按平台口径评估时验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.3892 / 0.3850（146 0.4346 / 0.4256）；
      IC 0.08417 / 0.06863（146 0.08398 / 0.06917）；
      ICIR 1.7965 / 1.1614（146 1.7811 / 1.1693）；
      Sharpe 53.06 / 35.48（146 53.27 / 35.88）；
      压力 1.7921 / 1.1388（146 1.7593 / 1.1520）。
    以下为 151 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十二分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    151 在 150 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_calm : 低振幅加权价格锚点偏离 ln(锚点 / 当前价)，锚点为最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权的均值。
    价格稳定（振幅小）的窗口里成交价更接近共识价值，权重更大；剧烈波动窗口的价格噪声大，权重更小。
    窗口长度沿用已有的 8，0.001 只是防止零振幅除零的下限，无新增可调参数。12 分量 x 四网络 = 48 列做 IQR Winsor 均值。
    这一分量来自第十七批“换权重口径锚点”信号的分量级筛选（逐个作为新分量放进 150 的 IQR），
    验证段与保留段同时为正且验证段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4174 / 0.4117（150 0.4267 / 0.4194）；
      IC 0.07968 / 0.06546（150 0.08003 / 0.06574）；
      ICIR 1.6943 / 1.1368（150 1.7005 / 1.1415）；
      Sharpe 47.72 / 34.28（150 47.56 / 34.33）；
      压力 1.6870 / 1.1431（150 1.6900 / 1.1467）。
    以下为 150 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 十一分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    150 在 149 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lomed : 最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)。
    r_lo 只看当日累计最低点这一个极值，r_lomed 取近 8 个窗口各自低点的中位数，是稳健的近期支撑位；
    窗口长度沿用已有的 8，无新增参数。11 分量 x 四网络 = 44 列做 IQR Winsor 均值。
    这一分量来自第十四至十八批“锚点”信号的分量级筛选（逐个作为新分量放进 149 的 IQR，共 20 个），
    验证段与保留段同时为正且两段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4267 / 0.4194（149 0.4303 / 0.4236）；
      IC 0.08003 / 0.06574（149 0.07995 / 0.06575）；
      ICIR 1.7005 / 1.1415（149 1.6975 / 1.1403）；
      Sharpe 47.56 / 34.33（149 47.31 / 34.16）；
      压力 1.6900 / 1.1467（149 1.6915 / 1.1474）。
    以下为 149 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格与区间中点中位数偏离 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    149 在 148 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_mid : 区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)。
    r_med 的锚点只用每个窗口末的一个报价，r_mid 用的是每个窗口内全部成交覆盖的价格区间的中点，
    反映窗口内的成交中枢，而不只是窗口收盘时刻的价格。窗口长度沿用已有的 8，无新增参数。10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    这一分量来自第十一批“稳健锚点”信号的分量级筛选（逐个作为新分量放进 148 的 IQR），
    4 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4303 / 0.4236（148 0.4365 / 0.4285）；
      IC 0.07995 / 0.06575（148 0.07984 / 0.06590）；
      ICIR 1.6975 / 1.1403（148 1.6944 / 1.1434）；
      Sharpe 47.31 / 34.16（148 47.28 / 33.82）；
      压力 1.6915 / 1.1474（148 1.6929 / 1.1516）。
    以下为 148 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格中位数偏离 九分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    148 在 147 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_med : 价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)。
    阶梯中的 r_rv8 以同样 8 个窗口的成交量加权均价为锚，会被单个放量窗口拉动；中位数锚点不受成交量和单个极端价格影响，
    衡量的是当前价相对近期“典型价格”的偏离。窗口长度沿用阶梯中已有的 8，无新增参数。9 分量 x 四网络 = 36 列做 IQR Winsor 均值。
    这一分量来自第十批 16 个“换锚点”反转信号的分量级筛选（逐个作为新分量放进 147 的 IQR），
    16 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4365 / 0.4285（147 0.4389 / 0.4314）；
      IC 0.07984 / 0.06590（147 0.07933 / 0.06569）；
      ICIR 1.6944 / 1.1434（147 1.6821 / 1.1417）；
      Sharpe 47.28 / 33.82（147 46.51 / 33.45）；
      压力 1.6929 / 1.1516（147 1.6729 / 1.1436）。
    以下为 147 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 八分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    147 在 146 号基础上只在 IQR 内部加两个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lo    : 当日低点距离 ln(min(当日累计窗口最低价, 昨收) / 当前价)，衡量从当日低点（或昨收）以来的反弹幅度；
      r_slope : 窗口 VWAP 斜率 ln(滚动 8 窗口 VWAP / 当前窗口 VWAP)，衡量当前窗口成交均价相对近期均价的抬升。
    两者与阶梯中的“价格相对 VWAP 偏离”口径不同：前者以当日极值为锚，后者只看成交均价、不含当前报价，
    与已有分量相关较低。8 分量 x 四网络 = 32 列做 IQR Winsor 均值。
    这两个分量来自对 215 个结构不同信号的批量分量级筛选（逐个作为新分量放进 146 的 IQR），验证段与保留段同时为正；
    所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4389 / 0.4314（146 0.4798 / 0.4698）；
      IC 0.07933 / 0.06569（146 0.08049 / 0.06665）；
      ICIR 1.6821 / 1.1417（146 1.7022 / 1.1562）；
      Sharpe 46.51 / 33.45（146 47.27 / 34.23）；
      压力 1.6729 / 1.1436（146 1.6992 / 1.1468）。
    以下为 146 的说明。
    更长倍增阶梯四网络同伴排名 IQR（144）+ 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。
    146 在 144 号基础上只加一个与反转正交的慢特征组合：
      c_park  : 低波动，最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值取负；
      c_size  : 小盘，-ln(1 + 最近 40 个窗口清洗后成交额之和)（144 中已有，用于中性化）；
      c_lowpx : 低价，-ln(当前价)（144 中已有，用于中性化）。
    三者截面排名等权相加，再与 144 的 IQR 输出按固定的 1 : 0.5 做截面排名融合。三者都是 A 股文献中的经典异象
    （低波动、小市值、低价），与反转主信号的风格剔除后相关约为 0；它们几乎不随时间变化，融合后换手下降。
    这一组合来自对 130 个结构不同信号的批量筛选，验证段与保留段同时为正；融合权重对全部信号统一取 0.5，未逐个调参。
      验证 / 保留两段换手 0.4798 / 0.4698（144 0.4969 / 0.4883）；
      IC 0.08049 / 0.06665（144 0.07809 / 0.06452）；
      ICIR 1.7022 / 1.1562（144 1.6088 / 1.1070）；
      Sharpe 47.27 / 34.23（144 49.77 / 35.09）；
      压力 1.6992 / 1.1468（144 1.5654 / 1.0782）。
    以下为 144 的说明。
    平台原始价格口径下的更长倍增阶梯超额四网络同伴排名，IQR Winsor 均值聚合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    144 在 143 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 32 窗口（约 4 个交易日，
    接近一周）清洗后成交 VWAP 偏离 r_rv32。六个分量为 2 / 4 / 8 / 16 / 32 窗口 VWAP 偏离 + 日内反转，
    x 四张同伴网络，24 列做 IQR Winsor 均值。周尺度锚点更新更慢，合成信号相邻截面更连续，换手继续下降；
    窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.4969 / 0.4883（143 0.5223 / 0.5155）；
      IC 0.07809 / 0.06452（143 0.07838 / 0.06514）；
      ICIR 1.6088 / 1.1070（143 1.6135 / 1.1239）；
      Sharpe 49.77 / 35.09（143 50.83 / 35.84）；
      压力 1.5654 / 1.0782（143 1.5519 / 1.0812）。
    以下为 143 的说明。
    143 在 142 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 16 窗口（约 2 个交易日）
    清洗后成交 VWAP 偏离 r_rv16。五个分量为 2 / 4 / 8 / 16 窗口 VWAP 偏离 + 日内反转，x 四张同伴网络，
    20 列做 IQR Winsor 均值。慢端锚点跨日更新缓慢，加入后合成信号的截面变化更平稳，同时多日尺度的
    VWAP 偏离本身也带有反转信息；窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.5223 / 0.5155（142 0.5688 / 0.5623）；
      IC 0.07838 / 0.06514（142 0.07808 / 0.06547）；
      ICIR 1.6135 / 1.1239（142 1.6142 / 1.1341）；
      Sharpe 50.83 / 35.84（142 51.17 / 36.54）；
      压力 1.5519 / 1.0812（142 1.5298 / 1.0755）。
    以下为 142 的说明。
    142 在 140 号基础上只换一个分量：30 分钟反转 r_30（只看上一窗口末的单个价格点）换成滚动 2 窗口（1 小时）
    清洗后成交 VWAP 偏离 r_rv2。四个分量成为 2 / 4 / 8 窗口 VWAP 偏离的倍增阶梯 + 日内反转，尺度均匀覆盖，
    且短端由成交量加权的均价代替单点价格，相邻截面之间更连续；首窗不再与 r_day 重复，无需再做首窗去重。
    各分量 x 四张同伴网络，16 列做 IQR Winsor 均值。窗口长度沿用已有的倍增序列，无新增参数。
      验证 / 保留两段换手 0.5688 / 0.5623（140 0.6005 / 0.5949）；
      IC 0.07808 / 0.06547（140 0.07906 / 0.06626）；
      ICIR 1.6142 / 1.1341（140 1.6303 / 1.1475）；
      Sharpe 51.17 / 36.54（140 51.55 / 36.53）；
      压力 1.5298 / 1.0755（140 1.5472 / 1.0809）。
    以下为 141 的说明（141 去掉 r_30 只留三个慢分量，换手降得更多但 IC 也掉得更多，142 是两者之间的折中）。
    141 在 140 号基础上只做一个无参结构改动：去掉 30 分钟反转分量 r_30，只保留滚动 VWAP 4/8 窗口偏离与
    日内反转三个慢分量（各 x 四张同伴网络，12 列做 IQR Winsor 均值）。r_30 只看最近一个窗口，相邻截面之间
    几乎完全更新，是多空两腿成分替换的主要来源；平台五项得分中换手的权重最大，r_30 的边际 IC 不足以抵消它带来的换手。
    r_30 仍用于构建 30 分钟同伴网络（窗口收益相关），只是不再作为合成分量。无新增参数。
      验证 / 保留两段换手 0.4775 / 0.4696（140 0.6005 / 0.5949，降约 0.12）；
      IC 0.07646 / 0.06347（140 0.07906 / 0.06626）；
      ICIR 1.5766 / 1.1041（140 1.6303 / 1.1475）；
      Sharpe 49.52 / 34.85（140 51.55 / 36.53）；
      压力 1.5145 / 1.0732（140 1.5472 / 1.0809）。
    以下为 140 的说明。
    140 在 137 号基础上只改最后一步聚合：16 列中性化排名由 Midhinge
    （25%/75% 分位点均值）改为 IQR Winsor 均值（各列截到本行四分位区间后取均值）。Midhinge 只由两个分位点决定，
    任一列的小幅变动都可能让分位点在相邻列之间跳动，截面排名随之抖动；Winsor 均值让中间一半的列都连续地参与，
    相邻截面更平滑。无新增参数。平台五项得分对换手最敏感之一，此改动的主要收益是降换手：
      开发 / 验证 / 保留三段换手 0.6009 / 0.6005 / 0.5949（137 0.6211 / 0.6194 / 0.6132，三段均降约 0.02）；
      IC 0.08895 / 0.07906 / 0.06626（137 0.08917 / 0.07937 / 0.06643）；
      ICIR 1.4783 / 1.6303 / 1.1475（137 1.4816 / 1.6362 / 1.1496）；
      Sharpe 49.77 / 51.55 / 36.53（137 50.05 / 52.27 / 36.73）；
      压力 1.4021 / 1.5472 / 1.0809（137 1.4049 / 1.5532 / 1.0807）。
    以下为 137 的说明。
    在 136 号（首窗去重的 4 个反转分量 x 日频/30 分钟两张同伴网络，各做“中位数残差 + 同伴内排名”融合与
    四特征中性化，8 列取 Midhinge）基础上只加一个无参结构改动：
      另建两张“超额收益”同伴网络：日收益 / 30 分钟窗口收益先减去同一时点的截面均值，再按相关性取前 30 名同伴。
      原始收益相关里很大一部分是共同的市场涨跌，高贝塔股票会因此互为同伴；超额网络只保留真实的同类联动。
      原始 / 超额 x 日频 / 30 分钟四张网络并列，每个分量在每张网络上照旧做融合与中性化，16 列取 Midhinge
      （09:30 截面去掉 r_30 的 4 列，用 12 列）。不新增任何参数，同伴数、回看期、最少观测都与 136 相同。
    相对 136，本地开发 / 验证 / 保留三段：
      IC 0.08917 / 0.07937 / 0.06643（136 0.08910 / 0.07944 / 0.06629）；
      ICIR 1.4816 / 1.6362 / 1.1496（136 1.4692 / 1.6274 / 1.1277）；
      Sharpe 50.05 / 52.27 / 36.73（136 49.94 / 51.94 / 36.93）；
      压力 1.4049 / 1.5532 / 1.0807（136 1.3931 / 1.5425 / 1.0629）；
      换手 0.6211 / 0.6194 / 0.6132（136 0.6188 / 0.6175 / 0.6102）。
    136 的改动：当日首个窗口的 30 分钟反转以昨收为基准，与 r_day 完全相同，首窗把 r_30 各列置为缺失，避免重复计票；
    r_30 仍保留首窗定义用于构建 30 分钟同伴网络。
    公共构造（103–107 共用同一段代码，只有最后的合成方式不同）：
      反转分量（与 90 号相同，均只用过去数据）：
        r_30  : 30 分钟反转 ln(上一窗口末价 / 当前价)，当日首个窗口以昨收为基准（57 号）
        r_rv4 / r_rv8 : 滚动 4/8 个窗口（2/4 小时，跨日）清洗后成交 VWAP 偏离 ln(滚动 VWAP / 当前价)
        r_day : 日内反转 ln(昨收 / 当前价)（56 号）
      同伴（伪行业）残差：每个自然月月初，用此前 60 个交易日的日收益 ln(收盘段价格 / 昨收) 计算股票池内两两相关系数，
        每只股票取相关性最高的 30 只为同伴（至少 30 个共同观测；只用过去数据）。分量残差 = 分量 - 同伴中位数
        （同伴有效值不足 3 个时减截面均值）。思路：行业内反转强于全市场反转（Da, Liu & Schaumburg 2014），
        去掉“整组一起涨跌”的部分后，只保留个股相对同类股票的过度反应。
      四特征中性化：因子截面排名对 规模（-ln 最近 40 窗口成交额）、特质波动、非流动性（Amihud）、低价 四个特征的截面排名
        做逐截面回归，取残差。反转信号天然偏向小盘、高波动股票，而这些暴露本身会拖累排序；剔除后 IC、IC_IR、多空夏普同时提升。
      数据清洗：成交均价落在该分钟 [最低价 x 0.98, 最高价 x 1.02] 之外的分钟不计入任何 VWAP / 成交额累加。

    稳健性：最终缺失补 0（中性），每个截面缺失率为 0；向前取不到足够历史时同伴退化为截面均值、特征排名记为中性。
    性能：纯 SQL 向量化（单次 GROUP BY，无窗口函数、无 Python UDF）；相关矩阵每月计算一次（约 1200 x 1200），
      同伴均值按月份用数组索引批量计算。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 150
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS = 30, 60, 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(bid_volume1, 0) AS DOUBLE) AS bv1,
            CAST(COALESCE(ask_volume1, 0) AS DOUBLE) AS av1,
            CAST(COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                 + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) AS DOUBLE) AS dep3,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            px_close,
            bp1,
            ap1,
            bv1,
            av1,
            dep3,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px ELSE 0 END) AS last_px,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment AND bp1 > 0 AND ap1 >= bp1
                     THEN (bp1 + ap1) / 2.0 ELSE 0 END) AS last_qmid,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment AND bp1 > 0 AND ap1 >= bp1 AND bv1 + av1 > 0
                     THEN (bp1 * av1 + ap1 * bv1) / (bv1 + av1) ELSE 0 END) AS last_micro,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px_close ELSE 0 END) AS last_close,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN dep3 ELSE 0 END) AS last_depth,
            MAX(pre_close) AS pre_close,
            MAX(px_high) AS bar_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS bar_low
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.last_px, f.last_qmid, f.last_micro, f.last_close, f.last_depth,
        f.pre_close, f.bar_high, f.bar_low
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "last_px", "last_qmid", "last_micro", "last_close", "last_depth",
              "pre_close", "bar_high", "bar_low"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    # 155：盘口报价与没有当前价（09:30 截面）时一律缺失。
    for c in ["last_qmid", "last_micro", "last_close", "last_depth"]:
        df[c] = df[c].where((df[c] > 0) & df["last_px"].notna())
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv2"] = rolling_vwap_rev(2)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_rv32"] = rolling_vwap_rev(32)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)
    # 148：价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)，以稳健的近期价格中枢为锚。
    df["r_med"] = safe_log_ratio(roll(px, 8, "median", 2), px)
    # 149：区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)，以窗口内全部成交区间的中枢为锚。
    df["r_mid"] = safe_log_ratio(roll(((df["bar_high"] + df["bar_low"]) / 2).where(df["bar_low"] > 0), 8, "median", 2), px)
    # 150：最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)，以近期各窗口低点的稳健中枢为支撑锚。
    df["r_lomed"] = safe_log_ratio(roll(df["bar_low"].where(df["bar_low"] > 0), 8, "median", 2), px)
    # 151：低振幅加权价格锚点偏离，最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权平均，振幅小的窗口权重大。
    calm_w = 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)
    calm_den = roll(calm_w.where(px > 0), 8, "sum", 2)
    df["r_calm"] = safe_log_ratio(roll(px.where(px > 0) * calm_w, 8, "sum", 2) / calm_den.where(calm_den > 0), px)
    # 155：一档中间价锚点偏离 ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)。
    df["r_qmid"] = safe_log_ratio(roll(df["last_qmid"], 8, "mean", 2), df["last_qmid"])
    # 155：成交价相对报价中枢，最近 40 个窗口 ln(窗口末成交价 / 窗口末当前价) 的均值（至少 10 个），没有当前价时缺失。
    df["r_cmid"] = roll(safe_log_ratio(df["last_close"], px), 40, "mean", 10).where(px.notna())
    # 155：挂单深度加权价格锚点偏离，最近 8 个窗口末价按三档挂单总量加权平均。
    deep_ok = (px > 0) & (df["last_depth"] > 0)
    deep_den = roll(df["last_depth"].where(deep_ok), 8, "sum", 2)
    df["r_deep"] = safe_log_ratio(roll((px * df["last_depth"]).where(deep_ok), 8, "sum", 2) / deep_den.where(deep_den > 0), px)
    # 155：滚动 VWAP 相对微价格 ln(最近 8 个窗口成交 VWAP / 当前一档微价格)。
    df["r_micro"] = safe_log_ratio(vwap_of(roll(df["sum_amt"], 8, "sum", 1), roll(df["sum_vol"], 8, "sum", 1)), df["last_micro"])

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 低波动（取负）：最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值。
    df["c_park"] = -roll(safe_log_ratio(df["bar_high"], df["bar_low"]) ** 2, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的收益计算相关系数，两种频率各建一张网络：
    #   日频：日收益（收盘段价格 / 昨收），至少 PEER_MIN_OBS 个共同观测；
    #   30 分钟：窗口收益（上一窗口末价，当日首个窗口用昨收），取回看期内全部窗口，至少 4 x PEER_MIN_OBS 个共同观测。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    win_ret30 = pd.Series(
        (-df["r_30"]).values, index=pd.MultiIndex.from_arrays([df["date"].values, df["instrument"].values])
    ).unstack().sort_index().reindex(columns=peer_insts)

    def build_peers(ret_wide, row_day, need):
        """返回 {月份: 同伴下标矩阵}；ret_wide 每行是一个收益观测，row_day 为该行所属交易日。"""
        peers = {}
        for m in out_months:
            in_month = all_days[day_month == m]
            if len(in_month) == 0:
                continue
            prior = all_days[all_days < in_month[0]][-PEER_LOOKBACK:]
            if len(prior) < PEER_MIN_OBS:
                continue
            hist = ret_wide[(row_day < in_month[0]) & (row_day >= prior[0])]
            ok = (hist.notna().sum() >= need).values
            corr = hist.loc[:, ok].corr(min_periods=need).values
            np.fill_diagonal(corr, -np.inf)
            corr = np.nan_to_num(corr, nan=-np.inf)
            sub = np.where(ok)[0]
            idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
            idx[sub] = sub[np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]]
            peers[m] = idx
        return peers

    # 超额网络：每个收益观测先减去同一时点的截面均值，再按超额收益的相关性挑同伴（去掉市场共振带来的伪同伴）。
    peers_by_freq = {
        "day": build_peers(daily_ret, all_days, PEER_MIN_OBS),
        "30": build_peers(win_ret30, win_ret30.index.normalize(), PEER_MIN_OBS * 4),
        "day_excess": build_peers(daily_ret.sub(daily_ret.mean(axis=1), axis=0), all_days, PEER_MIN_OBS),
        "30_excess": build_peers(win_ret30.sub(win_ret30.mean(axis=1), axis=0), win_ret30.index.normalize(), PEER_MIN_OBS * 4),
    }

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_park"]
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col, peers_by_month):
        """每个截面上同伴的 col 中位数（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            cnt = (~np.isnan(g)).sum(axis=2)
            pm = np.where(cnt >= 3, np.nanmedian(g, axis=2), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def peer_rank(col, peers_by_month):
        """个股 col 在其同伴中的相对排名（低于自身的同伴占比，平局记半），居中到 [-0.5, 0.5]；
        同伴有效值不足 3 个或没有同伴信息时回退为截面居中排名。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            x = wide.values
            valid = idx >= 0
            g = x[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            own = x[:, :, None]
            ok = ~np.isnan(g) & ~np.isnan(own)
            cnt = ok.sum(axis=2)
            score = ((g < own) & ok).sum(axis=2) + 0.5 * ((g == own) & ok).sum(axis=2)
            rel = np.where(cnt >= 3, score / np.maximum(cnt, 1) - 0.5, np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, rel[r_pos, np.maximum(c_pos, 0)], np.nan)
        fallback = result.groupby("date")[col].rank(pct=True) - 0.5
        return pd.Series(out, index=result.index).fillna(fallback)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx"]
    neutral_cols = []
    for c in ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro"]:
        cross_mean = result.groupby("date")[c].transform("mean")
        for f, peers in peers_by_freq.items():
            # 每个分量 x 每张同伴网络：同伴中位数残差与同伴内相对排名各自截面排名后等权融合，再做四特征中性化。
            result[f"{c}_p{f}"] = result[c] - peer_mean(c, peers).fillna(cross_mean)
            result[f"{c}_r{f}"] = peer_rank(c, peers)
            result[f"{c}_b{f}"] = rank_blend([f"{c}_p{f}", f"{c}_r{f}"])
            name = f"{c}_n{f}"
            result[name] = neutralize(f"{c}_b{f}", CHARS4)
            neutral_cols.append(name)
    neutral_ranks = result.groupby("date")[neutral_cols].rank(pct=True) - 0.5
    lower = neutral_ranks.quantile(0.25, axis=1)
    upper = neutral_ranks.quantile(0.75, axis=1)
    # IQR Winsor 均值：各列先截到本行四分位区间，再取均值（中间一半的票全部参与，而不只看两个分位点）。
    result["iqr"] = neutral_ranks.clip(lower=lower, upper=upper, axis=0).mean(axis=1, skipna=True)
    # 146：慢特征组合 = 低波动 / 小盘 / 低价三者截面排名等权相加，再与反转主信号按 1 : 0.5 做截面排名融合。
    char_ranks = result.groupby("date")[["c_park", "c_size", "c_lowpx"]].rank(pct=True)
    result["slow"] = char_ranks["c_park"] + char_ranks["c_size"] + char_ranks["c_lowpx"]
    result["factor"] = rank_blend(["iqr", "slow"], [1.0, 0.5])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def main_b8(datasources, start_date, end_date):
    """
    平台原始 IQR 结构（同伴残差 + 四特征中性化 + IQR Winsor 均值）的首窗去重四网络中位数版本。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1（0.80917），不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    在 138 号（首窗去重 + 原始 / 超额 x 日频 / 30 分钟四张同伴网络，16 列 IQR）基础上只改一处：
      同伴基准由等权均值改为中位数（132 号已验证的无参稳健化），单只同伴的异常涨跌不再拖动整组基准。
    138 的两个无参结构改动保持不变：
      1. 首窗去重：当日首个窗口的 30 分钟反转以昨收为基准，与 r_day 完全相同，首窗把 r_30 置为缺失，避免重复计票；
      2. 四张同伴网络：超额网络先把每个收益观测减去同一时点的截面均值，再按相关性取前 30 名同伴；
         每个分量在每张网络上各做一次同伴中位数残差与四特征中性化，16 列做 IQR（09:30 截面 12 列）。
    不新增任何参数，同伴数、回看期、最少观测都与原始 IQR 相同。
    本地开发 / 验证 / 保留三段（括号内依次为 138、原始 IQR）：
      IC 0.08863 / 0.07916 / 0.06639（0.08780 / 0.07828 / 0.06572；0.08636 / 0.07702 / 0.06470）；
      ICIR 1.4657 / 1.6266 / 1.1415（1.4655 / 1.6199 / 1.1361；1.4443 / 1.5882 / 1.1182）；
      Sharpe 49.37 / 50.89 / 36.40（49.13 / 50.62 / 36.17；48.54 / 49.99 / 35.71）；
      压力 1.3877 / 1.5418 / 1.0743（1.3895 / 1.5402 / 1.0632；1.3681 / 1.5036 / 1.0460）；
      换手 0.5930 / 0.5934 / 0.5895（0.5916 / 0.5923 / 0.5880；0.5896 / 0.5911 / 0.5861）。
    公共构造：
      反转分量（与 90 号相同，均只用过去数据）：
        r_30  : 30 分钟反转 ln(上一窗口末价 / 当前价)，当日首个窗口以昨收为基准（57 号）
        r_rv4 / r_rv8 : 滚动 4/8 个窗口（2/4 小时，跨日）清洗后成交 VWAP 偏离 ln(滚动 VWAP / 当前价)
        r_day : 日内反转 ln(昨收 / 当前价)（56 号）
      同伴（伪行业）残差：每个自然月月初，用此前 60 个交易日的日收益 ln(收盘段价格 / 昨收) 计算股票池内两两相关系数，
        每只股票取相关性最高的 30 只为同伴（至少 30 个共同观测；只用过去数据）。分量残差 = 分量 - 同伴中位数
        （同伴有效值不足 3 个时减截面均值）。思路：行业内反转强于全市场反转（Da, Liu & Schaumburg 2014），
        去掉“整组一起涨跌”的部分后，只保留个股相对同类股票的过度反应。
      四特征中性化：因子截面排名对 规模（-ln 最近 40 窗口成交额）、特质波动、非流动性（Amihud）、低价 四个特征的截面排名
        做逐截面回归，取残差。反转信号天然偏向小盘、高波动股票，而这些暴露本身会拖累排序；剔除后 IC、IC_IR、多空夏普同时提升。
      数据清洗：成交均价落在该分钟 [最低价 x 0.98, 最高价 x 1.02] 之外的分钟不计入任何 VWAP / 成交额累加。

    稳健性：最终缺失补 0（中性），每个截面缺失率为 0；向前取不到足够历史时同伴退化为截面均值、特征排名记为中性。
    性能：纯 SQL 向量化（单次 GROUP BY，无窗口函数、无 Python UDF）；相关矩阵每月计算一次（约 1200 x 1200），
      同伴均值按月份用数组索引批量计算。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 150
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS = 30, 60, 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px ELSE 0 END) AS last_px,
            MAX(pre_close) AS pre_close
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.last_px, f.pre_close
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "last_px", "pre_close"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)
    # 首窗 r_30 与 r_day 相同，参与合成的分量在首窗置缺失，避免重复计票。
    df["r_30d"] = df["r_30"].where(same_day)

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))
    df["c_max"] = -roll(win_ret, 40, "max", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的收益计算相关系数，两种频率各建一张网络：
    #   日频：日收益（收盘段价格 / 昨收），至少 PEER_MIN_OBS 个共同观测；
    #   30 分钟：窗口收益（上一窗口末价，当日首个窗口用昨收），取回看期内全部窗口，至少 4 x PEER_MIN_OBS 个共同观测。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    win_ret30 = pd.Series(
        (-df["r_30"]).values, index=pd.MultiIndex.from_arrays([df["date"].values, df["instrument"].values])
    ).unstack().sort_index().reindex(columns=peer_insts)

    def build_peers(ret_wide, row_day, need):
        """返回 {月份: 同伴下标矩阵}；ret_wide 每行是一个收益观测，row_day 为该行所属交易日。"""
        peers = {}
        for m in out_months:
            in_month = all_days[day_month == m]
            if len(in_month) == 0:
                continue
            prior = all_days[all_days < in_month[0]][-PEER_LOOKBACK:]
            if len(prior) < PEER_MIN_OBS:
                continue
            hist = ret_wide[(row_day < in_month[0]) & (row_day >= prior[0])]
            ok = (hist.notna().sum() >= need).values
            corr = hist.loc[:, ok].corr(min_periods=need).values
            np.fill_diagonal(corr, -np.inf)
            corr = np.nan_to_num(corr, nan=-np.inf)
            sub = np.where(ok)[0]
            idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
            idx[sub] = sub[np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]]
            peers[m] = idx
        return peers

    # 超额网络：每个收益观测先减去同一时点的截面均值，再按超额收益的相关性挑同伴（去掉市场共振带来的伪同伴）。
    peers_by_freq = {
        "day": build_peers(daily_ret, all_days, PEER_MIN_OBS),
        "30": build_peers(win_ret30, win_ret30.index.normalize(), PEER_MIN_OBS * 4),
        "day_excess": build_peers(daily_ret.sub(daily_ret.mean(axis=1), axis=0), all_days, PEER_MIN_OBS),
        "30_excess": build_peers(win_ret30.sub(win_ret30.mean(axis=1), axis=0), win_ret30.index.normalize(), PEER_MIN_OBS * 4),
    }

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_30d", "r_rv4", "r_rv8", "r_day", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max"]
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_median(col, peers_by_month):
        """每个截面上同伴的 col 中位数（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            cnt = (~np.isnan(g)).sum(axis=2)
            pm = np.where(cnt >= 3, np.nanmedian(g, axis=2), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx"]
    neutral_cols = []
    for c in ["r_30d", "r_rv4", "r_rv8", "r_day"]:
        cross_mean = result.groupby("date")[c].transform("mean")
        for f, peers in peers_by_freq.items():
            # 每个分量 x 每张同伴网络：同伴中位数残差后做四特征中性化。
            result[f"{c}_p{f}"] = (result[c] - peer_median(c, peers).fillna(cross_mean)).where(result[c].notna())
            name = f"{c}_n{f}"
            result[name] = neutralize(f"{c}_p{f}", CHARS4)
            neutral_cols.append(name)
    # IQR Winsor 均值：逐行把各列截到该行 [Q1, Q3] 再取均值（与平台原始 IQR 相同）。
    neutral_ranks = result.groupby("date")[neutral_cols].rank(pct=True) - 0.5
    lower = neutral_ranks.quantile(0.25, axis=1)
    upper = neutral_ranks.quantile(0.75, axis=1)
    result["factor"] = neutral_ranks.clip(lower=lower, upper=upper, axis=0).mean(axis=1, skipna=True)
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def main_b9(datasources, start_date, end_date):
    """
    152（倍增阶梯 + 四类价格锚点四网络 IQR + 低波 / 小盘 / 低价）的 IQR 内加 5 个盘口 / 价格锚点分量。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    161 在 152 号基础上只在 IQR 内部加以下反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_qmid : ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)，涨跌停单边盘口的窗口不参与锚点、当期缺失
      r_cmid : 最近 40 个窗口 ln(窗口末当前价 / 窗口末成交价) 的均值（成交持续落在买一侧即卖压时为负）
      r_deep : ln(最近 8 个窗口末价按三档挂单总量加权的均价 / 当前价)，挂单厚的窗口价格更可信
      r_micro : ln(最近 8 个窗口成交 VWAP / 当前一档微价格)，微价格按对侧挂单量加权买一 / 卖一价
      r_dealpx : ln(最近 8 个窗口末价按窗口成交笔数加权的均价 / 当前价)
    155 的四个盘口锚点分量再加 deal_px8，来自 155 组合上的逐个加分量筛选（验证段）。
    窗口长度沿用已有的 8 与 40，无新增参数，所有候选统一用同一套网络与聚合。
    没有当前价（09:30 截面）时新分量都缺失，该截面与 152 一样整体补 0。15 分量 x 四网络 = 60 列做 IQR Winsor 均值。
    以下为 152 的说明。
    更长倍增阶梯 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    152 在 151 号基础上去掉两个分量 r_lo（当日低点距离）与 r_slope（VWAP 斜率），其余（同伴网络、中性化、聚合、慢特征融合）全部不变，
    即 146 的 6 个阶梯分量 + r_med / r_mid / r_lomed / r_calm 四个价格锚点，10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    原因：窗口末价格取截面那一分钟，09:30 截面没有这一分钟（1 分钟数据只有 09:25 竞价和 09:31 之后），当前价缺失。
    146 在该截面所有分量缺失、整体补 0 而不参与评估；r_slope 只用成交额与成交量、不含当前价，147–151 在该截面只剩它一个分量，
    该截面 IC 仅约 0.02，拖低整体（本地按同一口径复现：147 相对 146 估计分差 -0.13，与平台 -0.11 一致）。
    152 的所有分量都以当前价为分母，09:30 截面与 146 相同地整体补 0。四个锚点作为一组加入 146 的 IQR，
    在 09:30 截面按平台口径评估时验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.3892 / 0.3850（146 0.4346 / 0.4256）；
      IC 0.08417 / 0.06863（146 0.08398 / 0.06917）；
      ICIR 1.7965 / 1.1614（146 1.7811 / 1.1693）；
      Sharpe 53.06 / 35.48（146 53.27 / 35.88）；
      压力 1.7921 / 1.1388（146 1.7593 / 1.1520）。
    以下为 151 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十二分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    151 在 150 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_calm : 低振幅加权价格锚点偏离 ln(锚点 / 当前价)，锚点为最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权的均值。
    价格稳定（振幅小）的窗口里成交价更接近共识价值，权重更大；剧烈波动窗口的价格噪声大，权重更小。
    窗口长度沿用已有的 8，0.001 只是防止零振幅除零的下限，无新增可调参数。12 分量 x 四网络 = 48 列做 IQR Winsor 均值。
    这一分量来自第十七批“换权重口径锚点”信号的分量级筛选（逐个作为新分量放进 150 的 IQR），
    验证段与保留段同时为正且验证段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4174 / 0.4117（150 0.4267 / 0.4194）；
      IC 0.07968 / 0.06546（150 0.08003 / 0.06574）；
      ICIR 1.6943 / 1.1368（150 1.7005 / 1.1415）；
      Sharpe 47.72 / 34.28（150 47.56 / 34.33）；
      压力 1.6870 / 1.1431（150 1.6900 / 1.1467）。
    以下为 150 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 十一分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    150 在 149 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lomed : 最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)。
    r_lo 只看当日累计最低点这一个极值，r_lomed 取近 8 个窗口各自低点的中位数，是稳健的近期支撑位；
    窗口长度沿用已有的 8，无新增参数。11 分量 x 四网络 = 44 列做 IQR Winsor 均值。
    这一分量来自第十四至十八批“锚点”信号的分量级筛选（逐个作为新分量放进 149 的 IQR，共 20 个），
    验证段与保留段同时为正且两段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4267 / 0.4194（149 0.4303 / 0.4236）；
      IC 0.08003 / 0.06574（149 0.07995 / 0.06575）；
      ICIR 1.7005 / 1.1415（149 1.6975 / 1.1403）；
      Sharpe 47.56 / 34.33（149 47.31 / 34.16）；
      压力 1.6900 / 1.1467（149 1.6915 / 1.1474）。
    以下为 149 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格与区间中点中位数偏离 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    149 在 148 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_mid : 区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)。
    r_med 的锚点只用每个窗口末的一个报价，r_mid 用的是每个窗口内全部成交覆盖的价格区间的中点，
    反映窗口内的成交中枢，而不只是窗口收盘时刻的价格。窗口长度沿用已有的 8，无新增参数。10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    这一分量来自第十一批“稳健锚点”信号的分量级筛选（逐个作为新分量放进 148 的 IQR），
    4 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4303 / 0.4236（148 0.4365 / 0.4285）；
      IC 0.07995 / 0.06575（148 0.07984 / 0.06590）；
      ICIR 1.6975 / 1.1403（148 1.6944 / 1.1434）；
      Sharpe 47.31 / 34.16（148 47.28 / 33.82）；
      压力 1.6915 / 1.1474（148 1.6929 / 1.1516）。
    以下为 148 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格中位数偏离 九分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    148 在 147 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_med : 价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)。
    阶梯中的 r_rv8 以同样 8 个窗口的成交量加权均价为锚，会被单个放量窗口拉动；中位数锚点不受成交量和单个极端价格影响，
    衡量的是当前价相对近期“典型价格”的偏离。窗口长度沿用阶梯中已有的 8，无新增参数。9 分量 x 四网络 = 36 列做 IQR Winsor 均值。
    这一分量来自第十批 16 个“换锚点”反转信号的分量级筛选（逐个作为新分量放进 147 的 IQR），
    16 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4365 / 0.4285（147 0.4389 / 0.4314）；
      IC 0.07984 / 0.06590（147 0.07933 / 0.06569）；
      ICIR 1.6944 / 1.1434（147 1.6821 / 1.1417）；
      Sharpe 47.28 / 33.82（147 46.51 / 33.45）；
      压力 1.6929 / 1.1516（147 1.6729 / 1.1436）。
    以下为 147 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 八分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    147 在 146 号基础上只在 IQR 内部加两个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lo    : 当日低点距离 ln(min(当日累计窗口最低价, 昨收) / 当前价)，衡量从当日低点（或昨收）以来的反弹幅度；
      r_slope : 窗口 VWAP 斜率 ln(滚动 8 窗口 VWAP / 当前窗口 VWAP)，衡量当前窗口成交均价相对近期均价的抬升。
    两者与阶梯中的“价格相对 VWAP 偏离”口径不同：前者以当日极值为锚，后者只看成交均价、不含当前报价，
    与已有分量相关较低。8 分量 x 四网络 = 32 列做 IQR Winsor 均值。
    这两个分量来自对 215 个结构不同信号的批量分量级筛选（逐个作为新分量放进 146 的 IQR），验证段与保留段同时为正；
    所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4389 / 0.4314（146 0.4798 / 0.4698）；
      IC 0.07933 / 0.06569（146 0.08049 / 0.06665）；
      ICIR 1.6821 / 1.1417（146 1.7022 / 1.1562）；
      Sharpe 46.51 / 33.45（146 47.27 / 34.23）；
      压力 1.6729 / 1.1436（146 1.6992 / 1.1468）。
    以下为 146 的说明。
    更长倍增阶梯四网络同伴排名 IQR（144）+ 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。
    146 在 144 号基础上只加一个与反转正交的慢特征组合：
      c_park  : 低波动，最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值取负；
      c_size  : 小盘，-ln(1 + 最近 40 个窗口清洗后成交额之和)（144 中已有，用于中性化）；
      c_lowpx : 低价，-ln(当前价)（144 中已有，用于中性化）。
    三者截面排名等权相加，再与 144 的 IQR 输出按固定的 1 : 0.5 做截面排名融合。三者都是 A 股文献中的经典异象
    （低波动、小市值、低价），与反转主信号的风格剔除后相关约为 0；它们几乎不随时间变化，融合后换手下降。
    这一组合来自对 130 个结构不同信号的批量筛选，验证段与保留段同时为正；融合权重对全部信号统一取 0.5，未逐个调参。
      验证 / 保留两段换手 0.4798 / 0.4698（144 0.4969 / 0.4883）；
      IC 0.08049 / 0.06665（144 0.07809 / 0.06452）；
      ICIR 1.7022 / 1.1562（144 1.6088 / 1.1070）；
      Sharpe 47.27 / 34.23（144 49.77 / 35.09）；
      压力 1.6992 / 1.1468（144 1.5654 / 1.0782）。
    以下为 144 的说明。
    平台原始价格口径下的更长倍增阶梯超额四网络同伴排名，IQR Winsor 均值聚合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    144 在 143 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 32 窗口（约 4 个交易日，
    接近一周）清洗后成交 VWAP 偏离 r_rv32。六个分量为 2 / 4 / 8 / 16 / 32 窗口 VWAP 偏离 + 日内反转，
    x 四张同伴网络，24 列做 IQR Winsor 均值。周尺度锚点更新更慢，合成信号相邻截面更连续，换手继续下降；
    窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.4969 / 0.4883（143 0.5223 / 0.5155）；
      IC 0.07809 / 0.06452（143 0.07838 / 0.06514）；
      ICIR 1.6088 / 1.1070（143 1.6135 / 1.1239）；
      Sharpe 49.77 / 35.09（143 50.83 / 35.84）；
      压力 1.5654 / 1.0782（143 1.5519 / 1.0812）。
    以下为 143 的说明。
    143 在 142 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 16 窗口（约 2 个交易日）
    清洗后成交 VWAP 偏离 r_rv16。五个分量为 2 / 4 / 8 / 16 窗口 VWAP 偏离 + 日内反转，x 四张同伴网络，
    20 列做 IQR Winsor 均值。慢端锚点跨日更新缓慢，加入后合成信号的截面变化更平稳，同时多日尺度的
    VWAP 偏离本身也带有反转信息；窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.5223 / 0.5155（142 0.5688 / 0.5623）；
      IC 0.07838 / 0.06514（142 0.07808 / 0.06547）；
      ICIR 1.6135 / 1.1239（142 1.6142 / 1.1341）；
      Sharpe 50.83 / 35.84（142 51.17 / 36.54）；
      压力 1.5519 / 1.0812（142 1.5298 / 1.0755）。
    以下为 142 的说明。
    142 在 140 号基础上只换一个分量：30 分钟反转 r_30（只看上一窗口末的单个价格点）换成滚动 2 窗口（1 小时）
    清洗后成交 VWAP 偏离 r_rv2。四个分量成为 2 / 4 / 8 窗口 VWAP 偏离的倍增阶梯 + 日内反转，尺度均匀覆盖，
    且短端由成交量加权的均价代替单点价格，相邻截面之间更连续；首窗不再与 r_day 重复，无需再做首窗去重。
    各分量 x 四张同伴网络，16 列做 IQR Winsor 均值。窗口长度沿用已有的倍增序列，无新增参数。
      验证 / 保留两段换手 0.5688 / 0.5623（140 0.6005 / 0.5949）；
      IC 0.07808 / 0.06547（140 0.07906 / 0.06626）；
      ICIR 1.6142 / 1.1341（140 1.6303 / 1.1475）；
      Sharpe 51.17 / 36.54（140 51.55 / 36.53）；
      压力 1.5298 / 1.0755（140 1.5472 / 1.0809）。
    以下为 141 的说明（141 去掉 r_30 只留三个慢分量，换手降得更多但 IC 也掉得更多，142 是两者之间的折中）。
    141 在 140 号基础上只做一个无参结构改动：去掉 30 分钟反转分量 r_30，只保留滚动 VWAP 4/8 窗口偏离与
    日内反转三个慢分量（各 x 四张同伴网络，12 列做 IQR Winsor 均值）。r_30 只看最近一个窗口，相邻截面之间
    几乎完全更新，是多空两腿成分替换的主要来源；平台五项得分中换手的权重最大，r_30 的边际 IC 不足以抵消它带来的换手。
    r_30 仍用于构建 30 分钟同伴网络（窗口收益相关），只是不再作为合成分量。无新增参数。
      验证 / 保留两段换手 0.4775 / 0.4696（140 0.6005 / 0.5949，降约 0.12）；
      IC 0.07646 / 0.06347（140 0.07906 / 0.06626）；
      ICIR 1.5766 / 1.1041（140 1.6303 / 1.1475）；
      Sharpe 49.52 / 34.85（140 51.55 / 36.53）；
      压力 1.5145 / 1.0732（140 1.5472 / 1.0809）。
    以下为 140 的说明。
    140 在 137 号基础上只改最后一步聚合：16 列中性化排名由 Midhinge
    （25%/75% 分位点均值）改为 IQR Winsor 均值（各列截到本行四分位区间后取均值）。Midhinge 只由两个分位点决定，
    任一列的小幅变动都可能让分位点在相邻列之间跳动，截面排名随之抖动；Winsor 均值让中间一半的列都连续地参与，
    相邻截面更平滑。无新增参数。平台五项得分对换手最敏感之一，此改动的主要收益是降换手：
      开发 / 验证 / 保留三段换手 0.6009 / 0.6005 / 0.5949（137 0.6211 / 0.6194 / 0.6132，三段均降约 0.02）；
      IC 0.08895 / 0.07906 / 0.06626（137 0.08917 / 0.07937 / 0.06643）；
      ICIR 1.4783 / 1.6303 / 1.1475（137 1.4816 / 1.6362 / 1.1496）；
      Sharpe 49.77 / 51.55 / 36.53（137 50.05 / 52.27 / 36.73）；
      压力 1.4021 / 1.5472 / 1.0809（137 1.4049 / 1.5532 / 1.0807）。
    以下为 137 的说明。
    在 136 号（首窗去重的 4 个反转分量 x 日频/30 分钟两张同伴网络，各做“中位数残差 + 同伴内排名”融合与
    四特征中性化，8 列取 Midhinge）基础上只加一个无参结构改动：
      另建两张“超额收益”同伴网络：日收益 / 30 分钟窗口收益先减去同一时点的截面均值，再按相关性取前 30 名同伴。
      原始收益相关里很大一部分是共同的市场涨跌，高贝塔股票会因此互为同伴；超额网络只保留真实的同类联动。
      原始 / 超额 x 日频 / 30 分钟四张网络并列，每个分量在每张网络上照旧做融合与中性化，16 列取 Midhinge
      （09:30 截面去掉 r_30 的 4 列，用 12 列）。不新增任何参数，同伴数、回看期、最少观测都与 136 相同。
    相对 136，本地开发 / 验证 / 保留三段：
      IC 0.08917 / 0.07937 / 0.06643（136 0.08910 / 0.07944 / 0.06629）；
      ICIR 1.4816 / 1.6362 / 1.1496（136 1.4692 / 1.6274 / 1.1277）；
      Sharpe 50.05 / 52.27 / 36.73（136 49.94 / 51.94 / 36.93）；
      压力 1.4049 / 1.5532 / 1.0807（136 1.3931 / 1.5425 / 1.0629）；
      换手 0.6211 / 0.6194 / 0.6132（136 0.6188 / 0.6175 / 0.6102）。
    136 的改动：当日首个窗口的 30 分钟反转以昨收为基准，与 r_day 完全相同，首窗把 r_30 各列置为缺失，避免重复计票；
    r_30 仍保留首窗定义用于构建 30 分钟同伴网络。
    公共构造（103–107 共用同一段代码，只有最后的合成方式不同）：
      反转分量（与 90 号相同，均只用过去数据）：
        r_30  : 30 分钟反转 ln(上一窗口末价 / 当前价)，当日首个窗口以昨收为基准（57 号）
        r_rv4 / r_rv8 : 滚动 4/8 个窗口（2/4 小时，跨日）清洗后成交 VWAP 偏离 ln(滚动 VWAP / 当前价)
        r_day : 日内反转 ln(昨收 / 当前价)（56 号）
      同伴（伪行业）残差：每个自然月月初，用此前 60 个交易日的日收益 ln(收盘段价格 / 昨收) 计算股票池内两两相关系数，
        每只股票取相关性最高的 30 只为同伴（至少 30 个共同观测；只用过去数据）。分量残差 = 分量 - 同伴中位数
        （同伴有效值不足 3 个时减截面均值）。思路：行业内反转强于全市场反转（Da, Liu & Schaumburg 2014），
        去掉“整组一起涨跌”的部分后，只保留个股相对同类股票的过度反应。
      四特征中性化：因子截面排名对 规模（-ln 最近 40 窗口成交额）、特质波动、非流动性（Amihud）、低价 四个特征的截面排名
        做逐截面回归，取残差。反转信号天然偏向小盘、高波动股票，而这些暴露本身会拖累排序；剔除后 IC、IC_IR、多空夏普同时提升。
      数据清洗：成交均价落在该分钟 [最低价 x 0.98, 最高价 x 1.02] 之外的分钟不计入任何 VWAP / 成交额累加。

    稳健性：最终缺失补 0（中性），每个截面缺失率为 0；向前取不到足够历史时同伴退化为截面均值、特征排名记为中性。
    性能：纯 SQL 向量化（单次 GROUP BY，无窗口函数、无 Python UDF）；相关矩阵每月计算一次（约 1200 x 1200），
      同伴均值按月份用数组索引批量计算。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 150
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS = 30, 60, 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(ask_volume1, 0) AS DOUBLE) AS av1,
            CAST(COALESCE(ask_volume2, 0) AS DOUBLE) AS av2,
            CAST(COALESCE(ask_volume3, 0) AS DOUBLE) AS av3,
            CAST(COALESCE(bid_volume1, 0) AS DOUBLE) AS bv1,
            CAST(COALESCE(bid_volume2, 0) AS DOUBLE) AS bv2,
            CAST(COALESCE(bid_volume3, 0) AS DOUBLE) AS bv3,
            CAST(COALESCE(deal_number, 0) AS DOUBLE) AS deal,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            ap1,
            av1,
            av2,
            av3,
            bp1,
            bv1,
            bv2,
            bv3,
            px_close,
            deal,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px ELSE 0 END) AS last_px,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN ap1 ELSE 0 END) AS last_ap1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av1 ELSE 0 END) AS last_av1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av2 ELSE 0 END) AS last_av2,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av3 ELSE 0 END) AS last_av3,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bp1 ELSE 0 END) AS last_bp1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv1 ELSE 0 END) AS last_bv1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv2 ELSE 0 END) AS last_bv2,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv3 ELSE 0 END) AS last_bv3,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px_close ELSE 0 END) AS last_close,
            SUM(deal) AS sum_deal,
            MAX(pre_close) AS pre_close,
            MAX(px_high) AS bar_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS bar_low
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.last_px, f.pre_close, f.bar_high, f.bar_low,
        f.last_ap1, f.last_av1, f.last_av2, f.last_av3, f.last_bp1, f.last_bv1, f.last_bv2, f.last_bv3, f.last_close, f.sum_deal
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "last_px", "pre_close", "bar_high", "bar_low",
              "last_ap1", "last_av1", "last_av2", "last_av3", "last_bp1", "last_bv1", "last_bv2", "last_bv3", "last_close", "sum_deal"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    # 161：窗口末盘口 / 成交价在没有当前价（09:30 截面）时一律缺失（盘口为空记 0，由各分量自行判断有效性）。
    for c in ["last_ap1", "last_av1", "last_av2", "last_av3", "last_bp1", "last_bv1", "last_bv2", "last_bv3", "last_close"]:
        df[c] = df[c].where(df["last_px"].notna())
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv2"] = rolling_vwap_rev(2)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_rv32"] = rolling_vwap_rev(32)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)
    # 148：价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)，以稳健的近期价格中枢为锚。
    df["r_med"] = safe_log_ratio(roll(px, 8, "median", 2), px)
    # 149：区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)，以窗口内全部成交区间的中枢为锚。
    df["r_mid"] = safe_log_ratio(roll(((df["bar_high"] + df["bar_low"]) / 2).where(df["bar_low"] > 0), 8, "median", 2), px)
    # 150：最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)，以近期各窗口低点的稳健中枢为支撑锚。
    df["r_lomed"] = safe_log_ratio(roll(df["bar_low"].where(df["bar_low"] > 0), 8, "median", 2), px)
    # 151：低振幅加权价格锚点偏离，最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权平均，振幅小的窗口权重大。
    calm_w = 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)
    calm_den = roll(calm_w.where(px > 0), 8, "sum", 2)
    df["r_calm"] = safe_log_ratio(roll(px.where(px > 0) * calm_w, 8, "sum", 2) / calm_den.where(calm_den > 0), px)
    # 161：新增分量的公共工具。
    def lag(s, k):
        return s.groupby(df["instrument"], sort=False).shift(k)

    def B(c):
        return df["last_" + c]

    def dvwap():
        g = df.groupby([df["instrument"], df["day"]], sort=False)
        return vwap_of(g["sum_amt"].cumsum(), g["sum_vol"].cumsum())

    def side_px3(x):
        """单侧三档挂单量加权报价（x 为 "b" 或 "a"）。"""
        v = sum(B(f"{x}v{i}") for i in (1, 2, 3))
        return (sum(B(f"{x}p{i}") * B(f"{x}v{i}") for i in (1, 2, 3)) / v.where(v > 0)).where(book_ok())

    def book_ok():
        return (B("bp1") > 0) & (B("ap1") >= B("bp1"))

    def qmid():
        return ((B("bp1") + B("ap1")) / 2).where(book_ok())

    def micro():
        bv, av = B("bv1"), B("av1")
        return ((B("bp1") * av + B("ap1") * bv) / (bv + av).where(bv + av > 0)).where(book_ok())

    def depth3(sides=("bv", "av")):
        s = sum(B(f"{x}{i}") for x in sides for i in (1, 2, 3))
        return s.where(s > 0)

    def book3():
        num = sum(B(f"bp{i}") * B(f"bv{i}") + B(f"ap{i}") * B(f"av{i}") for i in (1, 2, 3))
        return num / depth3()

    def rvwap(k):
        return vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1))

    def calm_w():
        return 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)

    def wanchor_of(q, w, n=8):
        q = q.where(q > 0)
        ok = q.notna() & w.notna() & (w > 0)
        den = roll(w.where(ok), n, "sum", 2)
        return safe_log_ratio(roll((q * w).where(ok), n, "sum", 2) / den.where(den > 0), q)

    def self_anchor(q, fn="mean", n=8):
        q = q.where(q > 0)
        return safe_log_ratio(roll(q, n, fn, 2), q)

    def px_at_max(key, n=8):
        """最近 n 个窗口中 key 最大那个窗口的成交 VWAP（至少 2 个有效窗口）。"""
        vw = vwap_of(df["sum_amt"], df["sum_vol"])
        ks = np.column_stack([lag(key, k).to_numpy(np.float64) for k in range(n)])
        vs = np.column_stack([lag(vw, k).to_numpy(np.float64) for k in range(n)])
        ks = np.where(np.isfinite(ks) & np.isfinite(vs) & (vs > 0), ks, -np.inf)
        j = ks.argmax(1)
        out = vs[np.arange(len(j)), j]
        return pd.Series(np.where(np.isfinite(ks).sum(1) >= 2, out, np.nan), index=df.index)

    def day_mean(x):
        """当日截至当前窗口 x 的累计均值（跳过缺失）。"""
        k = [df["instrument"], df["day"]]
        n = x.notna().astype(np.float64).groupby(k, sort=False).cumsum()
        return x.fillna(0).groupby(k, sort=False).cumsum() / n.where(n > 0)

    def day_first(x):
        """当日 x 的第一个有效值。"""
        return x.groupby([df["instrument"], df["day"]], sort=False).transform("first")
    # 161：ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)，涨跌停单边盘口的窗口不参与锚点、当期缺失，没有当前价时缺失。
    df["r_qmid"] = ((self_anchor(qmid()))).where(px.notna())
    # 161：最近 40 个窗口 ln(窗口末当前价 / 窗口末成交价) 的均值（成交持续落在买一侧即卖压时为负）；方向取负，没有当前价时缺失。
    df["r_cmid"] = (-(roll(safe_log_ratio(px, B("close")), 40, "mean", 10))).where(px.notna())
    # 161：ln(最近 8 个窗口末价按三档挂单总量加权的均价 / 当前价)，挂单厚的窗口价格更可信，没有当前价时缺失。
    df["r_deep"] = ((wanchor_of(px, depth3()))).where(px.notna())
    # 161：ln(最近 8 个窗口成交 VWAP / 当前一档微价格)，微价格按对侧挂单量加权买一 / 卖一价，没有当前价时缺失。
    df["r_micro"] = ((safe_log_ratio(rvwap(8), micro()))).where(px.notna())
    # 161：ln(最近 8 个窗口末价按窗口成交笔数加权的均价 / 当前价)，没有当前价时缺失。
    df["r_dealpx"] = ((wanchor_of(px, df["sum_deal"].where(df["sum_deal"] > 0)))).where(px.notna())

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 低波动（取负）：最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值。
    df["c_park"] = -roll(safe_log_ratio(df["bar_high"], df["bar_low"]) ** 2, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的收益计算相关系数，两种频率各建一张网络：
    #   日频：日收益（收盘段价格 / 昨收），至少 PEER_MIN_OBS 个共同观测；
    #   30 分钟：窗口收益（上一窗口末价，当日首个窗口用昨收），取回看期内全部窗口，至少 4 x PEER_MIN_OBS 个共同观测。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    win_ret30 = pd.Series(
        (-df["r_30"]).values, index=pd.MultiIndex.from_arrays([df["date"].values, df["instrument"].values])
    ).unstack().sort_index().reindex(columns=peer_insts)

    def build_peers(ret_wide, row_day, need):
        """返回 {月份: 同伴下标矩阵}；ret_wide 每行是一个收益观测，row_day 为该行所属交易日。"""
        peers = {}
        for m in out_months:
            in_month = all_days[day_month == m]
            if len(in_month) == 0:
                continue
            prior = all_days[all_days < in_month[0]][-PEER_LOOKBACK:]
            if len(prior) < PEER_MIN_OBS:
                continue
            hist = ret_wide[(row_day < in_month[0]) & (row_day >= prior[0])]
            ok = (hist.notna().sum() >= need).values
            corr = hist.loc[:, ok].corr(min_periods=need).values
            np.fill_diagonal(corr, -np.inf)
            corr = np.nan_to_num(corr, nan=-np.inf)
            sub = np.where(ok)[0]
            idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
            idx[sub] = sub[np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]]
            peers[m] = idx
        return peers

    # 超额网络：每个收益观测先减去同一时点的截面均值，再按超额收益的相关性挑同伴（去掉市场共振带来的伪同伴）。
    peers_by_freq = {
        "day": build_peers(daily_ret, all_days, PEER_MIN_OBS),
        "30": build_peers(win_ret30, win_ret30.index.normalize(), PEER_MIN_OBS * 4),
        "day_excess": build_peers(daily_ret.sub(daily_ret.mean(axis=1), axis=0), all_days, PEER_MIN_OBS),
        "30_excess": build_peers(win_ret30.sub(win_ret30.mean(axis=1), axis=0), win_ret30.index.normalize(), PEER_MIN_OBS * 4),
    }

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro", "r_dealpx", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_park"]
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col, peers_by_month):
        """每个截面上同伴的 col 中位数（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            cnt = (~np.isnan(g)).sum(axis=2)
            pm = np.where(cnt >= 3, np.nanmedian(g, axis=2), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def peer_rank(col, peers_by_month):
        """个股 col 在其同伴中的相对排名（低于自身的同伴占比，平局记半），居中到 [-0.5, 0.5]；
        同伴有效值不足 3 个或没有同伴信息时回退为截面居中排名。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            x = wide.values
            valid = idx >= 0
            g = x[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            own = x[:, :, None]
            ok = ~np.isnan(g) & ~np.isnan(own)
            cnt = ok.sum(axis=2)
            score = ((g < own) & ok).sum(axis=2) + 0.5 * ((g == own) & ok).sum(axis=2)
            rel = np.where(cnt >= 3, score / np.maximum(cnt, 1) - 0.5, np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, rel[r_pos, np.maximum(c_pos, 0)], np.nan)
        fallback = result.groupby("date")[col].rank(pct=True) - 0.5
        return pd.Series(out, index=result.index).fillna(fallback)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx"]
    neutral_cols = []
    for c in ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro", "r_dealpx"]:
        cross_mean = result.groupby("date")[c].transform("mean")
        for f, peers in peers_by_freq.items():
            # 每个分量 x 每张同伴网络：同伴中位数残差与同伴内相对排名各自截面排名后等权融合，再做四特征中性化。
            result[f"{c}_p{f}"] = result[c] - peer_mean(c, peers).fillna(cross_mean)
            result[f"{c}_r{f}"] = peer_rank(c, peers)
            result[f"{c}_b{f}"] = rank_blend([f"{c}_p{f}", f"{c}_r{f}"])
            name = f"{c}_n{f}"
            result[name] = neutralize(f"{c}_b{f}", CHARS4)
            neutral_cols.append(name)
    neutral_ranks = result.groupby("date")[neutral_cols].rank(pct=True) - 0.5
    lower = neutral_ranks.quantile(0.25, axis=1)
    upper = neutral_ranks.quantile(0.75, axis=1)
    # IQR Winsor 均值：各列先截到本行四分位区间，再取均值（中间一半的票全部参与，而不只看两个分位点）。
    result["iqr"] = neutral_ranks.clip(lower=lower, upper=upper, axis=0).mean(axis=1, skipna=True)
    # 146：慢特征组合 = 低波动 / 小盘 / 低价三者截面排名等权相加，再与反转主信号按 1 : 0.5 做截面排名融合。
    char_ranks = result.groupby("date")[["c_park", "c_size", "c_lowpx"]].rank(pct=True)
    result["slow"] = char_ranks["c_park"] + char_ranks["c_size"] + char_ranks["c_lowpx"]
    result["factor"] = rank_blend(["iqr", "slow"], [1.0, 0.5])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def main_b10(datasources, start_date, end_date):
    """【1397：临时峰度反向】新根单独版（公榜探针），STAT = A_kurt，方向 反向。
    根 = 1370_分布形态框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b10(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "A_kurt"))
    out = pd.concat(parts, ignore_index=True)
    out["factor"] = -out["factor"]                                           # 反向交付（09:30 仍为 0）
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b10(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z", "A"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(r * r) OVER w AS crr, COUNT(r) OVER w AS cnr, SUM(r * r * r * r) OVER w AS cr4
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "A_kurt": lambda: df["cnr"] * df["cr4"] / (df["crr"] ** 2).where(df["crr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b11(datasources, start_date, end_date):
    """【1397：临时五日偏度反向】新根单独版（公榜探针），STAT = A_skew5，方向 反向。
    根 = 1370_分布形态框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b11(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "A_skew5"))
    out = pd.concat(parts, ignore_index=True)
    out["factor"] = -out["factor"]                                           # 反向交付（09:30 仍为 0）
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b11(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z", "A"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * r * r) OVER w5 AS cr3_5, SUM(r * r) OVER w5 AS crr_5, COUNT(r) OVER w5 AS n5r
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "A_skew5": lambda: np.sqrt(df["n5r"]) * df["cr3_5"] / (df["crr_5"] ** 1.5).where(df["crr_5"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b12(datasources, start_date, end_date):
    """【1406：前日收盘竞价价格冲击】新根单独版（公榜探针），STAT = O_cauc，方向 正向。
    根 = 1400_集合竞价框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b12(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "O_cauc"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b12(datasources, start_date, end_date, STAT):
    """【新根因子 第十轮：集合竞价与隔夜框架（1400 起）】一份代码覆盖开盘集合竞价、隔夜、前一日收盘集合竞价、首半小时，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    数据口径（1 分钟表）：09:16–09:24 的 bar 成交量为 0，收盘价是集合竞价的虚拟撮合价；一档买、卖价量相同（= 撮合价与可撮合量），
    未匹配量只出现在二档的一侧（价格记 0）；09:25 的 bar 是开盘集合竞价的成交（约 55% 的股票这一 bar 有成交量，其余为 0，
    竞价量类变体只覆盖前者）；15:00 的 bar 含收盘集合竞价成交，14:57 的 bar 收盘价是收盘竞价前的最后连续竞价价。
    当日只用 10:00 及之前的 bar（所有截面都在 10:00 及之后）；当日收盘、全天成交量等一律取前一交易日（按股票 shift(1)）。
    每个交易日一个值，7 个截面相同（日内常数，换手低）。不读 vwap 表、不读暴露表。

    STAT：
      O_gap     隔夜跳空：开盘竞价价 ÷ 前一日收盘 − 1
      O_aucv20  开盘竞价成交量 ÷ 过去 20 个交易日开盘竞价成交量均值（对数）
      O_aucs    开盘竞价成交量 ÷ 前一日全天成交量
      O_pre     竞价后段价格变动：开盘竞价价 ÷ 09:20 虚拟撮合价 − 1（09:20 之后不能撤单）
      O_imb     09:24 未匹配量不平衡：(未匹配买 − 未匹配卖) ÷ (可撮合量 + 未匹配买 + 未匹配卖)
      O_cauc    前一日收盘竞价的价格冲击：前一日收盘 ÷ 前一日 14:57 价 − 1
      O_caucs   前一日收盘竞价成交量占全天比例
      O_f30     首半小时收益：10:00 价 ÷ 开盘竞价价 − 1
      O_pre1    竞价前段（可撤单阶段）价格变动：09:20 虚拟撮合价 ÷ 09:16 虚拟撮合价 − 1
      O_lim1    前一日收盘涨停（1）／跌停（−1）／其他（0）；涨跌幅限制按代码：688、300、301 开头 20%，8、4 开头 30%，其余 10%（ST 无法识别，按 10%）
      O_lim5    过去 5 个交易日收盘涨停次数 − 跌停次数
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=45)).strftime("%Y-%m-%d %H:%M:%S")   # 20 个交易日回看 + 前一日
    sql = f"""
    WITH b AS (
        SELECT b.instrument_id, ai.instrument, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M%S') AS t,
            CAST(b.close AS DOUBLE) AS c, CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v,
            CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.bid_volume2, 0) AS DOUBLE) AS bv2,
            CAST(COALESCE(b.ask_volume2, 0) AS DOUBLE) AS av2
        FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
        WHERE b.close > 0
    )
    SELECT instrument_id, MAX(instrument) AS instrument, d,
        MAX(CASE WHEN t = '091600' THEN c END) AS px0916, MAX(CASE WHEN t = '092000' THEN c END) AS px0920,
        MAX(CASE WHEN t = '092400' THEN bv1 END) AS m0924, MAX(CASE WHEN t = '092400' THEN bv2 END) AS ub0924,
        MAX(CASE WHEN t = '092400' THEN av2 END) AS ua0924,
        MAX(CASE WHEN t = '092500' THEN c END) AS apx, MAX(CASE WHEN t = '092500' THEN v END) AS aucv,
        MAX(CASE WHEN t = '100000' THEN c END) AS c1000,
        MAX(CASE WHEN t = '145700' THEN c END) AS c1457,
        MAX(CASE WHEN t = '150000' THEN c END) AS cls, MAX(CASE WHEN t = '150000' THEN v END) AS cauv,
        SUM(v) AS dv
    FROM b GROUP BY instrument_id, d
    """
    g = dai.query(sql, filters={"date": [q0, end_date]}, compression=True).df()
    if not len(g):
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    g["instrument"] = g["instrument"].astype(str)
    g["d"] = pd.to_datetime(g["d"])
    g = g.sort_values(["instrument", "d"]).reset_index(drop=True)
    grp = g.groupby("instrument", sort=False)
    with np.errstate(all="ignore"):
        prev_cls = grp["cls"].shift(1)                                   # 只用前一交易日的收盘、全天量、收盘竞价
        prev_dv = grp["dv"].shift(1)
        prev_c1457 = grp["c1457"].shift(1)
        prev_cauv = grp["cauv"].shift(1)
        mu20 = grp["aucv"].transform(lambda s: s.shift(1).rolling(20, min_periods=10).mean())
        tot = g["m0924"] + g["ub0924"] + g["ua0924"]
        _code = g["instrument"].str[:3]
        _lim = np.where(_code.isin(["688", "300", "301"]), 0.20, np.where(g["instrument"].str[:1].isin(["8", "4"]), 0.30, 0.10))
        _pc = grp["cls"].shift(1)                                        # 涨跌停判定用"当日收盘 vs 前一日收盘"，再整体后移一天才用
        _up = (g["cls"] >= np.round(_pc * (1 + _lim), 2) - 1e-6).astype(float).where(_pc > 0)
        _dn = (g["cls"] <= np.round(_pc * (1 - _lim), 2) + 1e-6).astype(float).where(_pc > 0)
        g["_lstate"] = _up - _dn
        lim1 = g.groupby("instrument", sort=False)["_lstate"].shift(1)
        lim5 = g.groupby("instrument", sort=False)["_lstate"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).sum())
        sig = {
            "O_gap": g["apx"] / prev_cls.where(prev_cls > 0) - 1,
            "O_aucv20": np.log(g["aucv"].where(g["aucv"] > 0) / mu20.where(mu20 > 0)),
            "O_aucs": g["aucv"] / prev_dv.where(prev_dv > 0),
            "O_pre": g["apx"] / g["px0920"].where(g["px0920"] > 0) - 1,
            "O_imb": (g["ub0924"] - g["ua0924"]) / tot.where(tot > 0),
            "O_cauc": prev_cls / prev_c1457.where(prev_c1457 > 0) - 1,
            "O_caucs": prev_cauv / prev_dv.where(prev_dv > 0),
            "O_f30": g["c1000"] / g["apx"].where(g["apx"] > 0) - 1,
            "O_pre1": g["px0920"] / g["px0916"].where(g["px0916"] > 0) - 1,
            "O_lim1": lim1,
            "O_lim5": lim5,
        }[STAT]
    g["signal"] = np.asarray(sig, dtype=float)
    g = g[g["d"] >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
    rows = []
    for t in SLOTS:
        s = g[["d", "instrument", "signal"]].copy()
        s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
        rows.append(s[["date", "instrument", "signal"]])
    df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b13(datasources, start_date, end_date):
    """
    更长倍增阶梯 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    152 在 151 号基础上去掉两个分量 r_lo（当日低点距离）与 r_slope（VWAP 斜率），其余（同伴网络、中性化、聚合、慢特征融合）全部不变，
    即 146 的 6 个阶梯分量 + r_med / r_mid / r_lomed / r_calm 四个价格锚点，10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    原因：窗口末价格取截面那一分钟，09:30 截面没有这一分钟（1 分钟数据只有 09:25 竞价和 09:31 之后），当前价缺失。
    146 在该截面所有分量缺失、整体补 0 而不参与评估；r_slope 只用成交额与成交量、不含当前价，147–151 在该截面只剩它一个分量，
    该截面 IC 仅约 0.02，拖低整体（本地按同一口径复现：147 相对 146 估计分差 -0.13，与平台 -0.11 一致）。
    152 的所有分量都以当前价为分母，09:30 截面与 146 相同地整体补 0。四个锚点作为一组加入 146 的 IQR，
    在 09:30 截面按平台口径评估时验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.3892 / 0.3850（146 0.4346 / 0.4256）；
      IC 0.08417 / 0.06863（146 0.08398 / 0.06917）；
      ICIR 1.7965 / 1.1614（146 1.7811 / 1.1693）；
      Sharpe 53.06 / 35.48（146 53.27 / 35.88）；
      压力 1.7921 / 1.1388（146 1.7593 / 1.1520）。
    以下为 151 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十二分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    151 在 150 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_calm : 低振幅加权价格锚点偏离 ln(锚点 / 当前价)，锚点为最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权的均值。
    价格稳定（振幅小）的窗口里成交价更接近共识价值，权重更大；剧烈波动窗口的价格噪声大，权重更小。
    窗口长度沿用已有的 8，0.001 只是防止零振幅除零的下限，无新增可调参数。12 分量 x 四网络 = 48 列做 IQR Winsor 均值。
    这一分量来自第十七批“换权重口径锚点”信号的分量级筛选（逐个作为新分量放进 150 的 IQR），
    验证段与保留段同时为正且验证段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4174 / 0.4117（150 0.4267 / 0.4194）；
      IC 0.07968 / 0.06546（150 0.08003 / 0.06574）；
      ICIR 1.6943 / 1.1368（150 1.7005 / 1.1415）；
      Sharpe 47.72 / 34.28（150 47.56 / 34.33）；
      压力 1.6870 / 1.1431（150 1.6900 / 1.1467）。
    以下为 150 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 十一分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    150 在 149 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lomed : 最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)。
    r_lo 只看当日累计最低点这一个极值，r_lomed 取近 8 个窗口各自低点的中位数，是稳健的近期支撑位；
    窗口长度沿用已有的 8，无新增参数。11 分量 x 四网络 = 44 列做 IQR Winsor 均值。
    这一分量来自第十四至十八批“锚点”信号的分量级筛选（逐个作为新分量放进 149 的 IQR，共 20 个），
    验证段与保留段同时为正且两段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4267 / 0.4194（149 0.4303 / 0.4236）；
      IC 0.08003 / 0.06574（149 0.07995 / 0.06575）；
      ICIR 1.7005 / 1.1415（149 1.6975 / 1.1403）；
      Sharpe 47.56 / 34.33（149 47.31 / 34.16）；
      压力 1.6900 / 1.1467（149 1.6915 / 1.1474）。
    以下为 149 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格与区间中点中位数偏离 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    149 在 148 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_mid : 区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)。
    r_med 的锚点只用每个窗口末的一个报价，r_mid 用的是每个窗口内全部成交覆盖的价格区间的中点，
    反映窗口内的成交中枢，而不只是窗口收盘时刻的价格。窗口长度沿用已有的 8，无新增参数。10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    这一分量来自第十一批“稳健锚点”信号的分量级筛选（逐个作为新分量放进 148 的 IQR），
    4 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4303 / 0.4236（148 0.4365 / 0.4285）；
      IC 0.07995 / 0.06575（148 0.07984 / 0.06590）；
      ICIR 1.6975 / 1.1403（148 1.6944 / 1.1434）；
      Sharpe 47.31 / 34.16（148 47.28 / 33.82）；
      压力 1.6915 / 1.1474（148 1.6929 / 1.1516）。
    以下为 148 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格中位数偏离 九分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    148 在 147 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_med : 价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)。
    阶梯中的 r_rv8 以同样 8 个窗口的成交量加权均价为锚，会被单个放量窗口拉动；中位数锚点不受成交量和单个极端价格影响，
    衡量的是当前价相对近期“典型价格”的偏离。窗口长度沿用阶梯中已有的 8，无新增参数。9 分量 x 四网络 = 36 列做 IQR Winsor 均值。
    这一分量来自第十批 16 个“换锚点”反转信号的分量级筛选（逐个作为新分量放进 147 的 IQR），
    16 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4365 / 0.4285（147 0.4389 / 0.4314）；
      IC 0.07984 / 0.06590（147 0.07933 / 0.06569）；
      ICIR 1.6944 / 1.1434（147 1.6821 / 1.1417）；
      Sharpe 47.28 / 33.82（147 46.51 / 33.45）；
      压力 1.6929 / 1.1516（147 1.6729 / 1.1436）。
    以下为 147 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 八分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    147 在 146 号基础上只在 IQR 内部加两个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lo    : 当日低点距离 ln(min(当日累计窗口最低价, 昨收) / 当前价)，衡量从当日低点（或昨收）以来的反弹幅度；
      r_slope : 窗口 VWAP 斜率 ln(滚动 8 窗口 VWAP / 当前窗口 VWAP)，衡量当前窗口成交均价相对近期均价的抬升。
    两者与阶梯中的“价格相对 VWAP 偏离”口径不同：前者以当日极值为锚，后者只看成交均价、不含当前报价，
    与已有分量相关较低。8 分量 x 四网络 = 32 列做 IQR Winsor 均值。
    这两个分量来自对 215 个结构不同信号的批量分量级筛选（逐个作为新分量放进 146 的 IQR），验证段与保留段同时为正；
    所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4389 / 0.4314（146 0.4798 / 0.4698）；
      IC 0.07933 / 0.06569（146 0.08049 / 0.06665）；
      ICIR 1.6821 / 1.1417（146 1.7022 / 1.1562）；
      Sharpe 46.51 / 33.45（146 47.27 / 34.23）；
      压力 1.6729 / 1.1436（146 1.6992 / 1.1468）。
    以下为 146 的说明。
    更长倍增阶梯四网络同伴排名 IQR（144）+ 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。
    146 在 144 号基础上只加一个与反转正交的慢特征组合：
      c_park  : 低波动，最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值取负；
      c_size  : 小盘，-ln(1 + 最近 40 个窗口清洗后成交额之和)（144 中已有，用于中性化）；
      c_lowpx : 低价，-ln(当前价)（144 中已有，用于中性化）。
    三者截面排名等权相加，再与 144 的 IQR 输出按固定的 1 : 0.5 做截面排名融合。三者都是 A 股文献中的经典异象
    （低波动、小市值、低价），与反转主信号的风格剔除后相关约为 0；它们几乎不随时间变化，融合后换手下降。
    这一组合来自对 130 个结构不同信号的批量筛选，验证段与保留段同时为正；融合权重对全部信号统一取 0.5，未逐个调参。
      验证 / 保留两段换手 0.4798 / 0.4698（144 0.4969 / 0.4883）；
      IC 0.08049 / 0.06665（144 0.07809 / 0.06452）；
      ICIR 1.7022 / 1.1562（144 1.6088 / 1.1070）；
      Sharpe 47.27 / 34.23（144 49.77 / 35.09）；
      压力 1.6992 / 1.1468（144 1.5654 / 1.0782）。
    以下为 144 的说明。
    平台原始价格口径下的更长倍增阶梯超额四网络同伴排名，IQR Winsor 均值聚合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    144 在 143 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 32 窗口（约 4 个交易日，
    接近一周）清洗后成交 VWAP 偏离 r_rv32。六个分量为 2 / 4 / 8 / 16 / 32 窗口 VWAP 偏离 + 日内反转，
    x 四张同伴网络，24 列做 IQR Winsor 均值。周尺度锚点更新更慢，合成信号相邻截面更连续，换手继续下降；
    窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.4969 / 0.4883（143 0.5223 / 0.5155）；
      IC 0.07809 / 0.06452（143 0.07838 / 0.06514）；
      ICIR 1.6088 / 1.1070（143 1.6135 / 1.1239）；
      Sharpe 49.77 / 35.09（143 50.83 / 35.84）；
      压力 1.5654 / 1.0782（143 1.5519 / 1.0812）。
    以下为 143 的说明。
    143 在 142 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 16 窗口（约 2 个交易日）
    清洗后成交 VWAP 偏离 r_rv16。五个分量为 2 / 4 / 8 / 16 窗口 VWAP 偏离 + 日内反转，x 四张同伴网络，
    20 列做 IQR Winsor 均值。慢端锚点跨日更新缓慢，加入后合成信号的截面变化更平稳，同时多日尺度的
    VWAP 偏离本身也带有反转信息；窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.5223 / 0.5155（142 0.5688 / 0.5623）；
      IC 0.07838 / 0.06514（142 0.07808 / 0.06547）；
      ICIR 1.6135 / 1.1239（142 1.6142 / 1.1341）；
      Sharpe 50.83 / 35.84（142 51.17 / 36.54）；
      压力 1.5519 / 1.0812（142 1.5298 / 1.0755）。
    以下为 142 的说明。
    142 在 140 号基础上只换一个分量：30 分钟反转 r_30（只看上一窗口末的单个价格点）换成滚动 2 窗口（1 小时）
    清洗后成交 VWAP 偏离 r_rv2。四个分量成为 2 / 4 / 8 窗口 VWAP 偏离的倍增阶梯 + 日内反转，尺度均匀覆盖，
    且短端由成交量加权的均价代替单点价格，相邻截面之间更连续；首窗不再与 r_day 重复，无需再做首窗去重。
    各分量 x 四张同伴网络，16 列做 IQR Winsor 均值。窗口长度沿用已有的倍增序列，无新增参数。
      验证 / 保留两段换手 0.5688 / 0.5623（140 0.6005 / 0.5949）；
      IC 0.07808 / 0.06547（140 0.07906 / 0.06626）；
      ICIR 1.6142 / 1.1341（140 1.6303 / 1.1475）；
      Sharpe 51.17 / 36.54（140 51.55 / 36.53）；
      压力 1.5298 / 1.0755（140 1.5472 / 1.0809）。
    以下为 141 的说明（141 去掉 r_30 只留三个慢分量，换手降得更多但 IC 也掉得更多，142 是两者之间的折中）。
    141 在 140 号基础上只做一个无参结构改动：去掉 30 分钟反转分量 r_30，只保留滚动 VWAP 4/8 窗口偏离与
    日内反转三个慢分量（各 x 四张同伴网络，12 列做 IQR Winsor 均值）。r_30 只看最近一个窗口，相邻截面之间
    几乎完全更新，是多空两腿成分替换的主要来源；平台五项得分中换手的权重最大，r_30 的边际 IC 不足以抵消它带来的换手。
    r_30 仍用于构建 30 分钟同伴网络（窗口收益相关），只是不再作为合成分量。无新增参数。
      验证 / 保留两段换手 0.4775 / 0.4696（140 0.6005 / 0.5949，降约 0.12）；
      IC 0.07646 / 0.06347（140 0.07906 / 0.06626）；
      ICIR 1.5766 / 1.1041（140 1.6303 / 1.1475）；
      Sharpe 49.52 / 34.85（140 51.55 / 36.53）；
      压力 1.5145 / 1.0732（140 1.5472 / 1.0809）。
    以下为 140 的说明。
    140 在 137 号基础上只改最后一步聚合：16 列中性化排名由 Midhinge
    （25%/75% 分位点均值）改为 IQR Winsor 均值（各列截到本行四分位区间后取均值）。Midhinge 只由两个分位点决定，
    任一列的小幅变动都可能让分位点在相邻列之间跳动，截面排名随之抖动；Winsor 均值让中间一半的列都连续地参与，
    相邻截面更平滑。无新增参数。平台五项得分对换手最敏感之一，此改动的主要收益是降换手：
      开发 / 验证 / 保留三段换手 0.6009 / 0.6005 / 0.5949（137 0.6211 / 0.6194 / 0.6132，三段均降约 0.02）；
      IC 0.08895 / 0.07906 / 0.06626（137 0.08917 / 0.07937 / 0.06643）；
      ICIR 1.4783 / 1.6303 / 1.1475（137 1.4816 / 1.6362 / 1.1496）；
      Sharpe 49.77 / 51.55 / 36.53（137 50.05 / 52.27 / 36.73）；
      压力 1.4021 / 1.5472 / 1.0809（137 1.4049 / 1.5532 / 1.0807）。
    以下为 137 的说明。
    在 136 号（首窗去重的 4 个反转分量 x 日频/30 分钟两张同伴网络，各做“中位数残差 + 同伴内排名”融合与
    四特征中性化，8 列取 Midhinge）基础上只加一个无参结构改动：
      另建两张“超额收益”同伴网络：日收益 / 30 分钟窗口收益先减去同一时点的截面均值，再按相关性取前 30 名同伴。
      原始收益相关里很大一部分是共同的市场涨跌，高贝塔股票会因此互为同伴；超额网络只保留真实的同类联动。
      原始 / 超额 x 日频 / 30 分钟四张网络并列，每个分量在每张网络上照旧做融合与中性化，16 列取 Midhinge
      （09:30 截面去掉 r_30 的 4 列，用 12 列）。不新增任何参数，同伴数、回看期、最少观测都与 136 相同。
    相对 136，本地开发 / 验证 / 保留三段：
      IC 0.08917 / 0.07937 / 0.06643（136 0.08910 / 0.07944 / 0.06629）；
      ICIR 1.4816 / 1.6362 / 1.1496（136 1.4692 / 1.6274 / 1.1277）；
      Sharpe 50.05 / 52.27 / 36.73（136 49.94 / 51.94 / 36.93）；
      压力 1.4049 / 1.5532 / 1.0807（136 1.3931 / 1.5425 / 1.0629）；
      换手 0.6211 / 0.6194 / 0.6132（136 0.6188 / 0.6175 / 0.6102）。
    136 的改动：当日首个窗口的 30 分钟反转以昨收为基准，与 r_day 完全相同，首窗把 r_30 各列置为缺失，避免重复计票；
    r_30 仍保留首窗定义用于构建 30 分钟同伴网络。
    公共构造（103–107 共用同一段代码，只有最后的合成方式不同）：
      反转分量（与 90 号相同，均只用过去数据）：
        r_30  : 30 分钟反转 ln(上一窗口末价 / 当前价)，当日首个窗口以昨收为基准（57 号）
        r_rv4 / r_rv8 : 滚动 4/8 个窗口（2/4 小时，跨日）清洗后成交 VWAP 偏离 ln(滚动 VWAP / 当前价)
        r_day : 日内反转 ln(昨收 / 当前价)（56 号）
      同伴（伪行业）残差：每个自然月月初，用此前 60 个交易日的日收益 ln(收盘段价格 / 昨收) 计算股票池内两两相关系数，
        每只股票取相关性最高的 30 只为同伴（至少 30 个共同观测；只用过去数据）。分量残差 = 分量 - 同伴中位数
        （同伴有效值不足 3 个时减截面均值）。思路：行业内反转强于全市场反转（Da, Liu & Schaumburg 2014），
        去掉“整组一起涨跌”的部分后，只保留个股相对同类股票的过度反应。
      四特征中性化：因子截面排名对 规模（-ln 最近 40 窗口成交额）、特质波动、非流动性（Amihud）、低价 四个特征的截面排名
        做逐截面回归，取残差。反转信号天然偏向小盘、高波动股票，而这些暴露本身会拖累排序；剔除后 IC、IC_IR、多空夏普同时提升。
      数据清洗：成交均价落在该分钟 [最低价 x 0.98, 最高价 x 1.02] 之外的分钟不计入任何 VWAP / 成交额累加。

    稳健性：最终缺失补 0（中性），每个截面缺失率为 0；向前取不到足够历史时同伴退化为截面均值、特征排名记为中性。
    性能：纯 SQL 向量化（单次 GROUP BY，无窗口函数、无 Python UDF）；相关矩阵每月计算一次（约 1200 x 1200），
      同伴均值按月份用数组索引批量计算。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 150
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS = 30, 60, 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px ELSE 0 END) AS last_px,
            MAX(pre_close) AS pre_close,
            MAX(px_high) AS bar_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS bar_low
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.last_px, f.pre_close, f.bar_high, f.bar_low
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "last_px", "pre_close", "bar_high", "bar_low"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv2"] = rolling_vwap_rev(2)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_rv32"] = rolling_vwap_rev(32)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)
    # 148：价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)，以稳健的近期价格中枢为锚。
    df["r_med"] = safe_log_ratio(roll(px, 8, "median", 2), px)
    # 149：区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)，以窗口内全部成交区间的中枢为锚。
    df["r_mid"] = safe_log_ratio(roll(((df["bar_high"] + df["bar_low"]) / 2).where(df["bar_low"] > 0), 8, "median", 2), px)
    # 150：最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)，以近期各窗口低点的稳健中枢为支撑锚。
    df["r_lomed"] = safe_log_ratio(roll(df["bar_low"].where(df["bar_low"] > 0), 8, "median", 2), px)
    # 151：低振幅加权价格锚点偏离，最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权平均，振幅小的窗口权重大。
    calm_w = 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)
    calm_den = roll(calm_w.where(px > 0), 8, "sum", 2)
    df["r_calm"] = safe_log_ratio(roll(px.where(px > 0) * calm_w, 8, "sum", 2) / calm_den.where(calm_den > 0), px)

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 低波动（取负）：最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值。
    df["c_park"] = -roll(safe_log_ratio(df["bar_high"], df["bar_low"]) ** 2, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的收益计算相关系数，两种频率各建一张网络：
    #   日频：日收益（收盘段价格 / 昨收），至少 PEER_MIN_OBS 个共同观测；
    #   30 分钟：窗口收益（上一窗口末价，当日首个窗口用昨收），取回看期内全部窗口，至少 4 x PEER_MIN_OBS 个共同观测。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    win_ret30 = pd.Series(
        (-df["r_30"]).values, index=pd.MultiIndex.from_arrays([df["date"].values, df["instrument"].values])
    ).unstack().sort_index().reindex(columns=peer_insts)

    def build_peers(ret_wide, row_day, need):
        """返回 {月份: 同伴下标矩阵}；ret_wide 每行是一个收益观测，row_day 为该行所属交易日。"""
        peers = {}
        for m in out_months:
            in_month = all_days[day_month == m]
            if len(in_month) == 0:
                continue
            prior = all_days[all_days < in_month[0]][-PEER_LOOKBACK:]
            if len(prior) < PEER_MIN_OBS:
                continue
            hist = ret_wide[(row_day < in_month[0]) & (row_day >= prior[0])]
            ok = (hist.notna().sum() >= need).values
            corr = hist.loc[:, ok].corr(min_periods=need).values
            np.fill_diagonal(corr, -np.inf)
            corr = np.nan_to_num(corr, nan=-np.inf)
            sub = np.where(ok)[0]
            idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
            idx[sub] = sub[np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]]
            peers[m] = idx
        return peers

    # 超额网络：每个收益观测先减去同一时点的截面均值，再按超额收益的相关性挑同伴（去掉市场共振带来的伪同伴）。
    peers_by_freq = {
        "day": build_peers(daily_ret, all_days, PEER_MIN_OBS),
        "30": build_peers(win_ret30, win_ret30.index.normalize(), PEER_MIN_OBS * 4),
        "day_excess": build_peers(daily_ret.sub(daily_ret.mean(axis=1), axis=0), all_days, PEER_MIN_OBS),
        "30_excess": build_peers(win_ret30.sub(win_ret30.mean(axis=1), axis=0), win_ret30.index.normalize(), PEER_MIN_OBS * 4),
    }

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_park"]
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col, peers_by_month):
        """每个截面上同伴的 col 中位数（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            cnt = (~np.isnan(g)).sum(axis=2)
            pm = np.where(cnt >= 3, np.nanmedian(g, axis=2), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def peer_rank(col, peers_by_month):
        """个股 col 在其同伴中的相对排名（低于自身的同伴占比，平局记半），居中到 [-0.5, 0.5]；
        同伴有效值不足 3 个或没有同伴信息时回退为截面居中排名。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            x = wide.values
            valid = idx >= 0
            g = x[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            own = x[:, :, None]
            ok = ~np.isnan(g) & ~np.isnan(own)
            cnt = ok.sum(axis=2)
            score = ((g < own) & ok).sum(axis=2) + 0.5 * ((g == own) & ok).sum(axis=2)
            rel = np.where(cnt >= 3, score / np.maximum(cnt, 1) - 0.5, np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, rel[r_pos, np.maximum(c_pos, 0)], np.nan)
        fallback = result.groupby("date")[col].rank(pct=True) - 0.5
        return pd.Series(out, index=result.index).fillna(fallback)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx"]
    neutral_cols = []
    for c in ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm"]:
        cross_mean = result.groupby("date")[c].transform("mean")
        for f, peers in peers_by_freq.items():
            # 每个分量 x 每张同伴网络：同伴中位数残差与同伴内相对排名各自截面排名后等权融合，再做四特征中性化。
            result[f"{c}_p{f}"] = result[c] - peer_mean(c, peers).fillna(cross_mean)
            result[f"{c}_r{f}"] = peer_rank(c, peers)
            result[f"{c}_b{f}"] = rank_blend([f"{c}_p{f}", f"{c}_r{f}"])
            name = f"{c}_n{f}"
            result[name] = neutralize(f"{c}_b{f}", CHARS4)
            neutral_cols.append(name)
    neutral_ranks = result.groupby("date")[neutral_cols].rank(pct=True) - 0.5
    lower = neutral_ranks.quantile(0.25, axis=1)
    upper = neutral_ranks.quantile(0.75, axis=1)
    # IQR Winsor 均值：各列先截到本行四分位区间，再取均值（中间一半的票全部参与，而不只看两个分位点）。
    result["iqr"] = neutral_ranks.clip(lower=lower, upper=upper, axis=0).mean(axis=1, skipna=True)
    # 146：慢特征组合 = 低波动 / 小盘 / 低价三者截面排名等权相加，再与反转主信号按 1 : 0.5 做截面排名融合。
    char_ranks = result.groupby("date")[["c_park", "c_size", "c_lowpx"]].rank(pct=True)
    result["slow"] = char_ranks["c_park"] + char_ranks["c_size"] + char_ranks["c_lowpx"]
    result["factor"] = rank_blend(["iqr", "slow"], [1.0, 0.5])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def main_b14(datasources, start_date, end_date):
    """【收盘段季节性 L_close20】最后半小时 (14:30,15:00]（收盘集合竞价决定该段收盘价）的"段收盘 ÷ 段 VWAP − 1"在过去 20 个交易日的均值；
    同一交易日内各截面（10:00…14:30）取值相同，逐截面百分位排名 − 0.5，09:30 置 0。只用截面所在交易日之前的完整交易日。
    30 分钟段与 1232（同时段季节性）同一段 SQL：1 分钟 bar 聚成 (09:30,10:00] … (14:30,15:00]。按自然月分批调用，控制内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_close_impl_b14(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), 20))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _close_impl_b14(datasources, start_date, end_date, N):
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
    sql_seg = f"""
    WITH b AS (
        SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
            CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
            CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
            CASE
                WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
            END AS seg
        FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
        WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
          AND b.close > 0 AND b.open > 0
    )
    SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
        SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
    FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
    """
    sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
    if not len(sg):
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    sg["instrument"] = sg["instrument"].astype(str)
    sg["d"] = pd.to_datetime(sg["d"])
    sg = sg[sg["seg"] == "1500"]
    with np.errstate(all="ignore"):
        sg = sg.assign(lab=sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1)
    piv = sg.pivot_table(index="d", columns="instrument", values="lab", aggfunc="last").sort_index()
    mu = piv.rolling(N, min_periods=N // 2).mean().shift(1)                  # 只用之前的交易日
    out_days = mu.index[mu.index >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
    s0 = mu.loc[out_days].stack().rename("signal").reset_index()
    rows = []
    for t in SLOTS:
        s = s0.copy()
        s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
        rows.append(s[["date", "instrument", "signal"]])
    df = pd.concat(rows, ignore_index=True)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b15(datasources, start_date, end_date):
    """【收盘位置 CLOSEPOS20】(当日收盘价 ÷ 当日 8 个半小时段收盘价的最高者 − 1) 在过去 20 个交易日的均值（至少 10 日，按同一股票自己的交易日滚动，错开 1 日）。
    同一交易日内各截面（10:00…14:30）取值相同（"日内不变锚"，压换手），逐截面百分位排名 − 0.5，09:30 置 0。口径同 因子实验/k1_anchor.py。
    30 分钟段与 1232 同一段 SQL：1 分钟 bar 聚成 (09:30,10:00] … (14:30,15:00]。按自然月分批调用，控制内存。不用成交笔数。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_cp_impl_b15(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), 20))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _cp_impl_b15(datasources, start_date, end_date, N):
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    SEGS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=60)).strftime("%Y-%m-%d %H:%M:%S")
    sql_seg = f"""
    WITH b AS (
        SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
            CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
            CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
            CASE
                WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
            END AS seg
        FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
        WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
          AND b.close > 0 AND b.open > 0
    )
    SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
        SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
    FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
    """
    sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
    if not len(sg):
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    sg["instrument"] = sg["instrument"].astype(str)
    sg["d"] = pd.to_datetime(sg["d"])
    cl = sg.pivot_table(index=["instrument", "d"], columns="seg", values="c_last", aggfunc="last").reindex(columns=SEGS).sort_index()
    with np.errstate(all="ignore"):
        cp = cl["1500"] / cl[SEGS].max(axis=1) - 1.0
    g = cp.groupby(level=0, sort=False)
    m = g.rolling(N, min_periods=N // 2).mean()
    m.index = m.index.droplevel(0)
    m = m.reindex(cp.index).groupby(level=0, sort=False).shift(1)            # 只用之前的交易日
    d0 = pd.Timestamp(pd.to_datetime(start_date).normalize())
    s0 = m[m.index.get_level_values(1) >= d0].rename("signal").reset_index()
    rows = []
    for t in SLOTS:
        s = s0.copy()
        s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
        rows.append(s[["date", "instrument", "signal"]])
    df = pd.concat(rows, ignore_index=True)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)
